In [ ]:
import torch
import gpytorch
import pandas as pd
import pickle
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from torch.utils.data import TensorDataset, DataLoader
from pyproj import Transformer
from sklearn.metrics import pairwise_distances
from scipy.interpolate import RegularGridInterpolator
from torch_geometric.data import Data




In [ ]:
#compare the graph vs no graph model on 10 seeds. 

In [1]:
# ================================================================
# 10-SEED COMPARISON: wind_graph (region-mean wind) vs no_graph
# Continuous 36h trajectory, same train/val/test split (2016-17/2018/2019)
# no_graph = identical architecture, edge weights forced to zero
# ================================================================
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import os, time, copy, gc, json
from sklearn.metrics import average_precision_score, roc_auc_score
from scipy import stats

BASE = "/Users/drewbaldwin/PM2_5 Research"
df = pd.read_pickle(f"{BASE}/air_korea_final_imputed_with_blh.pkl")
station_order = sorted(df["Station_ID"].unique())
n_stations = len(station_order)

stations = df[["Station_ID", "lat", "lon"]].drop_duplicates("Station_ID").set_index("Station_ID").loc[station_order]
lats, lons = stations["lat"].to_numpy(), stations["lon"].to_numpy()

def haversine_km(lat1, lon1, lat2, lon2):
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
    dlat, dlon = lat2 - lat1, lon2 - lon1
    a = np.sin(dlat/2)**2 + np.cos(lat1)*np.cos(lat2)*np.sin(dlon/2)**2
    return 2 * 6371.0 * np.arcsin(np.sqrt(a))

def bearing_matrix(lat, lon):
    lat_r, lon_r = np.radians(lat), np.radians(lon)
    lat1, lat2 = lat_r[:, None], lat_r[None, :]
    dlon = lon_r[None, :] - lon_r[:, None]
    x = np.sin(dlon) * np.cos(lat2)
    y = np.cos(lat1) * np.sin(lat2) - np.sin(lat1) * np.cos(lat2) * np.cos(dlon)
    return (np.degrees(np.arctan2(x, y)) + 360) % 360

dist_km = haversine_km(lats[:, None], lons[:, None], lats[None, :], lons[None, :])
DIST_CUTOFF, RHO_KM = 250.0, 250.0
HYBRID_THRESHOLD_KM = 20.0
dist_edges = (dist_km <= DIST_CUTOFF) & (dist_km > 0)
bearing_from = bearing_matrix(lats, lons)

src_idx, dst_idx = np.nonzero(dist_edges)
edge_index_np = np.stack([src_idx, dst_idx])
decay_edge = np.exp(-dist_km[src_idx, dst_idx] / RHO_KM).astype(np.float32)
bearing_edge = bearing_from[src_idx, dst_idx].astype(np.float32)
close_edge_mask = (dist_km[src_idx, dst_idx] <= HYBRID_THRESHOLD_KM)

REGION_CENTROIDS = {
    "Seoul": (37.566, 126.978), "Busan": (35.180, 129.075), "Daegu": (35.872, 128.602),
    "Incheon": (37.483, 126.633), "Gwangju": (35.155, 126.916), "Daejeon": (36.350, 127.385),
    "Ulsan": (35.550, 129.317), "Sejong": (36.487, 127.282), "Gyeonggi": (37.500, 127.250),
    "Gangwon": (37.867, 127.733), "Chungbuk": (36.633, 127.483), "Chungnam": (36.500, 126.750),
    "Jeonbuk": (35.824, 127.148), "Jeonnam": (34.750, 127.000), "Gyeongbuk": (36.559, 128.729),
    "Gyeongnam": (35.271, 128.663), "Jeju": (33.513, 126.523),
}
region_names = list(REGION_CENTROIDS.keys())
n_regions = len(region_names)
region_lats = np.array([REGION_CENTROIDS[r][0] for r in region_names])
region_lons = np.array([REGION_CENTROIDS[r][1] for r in region_names])
dist_to_region = haversine_km(lats[:, None], lons[:, None], region_lats[None, :], region_lons[None, :])
station_region_idx = dist_to_region.argmin(axis=1)

region_membership = np.zeros((n_stations, n_regions), dtype=np.float32)
region_membership[np.arange(n_stations), station_region_idx] = 1.0
region_membership_t = torch.tensor(region_membership)

region_dist_km = haversine_km(region_lats[:, None], region_lons[:, None], region_lats[None, :], region_lons[None, :])
region_bearing = bearing_matrix(region_lats, region_lons)
r_src_idx, r_dst_idx = np.nonzero(~np.eye(n_regions, dtype=bool))
region_edge_index_np = np.stack([r_src_idx, r_dst_idx])
region_decay_edge = np.exp(-region_dist_km[r_src_idx, r_dst_idx] / RHO_KM).astype(np.float32)
region_bearing_edge = region_bearing[r_src_idx, r_dst_idx].astype(np.float32)

WINDOW, HORIZON = 36, 36
GRAPH_RECENT_HOURS = 18
EVENT_THRESHOLD = 75.0
SUSTAIN_HOURS = 2
QUANTILES = [0.50, 0.75, 0.90, 0.95, 0.99]
N_QUANTILES = len(QUANTILES)
TIME_FEATS = ["SO2", "CO", "NO2", "O3", "PM10", "PM25"]
STATIC_COLS = ["elevation_m", "urban_landuse_area_m2_3km", "green_space_area_3km",
               "building_footprint_area_3km", "railway_length_3km", "dist_to_coast_km",
               "dist_to_major_road_km", "industrial_area_m2_3km", "traffic_points_count_3km",
               "major_roads_count_3km", "total_road_length_3km"]

time_panels = {c: df.pivot(index="Datetime", columns="Station_ID", values=c)[station_order] for c in TIME_FEATS}
dt_index = time_panels["PM25"].index
n_time = len(dt_index)
years = dt_index.year.to_numpy()
doy = dt_index.dayofyear.to_numpy().astype(float)

split_id_per_hour = np.where((years == 2016) | (years == 2017), 0, np.where(years == 2018, 1, np.where(years == 2019, 2, -1)))
TRAIN_MASK = split_id_per_hour == 0

wind_dir_arr = df.pivot(index="Datetime", columns="Station_ID", values="winddirection_10m")[station_order].reindex(dt_index).to_numpy().astype(float)
wind_speed_arr = df.pivot(index="Datetime", columns="Station_ID", values="windspeed_10m")[station_order].reindex(dt_index).to_numpy().astype(float)
blh_arr = df.pivot(index="Datetime", columns="Station_ID", values="boundary_layer_height")[station_order].reindex(dt_index).to_numpy().astype(float)
pm25_raw_arr = time_panels["PM25"].to_numpy()

def regional_flat_mean(arr):
    out = np.zeros((n_time, n_regions), dtype=np.float32)
    for r in range(n_regions):
        cols = station_region_idx == r
        out[:, r] = np.nanmean(arr[:, cols], axis=1)
    return out

region_pm25 = regional_flat_mean(pm25_raw_arr)
wdir_sin_station = np.sin(np.radians(wind_dir_arr))
wdir_cos_station = np.cos(np.radians(wind_dir_arr))
region_windspeed = regional_flat_mean(wind_speed_arr)
region_wdir_sin = regional_flat_mean(wdir_sin_station)
region_wdir_cos = regional_flat_mean(wdir_cos_station)
region_wind_dir_deg = (np.degrees(np.arctan2(region_wdir_sin, region_wdir_cos)) + 360) % 360
region_wind_blows_toward = (region_wind_dir_deg + 180) % 360

season_sin_1d = np.sin(2 * np.pi * doy / 365.25)
season_cos_1d = np.cos(2 * np.pi * doy / 365.25)
season_sin = np.tile(season_sin_1d[:, None], (1, n_stations))
season_cos = np.tile(season_cos_1d[:, None], (1, n_stations))

TIME_FEATS_FULL = TIME_FEATS + ["windspeed_10m", "wdir_sin", "wdir_cos", "boundary_layer_height", "season_sin", "season_cos"]
n_time_feats, n_static_feats = len(TIME_FEATS_FULL), len(STATIC_COLS)
n_feats = n_time_feats + n_static_feats
pm25_col_idx = TIME_FEATS_FULL.index("PM25")

time_arr_raw = np.stack([time_panels[c].to_numpy() for c in TIME_FEATS] +
                         [wind_speed_arr, wdir_sin_station, wdir_cos_station, blh_arr, season_sin, season_cos], axis=-1)
del time_panels, season_sin, season_cos, blh_arr, pm25_raw_arr
gc.collect()

static_df = df[["Station_ID"] + STATIC_COLS].drop_duplicates("Station_ID").set_index("Station_ID").loc[station_order]
static_arr = static_df[STATIC_COLS].to_numpy()
del df
gc.collect()

rev = region_pm25[::-1]
roll_min_rev = pd.DataFrame(rev).rolling(window=SUSTAIN_HOURS, min_periods=SUSTAIN_HOURS).min().to_numpy()
region_episode_label = (roll_min_rev[::-1] >= EVENT_THRESHOLD).astype(np.float32)
del rev, roll_min_rev
gc.collect()

def pinball_loss(preds, target, quantiles):
    target_exp = target.unsqueeze(-1)
    diff = target_exp - preds
    q_tensor = torch.tensor(quantiles, device=preds.device, dtype=preds.dtype).view(*([1] * (preds.dim() - 1)), -1)
    return torch.max(q_tensor * diff, (q_tensor - 1) * diff).mean()

def monotonic_quantiles(raw):
    first = raw[..., :1]
    deltas = F.softplus(raw[..., 1:])
    return torch.cat([first, first + torch.cumsum(deltas, dim=-1)], dim=-1)

class WindConvLayer(nn.Module):
    def __init__(self, in_dim, out_dim):
        super().__init__()
        self.lin_self = nn.Linear(in_dim, out_dim)
        self.lin_neigh = nn.Linear(in_dim, out_dim)
        self.lin_connectivity = nn.Linear(1, out_dim)

    def forward(self, x, edge_index, edge_weight, num_nodes):
        src, dst = edge_index[0], edge_index[1]
        messages = x[src] * edge_weight.unsqueeze(-1)
        agg_sum = x.new_zeros(num_nodes, x.size(-1))
        agg_sum.index_add_(0, dst, messages)
        weight_sum = x.new_zeros(num_nodes)
        weight_sum.index_add_(0, dst, edge_weight)
        agg_mean = agg_sum / (weight_sum.unsqueeze(-1) + 1e-8)
        connectivity = torch.log1p(weight_sum.clamp(min=0)).unsqueeze(-1)
        return self.lin_self(x) + self.lin_neigh(agg_mean) + self.lin_connectivity(connectivity)

class AttentionPool(nn.Module):
    def __init__(self, hidden):
        super().__init__()
        self.attn_score = nn.Linear(hidden, 1)

    def forward(self, h_station, region_membership_t_local):
        B, N, H = h_station.shape
        scores = self.attn_score(h_station).squeeze(-1)
        scores = scores - scores.max(dim=1, keepdim=True).values
        exp_scores = torch.exp(scores)
        weighted_exp = exp_scores.unsqueeze(-1) * region_membership_t_local.unsqueeze(0)
        region_denom = weighted_exp.sum(dim=1)
        region_numer = torch.einsum('bnr,bnh->brh', weighted_exp, h_station)
        return region_numer / (region_denom.unsqueeze(-1) + 1e-8)

class StationQuantileGCN(nn.Module):
    def __init__(self, in_dim, dropout, hidden=32, gru_hidden=32):
        super().__init__()
        self.station_conv = WindConvLayer(in_dim, hidden)
        self.drop = nn.Dropout(dropout)
        self.gru = nn.GRU(hidden, gru_hidden, batch_first=True)
        self.head = nn.Linear(gru_hidden, N_QUANTILES)

    def forward(self, x_window, edge_index, edge_weight_seq):
        B, W, N, Fin = x_window.shape
        ei_b = torch.cat([edge_index + i * N for i in range(B)], dim=1)
        num_nodes = B * N
        h_seq = []
        for w in range(W):
            xt = x_window[:, w].reshape(B * N, Fin)
            ew_b = edge_weight_seq[:, w].reshape(-1)
            h = torch.relu(self.station_conv(xt, ei_b, ew_b, num_nodes))
            h = self.drop(h)
            h_seq.append(h.reshape(B, N, -1))
        h_seq = torch.stack(h_seq, dim=1).permute(0, 2, 1, 3).reshape(B * N, W, -1)
        _, h_final = self.gru(h_seq)
        embed = self.drop(h_final.squeeze(0).reshape(B, N, -1))
        return monotonic_quantiles(self.head(embed))

class RegionFromTrajectoryGCN(nn.Module):
    def __init__(self, station_conv, region_membership_t_local, dropout, hidden=32, gru_hidden=32):
        super().__init__()
        self.station_conv = station_conv
        self.attn_pool = AttentionPool(hidden)
        self.region_conv = WindConvLayer(hidden, hidden)
        self.drop = nn.Dropout(dropout)
        self.region_gru = nn.GRU(hidden, gru_hidden, batch_first=True)
        self.region_head = nn.Linear(gru_hidden, HORIZON)
        self.rmem = region_membership_t_local

    def forward(self, x_window, station_edge_index, station_edge_weight_seq, region_edge_index, region_edge_weight_seq, n_reg):
        B, W, N, Fin = x_window.shape
        station_ei_b = torch.cat([station_edge_index + i * N for i in range(B)], dim=1)
        region_ei_b = torch.cat([region_edge_index + i * n_reg for i in range(B)], dim=1)
        num_station_nodes = B * N
        num_region_nodes = B * n_reg
        h_region_seq = []
        for w in range(W):
            xt = x_window[:, w].reshape(B * N, Fin)
            ew_station_b = station_edge_weight_seq[:, w].reshape(-1)
            h_station = torch.relu(self.station_conv(xt, station_ei_b, ew_station_b, num_station_nodes))
            h_station = self.drop(h_station).reshape(B, N, -1)
            h_region_pooled = self.attn_pool(h_station, self.rmem).reshape(B * n_reg, -1)
            ew_region_b = region_edge_weight_seq[:, w].reshape(-1)
            h_region = torch.relu(self.region_conv(h_region_pooled, region_ei_b, ew_region_b, num_region_nodes))
            h_region = self.drop(h_region)
            h_region_seq.append(h_region.reshape(B, n_reg, -1))
        h_region_seq = torch.stack(h_region_seq, dim=1).permute(0, 2, 1, 3).reshape(B * n_reg, W, -1)
        _, h_final = self.region_gru(h_region_seq)
        embed = self.drop(h_final.squeeze(0).reshape(B, n_reg, -1))
        return self.region_head(embed)  # (B, n_reg, HORIZON) logits

DEVICE = torch.device("cuda" if torch.cuda.is_available() else ("mps" if torch.backends.mps.is_available() else "cpu"))
MICRO_BATCH, ACCUM_STEPS = 16, 4
MAX_EPOCHS_P1, MAX_EPOCHS_P2 = 2, 2
print(f"device={DEVICE}")

region_membership_t = region_membership_t.to(DEVICE)
edge_index = torch.tensor(edge_index_np, dtype=torch.long).to(DEVICE)
region_edge_index = torch.tensor(region_edge_index_np, dtype=torch.long).to(DEVICE)

def add_static_fn(x_time_batch, static_tensor_local):
    B, W, N, _ = x_time_batch.shape
    static_b = static_tensor_local.unsqueeze(0).unsqueeze(0).expand(B, W, N, n_static_feats)
    return torch.cat([x_time_batch, static_b], dim=-1)

def gather_seq(ew_by_hour_t, starts_subset):
    idx = starts_subset.unsqueeze(1) + torch.arange(WINDOW).unsqueeze(0)
    ew = ew_by_hour_t[idx]
    ew = ew.clone()
    ew[:, :WINDOW - GRAPH_RECENT_HOURS, :] = 0.0
    return ew

print(f"train hours (2016-2017): {TRAIN_MASK.sum()}  val hours (2018): {(split_id_per_hour==1).sum()}  test hours (2019): {(split_id_per_hour==2).sum()}")

# ---- wind edges ----
wind_blows_toward = (wind_dir_arr + 180) % 360
wbt_src = wind_blows_toward[:, src_idx]
cos_align = np.maximum(np.cos(np.radians(wbt_src - bearing_edge[None, :])), 0.0)
speed_src = wind_speed_arr[:, src_idx]
wind_component = (cos_align * speed_src).astype(np.float32)
del wbt_src, cos_align, speed_src
gc.collect()
ref_speed = np.float32(np.nanmean(wind_speed_arr[TRAIN_MASK]))
component = np.where(close_edge_mask[None, :], ref_speed, wind_component)
station_wind_raw = np.nan_to_num(decay_edge[None, :] * component, nan=0.0).astype(np.float32)
del component
gc.collect()

r_wbt_src = region_wind_blows_toward[:, r_src_idx]
r_cos_align = np.maximum(np.cos(np.radians(r_wbt_src - region_bearing_edge[None, :])), 0.0)
r_speed_src = region_windspeed[:, r_src_idx]
region_wind_raw = np.nan_to_num(region_decay_edge[None, :] * r_cos_align * r_speed_src, nan=0.0).astype(np.float32)
del r_wbt_src, r_cos_align, r_speed_src
gc.collect()

train_nonzero = station_wind_raw[TRAIN_MASK][station_wind_raw[TRAIN_MASK] > 0]
wind_scale_s = train_nonzero.std()
station_wind_edge_weight = (station_wind_raw / wind_scale_s).astype(np.float32)
region_train_nonzero = region_wind_raw[TRAIN_MASK][region_wind_raw[TRAIN_MASK] > 0]
wind_scale_r = region_train_nonzero.std()
region_wind_edge_weight = (region_wind_raw / wind_scale_r).astype(np.float32)
del train_nonzero, region_train_nonzero, station_wind_raw, region_wind_raw
gc.collect()

t_mean = np.nanmean(time_arr_raw[TRAIN_MASK], axis=(0, 1), keepdims=True)
t_std = np.nanstd(time_arr_raw[TRAIN_MASK], axis=(0, 1), keepdims=True) + 1e-6
time_arr_std = np.nan_to_num((time_arr_raw - t_mean) / t_std, nan=0.0)
s_mean, s_std = static_arr.mean(axis=0, keepdims=True), static_arr.std(axis=0, keepdims=True) + 1e-6
static_tensor = torch.tensor((static_arr - s_mean) / s_std, dtype=torch.float32).to(DEVICE)

p_buckets = {0: ([], [], [], []), 1: ([], [], [], []), 2: ([], [], [], [])}
for t in range(0, n_time - WINDOW - HORIZON + 1):
    target_t = t + WINDOW + HORIZON - 1
    s_start, s_target = split_id_per_hour[t], split_id_per_hour[target_t]
    if s_start != s_target or s_start == -1:
        continue
    x_win = time_arr_std[t:t + WINDOW]
    traj = region_episode_label[t + WINDOW: t + WINDOW + HORIZON]
    Xl, yregl, ytrajl, sl = p_buckets[s_start]
    Xl.append(x_win)
    yregl.append(time_arr_std[target_t, :, pm25_col_idx])
    ytrajl.append(traj.T)
    sl.append(t)

X0, yreg0, ytraj0, starts0 = (np.stack(v) for v in p_buckets[0])
X1, yreg1, ytraj1, starts1 = (np.stack(v) for v in p_buckets[1])
X2, yreg2, ytraj2, starts2 = (np.stack(v) for v in p_buckets[2])
del p_buckets, time_arr_std
gc.collect()
print(f"windows: train={len(X0)} val={len(X1)} test={len(X2)}")

X0_t = torch.tensor(X0, dtype=torch.float32); del X0
X1_t = torch.tensor(X1, dtype=torch.float32); del X1
X2_t = torch.tensor(X2, dtype=torch.float32); del X2
gc.collect()

yreg0_t, yreg1_t = torch.tensor(yreg0, dtype=torch.float32), torch.tensor(yreg1, dtype=torch.float32)
ytraj0_t = torch.tensor(ytraj0, dtype=torch.float32)
ytraj1_t = torch.tensor(ytraj1, dtype=torch.float32)
starts0_t, starts1_t, starts2_t = (torch.tensor(a, dtype=torch.long) for a in (starts0, starts1, starts2))
del yreg0, yreg1
gc.collect()

POS_WEIGHT = min(float((ytraj0_t.numel() - ytraj0_t.sum()) / ytraj0_t.sum().clamp(min=1)), 50.0)
print(f"pos_weight (per-hour trajectory): {POS_WEIGHT:.2f}")

wind_ewt_s = torch.tensor(station_wind_edge_weight)
wind_ewt_r = torch.tensor(region_wind_edge_weight)
del station_wind_edge_weight, region_wind_edge_weight
gc.collect()

# no_graph: identical topology, edge weights forced to zero -> WindConvLayer's
# neighbor/connectivity terms vanish, leaving only the self-transform.
zero_ewt_s = torch.zeros_like(wind_ewt_s)
zero_ewt_r = torch.zeros_like(wind_ewt_r)

region_criterion = nn.BCEWithLogitsLoss(pos_weight=torch.tensor(POS_WEIGHT))

def run_p1_epoch(model, ewt, X, y, starts, optimizer, train):
    n = X.shape[0]
    idx = torch.randperm(n) if train else torch.arange(n)
    model.train(train)
    total_loss, total_n = 0.0, 0
    eff_batch = MICRO_BATCH * ACCUM_STEPS
    for start in range(0, n, eff_batch):
        if train: optimizer.zero_grad()
        batch_idx = idx[start:start + eff_batch]
        for ms in range(0, len(batch_idx), MICRO_BATCH):
            mb_idx = batch_idx[ms:ms + MICRO_BATCH]
            if len(mb_idx) == 0: continue
            xb = add_static_fn(X[mb_idx].to(DEVICE), static_tensor)
            yb = y[mb_idx].to(DEVICE)
            ew_seq = gather_seq(ewt, starts[mb_idx]).to(DEVICE)
            with torch.set_grad_enabled(train):
                pred = model(xb, edge_index, ew_seq)
                loss = pinball_loss(pred, yb, QUANTILES)
            if train: (loss * len(mb_idx) / len(batch_idx)).backward()
            total_loss += loss.item() * len(mb_idx); total_n += len(mb_idx)
        if train: optimizer.step()
    return total_loss / total_n

def run_p2_traj_epoch(model, ewt_s, ewt_r, X, y_traj, starts, optimizer, train):
    n = X.shape[0]
    idx = torch.randperm(n) if train else torch.arange(n)
    model.train(train)
    total_loss, total_n = 0.0, 0
    eff_batch = MICRO_BATCH * ACCUM_STEPS
    for start in range(0, n, eff_batch):
        if train: optimizer.zero_grad()
        batch_idx = idx[start:start + eff_batch]
        for ms in range(0, len(batch_idx), MICRO_BATCH):
            mb_idx = batch_idx[ms:ms + MICRO_BATCH]
            if len(mb_idx) == 0: continue
            xb = add_static_fn(X[mb_idx].to(DEVICE), static_tensor)
            yb = y_traj[mb_idx].to(DEVICE)
            ew_s = gather_seq(ewt_s, starts[mb_idx]).to(DEVICE)
            ew_r = gather_seq(ewt_r, starts[mb_idx]).to(DEVICE)
            with torch.set_grad_enabled(train):
                logits = model(xb, edge_index, ew_s, region_edge_index, ew_r, n_regions)
                loss = region_criterion(logits, yb)
            if train: loss.backward()
            total_loss += loss.item() * len(mb_idx); total_n += len(mb_idx)
        if train: optimizer.step()
    return total_loss / max(total_n, 1)

def predict_traj_probs(model, ewt_s, ewt_r, X, starts, batch_size=64):
    model.eval()
    n = X.shape[0]
    out = np.zeros((n, n_regions, HORIZON), dtype=np.float32)
    with torch.no_grad():
        for start in range(0, n, batch_size):
            idx = torch.arange(start, min(start + batch_size, n))
            xb = add_static_fn(X[idx].to(DEVICE), static_tensor)
            ew_s = gather_seq(ewt_s, starts[idx]).to(DEVICE)
            ew_r = gather_seq(ewt_r, starts[idx]).to(DEVICE)
            logits = model(xb, edge_index, ew_s, region_edge_index, ew_r, n_regions)
            out[idx.numpy()] = torch.sigmoid(logits).cpu().numpy()
    return out

def train_one_seed_final(seed, ewt_s, ewt_r, lr=3e-3, dropout=0.5, weight_decay=5e-4):
    torch.manual_seed(seed); np.random.seed(seed)
    p1 = StationQuantileGCN(n_feats, dropout).to(DEVICE)
    opt1 = torch.optim.Adam(p1.parameters(), lr=lr, weight_decay=weight_decay)
    best_p1_val, best_p1_state = float("inf"), None
    for epoch in range(1, MAX_EPOCHS_P1 + 1):
        run_p1_epoch(p1, ewt_s, X0_t, yreg0_t, starts0_t, opt1, True)
        vl = run_p1_epoch(p1, ewt_s, X1_t, yreg1_t, starts1_t, opt1, False)
        if vl < best_p1_val:
            best_p1_val, best_p1_state = vl, copy.deepcopy(p1.state_dict())
    p1.load_state_dict(best_p1_state)
    encoder_copy = copy.deepcopy(p1.station_conv)
    p2 = RegionFromTrajectoryGCN(encoder_copy, region_membership_t, dropout).to(DEVICE)
    del p1; gc.collect()
    if DEVICE.type == "mps": torch.mps.empty_cache()

    opt2 = torch.optim.Adam(p2.parameters(), lr=lr, weight_decay=weight_decay)
    y_val_agg = ytraj1_t.numpy().max(axis=-1).reshape(-1)
    best_val_aucpr, best_state = -1.0, None
    for epoch in range(1, MAX_EPOCHS_P2 + 1):
        run_p2_traj_epoch(p2, ewt_s, ewt_r, X0_t, ytraj0_t, starts0_t, opt2, True)
        run_p2_traj_epoch(p2, ewt_s, ewt_r, X1_t, ytraj1_t, starts1_t, opt2, False)
        val_probs = predict_traj_probs(p2, ewt_s, ewt_r, X1_t, starts1_t)
        val_agg_score = val_probs.max(axis=-1).reshape(-1)
        va = average_precision_score(y_val_agg, val_agg_score)
        if va > best_val_aucpr:
            best_val_aucpr, best_state = va, copy.deepcopy(p2.state_dict())
    p2.load_state_dict(best_state)
    p2.eval()
    return p2, best_val_aucpr

# ================================================================
# RUN: 10 seeds x {wind_graph, no_graph}
# ================================================================
N_SEEDS = 10
results = {"wind_graph": [], "no_graph": []}

for seed in range(N_SEEDS):
    for model_name, ewt_s, ewt_r in [("wind_graph", wind_ewt_s, wind_ewt_r), ("no_graph", zero_ewt_s, zero_ewt_r)]:
        print(f"\n{'='*10} seed={seed}  model={model_name} {'='*10}")
        t0 = time.time()
        p2, val_aucpr = train_one_seed_final(seed, ewt_s, ewt_r)
        test_probs = predict_traj_probs(p2, ewt_s, ewt_r, X2_t, starts2_t)
        y_test_agg = ytraj2.max(axis=-1).reshape(-1)
        test_agg_score = test_probs.max(axis=-1).reshape(-1)
        test_aucroc = roc_auc_score(y_test_agg, test_agg_score)
        test_aucpr = average_precision_score(y_test_agg, test_agg_score)
        print(f"  trained in {time.time()-t0:.0f}s  val_AUCPR={val_aucpr:.4f}  "
              f"test_AUCROC={test_aucroc:.4f}  test_AUCPR={test_aucpr:.4f}")
        results[model_name].append({"seed": seed, "val_aucpr": float(val_aucpr),
                                     "test_aucroc": float(test_aucroc), "test_aucpr": float(test_aucpr)})
        del p2; gc.collect()
        if DEVICE.type == "mps": torch.mps.empty_cache()

# ================================================================
# SUMMARY + PAIRED SIGNIFICANCE TESTS
# ================================================================
wind_aucpr = np.array([r["test_aucpr"] for r in results["wind_graph"]])
nograph_aucpr = np.array([r["test_aucpr"] for r in results["no_graph"]])
wind_aucroc = np.array([r["test_aucroc"] for r in results["wind_graph"]])
nograph_aucroc = np.array([r["test_aucroc"] for r in results["no_graph"]])

print(f"\n{'='*20} SUMMARY ACROSS {N_SEEDS} SEEDS {'='*20}")
print(f"wind_graph  AUC-PR:  mean={wind_aucpr.mean():.4f}  std={wind_aucpr.std():.4f}  values={wind_aucpr.round(4)}")
print(f"no_graph    AUC-PR:  mean={nograph_aucpr.mean():.4f}  std={nograph_aucpr.std():.4f}  values={nograph_aucpr.round(4)}")
print(f"wind_graph  AUC-ROC: mean={wind_aucroc.mean():.4f}  std={wind_aucroc.std():.4f}  values={wind_aucroc.round(4)}")
print(f"no_graph    AUC-ROC: mean={nograph_aucroc.mean():.4f}  std={nograph_aucroc.std():.4f}  values={nograph_aucroc.round(4)}")

t_stat_pr, p_ttest_pr = stats.ttest_rel(wind_aucpr, nograph_aucpr)
w_stat_pr, p_wilcoxon_pr = stats.wilcoxon(wind_aucpr, nograph_aucpr)
t_stat_roc, p_ttest_roc = stats.ttest_rel(wind_aucroc, nograph_aucroc)
w_stat_roc, p_wilcoxon_roc = stats.wilcoxon(wind_aucroc, nograph_aucroc)

n_wins_pr = int((wind_aucpr > nograph_aucpr).sum())
n_wins_roc = int((wind_aucroc > nograph_aucroc).sum())

print(f"\nAUC-PR:  paired t-test t={t_stat_pr:.3f} p={p_ttest_pr:.4f}  |  Wilcoxon p={p_wilcoxon_pr:.4f}  |  wind_graph wins {n_wins_pr}/{N_SEEDS} seeds")
print(f"AUC-ROC: paired t-test t={t_stat_roc:.3f} p={p_ttest_roc:.4f}  |  Wilcoxon p={p_wilcoxon_roc:.4f}  |  wind_graph wins {n_wins_roc}/{N_SEEDS} seeds")

with open(f"{BASE}/wind_vs_nograph_10seed.json", "w") as f:
    json.dump({"wind_graph": results["wind_graph"], "no_graph": results["no_graph"],
               "aucpr_paired_ttest": {"t": float(t_stat_pr), "p": float(p_ttest_pr)},
               "aucpr_wilcoxon": {"stat": float(w_stat_pr), "p": float(p_wilcoxon_pr)},
               "aucroc_paired_ttest": {"t": float(t_stat_roc), "p": float(p_ttest_roc)},
               "aucroc_wilcoxon": {"stat": float(w_stat_roc), "p": float(p_wilcoxon_roc)}}, f, indent=2)
print(f"\nsaved to {BASE}/wind_vs_nograph_10seed.json")


device=mps
train hours (2016-2017): 17544  val hours (2018): 8760  test hours (2019): 8760
windows: train=17473 val=8689 test=8689
pos_weight (per-hour trajectory): 50.00

========== seed=0  model=wind_graph ==========
  trained in 653s  val_AUCPR=0.5023  test_AUCROC=0.9583  test_AUCPR=0.7157

========== seed=0  model=no_graph ==========
  trained in 643s  val_AUCPR=0.4604  test_AUCROC=0.9466  test_AUCPR=0.6752

========== seed=1  model=wind_graph ==========
  trained in 640s  val_AUCPR=0.4873  test_AUCROC=0.9572  test_AUCPR=0.7033

========== seed=1  model=no_graph ==========
  trained in 641s  val_AUCPR=0.4716  test_AUCROC=0.9504  test_AUCPR=0.6874

========== seed=2  model=wind_graph ==========
  trained in 640s  val_AUCPR=0.4894  test_AUCROC=0.9631  test_AUCPR=0.7193

========== seed=2  model=no_graph ==========
  trained in 645s  val_AUCPR=0.4693  test_AUCROC=0.9501  test_AUCPR=0.6889

========== seed=3  model=wind_graph ==========
  trained in 640s  val_AUCPR=0.5026  test_AUCROC=

In [1]:
#GRU comparison 

# ================================================================
# MINIMAL GRU BASELINE (5 seeds) -- true no-graph comparison
# No WindConvLayer anywhere: per-node Linear -> GRU, same AttentionPool
# station->region step as wind_graph/no_graph, same everything else.
# Compared against the already-saved wind_graph 10-seed results (seeds 0-4).
# ================================================================
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import os, time, copy, gc, json
from sklearn.metrics import average_precision_score, roc_auc_score
from scipy import stats

BASE = "/Users/drewbaldwin/PM2_5 Research"
df = pd.read_pickle(f"{BASE}/air_korea_final_imputed_with_blh.pkl")
station_order = sorted(df["Station_ID"].unique())
n_stations = len(station_order)

stations = df[["Station_ID", "lat", "lon"]].drop_duplicates("Station_ID").set_index("Station_ID").loc[station_order]
lats, lons = stations["lat"].to_numpy(), stations["lon"].to_numpy()

def haversine_km(lat1, lon1, lat2, lon2):
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
    dlat, dlon = lat2 - lat1, lon2 - lon1
    a = np.sin(dlat/2)**2 + np.cos(lat1)*np.cos(lat2)*np.sin(dlon/2)**2
    return 2 * 6371.0 * np.arcsin(np.sqrt(a))

REGION_CENTROIDS = {
    "Seoul": (37.566, 126.978), "Busan": (35.180, 129.075), "Daegu": (35.872, 128.602),
    "Incheon": (37.483, 126.633), "Gwangju": (35.155, 126.916), "Daejeon": (36.350, 127.385),
    "Ulsan": (35.550, 129.317), "Sejong": (36.487, 127.282), "Gyeonggi": (37.500, 127.250),
    "Gangwon": (37.867, 127.733), "Chungbuk": (36.633, 127.483), "Chungnam": (36.500, 126.750),
    "Jeonbuk": (35.824, 127.148), "Jeonnam": (34.750, 127.000), "Gyeongbuk": (36.559, 128.729),
    "Gyeongnam": (35.271, 128.663), "Jeju": (33.513, 126.523),
}
region_names = list(REGION_CENTROIDS.keys())
n_regions = len(region_names)
region_lats = np.array([REGION_CENTROIDS[r][0] for r in region_names])
region_lons = np.array([REGION_CENTROIDS[r][1] for r in region_names])
dist_to_region = haversine_km(lats[:, None], lons[:, None], region_lats[None, :], region_lons[None, :])
station_region_idx = dist_to_region.argmin(axis=1)

region_membership = np.zeros((n_stations, n_regions), dtype=np.float32)
region_membership[np.arange(n_stations), station_region_idx] = 1.0
region_membership_t = torch.tensor(region_membership)

WINDOW, HORIZON = 36, 36
EVENT_THRESHOLD = 75.0
SUSTAIN_HOURS = 2
QUANTILES = [0.50, 0.75, 0.90, 0.95, 0.99]
N_QUANTILES = len(QUANTILES)
TIME_FEATS = ["SO2", "CO", "NO2", "O3", "PM10", "PM25"]
STATIC_COLS = ["elevation_m", "urban_landuse_area_m2_3km", "green_space_area_3km",
               "building_footprint_area_3km", "railway_length_3km", "dist_to_coast_km",
               "dist_to_major_road_km", "industrial_area_m2_3km", "traffic_points_count_3km",
               "major_roads_count_3km", "total_road_length_3km"]

time_panels = {c: df.pivot(index="Datetime", columns="Station_ID", values=c)[station_order] for c in TIME_FEATS}
dt_index = time_panels["PM25"].index
n_time = len(dt_index)
years = dt_index.year.to_numpy()
doy = dt_index.dayofyear.to_numpy().astype(float)

split_id_per_hour = np.where((years == 2016) | (years == 2017), 0, np.where(years == 2018, 1, np.where(years == 2019, 2, -1)))
TRAIN_MASK = split_id_per_hour == 0

wind_speed_arr = df.pivot(index="Datetime", columns="Station_ID", values="windspeed_10m")[station_order].reindex(dt_index).to_numpy().astype(float)
wind_dir_arr = df.pivot(index="Datetime", columns="Station_ID", values="winddirection_10m")[station_order].reindex(dt_index).to_numpy().astype(float)
blh_arr = df.pivot(index="Datetime", columns="Station_ID", values="boundary_layer_height")[station_order].reindex(dt_index).to_numpy().astype(float)
pm25_raw_arr = time_panels["PM25"].to_numpy()

def regional_flat_mean(arr):
    out = np.zeros((n_time, n_regions), dtype=np.float32)
    for r in range(n_regions):
        cols = station_region_idx == r
        out[:, r] = np.nanmean(arr[:, cols], axis=1)
    return out

region_pm25 = regional_flat_mean(pm25_raw_arr)
wdir_sin_station = np.sin(np.radians(wind_dir_arr))
wdir_cos_station = np.cos(np.radians(wind_dir_arr))

season_sin_1d = np.sin(2 * np.pi * doy / 365.25)
season_cos_1d = np.cos(2 * np.pi * doy / 365.25)
season_sin = np.tile(season_sin_1d[:, None], (1, n_stations))
season_cos = np.tile(season_cos_1d[:, None], (1, n_stations))

TIME_FEATS_FULL = TIME_FEATS + ["windspeed_10m", "wdir_sin", "wdir_cos", "boundary_layer_height", "season_sin", "season_cos"]
n_time_feats, n_static_feats = len(TIME_FEATS_FULL), len(STATIC_COLS)
n_feats = n_time_feats + n_static_feats
pm25_col_idx = TIME_FEATS_FULL.index("PM25")

time_arr_raw = np.stack([time_panels[c].to_numpy() for c in TIME_FEATS] +
                         [wind_speed_arr, wdir_sin_station, wdir_cos_station, blh_arr, season_sin, season_cos], axis=-1)
del time_panels, season_sin, season_cos, blh_arr, pm25_raw_arr
gc.collect()

static_df = df[["Station_ID"] + STATIC_COLS].drop_duplicates("Station_ID").set_index("Station_ID").loc[station_order]
static_arr = static_df[STATIC_COLS].to_numpy()
del df
gc.collect()

rev = region_pm25[::-1]
roll_min_rev = pd.DataFrame(rev).rolling(window=SUSTAIN_HOURS, min_periods=SUSTAIN_HOURS).min().to_numpy()
region_episode_label = (roll_min_rev[::-1] >= EVENT_THRESHOLD).astype(np.float32)
del rev, roll_min_rev
gc.collect()

def pinball_loss(preds, target, quantiles):
    target_exp = target.unsqueeze(-1)
    diff = target_exp - preds
    q_tensor = torch.tensor(quantiles, device=preds.device, dtype=preds.dtype).view(*([1] * (preds.dim() - 1)), -1)
    return torch.max(q_tensor * diff, (q_tensor - 1) * diff).mean()

def monotonic_quantiles(raw):
    first = raw[..., :1]
    deltas = F.softplus(raw[..., 1:])
    return torch.cat([first, first + torch.cumsum(deltas, dim=-1)], dim=-1)

class AttentionPool(nn.Module):
    def __init__(self, hidden):
        super().__init__()
        self.attn_score = nn.Linear(hidden, 1)

    def forward(self, h_station, region_membership_t_local):
        B, N, H = h_station.shape
        scores = self.attn_score(h_station).squeeze(-1)
        scores = scores - scores.max(dim=1, keepdim=True).values
        exp_scores = torch.exp(scores)
        weighted_exp = exp_scores.unsqueeze(-1) * region_membership_t_local.unsqueeze(0)
        region_denom = weighted_exp.sum(dim=1)
        region_numer = torch.einsum('bnr,bnh->brh', weighted_exp, h_station)
        return region_numer / (region_denom.unsqueeze(-1) + 1e-8)

class StationGRU(nn.Module):
    """No graph convolution at all -- per-station Linear -> GRU."""
    def __init__(self, in_dim, dropout, hidden=32, gru_hidden=32):
        super().__init__()
        self.station_lin = nn.Linear(in_dim, hidden)
        self.drop = nn.Dropout(dropout)
        self.gru = nn.GRU(hidden, gru_hidden, batch_first=True)
        self.head = nn.Linear(gru_hidden, N_QUANTILES)

    def forward(self, x_window):
        B, W, N, Fin = x_window.shape
        h_seq = []
        for w in range(W):
            xt = x_window[:, w].reshape(B * N, Fin)
            h = torch.relu(self.station_lin(xt))
            h = self.drop(h)
            h_seq.append(h.reshape(B, N, -1))
        h_seq = torch.stack(h_seq, dim=1).permute(0, 2, 1, 3).reshape(B * N, W, -1)
        _, h_final = self.gru(h_seq)
        embed = self.drop(h_final.squeeze(0).reshape(B, N, -1))
        return monotonic_quantiles(self.head(embed))

class RegionFromTrajectoryGRU(nn.Module):
    """Reuses phase-1 station encoder, pools to regions via attention (same as
    wind_graph/no_graph), then runs a region-level GRU directly -- NO region_conv,
    i.e. zero cross-region information flow, by construction rather than by masking."""
    def __init__(self, station_lin, region_membership_t_local, dropout, hidden=32, gru_hidden=32):
        super().__init__()
        self.station_lin = station_lin
        self.attn_pool = AttentionPool(hidden)
        self.drop = nn.Dropout(dropout)
        self.region_gru = nn.GRU(hidden, gru_hidden, batch_first=True)
        self.region_head = nn.Linear(gru_hidden, HORIZON)
        self.rmem = region_membership_t_local

    def forward(self, x_window, n_reg):
        B, W, N, Fin = x_window.shape
        h_region_seq = []
        for w in range(W):
            xt = x_window[:, w].reshape(B * N, Fin)
            h_station = torch.relu(self.station_lin(xt)).reshape(B, N, -1)
            h_station = self.drop(h_station)
            h_region_pooled = self.attn_pool(h_station, self.rmem)  # (B, n_reg, hidden)
            h_region_pooled = self.drop(h_region_pooled)
            h_region_seq.append(h_region_pooled)
        h_region_seq = torch.stack(h_region_seq, dim=1).permute(0, 2, 1, 3).reshape(B * n_reg, W, -1)
        _, h_final = self.region_gru(h_region_seq)
        embed = self.drop(h_final.squeeze(0).reshape(B, n_reg, -1))
        return self.region_head(embed)  # (B, n_reg, HORIZON) logits

DEVICE = torch.device("cuda" if torch.cuda.is_available() else ("mps" if torch.backends.mps.is_available() else "cpu"))
MICRO_BATCH, ACCUM_STEPS = 16, 4
MAX_EPOCHS_P1, MAX_EPOCHS_P2 = 2, 2
print(f"device={DEVICE}")

region_membership_t = region_membership_t.to(DEVICE)

def add_static_fn(x_time_batch, static_tensor_local):
    B, W, N, _ = x_time_batch.shape
    static_b = static_tensor_local.unsqueeze(0).unsqueeze(0).expand(B, W, N, n_static_feats)
    return torch.cat([x_time_batch, static_b], dim=-1)

print(f"train hours (2016-2017): {TRAIN_MASK.sum()}  val hours (2018): {(split_id_per_hour==1).sum()}  test hours (2019): {(split_id_per_hour==2).sum()}")

t_mean = np.nanmean(time_arr_raw[TRAIN_MASK], axis=(0, 1), keepdims=True)
t_std = np.nanstd(time_arr_raw[TRAIN_MASK], axis=(0, 1), keepdims=True) + 1e-6
time_arr_std = np.nan_to_num((time_arr_raw - t_mean) / t_std, nan=0.0)
s_mean, s_std = static_arr.mean(axis=0, keepdims=True), static_arr.std(axis=0, keepdims=True) + 1e-6
static_tensor = torch.tensor((static_arr - s_mean) / s_std, dtype=torch.float32).to(DEVICE)

p_buckets = {0: ([], [], []), 1: ([], [], []), 2: ([], [], [])}
for t in range(0, n_time - WINDOW - HORIZON + 1):
    target_t = t + WINDOW + HORIZON - 1
    s_start, s_target = split_id_per_hour[t], split_id_per_hour[target_t]
    if s_start != s_target or s_start == -1:
        continue
    x_win = time_arr_std[t:t + WINDOW]
    traj = region_episode_label[t + WINDOW: t + WINDOW + HORIZON]
    Xl, yregl, ytrajl = p_buckets[s_start]
    Xl.append(x_win)
    yregl.append(time_arr_std[target_t, :, pm25_col_idx])
    ytrajl.append(traj.T)

X0, yreg0, ytraj0 = (np.stack(v) for v in p_buckets[0])
X1, yreg1, ytraj1 = (np.stack(v) for v in p_buckets[1])
X2, yreg2, ytraj2 = (np.stack(v) for v in p_buckets[2])
del p_buckets, time_arr_std
gc.collect()
print(f"windows: train={len(X0)} val={len(X1)} test={len(X2)}")

X0_t = torch.tensor(X0, dtype=torch.float32); del X0
X1_t = torch.tensor(X1, dtype=torch.float32); del X1
X2_t = torch.tensor(X2, dtype=torch.float32); del X2
gc.collect()

yreg0_t, yreg1_t = torch.tensor(yreg0, dtype=torch.float32), torch.tensor(yreg1, dtype=torch.float32)
ytraj0_t = torch.tensor(ytraj0, dtype=torch.float32)
ytraj1_t = torch.tensor(ytraj1, dtype=torch.float32)
del yreg0, yreg1
gc.collect()

POS_WEIGHT = min(float((ytraj0_t.numel() - ytraj0_t.sum()) / ytraj0_t.sum().clamp(min=1)), 50.0)
print(f"pos_weight (per-hour trajectory): {POS_WEIGHT:.2f}")

region_criterion = nn.BCEWithLogitsLoss(pos_weight=torch.tensor(POS_WEIGHT))

def run_p1_epoch(model, X, y, optimizer, train):
    n = X.shape[0]
    idx = torch.randperm(n) if train else torch.arange(n)
    model.train(train)
    total_loss, total_n = 0.0, 0
    eff_batch = MICRO_BATCH * ACCUM_STEPS
    for start in range(0, n, eff_batch):
        if train: optimizer.zero_grad()
        batch_idx = idx[start:start + eff_batch]
        for ms in range(0, len(batch_idx), MICRO_BATCH):
            mb_idx = batch_idx[ms:ms + MICRO_BATCH]
            if len(mb_idx) == 0: continue
            xb = add_static_fn(X[mb_idx].to(DEVICE), static_tensor)
            yb = y[mb_idx].to(DEVICE)
            with torch.set_grad_enabled(train):
                pred = model(xb)
                loss = pinball_loss(pred, yb, QUANTILES)
            if train: (loss * len(mb_idx) / len(batch_idx)).backward()
            total_loss += loss.item() * len(mb_idx); total_n += len(mb_idx)
        if train: optimizer.step()
    return total_loss / total_n

def run_p2_traj_epoch(model, X, y_traj, optimizer, train):
    n = X.shape[0]
    idx = torch.randperm(n) if train else torch.arange(n)
    model.train(train)
    total_loss, total_n = 0.0, 0
    eff_batch = MICRO_BATCH * ACCUM_STEPS
    for start in range(0, n, eff_batch):
        if train: optimizer.zero_grad()
        batch_idx = idx[start:start + eff_batch]
        for ms in range(0, len(batch_idx), MICRO_BATCH):
            mb_idx = batch_idx[ms:ms + MICRO_BATCH]
            if len(mb_idx) == 0: continue
            xb = add_static_fn(X[mb_idx].to(DEVICE), static_tensor)
            yb = y_traj[mb_idx].to(DEVICE)
            with torch.set_grad_enabled(train):
                logits = model(xb, n_regions)
                loss = region_criterion(logits, yb)
            if train: loss.backward()
            total_loss += loss.item() * len(mb_idx); total_n += len(mb_idx)
        if train: optimizer.step()
    return total_loss / max(total_n, 1)

def predict_traj_probs(model, X, batch_size=64):
    model.eval()
    n = X.shape[0]
    out = np.zeros((n, n_regions, HORIZON), dtype=np.float32)
    with torch.no_grad():
        for start in range(0, n, batch_size):
            idx = torch.arange(start, min(start + batch_size, n))
            xb = add_static_fn(X[idx].to(DEVICE), static_tensor)
            logits = model(xb, n_regions)
            out[idx.numpy()] = torch.sigmoid(logits).cpu().numpy()
    return out

def train_one_seed_minimal(seed, lr=3e-3, dropout=0.5, weight_decay=5e-4):
    torch.manual_seed(seed); np.random.seed(seed)
    p1 = StationGRU(n_feats, dropout).to(DEVICE)
    opt1 = torch.optim.Adam(p1.parameters(), lr=lr, weight_decay=weight_decay)
    best_p1_val, best_p1_state = float("inf"), None
    for epoch in range(1, MAX_EPOCHS_P1 + 1):
        run_p1_epoch(p1, X0_t, yreg0_t, opt1, True)
        vl = run_p1_epoch(p1, X1_t, yreg1_t, opt1, False)
        if vl < best_p1_val:
            best_p1_val, best_p1_state = vl, copy.deepcopy(p1.state_dict())
    p1.load_state_dict(best_p1_state)
    encoder_copy = copy.deepcopy(p1.station_lin)
    p2 = RegionFromTrajectoryGRU(encoder_copy, region_membership_t, dropout).to(DEVICE)
    del p1; gc.collect()
    if DEVICE.type == "mps": torch.mps.empty_cache()

    opt2 = torch.optim.Adam(p2.parameters(), lr=lr, weight_decay=weight_decay)
    y_val_agg = ytraj1_t.numpy().max(axis=-1).reshape(-1)
    best_val_aucpr, best_state = -1.0, None
    for epoch in range(1, MAX_EPOCHS_P2 + 1):
        run_p2_traj_epoch(p2, X0_t, ytraj0_t, opt2, True)
        run_p2_traj_epoch(p2, X1_t, ytraj1_t, opt2, False)
        val_probs = predict_traj_probs(p2, X1_t)
        val_agg_score = val_probs.max(axis=-1).reshape(-1)
        va = average_precision_score(y_val_agg, val_agg_score)
        if va > best_val_aucpr:
            best_val_aucpr, best_state = va, copy.deepcopy(p2.state_dict())
    p2.load_state_dict(best_state)
    p2.eval()
    return p2, best_val_aucpr

# ================================================================
# RUN: 5 seeds, minimal_gru only
# ================================================================
N_SEEDS = 5
results_minimal = []

for seed in range(N_SEEDS):
    print(f"\n========== seed={seed}  model=minimal_gru ==========")
    t0 = time.time()
    p2, val_aucpr = train_one_seed_minimal(seed)
    test_probs = predict_traj_probs(p2, X2_t)
    y_test_agg = ytraj2.max(axis=-1).reshape(-1)
    test_agg_score = test_probs.max(axis=-1).reshape(-1)
    test_aucroc = roc_auc_score(y_test_agg, test_agg_score)
    test_aucpr = average_precision_score(y_test_agg, test_agg_score)
    print(f"  trained in {time.time()-t0:.0f}s  val_AUCPR={val_aucpr:.4f}  "
          f"test_AUCROC={test_aucroc:.4f}  test_AUCPR={test_aucpr:.4f}")
    results_minimal.append({"seed": seed, "val_aucpr": float(val_aucpr),
                             "test_aucroc": float(test_aucroc), "test_aucpr": float(test_aucpr)})
    del p2; gc.collect()
    if DEVICE.type == "mps": torch.mps.empty_cache()

# ================================================================
# COMPARE to already-saved wind_graph 10-seed results (seeds 0-4)
# ================================================================
with open(f"{BASE}/wind_vs_nograph_10seed.json") as f:
    prior = json.load(f)
wind_aucpr_5 = np.array([r["test_aucpr"] for r in prior["wind_graph"][:N_SEEDS]])
wind_aucroc_5 = np.array([r["test_aucroc"] for r in prior["wind_graph"][:N_SEEDS]])
minimal_aucpr = np.array([r["test_aucpr"] for r in results_minimal])
minimal_aucroc = np.array([r["test_aucroc"] for r in results_minimal])

print(f"\n{'='*20} SUMMARY: wind_graph vs minimal_gru (seeds 0-4) {'='*20}")
print(f"wind_graph   AUC-PR:  mean={wind_aucpr_5.mean():.4f}  values={wind_aucpr_5.round(4)}")
print(f"minimal_gru  AUC-PR:  mean={minimal_aucpr.mean():.4f}  values={minimal_aucpr.round(4)}")
print(f"wind_graph   AUC-ROC: mean={wind_aucroc_5.mean():.4f}  values={wind_aucroc_5.round(4)}")
print(f"minimal_gru  AUC-ROC: mean={minimal_aucroc.mean():.4f}  values={minimal_aucroc.round(4)}")

t_stat_pr, p_ttest_pr = stats.ttest_rel(wind_aucpr_5, minimal_aucpr)
w_stat_pr, p_wilcoxon_pr = stats.wilcoxon(wind_aucpr_5, minimal_aucpr)
n_wins_pr = int((wind_aucpr_5 > minimal_aucpr).sum())
print(f"\nAUC-PR: paired t-test t={t_stat_pr:.3f} p={p_ttest_pr:.4f}  |  Wilcoxon p={p_wilcoxon_pr:.4f}  |  wind_graph wins {n_wins_pr}/{N_SEEDS} seeds")

with open(f"{BASE}/wind_vs_minimal_gru_5seed.json", "w") as f:
    json.dump({"minimal_gru": results_minimal,
               "wind_graph_seeds_0to4": prior["wind_graph"][:N_SEEDS],
               "aucpr_paired_ttest": {"t": float(t_stat_pr), "p": float(p_ttest_pr)},
               "aucpr_wilcoxon": {"stat": float(w_stat_pr), "p": float(p_wilcoxon_pr)}}, f, indent=2)
print(f"\nsaved to {BASE}/wind_vs_minimal_gru_5seed.json")


device=mps
train hours (2016-2017): 17544  val hours (2018): 8760  test hours (2019): 8760
windows: train=17473 val=8689 test=8689
pos_weight (per-hour trajectory): 50.00

========== seed=0  model=minimal_gru ==========
  trained in 109s  val_AUCPR=0.4600  test_AUCROC=0.9507  test_AUCPR=0.6823

========== seed=1  model=minimal_gru ==========
  trained in 106s  val_AUCPR=0.4443  test_AUCROC=0.9469  test_AUCPR=0.6670

========== seed=2  model=minimal_gru ==========
  trained in 107s  val_AUCPR=0.4524  test_AUCROC=0.9467  test_AUCPR=0.6708

========== seed=3  model=minimal_gru ==========
  trained in 107s  val_AUCPR=0.4441  test_AUCROC=0.9422  test_AUCPR=0.6520

========== seed=4  model=minimal_gru ==========
  trained in 108s  val_AUCPR=0.4475  test_AUCROC=0.9443  test_AUCPR=0.6695

==================== SUMMARY: wind_graph vs minimal_gru (seeds 0-4) ====================
wind_graph   AUC-PR:  mean=0.7124  values=[0.7157 0.7033 0.7193 0.7107 0.7132]
minimal_gru  AUC-PR:  mean=0.6683  value

In [1]:
#change the definition of event to 8 hours of pm 
#over 75. 

# ================================================================
# LABEL-DURATION SENSITIVITY CHECK: 8-hour sustained episode
# (vs. the headline 2-hour definition) -- wind_graph, same 2016-2019 split
# Everything identical to the 2-hour final evaluation EXCEPT SUSTAIN_HOURS
# ================================================================
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import os, time, copy, gc, json
from sklearn.metrics import (average_precision_score, roc_auc_score,
                              precision_score, recall_score, matthews_corrcoef,
                              roc_curve)

BASE = "/Users/drewbaldwin/PM2_5 Research"
df = pd.read_pickle(f"{BASE}/air_korea_final_imputed_with_blh.pkl")
station_order = sorted(df["Station_ID"].unique())
n_stations = len(station_order)

stations = df[["Station_ID", "lat", "lon"]].drop_duplicates("Station_ID").set_index("Station_ID").loc[station_order]
lats, lons = stations["lat"].to_numpy(), stations["lon"].to_numpy()

def haversine_km(lat1, lon1, lat2, lon2):
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
    dlat, dlon = lat2 - lat1, lon2 - lon1
    a = np.sin(dlat/2)**2 + np.cos(lat1)*np.cos(lat2)*np.sin(dlon/2)**2
    return 2 * 6371.0 * np.arcsin(np.sqrt(a))

def bearing_matrix(lat, lon):
    lat_r, lon_r = np.radians(lat), np.radians(lon)
    lat1, lat2 = lat_r[:, None], lat_r[None, :]
    dlon = lon_r[None, :] - lon_r[:, None]
    x = np.sin(dlon) * np.cos(lat2)
    y = np.cos(lat1) * np.sin(lat2) - np.sin(lat1) * np.cos(lat2) * np.cos(dlon)
    return (np.degrees(np.arctan2(x, y)) + 360) % 360

dist_km = haversine_km(lats[:, None], lons[:, None], lats[None, :], lons[None, :])
DIST_CUTOFF, RHO_KM = 250.0, 250.0
HYBRID_THRESHOLD_KM = 20.0
dist_edges = (dist_km <= DIST_CUTOFF) & (dist_km > 0)
bearing_from = bearing_matrix(lats, lons)

src_idx, dst_idx = np.nonzero(dist_edges)
edge_index_np = np.stack([src_idx, dst_idx])
decay_edge = np.exp(-dist_km[src_idx, dst_idx] / RHO_KM).astype(np.float32)
bearing_edge = bearing_from[src_idx, dst_idx].astype(np.float32)
close_edge_mask = (dist_km[src_idx, dst_idx] <= HYBRID_THRESHOLD_KM)

REGION_CENTROIDS = {
    "Seoul": (37.566, 126.978), "Busan": (35.180, 129.075), "Daegu": (35.872, 128.602),
    "Incheon": (37.483, 126.633), "Gwangju": (35.155, 126.916), "Daejeon": (36.350, 127.385),
    "Ulsan": (35.550, 129.317), "Sejong": (36.487, 127.282), "Gyeonggi": (37.500, 127.250),
    "Gangwon": (37.867, 127.733), "Chungbuk": (36.633, 127.483), "Chungnam": (36.500, 126.750),
    "Jeonbuk": (35.824, 127.148), "Jeonnam": (34.750, 127.000), "Gyeongbuk": (36.559, 128.729),
    "Gyeongnam": (35.271, 128.663), "Jeju": (33.513, 126.523),
}
region_names = list(REGION_CENTROIDS.keys())
n_regions = len(region_names)
region_lats = np.array([REGION_CENTROIDS[r][0] for r in region_names])
region_lons = np.array([REGION_CENTROIDS[r][1] for r in region_names])
dist_to_region = haversine_km(lats[:, None], lons[:, None], region_lats[None, :], region_lons[None, :])
station_region_idx = dist_to_region.argmin(axis=1)

region_membership = np.zeros((n_stations, n_regions), dtype=np.float32)
region_membership[np.arange(n_stations), station_region_idx] = 1.0
region_membership_t = torch.tensor(region_membership)

region_dist_km = haversine_km(region_lats[:, None], region_lons[:, None], region_lats[None, :], region_lons[None, :])
region_bearing = bearing_matrix(region_lats, region_lons)
r_src_idx, r_dst_idx = np.nonzero(~np.eye(n_regions, dtype=bool))
region_edge_index_np = np.stack([r_src_idx, r_dst_idx])
region_decay_edge = np.exp(-region_dist_km[r_src_idx, r_dst_idx] / RHO_KM).astype(np.float32)
region_bearing_edge = region_bearing[r_src_idx, r_dst_idx].astype(np.float32)

WINDOW, HORIZON = 36, 36
GRAPH_RECENT_HOURS = 18
EVENT_THRESHOLD = 75.0
SUSTAIN_HOURS = 8  # <-- CHANGED from 2 to 8; everything else identical
QUANTILES = [0.50, 0.75, 0.90, 0.95, 0.99]
N_QUANTILES = len(QUANTILES)
TIME_FEATS = ["SO2", "CO", "NO2", "O3", "PM10", "PM25"]
STATIC_COLS = ["elevation_m", "urban_landuse_area_m2_3km", "green_space_area_3km",
               "building_footprint_area_3km", "railway_length_3km", "dist_to_coast_km",
               "dist_to_major_road_km", "industrial_area_m2_3km", "traffic_points_count_3km",
               "major_roads_count_3km", "total_road_length_3km"]

time_panels = {c: df.pivot(index="Datetime", columns="Station_ID", values=c)[station_order] for c in TIME_FEATS}
dt_index = time_panels["PM25"].index
n_time = len(dt_index)
years = dt_index.year.to_numpy()
doy = dt_index.dayofyear.to_numpy().astype(float)

split_id_per_hour = np.where((years == 2016) | (years == 2017), 0, np.where(years == 2018, 1, np.where(years == 2019, 2, -1)))
TRAIN_MASK = split_id_per_hour == 0

wind_dir_arr = df.pivot(index="Datetime", columns="Station_ID", values="winddirection_10m")[station_order].reindex(dt_index).to_numpy().astype(float)
wind_speed_arr = df.pivot(index="Datetime", columns="Station_ID", values="windspeed_10m")[station_order].reindex(dt_index).to_numpy().astype(float)
blh_arr = df.pivot(index="Datetime", columns="Station_ID", values="boundary_layer_height")[station_order].reindex(dt_index).to_numpy().astype(float)
pm25_raw_arr = time_panels["PM25"].to_numpy()

def regional_flat_mean(arr):
    out = np.zeros((n_time, n_regions), dtype=np.float32)
    for r in range(n_regions):
        cols = station_region_idx == r
        out[:, r] = np.nanmean(arr[:, cols], axis=1)
    return out

region_pm25 = regional_flat_mean(pm25_raw_arr)
wdir_sin_station = np.sin(np.radians(wind_dir_arr))
wdir_cos_station = np.cos(np.radians(wind_dir_arr))
region_windspeed = regional_flat_mean(wind_speed_arr)
region_wdir_sin = regional_flat_mean(wdir_sin_station)
region_wdir_cos = regional_flat_mean(wdir_cos_station)
region_wind_dir_deg = (np.degrees(np.arctan2(region_wdir_sin, region_wdir_cos)) + 360) % 360
region_wind_blows_toward = (region_wind_dir_deg + 180) % 360

season_sin_1d = np.sin(2 * np.pi * doy / 365.25)
season_cos_1d = np.cos(2 * np.pi * doy / 365.25)
season_sin = np.tile(season_sin_1d[:, None], (1, n_stations))
season_cos = np.tile(season_cos_1d[:, None], (1, n_stations))

TIME_FEATS_FULL = TIME_FEATS + ["windspeed_10m", "wdir_sin", "wdir_cos", "boundary_layer_height", "season_sin", "season_cos"]
n_time_feats, n_static_feats = len(TIME_FEATS_FULL), len(STATIC_COLS)
n_feats = n_time_feats + n_static_feats
pm25_col_idx = TIME_FEATS_FULL.index("PM25")

time_arr_raw = np.stack([time_panels[c].to_numpy() for c in TIME_FEATS] +
                         [wind_speed_arr, wdir_sin_station, wdir_cos_station, blh_arr, season_sin, season_cos], axis=-1)
del time_panels, season_sin, season_cos, blh_arr, pm25_raw_arr
gc.collect()

static_df = df[["Station_ID"] + STATIC_COLS].drop_duplicates("Station_ID").set_index("Station_ID").loc[station_order]
static_arr = static_df[STATIC_COLS].to_numpy()
del df
gc.collect()

rev = region_pm25[::-1]
roll_min_rev = pd.DataFrame(rev).rolling(window=SUSTAIN_HOURS, min_periods=SUSTAIN_HOURS).min().to_numpy()
region_episode_label = (roll_min_rev[::-1] >= EVENT_THRESHOLD).astype(np.float32)
del rev, roll_min_rev
gc.collect()

def pinball_loss(preds, target, quantiles):
    target_exp = target.unsqueeze(-1)
    diff = target_exp - preds
    q_tensor = torch.tensor(quantiles, device=preds.device, dtype=preds.dtype).view(*([1] * (preds.dim() - 1)), -1)
    return torch.max(q_tensor * diff, (q_tensor - 1) * diff).mean()

def monotonic_quantiles(raw):
    first = raw[..., :1]
    deltas = F.softplus(raw[..., 1:])
    return torch.cat([first, first + torch.cumsum(deltas, dim=-1)], dim=-1)

class WindConvLayer(nn.Module):
    def __init__(self, in_dim, out_dim):
        super().__init__()
        self.lin_self = nn.Linear(in_dim, out_dim)
        self.lin_neigh = nn.Linear(in_dim, out_dim)
        self.lin_connectivity = nn.Linear(1, out_dim)

    def forward(self, x, edge_index, edge_weight, num_nodes):
        src, dst = edge_index[0], edge_index[1]
        messages = x[src] * edge_weight.unsqueeze(-1)
        agg_sum = x.new_zeros(num_nodes, x.size(-1))
        agg_sum.index_add_(0, dst, messages)
        weight_sum = x.new_zeros(num_nodes)
        weight_sum.index_add_(0, dst, edge_weight)
        agg_mean = agg_sum / (weight_sum.unsqueeze(-1) + 1e-8)
        connectivity = torch.log1p(weight_sum.clamp(min=0)).unsqueeze(-1)
        return self.lin_self(x) + self.lin_neigh(agg_mean) + self.lin_connectivity(connectivity)

class AttentionPool(nn.Module):
    def __init__(self, hidden):
        super().__init__()
        self.attn_score = nn.Linear(hidden, 1)

    def forward(self, h_station, region_membership_t_local):
        B, N, H = h_station.shape
        scores = self.attn_score(h_station).squeeze(-1)
        scores = scores - scores.max(dim=1, keepdim=True).values
        exp_scores = torch.exp(scores)
        weighted_exp = exp_scores.unsqueeze(-1) * region_membership_t_local.unsqueeze(0)
        region_denom = weighted_exp.sum(dim=1)
        region_numer = torch.einsum('bnr,bnh->brh', weighted_exp, h_station)
        return region_numer / (region_denom.unsqueeze(-1) + 1e-8)

class StationQuantileGCN(nn.Module):
    def __init__(self, in_dim, dropout, hidden=32, gru_hidden=32):
        super().__init__()
        self.station_conv = WindConvLayer(in_dim, hidden)
        self.drop = nn.Dropout(dropout)
        self.gru = nn.GRU(hidden, gru_hidden, batch_first=True)
        self.head = nn.Linear(gru_hidden, N_QUANTILES)

    def forward(self, x_window, edge_index, edge_weight_seq):
        B, W, N, Fin = x_window.shape
        ei_b = torch.cat([edge_index + i * N for i in range(B)], dim=1)
        num_nodes = B * N
        h_seq = []
        for w in range(W):
            xt = x_window[:, w].reshape(B * N, Fin)
            ew_b = edge_weight_seq[:, w].reshape(-1)
            h = torch.relu(self.station_conv(xt, ei_b, ew_b, num_nodes))
            h = self.drop(h)
            h_seq.append(h.reshape(B, N, -1))
        h_seq = torch.stack(h_seq, dim=1).permute(0, 2, 1, 3).reshape(B * N, W, -1)
        _, h_final = self.gru(h_seq)
        embed = self.drop(h_final.squeeze(0).reshape(B, N, -1))
        return monotonic_quantiles(self.head(embed))

class RegionFromTrajectoryGCN(nn.Module):
    def __init__(self, station_conv, region_membership_t_local, dropout, hidden=32, gru_hidden=32):
        super().__init__()
        self.station_conv = station_conv
        self.attn_pool = AttentionPool(hidden)
        self.region_conv = WindConvLayer(hidden, hidden)
        self.drop = nn.Dropout(dropout)
        self.region_gru = nn.GRU(hidden, gru_hidden, batch_first=True)
        self.region_head = nn.Linear(gru_hidden, HORIZON)
        self.rmem = region_membership_t_local

    def forward(self, x_window, station_edge_index, station_edge_weight_seq, region_edge_index, region_edge_weight_seq, n_reg):
        B, W, N, Fin = x_window.shape
        station_ei_b = torch.cat([station_edge_index + i * N for i in range(B)], dim=1)
        region_ei_b = torch.cat([region_edge_index + i * n_reg for i in range(B)], dim=1)
        num_station_nodes = B * N
        num_region_nodes = B * n_reg
        h_region_seq = []
        for w in range(W):
            xt = x_window[:, w].reshape(B * N, Fin)
            ew_station_b = station_edge_weight_seq[:, w].reshape(-1)
            h_station = torch.relu(self.station_conv(xt, station_ei_b, ew_station_b, num_station_nodes))
            h_station = self.drop(h_station).reshape(B, N, -1)
            h_region_pooled = self.attn_pool(h_station, self.rmem).reshape(B * n_reg, -1)
            ew_region_b = region_edge_weight_seq[:, w].reshape(-1)
            h_region = torch.relu(self.region_conv(h_region_pooled, region_ei_b, ew_region_b, num_region_nodes))
            h_region = self.drop(h_region)
            h_region_seq.append(h_region.reshape(B, n_reg, -1))
        h_region_seq = torch.stack(h_region_seq, dim=1).permute(0, 2, 1, 3).reshape(B * n_reg, W, -1)
        _, h_final = self.region_gru(h_region_seq)
        embed = self.drop(h_final.squeeze(0).reshape(B, n_reg, -1))
        return self.region_head(embed)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else ("mps" if torch.backends.mps.is_available() else "cpu"))
MICRO_BATCH, ACCUM_STEPS = 16, 4
MAX_EPOCHS_P1, MAX_EPOCHS_P2 = 2, 2
print(f"device={DEVICE}")

region_membership_t = region_membership_t.to(DEVICE)
edge_index = torch.tensor(edge_index_np, dtype=torch.long).to(DEVICE)
region_edge_index = torch.tensor(region_edge_index_np, dtype=torch.long).to(DEVICE)

def add_static_fn(x_time_batch, static_tensor_local):
    B, W, N, _ = x_time_batch.shape
    static_b = static_tensor_local.unsqueeze(0).unsqueeze(0).expand(B, W, N, n_static_feats)
    return torch.cat([x_time_batch, static_b], dim=-1)

def gather_seq(ew_by_hour_t, starts_subset):
    idx = starts_subset.unsqueeze(1) + torch.arange(WINDOW).unsqueeze(0)
    ew = ew_by_hour_t[idx]
    ew = ew.clone()
    ew[:, :WINDOW - GRAPH_RECENT_HOURS, :] = 0.0
    return ew

print(f"train hours (2016-2017): {TRAIN_MASK.sum()}  val hours (2018): {(split_id_per_hour==1).sum()}  test hours (2019): {(split_id_per_hour==2).sum()}")
print(f"SUSTAIN_HOURS={SUSTAIN_HOURS}  EVENT_THRESHOLD={EVENT_THRESHOLD}")

wind_blows_toward = (wind_dir_arr + 180) % 360
wbt_src = wind_blows_toward[:, src_idx]
cos_align = np.maximum(np.cos(np.radians(wbt_src - bearing_edge[None, :])), 0.0)
speed_src = wind_speed_arr[:, src_idx]
wind_component = (cos_align * speed_src).astype(np.float32)
del wbt_src, cos_align, speed_src
gc.collect()
ref_speed = np.float32(np.nanmean(wind_speed_arr[TRAIN_MASK]))
component = np.where(close_edge_mask[None, :], ref_speed, wind_component)
station_wind_raw = np.nan_to_num(decay_edge[None, :] * component, nan=0.0).astype(np.float32)
del component
gc.collect()

r_wbt_src = region_wind_blows_toward[:, r_src_idx]
r_cos_align = np.maximum(np.cos(np.radians(r_wbt_src - region_bearing_edge[None, :])), 0.0)
r_speed_src = region_windspeed[:, r_src_idx]
region_wind_raw = np.nan_to_num(region_decay_edge[None, :] * r_cos_align * r_speed_src, nan=0.0).astype(np.float32)
del r_wbt_src, r_cos_align, r_speed_src
gc.collect()

train_nonzero = station_wind_raw[TRAIN_MASK][station_wind_raw[TRAIN_MASK] > 0]
wind_scale_s = train_nonzero.std()
station_wind_edge_weight = (station_wind_raw / wind_scale_s).astype(np.float32)
region_train_nonzero = region_wind_raw[TRAIN_MASK][region_wind_raw[TRAIN_MASK] > 0]
wind_scale_r = region_train_nonzero.std()
region_wind_edge_weight = (region_wind_raw / wind_scale_r).astype(np.float32)
del train_nonzero, region_train_nonzero, station_wind_raw, region_wind_raw
gc.collect()

t_mean = np.nanmean(time_arr_raw[TRAIN_MASK], axis=(0, 1), keepdims=True)
t_std = np.nanstd(time_arr_raw[TRAIN_MASK], axis=(0, 1), keepdims=True) + 1e-6
time_arr_std = np.nan_to_num((time_arr_raw - t_mean) / t_std, nan=0.0)
s_mean, s_std = static_arr.mean(axis=0, keepdims=True), static_arr.std(axis=0, keepdims=True) + 1e-6
static_tensor = torch.tensor((static_arr - s_mean) / s_std, dtype=torch.float32).to(DEVICE)

p_buckets = {0: ([], [], [], []), 1: ([], [], [], []), 2: ([], [], [], [])}
for t in range(0, n_time - WINDOW - HORIZON + 1):
    target_t = t + WINDOW + HORIZON - 1
    s_start, s_target = split_id_per_hour[t], split_id_per_hour[target_t]
    if s_start != s_target or s_start == -1:
        continue
    x_win = time_arr_std[t:t + WINDOW]
    traj = region_episode_label[t + WINDOW: t + WINDOW + HORIZON]
    Xl, yregl, ytrajl, sl = p_buckets[s_start]
    Xl.append(x_win)
    yregl.append(time_arr_std[target_t, :, pm25_col_idx])
    ytrajl.append(traj.T)
    sl.append(t)

X0, yreg0, ytraj0, starts0 = (np.stack(v) for v in p_buckets[0])
X1, yreg1, ytraj1, starts1 = (np.stack(v) for v in p_buckets[1])
X2, yreg2, ytraj2, starts2 = (np.stack(v) for v in p_buckets[2])
del p_buckets, time_arr_std
gc.collect()
print(f"windows: train={len(X0)} val={len(X1)} test={len(X2)}")
print(f"positive rate (train, per hour-region): {ytraj0.mean():.5f}")

X0_t = torch.tensor(X0, dtype=torch.float32); del X0
X1_t = torch.tensor(X1, dtype=torch.float32); del X1
X2_t = torch.tensor(X2, dtype=torch.float32); del X2
gc.collect()

yreg0_t, yreg1_t = torch.tensor(yreg0, dtype=torch.float32), torch.tensor(yreg1, dtype=torch.float32)
ytraj0_t = torch.tensor(ytraj0, dtype=torch.float32)
ytraj1_t = torch.tensor(ytraj1, dtype=torch.float32)
starts0_t, starts1_t, starts2_t = (torch.tensor(a, dtype=torch.long) for a in (starts0, starts1, starts2))
del yreg0, yreg1
gc.collect()

POS_WEIGHT = min(float((ytraj0_t.numel() - ytraj0_t.sum()) / ytraj0_t.sum().clamp(min=1)), 250.0)
print(f"pos_weight (per-hour trajectory): {POS_WEIGHT:.2f}")

wind_ewt_s = torch.tensor(station_wind_edge_weight)
wind_ewt_r = torch.tensor(region_wind_edge_weight)
del station_wind_edge_weight, region_wind_edge_weight
gc.collect()

region_criterion = nn.BCEWithLogitsLoss(pos_weight=torch.tensor(POS_WEIGHT))

def run_p1_epoch(model, ewt, X, y, starts, optimizer, train):
    n = X.shape[0]
    idx = torch.randperm(n) if train else torch.arange(n)
    model.train(train)
    total_loss, total_n = 0.0, 0
    eff_batch = MICRO_BATCH * ACCUM_STEPS
    for start in range(0, n, eff_batch):
        if train: optimizer.zero_grad()
        batch_idx = idx[start:start + eff_batch]
        for ms in range(0, len(batch_idx), MICRO_BATCH):
            mb_idx = batch_idx[ms:ms + MICRO_BATCH]
            if len(mb_idx) == 0: continue
            xb = add_static_fn(X[mb_idx].to(DEVICE), static_tensor)
            yb = y[mb_idx].to(DEVICE)
            ew_seq = gather_seq(ewt, starts[mb_idx]).to(DEVICE)
            with torch.set_grad_enabled(train):
                pred = model(xb, edge_index, ew_seq)
                loss = pinball_loss(pred, yb, QUANTILES)
            if train: (loss * len(mb_idx) / len(batch_idx)).backward()
            total_loss += loss.item() * len(mb_idx); total_n += len(mb_idx)
        if train: optimizer.step()
    return total_loss / total_n

def run_p2_traj_epoch(model, ewt_s, ewt_r, X, y_traj, starts, optimizer, train):
    n = X.shape[0]
    idx = torch.randperm(n) if train else torch.arange(n)
    model.train(train)
    total_loss, total_n = 0.0, 0
    eff_batch = MICRO_BATCH * ACCUM_STEPS
    for start in range(0, n, eff_batch):
        if train: optimizer.zero_grad()
        batch_idx = idx[start:start + eff_batch]
        for ms in range(0, len(batch_idx), MICRO_BATCH):
            mb_idx = batch_idx[ms:ms + MICRO_BATCH]
            if len(mb_idx) == 0: continue
            xb = add_static_fn(X[mb_idx].to(DEVICE), static_tensor)
            yb = y_traj[mb_idx].to(DEVICE)
            ew_s = gather_seq(ewt_s, starts[mb_idx]).to(DEVICE)
            ew_r = gather_seq(ewt_r, starts[mb_idx]).to(DEVICE)
            with torch.set_grad_enabled(train):
                logits = model(xb, edge_index, ew_s, region_edge_index, ew_r, n_regions)
                loss = region_criterion(logits, yb)
            if train: loss.backward()
            total_loss += loss.item() * len(mb_idx); total_n += len(mb_idx)
        if train: optimizer.step()
    return total_loss / max(total_n, 1)

def predict_traj_probs(model, ewt_s, ewt_r, X, starts, batch_size=64):
    model.eval()
    n = X.shape[0]
    out = np.zeros((n, n_regions, HORIZON), dtype=np.float32)
    with torch.no_grad():
        for start in range(0, n, batch_size):
            idx = torch.arange(start, min(start + batch_size, n))
            xb = add_static_fn(X[idx].to(DEVICE), static_tensor)
            ew_s = gather_seq(ewt_s, starts[idx]).to(DEVICE)
            ew_r = gather_seq(ewt_r, starts[idx]).to(DEVICE)
            logits = model(xb, edge_index, ew_s, region_edge_index, ew_r, n_regions)
            out[idx.numpy()] = torch.sigmoid(logits).cpu().numpy()
    return out

def train_one_seed_final(seed, lr=3e-3, dropout=0.5, weight_decay=5e-4):
    torch.manual_seed(seed); np.random.seed(seed)
    p1 = StationQuantileGCN(n_feats, dropout).to(DEVICE)
    opt1 = torch.optim.Adam(p1.parameters(), lr=lr, weight_decay=weight_decay)
    best_p1_val, best_p1_state = float("inf"), None
    for epoch in range(1, MAX_EPOCHS_P1 + 1):
        run_p1_epoch(p1, wind_ewt_s, X0_t, yreg0_t, starts0_t, opt1, True)
        vl = run_p1_epoch(p1, wind_ewt_s, X1_t, yreg1_t, starts1_t, opt1, False)
        if vl < best_p1_val:
            best_p1_val, best_p1_state = vl, copy.deepcopy(p1.state_dict())
    p1.load_state_dict(best_p1_state)
    encoder_copy = copy.deepcopy(p1.station_conv)
    p2 = RegionFromTrajectoryGCN(encoder_copy, region_membership_t, dropout).to(DEVICE)
    del p1; gc.collect()
    if DEVICE.type == "mps": torch.mps.empty_cache()

    opt2 = torch.optim.Adam(p2.parameters(), lr=lr, weight_decay=weight_decay)
    y_val_agg = ytraj1_t.numpy().max(axis=-1).reshape(-1)
    best_val_aucpr, best_state = -1.0, None
    for epoch in range(1, MAX_EPOCHS_P2 + 1):
        run_p2_traj_epoch(p2, wind_ewt_s, wind_ewt_r, X0_t, ytraj0_t, starts0_t, opt2, True)
        run_p2_traj_epoch(p2, wind_ewt_s, wind_ewt_r, X1_t, ytraj1_t, starts1_t, opt2, False)
        val_probs = predict_traj_probs(p2, wind_ewt_s, wind_ewt_r, X1_t, starts1_t)
        val_agg_score = val_probs.max(axis=-1).reshape(-1)
        va = average_precision_score(y_val_agg, val_agg_score)
        if va > best_val_aucpr:
            best_val_aucpr, best_state = va, copy.deepcopy(p2.state_dict())
    p2.load_state_dict(best_state)
    p2.eval()
    return p2, best_val_aucpr

def contiguous_lead_hours(prob_traj, threshold):
    if prob_traj[0] < threshold:
        return 0
    o = 1
    while o < HORIZON and prob_traj[o] >= threshold:
        o += 1
    return o

# ================================================================
# RUN: 5 seeds, SUSTAIN_HOURS=8
# ================================================================
N_SEEDS = 5
seed_results = []

for seed in range(N_SEEDS):
    print(f"\n{'='*15} seed={seed} {'='*15}")
    t0 = time.time()
    p2, val_aucpr = train_one_seed_final(seed)
    print(f"  trained in {time.time()-t0:.0f}s, val_AUCPR(agg)={val_aucpr:.4f}")

    val_probs = predict_traj_probs(p2, wind_ewt_s, wind_ewt_r, X1_t, starts1_t)
    test_probs = predict_traj_probs(p2, wind_ewt_s, wind_ewt_r, X2_t, starts2_t)

    y_val_agg = ytraj1_t.numpy().max(axis=-1).reshape(-1)
    y_test_agg = ytraj2.max(axis=-1).reshape(-1)
    val_agg_score = val_probs.max(axis=-1).reshape(-1)
    test_agg_score = test_probs.max(axis=-1).reshape(-1)

    test_aucroc = roc_auc_score(y_test_agg, test_agg_score)
    test_aucpr = average_precision_score(y_test_agg, test_agg_score)

    fpr, tpr, thresholds = roc_curve(y_val_agg, val_agg_score)
    j_scores = tpr - fpr
    thresh_youden = thresholds[np.argmax(j_scores)]

    f1_scores = []
    for th in thresholds:
        pred = (val_agg_score >= th).astype(int)
        p = precision_score(y_val_agg, pred, zero_division=0)
        r = recall_score(y_val_agg, pred, zero_division=0)
        f1 = 2 * p * r / (p + r) if (p + r) > 0 else 0.0
        f1_scores.append(f1)
    thresh_f1 = thresholds[np.argmax(f1_scores)]

    def eval_at_threshold(th, label):
        pred = (test_agg_score >= th).astype(int)
        precision = precision_score(y_test_agg, pred, zero_division=0)
        recall = recall_score(y_test_agg, pred, zero_division=0)
        mcc = matthews_corrcoef(y_test_agg, pred)
        far = 1 - precision
        print(f"  [{label}] thresh={th:.4f}  precision={precision:.4f}  recall(POD)={recall:.4f}  "
              f"FAR={far:.4f}  MCC={mcc:.4f}")
        return {"threshold": float(th), "precision": float(precision), "recall": float(recall),
                "far": float(far), "mcc": float(mcc)}

    print(f"  test AUC-ROC={test_aucroc:.4f}  AUC-PR={test_aucpr:.4f}")
    result_youden = eval_at_threshold(thresh_youden, "Youden J")
    result_f1 = eval_at_threshold(thresh_f1, "F1-optimal")

    best_thresh = thresh_f1
    lead_hours_list, onset_hours_list = [], []
    n_windows, n_reg = ytraj2.shape[0], ytraj2.shape[1]
    for i in range(n_windows):
        for r in range(n_reg):
            if ytraj2[i, r].max() < 1:
                continue
            onset_h = int(np.argmax(ytraj2[i, r] >= 1)) + 1
            lead = contiguous_lead_hours(test_probs[i, r], best_thresh)
            lead_hours_list.append(lead)
            onset_hours_list.append(onset_h)
    lead_hours_arr = np.array(lead_hours_list)
    onset_hours_arr = np.array(onset_hours_list)

    print(f"  n_true_positive_windows={len(lead_hours_arr)}  mean_lead_hours={lead_hours_arr.mean():.2f}  "
          f"median_lead_hours={np.median(lead_hours_arr):.1f}  mean_onset_hour_in_window={onset_hours_arr.mean():.2f}")

    seed_results.append({
        "seed": seed, "val_aucpr": float(val_aucpr), "test_aucroc": float(test_aucroc),
        "test_aucpr": float(test_aucpr), "youden": result_youden, "f1_optimal": result_f1,
        "mean_lead_hours": float(lead_hours_arr.mean()), "median_lead_hours": float(np.median(lead_hours_arr)),
        "mean_onset_hour_in_window": float(onset_hours_arr.mean()), "n_true_positive_windows": int(len(lead_hours_arr)),
    })

    del p2; gc.collect()
    if DEVICE.type == "mps": torch.mps.empty_cache()

print(f"\n{'='*20} SUMMARY: SUSTAIN_HOURS=8, {N_SEEDS} seeds {'='*20}")
for metric in ["test_aucroc", "test_aucpr", "mean_lead_hours", "median_lead_hours"]:
    vals = np.array([r[metric] for r in seed_results])
    print(f"  {metric:<25} mean={vals.mean():.4f}  std={vals.std():.4f}  values={vals.round(4)}")
for op_label in ["youden", "f1_optimal"]:
    print(f"\n  --- {op_label} operating point ---")
    for metric in ["precision", "recall", "far", "mcc"]:
        vals = np.array([r[op_label][metric] for r in seed_results])
        print(f"  {metric:<25} mean={vals.mean():.4f}  std={vals.std():.4f}  values={vals.round(4)}")

with open(f"{BASE}/wind_graph_8hour_sustain_5seed.json", "w") as f:
    json.dump({"sustain_hours": SUSTAIN_HOURS, "results": seed_results}, f, indent=2)
print(f"\nsaved to {BASE}/wind_graph_8hour_sustain_5seed.json")


device=mps
train hours (2016-2017): 17544  val hours (2018): 8760  test hours (2019): 8760
SUSTAIN_HOURS=8  EVENT_THRESHOLD=75.0
windows: train=17473 val=8689 test=8689
positive rate (train, per hour-region): 0.00346
pos_weight (per-hour trajectory): 250.00

=============== seed=0 ===============
  trained in 613s, val_AUCPR(agg)=0.2792
  test AUC-ROC=0.9606  AUC-PR=0.5962
  [Youden J] thresh=0.0375  precision=0.1293  recall(POD)=0.9509  FAR=0.8707  MCC=0.3095
  [F1-optimal] thresh=0.9752  precision=0.5838  recall(POD)=0.5453  FAR=0.4162  MCC=0.5511
  n_true_positive_windows=4460  mean_lead_hours=13.78  median_lead_hours=2.0  mean_onset_hour_in_window=10.97

=============== seed=1 ===============
  trained in 618s, val_AUCPR(agg)=0.3079
  test AUC-ROC=0.9552  AUC-PR=0.5791
  [Youden J] thresh=0.0665  precision=0.1284  recall(POD)=0.9334  FAR=0.8716  MCC=0.3043
  [F1-optimal] thresh=0.9681  precision=0.5576  recall(POD)=0.5549  FAR=0.4424  MCC=0.5425
  n_true_positive_windows=4460  mean

In [1]:
#retrain and retest on 2020-2021 data. 

# ================================================================
# WIND_GRAPH: retrain + optimize on 2020-2021 (COVID-era, separate model)
# Split: train=2020 (full year), val=Jan-Jun 2021, test=Jul-Dec 2021
# Step 1: small hyperparameter grid (4 configs, 1 seed each)
# Step 2: retrain best config across 5 seeds
# ================================================================
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import os, time, copy, gc, json
from sklearn.metrics import average_precision_score, roc_auc_score

BASE = "/Users/drewbaldwin/PM2_5 Research"
df = pd.read_pickle(f"{BASE}/air_korea_final_imputed_with_blh.pkl")
station_order = sorted(df["Station_ID"].unique())
n_stations = len(station_order)

stations = df[["Station_ID", "lat", "lon"]].drop_duplicates("Station_ID").set_index("Station_ID").loc[station_order]
lats, lons = stations["lat"].to_numpy(), stations["lon"].to_numpy()

def haversine_km(lat1, lon1, lat2, lon2):
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
    dlat, dlon = lat2 - lat1, lon2 - lon1
    a = np.sin(dlat/2)**2 + np.cos(lat1)*np.cos(lat2)*np.sin(dlon/2)**2
    return 2 * 6371.0 * np.arcsin(np.sqrt(a))

def bearing_matrix(lat, lon):
    lat_r, lon_r = np.radians(lat), np.radians(lon)
    lat1, lat2 = lat_r[:, None], lat_r[None, :]
    dlon = lon_r[None, :] - lon_r[:, None]
    x = np.sin(dlon) * np.cos(lat2)
    y = np.cos(lat1) * np.sin(lat2) - np.sin(lat1) * np.cos(lat2) * np.cos(dlon)
    return (np.degrees(np.arctan2(x, y)) + 360) % 360

dist_km = haversine_km(lats[:, None], lons[:, None], lats[None, :], lons[None, :])
DIST_CUTOFF, RHO_KM = 250.0, 250.0
HYBRID_THRESHOLD_KM = 20.0
dist_edges = (dist_km <= DIST_CUTOFF) & (dist_km > 0)
bearing_from = bearing_matrix(lats, lons)

src_idx, dst_idx = np.nonzero(dist_edges)
edge_index_np = np.stack([src_idx, dst_idx])
decay_edge = np.exp(-dist_km[src_idx, dst_idx] / RHO_KM).astype(np.float32)
bearing_edge = bearing_from[src_idx, dst_idx].astype(np.float32)
close_edge_mask = (dist_km[src_idx, dst_idx] <= HYBRID_THRESHOLD_KM)

REGION_CENTROIDS = {
    "Seoul": (37.566, 126.978), "Busan": (35.180, 129.075), "Daegu": (35.872, 128.602),
    "Incheon": (37.483, 126.633), "Gwangju": (35.155, 126.916), "Daejeon": (36.350, 127.385),
    "Ulsan": (35.550, 129.317), "Sejong": (36.487, 127.282), "Gyeonggi": (37.500, 127.250),
    "Gangwon": (37.867, 127.733), "Chungbuk": (36.633, 127.483), "Chungnam": (36.500, 126.750),
    "Jeonbuk": (35.824, 127.148), "Jeonnam": (34.750, 127.000), "Gyeongbuk": (36.559, 128.729),
    "Gyeongnam": (35.271, 128.663), "Jeju": (33.513, 126.523),
}
region_names = list(REGION_CENTROIDS.keys())
n_regions = len(region_names)
region_lats = np.array([REGION_CENTROIDS[r][0] for r in region_names])
region_lons = np.array([REGION_CENTROIDS[r][1] for r in region_names])
dist_to_region = haversine_km(lats[:, None], lons[:, None], region_lats[None, :], region_lons[None, :])
station_region_idx = dist_to_region.argmin(axis=1)

region_membership = np.zeros((n_stations, n_regions), dtype=np.float32)
region_membership[np.arange(n_stations), station_region_idx] = 1.0
region_membership_t = torch.tensor(region_membership)

region_dist_km = haversine_km(region_lats[:, None], region_lons[:, None], region_lats[None, :], region_lons[None, :])
region_bearing = bearing_matrix(region_lats, region_lons)
r_src_idx, r_dst_idx = np.nonzero(~np.eye(n_regions, dtype=bool))
region_edge_index_np = np.stack([r_src_idx, r_dst_idx])
region_decay_edge = np.exp(-region_dist_km[r_src_idx, r_dst_idx] / RHO_KM).astype(np.float32)
region_bearing_edge = region_bearing[r_src_idx, r_dst_idx].astype(np.float32)

WINDOW, HORIZON = 36, 36
GRAPH_RECENT_HOURS = 18
EVENT_THRESHOLD = 75.0
SUSTAIN_HOURS = 2
QUANTILES = [0.50, 0.75, 0.90, 0.95, 0.99]
N_QUANTILES = len(QUANTILES)
TIME_FEATS = ["SO2", "CO", "NO2", "O3", "PM10", "PM25"]
STATIC_COLS = ["elevation_m", "urban_landuse_area_m2_3km", "green_space_area_3km",
               "building_footprint_area_3km", "railway_length_3km", "dist_to_coast_km",
               "dist_to_major_road_km", "industrial_area_m2_3km", "traffic_points_count_3km",
               "major_roads_count_3km", "total_road_length_3km"]

time_panels = {c: df.pivot(index="Datetime", columns="Station_ID", values=c)[station_order] for c in TIME_FEATS}
dt_index = time_panels["PM25"].index
n_time = len(dt_index)
years = dt_index.year.to_numpy()
months = dt_index.month.to_numpy()
doy = dt_index.dayofyear.to_numpy().astype(float)

# ---- NEW SPLIT: train=2020, val=Jan-Jun 2021, test=Jul-Dec 2021 ----
split_id_per_hour = np.where(
    years == 2020, 0,
    np.where((years == 2021) & (months <= 6), 1,
    np.where((years == 2021) & (months >= 7), 2, -1)))
TRAIN_MASK = split_id_per_hour == 0

wind_dir_arr = df.pivot(index="Datetime", columns="Station_ID", values="winddirection_10m")[station_order].reindex(dt_index).to_numpy().astype(float)
wind_speed_arr = df.pivot(index="Datetime", columns="Station_ID", values="windspeed_10m")[station_order].reindex(dt_index).to_numpy().astype(float)
blh_arr = df.pivot(index="Datetime", columns="Station_ID", values="boundary_layer_height")[station_order].reindex(dt_index).to_numpy().astype(float)
pm25_raw_arr = time_panels["PM25"].to_numpy()

def regional_flat_mean(arr):
    out = np.zeros((n_time, n_regions), dtype=np.float32)
    for r in range(n_regions):
        cols = station_region_idx == r
        out[:, r] = np.nanmean(arr[:, cols], axis=1)
    return out

region_pm25 = regional_flat_mean(pm25_raw_arr)
wdir_sin_station = np.sin(np.radians(wind_dir_arr))
wdir_cos_station = np.cos(np.radians(wind_dir_arr))
region_windspeed = regional_flat_mean(wind_speed_arr)
region_wdir_sin = regional_flat_mean(wdir_sin_station)
region_wdir_cos = regional_flat_mean(wdir_cos_station)
region_wind_dir_deg = (np.degrees(np.arctan2(region_wdir_sin, region_wdir_cos)) + 360) % 360
region_wind_blows_toward = (region_wind_dir_deg + 180) % 360

season_sin_1d = np.sin(2 * np.pi * doy / 365.25)
season_cos_1d = np.cos(2 * np.pi * doy / 365.25)
season_sin = np.tile(season_sin_1d[:, None], (1, n_stations))
season_cos = np.tile(season_cos_1d[:, None], (1, n_stations))

TIME_FEATS_FULL = TIME_FEATS + ["windspeed_10m", "wdir_sin", "wdir_cos", "boundary_layer_height", "season_sin", "season_cos"]
n_time_feats, n_static_feats = len(TIME_FEATS_FULL), len(STATIC_COLS)
n_feats = n_time_feats + n_static_feats
pm25_col_idx = TIME_FEATS_FULL.index("PM25")

time_arr_raw = np.stack([time_panels[c].to_numpy() for c in TIME_FEATS] +
                         [wind_speed_arr, wdir_sin_station, wdir_cos_station, blh_arr, season_sin, season_cos], axis=-1)
del time_panels, season_sin, season_cos, blh_arr, pm25_raw_arr
gc.collect()

static_df = df[["Station_ID"] + STATIC_COLS].drop_duplicates("Station_ID").set_index("Station_ID").loc[station_order]
static_arr = static_df[STATIC_COLS].to_numpy()
del df
gc.collect()

rev = region_pm25[::-1]
roll_min_rev = pd.DataFrame(rev).rolling(window=SUSTAIN_HOURS, min_periods=SUSTAIN_HOURS).min().to_numpy()
region_episode_label = (roll_min_rev[::-1] >= EVENT_THRESHOLD).astype(np.float32)
del rev, roll_min_rev
gc.collect()

def pinball_loss(preds, target, quantiles):
    target_exp = target.unsqueeze(-1)
    diff = target_exp - preds
    q_tensor = torch.tensor(quantiles, device=preds.device, dtype=preds.dtype).view(*([1] * (preds.dim() - 1)), -1)
    return torch.max(q_tensor * diff, (q_tensor - 1) * diff).mean()

def monotonic_quantiles(raw):
    first = raw[..., :1]
    deltas = F.softplus(raw[..., 1:])
    return torch.cat([first, first + torch.cumsum(deltas, dim=-1)], dim=-1)

class WindConvLayer(nn.Module):
    def __init__(self, in_dim, out_dim):
        super().__init__()
        self.lin_self = nn.Linear(in_dim, out_dim)
        self.lin_neigh = nn.Linear(in_dim, out_dim)
        self.lin_connectivity = nn.Linear(1, out_dim)

    def forward(self, x, edge_index, edge_weight, num_nodes):
        src, dst = edge_index[0], edge_index[1]
        messages = x[src] * edge_weight.unsqueeze(-1)
        agg_sum = x.new_zeros(num_nodes, x.size(-1))
        agg_sum.index_add_(0, dst, messages)
        weight_sum = x.new_zeros(num_nodes)
        weight_sum.index_add_(0, dst, edge_weight)
        agg_mean = agg_sum / (weight_sum.unsqueeze(-1) + 1e-8)
        connectivity = torch.log1p(weight_sum.clamp(min=0)).unsqueeze(-1)
        return self.lin_self(x) + self.lin_neigh(agg_mean) + self.lin_connectivity(connectivity)

class AttentionPool(nn.Module):
    def __init__(self, hidden):
        super().__init__()
        self.attn_score = nn.Linear(hidden, 1)

    def forward(self, h_station, region_membership_t_local):
        B, N, H = h_station.shape
        scores = self.attn_score(h_station).squeeze(-1)
        scores = scores - scores.max(dim=1, keepdim=True).values
        exp_scores = torch.exp(scores)
        weighted_exp = exp_scores.unsqueeze(-1) * region_membership_t_local.unsqueeze(0)
        region_denom = weighted_exp.sum(dim=1)
        region_numer = torch.einsum('bnr,bnh->brh', weighted_exp, h_station)
        return region_numer / (region_denom.unsqueeze(-1) + 1e-8)

class StationQuantileGCN(nn.Module):
    def __init__(self, in_dim, dropout, hidden=32, gru_hidden=32):
        super().__init__()
        self.station_conv = WindConvLayer(in_dim, hidden)
        self.drop = nn.Dropout(dropout)
        self.gru = nn.GRU(hidden, gru_hidden, batch_first=True)
        self.head = nn.Linear(gru_hidden, N_QUANTILES)

    def forward(self, x_window, edge_index, edge_weight_seq):
        B, W, N, Fin = x_window.shape
        ei_b = torch.cat([edge_index + i * N for i in range(B)], dim=1)
        num_nodes = B * N
        h_seq = []
        for w in range(W):
            xt = x_window[:, w].reshape(B * N, Fin)
            ew_b = edge_weight_seq[:, w].reshape(-1)
            h = torch.relu(self.station_conv(xt, ei_b, ew_b, num_nodes))
            h = self.drop(h)
            h_seq.append(h.reshape(B, N, -1))
        h_seq = torch.stack(h_seq, dim=1).permute(0, 2, 1, 3).reshape(B * N, W, -1)
        _, h_final = self.gru(h_seq)
        embed = self.drop(h_final.squeeze(0).reshape(B, N, -1))
        return monotonic_quantiles(self.head(embed))

class RegionFromTrajectoryGCN(nn.Module):
    def __init__(self, station_conv, region_membership_t_local, dropout, hidden=32, gru_hidden=32):
        super().__init__()
        self.station_conv = station_conv
        self.attn_pool = AttentionPool(hidden)
        self.region_conv = WindConvLayer(hidden, hidden)
        self.drop = nn.Dropout(dropout)
        self.region_gru = nn.GRU(hidden, gru_hidden, batch_first=True)
        self.region_head = nn.Linear(gru_hidden, HORIZON)
        self.rmem = region_membership_t_local

    def forward(self, x_window, station_edge_index, station_edge_weight_seq, region_edge_index, region_edge_weight_seq, n_reg):
        B, W, N, Fin = x_window.shape
        station_ei_b = torch.cat([station_edge_index + i * N for i in range(B)], dim=1)
        region_ei_b = torch.cat([region_edge_index + i * n_reg for i in range(B)], dim=1)
        num_station_nodes = B * N
        num_region_nodes = B * n_reg
        h_region_seq = []
        for w in range(W):
            xt = x_window[:, w].reshape(B * N, Fin)
            ew_station_b = station_edge_weight_seq[:, w].reshape(-1)
            h_station = torch.relu(self.station_conv(xt, station_ei_b, ew_station_b, num_station_nodes))
            h_station = self.drop(h_station).reshape(B, N, -1)
            h_region_pooled = self.attn_pool(h_station, self.rmem).reshape(B * n_reg, -1)
            ew_region_b = region_edge_weight_seq[:, w].reshape(-1)
            h_region = torch.relu(self.region_conv(h_region_pooled, region_ei_b, ew_region_b, num_region_nodes))
            h_region = self.drop(h_region)
            h_region_seq.append(h_region.reshape(B, n_reg, -1))
        h_region_seq = torch.stack(h_region_seq, dim=1).permute(0, 2, 1, 3).reshape(B * n_reg, W, -1)
        _, h_final = self.region_gru(h_region_seq)
        embed = self.drop(h_final.squeeze(0).reshape(B, n_reg, -1))
        return self.region_head(embed)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else ("mps" if torch.backends.mps.is_available() else "cpu"))
MICRO_BATCH, ACCUM_STEPS = 16, 4
MAX_EPOCHS_P1, MAX_EPOCHS_P2 = 2, 2
print(f"device={DEVICE}")

region_membership_t = region_membership_t.to(DEVICE)
edge_index = torch.tensor(edge_index_np, dtype=torch.long).to(DEVICE)
region_edge_index = torch.tensor(region_edge_index_np, dtype=torch.long).to(DEVICE)

def add_static_fn(x_time_batch, static_tensor_local):
    B, W, N, _ = x_time_batch.shape
    static_b = static_tensor_local.unsqueeze(0).unsqueeze(0).expand(B, W, N, n_static_feats)
    return torch.cat([x_time_batch, static_b], dim=-1)

def gather_seq(ew_by_hour_t, starts_subset):
    idx = starts_subset.unsqueeze(1) + torch.arange(WINDOW).unsqueeze(0)
    ew = ew_by_hour_t[idx]
    ew = ew.clone()
    ew[:, :WINDOW - GRAPH_RECENT_HOURS, :] = 0.0
    return ew

print(f"train hours (2020): {TRAIN_MASK.sum()}  val hours (Jan-Jun 2021): {(split_id_per_hour==1).sum()}  test hours (Jul-Dec 2021): {(split_id_per_hour==2).sum()}")

wind_blows_toward = (wind_dir_arr + 180) % 360
wbt_src = wind_blows_toward[:, src_idx]
cos_align = np.maximum(np.cos(np.radians(wbt_src - bearing_edge[None, :])), 0.0)
speed_src = wind_speed_arr[:, src_idx]
wind_component = (cos_align * speed_src).astype(np.float32)
del wbt_src, cos_align, speed_src
gc.collect()
ref_speed = np.float32(np.nanmean(wind_speed_arr[TRAIN_MASK]))
component = np.where(close_edge_mask[None, :], ref_speed, wind_component)
station_wind_raw = np.nan_to_num(decay_edge[None, :] * component, nan=0.0).astype(np.float32)
del component
gc.collect()

r_wbt_src = region_wind_blows_toward[:, r_src_idx]
r_cos_align = np.maximum(np.cos(np.radians(r_wbt_src - region_bearing_edge[None, :])), 0.0)
r_speed_src = region_windspeed[:, r_src_idx]
region_wind_raw = np.nan_to_num(region_decay_edge[None, :] * r_cos_align * r_speed_src, nan=0.0).astype(np.float32)
del r_wbt_src, r_cos_align, r_speed_src
gc.collect()

train_nonzero = station_wind_raw[TRAIN_MASK][station_wind_raw[TRAIN_MASK] > 0]
wind_scale_s = train_nonzero.std()
station_wind_edge_weight = (station_wind_raw / wind_scale_s).astype(np.float32)
region_train_nonzero = region_wind_raw[TRAIN_MASK][region_wind_raw[TRAIN_MASK] > 0]
wind_scale_r = region_train_nonzero.std()
region_wind_edge_weight = (region_wind_raw / wind_scale_r).astype(np.float32)
del train_nonzero, region_train_nonzero, station_wind_raw, region_wind_raw
gc.collect()

t_mean = np.nanmean(time_arr_raw[TRAIN_MASK], axis=(0, 1), keepdims=True)
t_std = np.nanstd(time_arr_raw[TRAIN_MASK], axis=(0, 1), keepdims=True) + 1e-6
time_arr_std = np.nan_to_num((time_arr_raw - t_mean) / t_std, nan=0.0)
s_mean, s_std = static_arr.mean(axis=0, keepdims=True), static_arr.std(axis=0, keepdims=True) + 1e-6
static_tensor = torch.tensor((static_arr - s_mean) / s_std, dtype=torch.float32).to(DEVICE)

p_buckets = {0: ([], [], [], []), 1: ([], [], [], []), 2: ([], [], [], [])}
for t in range(0, n_time - WINDOW - HORIZON + 1):
    target_t = t + WINDOW + HORIZON - 1
    s_start, s_target = split_id_per_hour[t], split_id_per_hour[target_t]
    if s_start != s_target or s_start == -1:
        continue
    x_win = time_arr_std[t:t + WINDOW]
    traj = region_episode_label[t + WINDOW: t + WINDOW + HORIZON]
    Xl, yregl, ytrajl, sl = p_buckets[s_start]
    Xl.append(x_win)
    yregl.append(time_arr_std[target_t, :, pm25_col_idx])
    ytrajl.append(traj.T)
    sl.append(t)

X0, yreg0, ytraj0, starts0 = (np.stack(v) for v in p_buckets[0])
X1, yreg1, ytraj1, starts1 = (np.stack(v) for v in p_buckets[1])
X2, yreg2, ytraj2, starts2 = (np.stack(v) for v in p_buckets[2])
del p_buckets, time_arr_std
gc.collect()
print(f"windows: train={len(X0)} val={len(X1)} test={len(X2)}")

X0_t = torch.tensor(X0, dtype=torch.float32); del X0
X1_t = torch.tensor(X1, dtype=torch.float32); del X1
X2_t = torch.tensor(X2, dtype=torch.float32); del X2
gc.collect()

yreg0_t, yreg1_t = torch.tensor(yreg0, dtype=torch.float32), torch.tensor(yreg1, dtype=torch.float32)
ytraj0_t = torch.tensor(ytraj0, dtype=torch.float32)
ytraj1_t = torch.tensor(ytraj1, dtype=torch.float32)
starts0_t, starts1_t, starts2_t = (torch.tensor(a, dtype=torch.long) for a in (starts0, starts1, starts2))
del yreg0, yreg1
gc.collect()

POS_WEIGHT = min(float((ytraj0_t.numel() - ytraj0_t.sum()) / ytraj0_t.sum().clamp(min=1)), 50.0)
print(f"pos_weight (per-hour trajectory): {POS_WEIGHT:.2f}")

wind_ewt_s = torch.tensor(station_wind_edge_weight)
wind_ewt_r = torch.tensor(region_wind_edge_weight)
del station_wind_edge_weight, region_wind_edge_weight
gc.collect()

def run_p1_epoch(model, ewt, X, y, starts, optimizer, train, criterion=None):
    n = X.shape[0]
    idx = torch.randperm(n) if train else torch.arange(n)
    model.train(train)
    total_loss, total_n = 0.0, 0
    eff_batch = MICRO_BATCH * ACCUM_STEPS
    for start in range(0, n, eff_batch):
        if train: optimizer.zero_grad()
        batch_idx = idx[start:start + eff_batch]
        for ms in range(0, len(batch_idx), MICRO_BATCH):
            mb_idx = batch_idx[ms:ms + MICRO_BATCH]
            if len(mb_idx) == 0: continue
            xb = add_static_fn(X[mb_idx].to(DEVICE), static_tensor)
            yb = y[mb_idx].to(DEVICE)
            ew_seq = gather_seq(ewt, starts[mb_idx]).to(DEVICE)
            with torch.set_grad_enabled(train):
                pred = model(xb, edge_index, ew_seq)
                loss = pinball_loss(pred, yb, QUANTILES)
            if train: (loss * len(mb_idx) / len(batch_idx)).backward()
            total_loss += loss.item() * len(mb_idx); total_n += len(mb_idx)
        if train: optimizer.step()
    return total_loss / total_n

def run_p2_traj_epoch(model, ewt_s, ewt_r, X, y_traj, starts, optimizer, train, criterion):
    n = X.shape[0]
    idx = torch.randperm(n) if train else torch.arange(n)
    model.train(train)
    total_loss, total_n = 0.0, 0
    eff_batch = MICRO_BATCH * ACCUM_STEPS
    for start in range(0, n, eff_batch):
        if train: optimizer.zero_grad()
        batch_idx = idx[start:start + eff_batch]
        for ms in range(0, len(batch_idx), MICRO_BATCH):
            mb_idx = batch_idx[ms:ms + MICRO_BATCH]
            if len(mb_idx) == 0: continue
            xb = add_static_fn(X[mb_idx].to(DEVICE), static_tensor)
            yb = y_traj[mb_idx].to(DEVICE)
            ew_s = gather_seq(ewt_s, starts[mb_idx]).to(DEVICE)
            ew_r = gather_seq(ewt_r, starts[mb_idx]).to(DEVICE)
            with torch.set_grad_enabled(train):
                logits = model(xb, edge_index, ew_s, region_edge_index, ew_r, n_regions)
                loss = criterion(logits, yb)
            if train: loss.backward()
            total_loss += loss.item() * len(mb_idx); total_n += len(mb_idx)
        if train: optimizer.step()
    return total_loss / max(total_n, 1)

def predict_traj_probs(model, ewt_s, ewt_r, X, starts, batch_size=64):
    model.eval()
    n = X.shape[0]
    out = np.zeros((n, n_regions, HORIZON), dtype=np.float32)
    with torch.no_grad():
        for start in range(0, n, batch_size):
            idx = torch.arange(start, min(start + batch_size, n))
            xb = add_static_fn(X[idx].to(DEVICE), static_tensor)
            ew_s = gather_seq(ewt_s, starts[idx]).to(DEVICE)
            ew_r = gather_seq(ewt_r, starts[idx]).to(DEVICE)
            logits = model(xb, edge_index, ew_s, region_edge_index, ew_r, n_regions)
            out[idx.numpy()] = torch.sigmoid(logits).cpu().numpy()
    return out

def train_one_config(seed, lr, dropout, weight_decay):
    torch.manual_seed(seed); np.random.seed(seed)
    criterion = nn.BCEWithLogitsLoss(pos_weight=torch.tensor(POS_WEIGHT))
    p1 = StationQuantileGCN(n_feats, dropout).to(DEVICE)
    opt1 = torch.optim.Adam(p1.parameters(), lr=lr, weight_decay=weight_decay)
    best_p1_val, best_p1_state = float("inf"), None
    for epoch in range(1, MAX_EPOCHS_P1 + 1):
        run_p1_epoch(p1, wind_ewt_s, X0_t, yreg0_t, starts0_t, opt1, True)
        vl = run_p1_epoch(p1, wind_ewt_s, X1_t, yreg1_t, starts1_t, opt1, False)
        if vl < best_p1_val:
            best_p1_val, best_p1_state = vl, copy.deepcopy(p1.state_dict())
    p1.load_state_dict(best_p1_state)
    encoder_copy = copy.deepcopy(p1.station_conv)
    p2 = RegionFromTrajectoryGCN(encoder_copy, region_membership_t, dropout).to(DEVICE)
    del p1; gc.collect()
    if DEVICE.type == "mps": torch.mps.empty_cache()

    opt2 = torch.optim.Adam(p2.parameters(), lr=lr, weight_decay=weight_decay)
    y_val_agg = ytraj1_t.numpy().max(axis=-1).reshape(-1)
    best_val_aucpr, best_state = -1.0, None
    for epoch in range(1, MAX_EPOCHS_P2 + 1):
        run_p2_traj_epoch(p2, wind_ewt_s, wind_ewt_r, X0_t, ytraj0_t, starts0_t, opt2, True, criterion)
        run_p2_traj_epoch(p2, wind_ewt_s, wind_ewt_r, X1_t, ytraj1_t, starts1_t, opt2, False, criterion)
        val_probs = predict_traj_probs(p2, wind_ewt_s, wind_ewt_r, X1_t, starts1_t)
        val_agg_score = val_probs.max(axis=-1).reshape(-1)
        va = average_precision_score(y_val_agg, val_agg_score)
        if va > best_val_aucpr:
            best_val_aucpr, best_state = va, copy.deepcopy(p2.state_dict())
    p2.load_state_dict(best_state)
    p2.eval()
    return p2, best_val_aucpr

# ================================================================
# STEP 1: small hyperparameter grid search (1 seed each)
# ================================================================
GRID = [
    {"lr": 0.003, "dropout": 0.5, "weight_decay": 0.0005},  # original best config, as reference
    {"lr": 0.001, "dropout": 0.5, "weight_decay": 0.0005},
    {"lr": 0.003, "dropout": 0.3, "weight_decay": 0.0005},
    {"lr": 0.003, "dropout": 0.5, "weight_decay": 0.001},
]
print(f"\n{'#'*15} HYPERPARAMETER SEARCH (4 configs, seed=0) {'#'*15}")
grid_results = []
for cfg in GRID:
    t0 = time.time()
    p2, val_aucpr = train_one_config(seed=0, **cfg)
    print(f"  {cfg}  ->  val_AUCPR={val_aucpr:.4f}  ({time.time()-t0:.0f}s)")
    grid_results.append({**cfg, "val_aucpr": float(val_aucpr)})
    del p2; gc.collect()
    if DEVICE.type == "mps": torch.mps.empty_cache()

best_cfg = max(grid_results, key=lambda r: r["val_aucpr"])
print(f"\nbest config: {best_cfg}")

# ================================================================
# STEP 2: retrain best config across 5 seeds
# ================================================================
N_SEEDS = 5
print(f"\n{'#'*15} FINAL: best config, {N_SEEDS} seeds {'#'*15}")
final_results = []
for seed in range(N_SEEDS):
    t0 = time.time()
    p2, val_aucpr = train_one_config(seed, best_cfg["lr"], best_cfg["dropout"], best_cfg["weight_decay"])
    test_probs = predict_traj_probs(p2, wind_ewt_s, wind_ewt_r, X2_t, starts2_t)
    y_test_agg = ytraj2.max(axis=-1).reshape(-1)
    test_agg_score = test_probs.max(axis=-1).reshape(-1)
    test_aucroc = roc_auc_score(y_test_agg, test_agg_score)
    test_aucpr = average_precision_score(y_test_agg, test_agg_score)
    print(f"  seed={seed}  trained in {time.time()-t0:.0f}s  val_AUCPR={val_aucpr:.4f}  "
          f"test_AUCROC={test_aucroc:.4f}  test_AUCPR={test_aucpr:.4f}")
    final_results.append({"seed": seed, "val_aucpr": float(val_aucpr),
                           "test_aucroc": float(test_aucroc), "test_aucpr": float(test_aucpr)})
    del p2; gc.collect()
    if DEVICE.type == "mps": torch.mps.empty_cache()

test_aucpr_arr = np.array([r["test_aucpr"] for r in final_results])
test_aucroc_arr = np.array([r["test_aucroc"] for r in final_results])
print(f"\n{'='*20} SUMMARY: wind_graph on 2020-2021 (COVID-era) {'='*20}")
print(f"test AUC-PR:  mean={test_aucpr_arr.mean():.4f}  std={test_aucpr_arr.std():.4f}  values={test_aucpr_arr.round(4)}")
print(f"test AUC-ROC: mean={test_aucroc_arr.mean():.4f}  std={test_aucroc_arr.std():.4f}  values={test_aucroc_arr.round(4)}")

with open(f"{BASE}/wind_graph_2020_2021_covid.json", "w") as f:
    json.dump({"split": "train=2020, val=Jan-Jun 2021, test=Jul-Dec 2021",
               "grid_search": grid_results, "best_config": best_cfg, "final_results": final_results}, f, indent=2)
print(f"\nsaved to {BASE}/wind_graph_2020_2021_covid.json")


device=mps
train hours (2020): 8784  val hours (Jan-Jun 2021): 4344  test hours (Jul-Dec 2021): 4416
windows: train=8713 val=4273 test=4345
pos_weight (per-hour trajectory): 50.00

############### HYPERPARAMETER SEARCH (4 configs, seed=0) ###############
  {'lr': 0.003, 'dropout': 0.5, 'weight_decay': 0.0005}  ->  val_AUCPR=0.2585  (303s)
  {'lr': 0.001, 'dropout': 0.5, 'weight_decay': 0.0005}  ->  val_AUCPR=0.3032  (299s)
  {'lr': 0.003, 'dropout': 0.3, 'weight_decay': 0.0005}  ->  val_AUCPR=0.2712  (301s)
  {'lr': 0.003, 'dropout': 0.5, 'weight_decay': 0.001}  ->  val_AUCPR=0.2827  (296s)

best config: {'lr': 0.001, 'dropout': 0.5, 'weight_decay': 0.0005, 'val_aucpr': 0.3032179274359956}

############### FINAL: best config, 5 seeds ###############
  seed=0  trained in 315s  val_AUCPR=0.3032  test_AUCROC=0.9508  test_AUCPR=0.4919
  seed=1  trained in 315s  val_AUCPR=0.2906  test_AUCROC=0.9500  test_AUCPR=0.4569
  seed=2  trained in 318s  val_AUCPR=0.3416  test_AUCROC=0.9534  test_AUCP

In [1]:
#2018 as test year. 

# ================================================================
# CROSS-YEAR CHECK: train=2016, val=2017, test=2018
# wind_graph vs minimal_gru (no graph at all), same 5 seeds,
# same hyperparameters (selected via grid search on 2016/2017 only)
# ================================================================
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import os, time, copy, gc, json
from sklearn.metrics import average_precision_score, roc_auc_score
from scipy import stats

BASE = "/Users/drewbaldwin/PM2_5 Research"
df = pd.read_pickle(f"{BASE}/air_korea_final_imputed_with_blh.pkl")
station_order = sorted(df["Station_ID"].unique())
n_stations = len(station_order)

stations = df[["Station_ID", "lat", "lon"]].drop_duplicates("Station_ID").set_index("Station_ID").loc[station_order]
lats, lons = stations["lat"].to_numpy(), stations["lon"].to_numpy()

def haversine_km(lat1, lon1, lat2, lon2):
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
    dlat, dlon = lat2 - lat1, lon2 - lon1
    a = np.sin(dlat/2)**2 + np.cos(lat1)*np.cos(lat2)*np.sin(dlon/2)**2
    return 2 * 6371.0 * np.arcsin(np.sqrt(a))

def bearing_matrix(lat, lon):
    lat_r, lon_r = np.radians(lat), np.radians(lon)
    lat1, lat2 = lat_r[:, None], lat_r[None, :]
    dlon = lon_r[None, :] - lon_r[:, None]
    x = np.sin(dlon) * np.cos(lat2)
    y = np.cos(lat1) * np.sin(lat2) - np.sin(lat1) * np.cos(lat2) * np.cos(dlon)
    return (np.degrees(np.arctan2(x, y)) + 360) % 360

dist_km = haversine_km(lats[:, None], lons[:, None], lats[None, :], lons[None, :])
DIST_CUTOFF, RHO_KM = 250.0, 250.0
HYBRID_THRESHOLD_KM = 20.0
dist_edges = (dist_km <= DIST_CUTOFF) & (dist_km > 0)
bearing_from = bearing_matrix(lats, lons)

src_idx, dst_idx = np.nonzero(dist_edges)
edge_index_np = np.stack([src_idx, dst_idx])
decay_edge = np.exp(-dist_km[src_idx, dst_idx] / RHO_KM).astype(np.float32)
bearing_edge = bearing_from[src_idx, dst_idx].astype(np.float32)
close_edge_mask = (dist_km[src_idx, dst_idx] <= HYBRID_THRESHOLD_KM)

REGION_CENTROIDS = {
    "Seoul": (37.566, 126.978), "Busan": (35.180, 129.075), "Daegu": (35.872, 128.602),
    "Incheon": (37.483, 126.633), "Gwangju": (35.155, 126.916), "Daejeon": (36.350, 127.385),
    "Ulsan": (35.550, 129.317), "Sejong": (36.487, 127.282), "Gyeonggi": (37.500, 127.250),
    "Gangwon": (37.867, 127.733), "Chungbuk": (36.633, 127.483), "Chungnam": (36.500, 126.750),
    "Jeonbuk": (35.824, 127.148), "Jeonnam": (34.750, 127.000), "Gyeongbuk": (36.559, 128.729),
    "Gyeongnam": (35.271, 128.663), "Jeju": (33.513, 126.523),
}
region_names = list(REGION_CENTROIDS.keys())
n_regions = len(region_names)
region_lats = np.array([REGION_CENTROIDS[r][0] for r in region_names])
region_lons = np.array([REGION_CENTROIDS[r][1] for r in region_names])
dist_to_region = haversine_km(lats[:, None], lons[:, None], region_lats[None, :], region_lons[None, :])
station_region_idx = dist_to_region.argmin(axis=1)

region_membership = np.zeros((n_stations, n_regions), dtype=np.float32)
region_membership[np.arange(n_stations), station_region_idx] = 1.0
region_membership_t = torch.tensor(region_membership)

region_dist_km = haversine_km(region_lats[:, None], region_lons[:, None], region_lats[None, :], region_lons[None, :])
region_bearing = bearing_matrix(region_lats, region_lons)
r_src_idx, r_dst_idx = np.nonzero(~np.eye(n_regions, dtype=bool))
region_edge_index_np = np.stack([r_src_idx, r_dst_idx])
region_decay_edge = np.exp(-region_dist_km[r_src_idx, r_dst_idx] / RHO_KM).astype(np.float32)
region_bearing_edge = region_bearing[r_src_idx, r_dst_idx].astype(np.float32)

WINDOW, HORIZON = 36, 36
GRAPH_RECENT_HOURS = 18
EVENT_THRESHOLD = 75.0
SUSTAIN_HOURS = 2
QUANTILES = [0.50, 0.75, 0.90, 0.95, 0.99]
N_QUANTILES = len(QUANTILES)
TIME_FEATS = ["SO2", "CO", "NO2", "O3", "PM10", "PM25"]
STATIC_COLS = ["elevation_m", "urban_landuse_area_m2_3km", "green_space_area_3km",
               "building_footprint_area_3km", "railway_length_3km", "dist_to_coast_km",
               "dist_to_major_road_km", "industrial_area_m2_3km", "traffic_points_count_3km",
               "major_roads_count_3km", "total_road_length_3km"]

time_panels = {c: df.pivot(index="Datetime", columns="Station_ID", values=c)[station_order] for c in TIME_FEATS}
dt_index = time_panels["PM25"].index
n_time = len(dt_index)
years = dt_index.year.to_numpy()
doy = dt_index.dayofyear.to_numpy().astype(float)

split_id_per_hour = np.where(years == 2016, 0, np.where(years == 2017, 1, np.where(years == 2018, 2, -1)))
TRAIN_MASK = split_id_per_hour == 0

wind_dir_arr = df.pivot(index="Datetime", columns="Station_ID", values="winddirection_10m")[station_order].reindex(dt_index).to_numpy().astype(float)
wind_speed_arr = df.pivot(index="Datetime", columns="Station_ID", values="windspeed_10m")[station_order].reindex(dt_index).to_numpy().astype(float)
blh_arr = df.pivot(index="Datetime", columns="Station_ID", values="boundary_layer_height")[station_order].reindex(dt_index).to_numpy().astype(float)
pm25_raw_arr = time_panels["PM25"].to_numpy()

def regional_flat_mean(arr):
    out = np.zeros((n_time, n_regions), dtype=np.float32)
    for r in range(n_regions):
        cols = station_region_idx == r
        out[:, r] = np.nanmean(arr[:, cols], axis=1)
    return out

region_pm25 = regional_flat_mean(pm25_raw_arr)
wdir_sin_station = np.sin(np.radians(wind_dir_arr))
wdir_cos_station = np.cos(np.radians(wind_dir_arr))
region_windspeed = regional_flat_mean(wind_speed_arr)
region_wdir_sin = regional_flat_mean(wdir_sin_station)
region_wdir_cos = regional_flat_mean(wdir_cos_station)
region_wind_dir_deg = (np.degrees(np.arctan2(region_wdir_sin, region_wdir_cos)) + 360) % 360
region_wind_blows_toward = (region_wind_dir_deg + 180) % 360

season_sin_1d = np.sin(2 * np.pi * doy / 365.25)
season_cos_1d = np.cos(2 * np.pi * doy / 365.25)
season_sin = np.tile(season_sin_1d[:, None], (1, n_stations))
season_cos = np.tile(season_cos_1d[:, None], (1, n_stations))

TIME_FEATS_FULL = TIME_FEATS + ["windspeed_10m", "wdir_sin", "wdir_cos", "boundary_layer_height", "season_sin", "season_cos"]
n_time_feats, n_static_feats = len(TIME_FEATS_FULL), len(STATIC_COLS)
n_feats = n_time_feats + n_static_feats
pm25_col_idx = TIME_FEATS_FULL.index("PM25")

time_arr_raw = np.stack([time_panels[c].to_numpy() for c in TIME_FEATS] +
                         [wind_speed_arr, wdir_sin_station, wdir_cos_station, blh_arr, season_sin, season_cos], axis=-1)
del time_panels, season_sin, season_cos, blh_arr, pm25_raw_arr
gc.collect()

static_df = df[["Station_ID"] + STATIC_COLS].drop_duplicates("Station_ID").set_index("Station_ID").loc[station_order]
static_arr = static_df[STATIC_COLS].to_numpy()
del df
gc.collect()

rev = region_pm25[::-1]
roll_min_rev = pd.DataFrame(rev).rolling(window=SUSTAIN_HOURS, min_periods=SUSTAIN_HOURS).min().to_numpy()
region_episode_label = (roll_min_rev[::-1] >= EVENT_THRESHOLD).astype(np.float32)
del rev, roll_min_rev
gc.collect()

def pinball_loss(preds, target, quantiles):
    target_exp = target.unsqueeze(-1)
    diff = target_exp - preds
    q_tensor = torch.tensor(quantiles, device=preds.device, dtype=preds.dtype).view(*([1] * (preds.dim() - 1)), -1)
    return torch.max(q_tensor * diff, (q_tensor - 1) * diff).mean()

def monotonic_quantiles(raw):
    first = raw[..., :1]
    deltas = F.softplus(raw[..., 1:])
    return torch.cat([first, first + torch.cumsum(deltas, dim=-1)], dim=-1)

class WindConvLayer(nn.Module):
    def __init__(self, in_dim, out_dim):
        super().__init__()
        self.lin_self = nn.Linear(in_dim, out_dim)
        self.lin_neigh = nn.Linear(in_dim, out_dim)
        self.lin_connectivity = nn.Linear(1, out_dim)

    def forward(self, x, edge_index, edge_weight, num_nodes):
        src, dst = edge_index[0], edge_index[1]
        messages = x[src] * edge_weight.unsqueeze(-1)
        agg_sum = x.new_zeros(num_nodes, x.size(-1))
        agg_sum.index_add_(0, dst, messages)
        weight_sum = x.new_zeros(num_nodes)
        weight_sum.index_add_(0, dst, edge_weight)
        agg_mean = agg_sum / (weight_sum.unsqueeze(-1) + 1e-8)
        connectivity = torch.log1p(weight_sum.clamp(min=0)).unsqueeze(-1)
        return self.lin_self(x) + self.lin_neigh(agg_mean) + self.lin_connectivity(connectivity)

class AttentionPool(nn.Module):
    def __init__(self, hidden):
        super().__init__()
        self.attn_score = nn.Linear(hidden, 1)

    def forward(self, h_station, region_membership_t_local):
        B, N, H = h_station.shape
        scores = self.attn_score(h_station).squeeze(-1)
        scores = scores - scores.max(dim=1, keepdim=True).values
        exp_scores = torch.exp(scores)
        weighted_exp = exp_scores.unsqueeze(-1) * region_membership_t_local.unsqueeze(0)
        region_denom = weighted_exp.sum(dim=1)
        region_numer = torch.einsum('bnr,bnh->brh', weighted_exp, h_station)
        return region_numer / (region_denom.unsqueeze(-1) + 1e-8)

class StationQuantileGCN(nn.Module):
    def __init__(self, in_dim, dropout, hidden=32, gru_hidden=32):
        super().__init__()
        self.station_conv = WindConvLayer(in_dim, hidden)
        self.drop = nn.Dropout(dropout)
        self.gru = nn.GRU(hidden, gru_hidden, batch_first=True)
        self.head = nn.Linear(gru_hidden, N_QUANTILES)

    def forward(self, x_window, edge_index, edge_weight_seq):
        B, W, N, Fin = x_window.shape
        ei_b = torch.cat([edge_index + i * N for i in range(B)], dim=1)
        num_nodes = B * N
        h_seq = []
        for w in range(W):
            xt = x_window[:, w].reshape(B * N, Fin)
            ew_b = edge_weight_seq[:, w].reshape(-1)
            h = torch.relu(self.station_conv(xt, ei_b, ew_b, num_nodes))
            h = self.drop(h)
            h_seq.append(h.reshape(B, N, -1))
        h_seq = torch.stack(h_seq, dim=1).permute(0, 2, 1, 3).reshape(B * N, W, -1)
        _, h_final = self.gru(h_seq)
        embed = self.drop(h_final.squeeze(0).reshape(B, N, -1))
        return monotonic_quantiles(self.head(embed))

class RegionFromTrajectoryGCN(nn.Module):
    def __init__(self, station_conv, region_membership_t_local, dropout, hidden=32, gru_hidden=32):
        super().__init__()
        self.station_conv = station_conv
        self.attn_pool = AttentionPool(hidden)
        self.region_conv = WindConvLayer(hidden, hidden)
        self.drop = nn.Dropout(dropout)
        self.region_gru = nn.GRU(hidden, gru_hidden, batch_first=True)
        self.region_head = nn.Linear(gru_hidden, HORIZON)
        self.rmem = region_membership_t_local

    def forward(self, x_window, station_edge_index, station_edge_weight_seq, region_edge_index, region_edge_weight_seq, n_reg):
        B, W, N, Fin = x_window.shape
        station_ei_b = torch.cat([station_edge_index + i * N for i in range(B)], dim=1)
        region_ei_b = torch.cat([region_edge_index + i * n_reg for i in range(B)], dim=1)
        num_station_nodes = B * N
        num_region_nodes = B * n_reg
        h_region_seq = []
        for w in range(W):
            xt = x_window[:, w].reshape(B * N, Fin)
            ew_station_b = station_edge_weight_seq[:, w].reshape(-1)
            h_station = torch.relu(self.station_conv(xt, station_ei_b, ew_station_b, num_station_nodes))
            h_station = self.drop(h_station).reshape(B, N, -1)
            h_region_pooled = self.attn_pool(h_station, self.rmem).reshape(B * n_reg, -1)
            ew_region_b = region_edge_weight_seq[:, w].reshape(-1)
            h_region = torch.relu(self.region_conv(h_region_pooled, region_ei_b, ew_region_b, num_region_nodes))
            h_region = self.drop(h_region)
            h_region_seq.append(h_region.reshape(B, n_reg, -1))
        h_region_seq = torch.stack(h_region_seq, dim=1).permute(0, 2, 1, 3).reshape(B * n_reg, W, -1)
        _, h_final = self.region_gru(h_region_seq)
        embed = self.drop(h_final.squeeze(0).reshape(B, n_reg, -1))
        return self.region_head(embed)

class StationGRU(nn.Module):
    """No graph convolution at all -- per-station Linear -> GRU."""
    def __init__(self, in_dim, dropout, hidden=32, gru_hidden=32):
        super().__init__()
        self.station_lin = nn.Linear(in_dim, hidden)
        self.drop = nn.Dropout(dropout)
        self.gru = nn.GRU(hidden, gru_hidden, batch_first=True)
        self.head = nn.Linear(gru_hidden, N_QUANTILES)

    def forward(self, x_window):
        B, W, N, Fin = x_window.shape
        h_seq = []
        for w in range(W):
            xt = x_window[:, w].reshape(B * N, Fin)
            h = torch.relu(self.station_lin(xt))
            h = self.drop(h)
            h_seq.append(h.reshape(B, N, -1))
        h_seq = torch.stack(h_seq, dim=1).permute(0, 2, 1, 3).reshape(B * N, W, -1)
        _, h_final = self.gru(h_seq)
        embed = self.drop(h_final.squeeze(0).reshape(B, N, -1))
        return monotonic_quantiles(self.head(embed))

class RegionFromTrajectoryGRU(nn.Module):
    """Same AttentionPool station->region step as wind_graph, but no region_conv
    (zero cross-region information flow) -- true minimal no-graph baseline."""
    def __init__(self, station_lin, region_membership_t_local, dropout, hidden=32, gru_hidden=32):
        super().__init__()
        self.station_lin = station_lin
        self.attn_pool = AttentionPool(hidden)
        self.drop = nn.Dropout(dropout)
        self.region_gru = nn.GRU(hidden, gru_hidden, batch_first=True)
        self.region_head = nn.Linear(gru_hidden, HORIZON)
        self.rmem = region_membership_t_local

    def forward(self, x_window, n_reg):
        B, W, N, Fin = x_window.shape
        h_region_seq = []
        for w in range(W):
            xt = x_window[:, w].reshape(B * N, Fin)
            h_station = torch.relu(self.station_lin(xt)).reshape(B, N, -1)
            h_station = self.drop(h_station)
            h_region_pooled = self.attn_pool(h_station, self.rmem)
            h_region_pooled = self.drop(h_region_pooled)
            h_region_seq.append(h_region_pooled)
        h_region_seq = torch.stack(h_region_seq, dim=1).permute(0, 2, 1, 3).reshape(B * n_reg, W, -1)
        _, h_final = self.region_gru(h_region_seq)
        embed = self.drop(h_final.squeeze(0).reshape(B, n_reg, -1))
        return self.region_head(embed)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else ("mps" if torch.backends.mps.is_available() else "cpu"))
MICRO_BATCH, ACCUM_STEPS = 16, 4
MAX_EPOCHS_P1, MAX_EPOCHS_P2 = 2, 2
print(f"device={DEVICE}")

region_membership_t = region_membership_t.to(DEVICE)
edge_index = torch.tensor(edge_index_np, dtype=torch.long).to(DEVICE)
region_edge_index = torch.tensor(region_edge_index_np, dtype=torch.long).to(DEVICE)

def add_static_fn(x_time_batch, static_tensor_local):
    B, W, N, _ = x_time_batch.shape
    static_b = static_tensor_local.unsqueeze(0).unsqueeze(0).expand(B, W, N, n_static_feats)
    return torch.cat([x_time_batch, static_b], dim=-1)

def gather_seq(ew_by_hour_t, starts_subset):
    idx = starts_subset.unsqueeze(1) + torch.arange(WINDOW).unsqueeze(0)
    ew = ew_by_hour_t[idx]
    ew = ew.clone()
    ew[:, :WINDOW - GRAPH_RECENT_HOURS, :] = 0.0
    return ew

print(f"train hours (2016): {TRAIN_MASK.sum()}  val hours (2017): {(split_id_per_hour==1).sum()}  test hours (2018): {(split_id_per_hour==2).sum()}")

wind_blows_toward = (wind_dir_arr + 180) % 360
wbt_src = wind_blows_toward[:, src_idx]
cos_align = np.maximum(np.cos(np.radians(wbt_src - bearing_edge[None, :])), 0.0)
speed_src = wind_speed_arr[:, src_idx]
wind_component = (cos_align * speed_src).astype(np.float32)
del wbt_src, cos_align, speed_src
gc.collect()
ref_speed = np.float32(np.nanmean(wind_speed_arr[TRAIN_MASK]))
component = np.where(close_edge_mask[None, :], ref_speed, wind_component)
station_wind_raw = np.nan_to_num(decay_edge[None, :] * component, nan=0.0).astype(np.float32)
del component
gc.collect()

r_wbt_src = region_wind_blows_toward[:, r_src_idx]
r_cos_align = np.maximum(np.cos(np.radians(r_wbt_src - region_bearing_edge[None, :])), 0.0)
r_speed_src = region_windspeed[:, r_src_idx]
region_wind_raw = np.nan_to_num(region_decay_edge[None, :] * r_cos_align * r_speed_src, nan=0.0).astype(np.float32)
del r_wbt_src, r_cos_align, r_speed_src
gc.collect()

train_nonzero = station_wind_raw[TRAIN_MASK][station_wind_raw[TRAIN_MASK] > 0]
wind_scale_s = train_nonzero.std()
station_wind_edge_weight = (station_wind_raw / wind_scale_s).astype(np.float32)
region_train_nonzero = region_wind_raw[TRAIN_MASK][region_wind_raw[TRAIN_MASK] > 0]
wind_scale_r = region_train_nonzero.std()
region_wind_edge_weight = (region_wind_raw / wind_scale_r).astype(np.float32)
del train_nonzero, region_train_nonzero, station_wind_raw, region_wind_raw
gc.collect()

t_mean = np.nanmean(time_arr_raw[TRAIN_MASK], axis=(0, 1), keepdims=True)
t_std = np.nanstd(time_arr_raw[TRAIN_MASK], axis=(0, 1), keepdims=True) + 1e-6
time_arr_std = np.nan_to_num((time_arr_raw - t_mean) / t_std, nan=0.0)
s_mean, s_std = static_arr.mean(axis=0, keepdims=True), static_arr.std(axis=0, keepdims=True) + 1e-6
static_tensor = torch.tensor((static_arr - s_mean) / s_std, dtype=torch.float32).to(DEVICE)

p_buckets = {0: ([], [], [], []), 1: ([], [], [], []), 2: ([], [], [], [])}
for t in range(0, n_time - WINDOW - HORIZON + 1):
    target_t = t + WINDOW + HORIZON - 1
    s_start, s_target = split_id_per_hour[t], split_id_per_hour[target_t]
    if s_start != s_target or s_start == -1:
        continue
    x_win = time_arr_std[t:t + WINDOW]
    traj = region_episode_label[t + WINDOW: t + WINDOW + HORIZON]
    Xl, yregl, ytrajl, sl = p_buckets[s_start]
    Xl.append(x_win)
    yregl.append(time_arr_std[target_t, :, pm25_col_idx])
    ytrajl.append(traj.T)
    sl.append(t)

X0, yreg0, ytraj0, starts0 = (np.stack(v) for v in p_buckets[0])
X1, yreg1, ytraj1, starts1 = (np.stack(v) for v in p_buckets[1])
X2, yreg2, ytraj2, starts2 = (np.stack(v) for v in p_buckets[2])
del p_buckets, time_arr_std
gc.collect()
print(f"windows: train={len(X0)} val={len(X1)} test={len(X2)}")

X0_t = torch.tensor(X0, dtype=torch.float32); del X0
X1_t = torch.tensor(X1, dtype=torch.float32); del X1
X2_t = torch.tensor(X2, dtype=torch.float32); del X2
gc.collect()

yreg0_t, yreg1_t = torch.tensor(yreg0, dtype=torch.float32), torch.tensor(yreg1, dtype=torch.float32)
ytraj0_t = torch.tensor(ytraj0, dtype=torch.float32)
ytraj1_t = torch.tensor(ytraj1, dtype=torch.float32)
starts0_t, starts1_t, starts2_t = (torch.tensor(a, dtype=torch.long) for a in (starts0, starts1, starts2))
del yreg0, yreg1
gc.collect()

POS_WEIGHT = min(float((ytraj0_t.numel() - ytraj0_t.sum()) / ytraj0_t.sum().clamp(min=1)), 50.0)
print(f"pos_weight (per-hour trajectory): {POS_WEIGHT:.2f}")

wind_ewt_s = torch.tensor(station_wind_edge_weight)
wind_ewt_r = torch.tensor(region_wind_edge_weight)
del station_wind_edge_weight, region_wind_edge_weight
gc.collect()

region_criterion = nn.BCEWithLogitsLoss(pos_weight=torch.tensor(POS_WEIGHT))

def run_p1_epoch_graph(model, ewt, X, y, starts, optimizer, train):
    n = X.shape[0]
    idx = torch.randperm(n) if train else torch.arange(n)
    model.train(train)
    total_loss, total_n = 0.0, 0
    eff_batch = MICRO_BATCH * ACCUM_STEPS
    for start in range(0, n, eff_batch):
        if train: optimizer.zero_grad()
        batch_idx = idx[start:start + eff_batch]
        for ms in range(0, len(batch_idx), MICRO_BATCH):
            mb_idx = batch_idx[ms:ms + MICRO_BATCH]
            if len(mb_idx) == 0: continue
            xb = add_static_fn(X[mb_idx].to(DEVICE), static_tensor)
            yb = y[mb_idx].to(DEVICE)
            ew_seq = gather_seq(ewt, starts[mb_idx]).to(DEVICE)
            with torch.set_grad_enabled(train):
                pred = model(xb, edge_index, ew_seq)
                loss = pinball_loss(pred, yb, QUANTILES)
            if train: (loss * len(mb_idx) / len(batch_idx)).backward()
            total_loss += loss.item() * len(mb_idx); total_n += len(mb_idx)
        if train: optimizer.step()
    return total_loss / total_n

def run_p2_epoch_graph(model, ewt_s, ewt_r, X, y_traj, starts, optimizer, train):
    n = X.shape[0]
    idx = torch.randperm(n) if train else torch.arange(n)
    model.train(train)
    total_loss, total_n = 0.0, 0
    eff_batch = MICRO_BATCH * ACCUM_STEPS
    for start in range(0, n, eff_batch):
        if train: optimizer.zero_grad()
        batch_idx = idx[start:start + eff_batch]
        for ms in range(0, len(batch_idx), MICRO_BATCH):
            mb_idx = batch_idx[ms:ms + MICRO_BATCH]
            if len(mb_idx) == 0: continue
            xb = add_static_fn(X[mb_idx].to(DEVICE), static_tensor)
            yb = y_traj[mb_idx].to(DEVICE)
            ew_s = gather_seq(ewt_s, starts[mb_idx]).to(DEVICE)
            ew_r = gather_seq(ewt_r, starts[mb_idx]).to(DEVICE)
            with torch.set_grad_enabled(train):
                logits = model(xb, edge_index, ew_s, region_edge_index, ew_r, n_regions)
                loss = region_criterion(logits, yb)
            if train: loss.backward()
            total_loss += loss.item() * len(mb_idx); total_n += len(mb_idx)
        if train: optimizer.step()
    return total_loss / max(total_n, 1)

def predict_probs_graph(model, ewt_s, ewt_r, X, starts, batch_size=64):
    model.eval()
    n = X.shape[0]
    out = np.zeros((n, n_regions, HORIZON), dtype=np.float32)
    with torch.no_grad():
        for start in range(0, n, batch_size):
            idx = torch.arange(start, min(start + batch_size, n))
            xb = add_static_fn(X[idx].to(DEVICE), static_tensor)
            ew_s = gather_seq(ewt_s, starts[idx]).to(DEVICE)
            ew_r = gather_seq(ewt_r, starts[idx]).to(DEVICE)
            logits = model(xb, edge_index, ew_s, region_edge_index, ew_r, n_regions)
            out[idx.numpy()] = torch.sigmoid(logits).cpu().numpy()
    return out

def run_p1_epoch_gru(model, X, y, optimizer, train):
    n = X.shape[0]
    idx = torch.randperm(n) if train else torch.arange(n)
    model.train(train)
    total_loss, total_n = 0.0, 0
    eff_batch = MICRO_BATCH * ACCUM_STEPS
    for start in range(0, n, eff_batch):
        if train: optimizer.zero_grad()
        batch_idx = idx[start:start + eff_batch]
        for ms in range(0, len(batch_idx), MICRO_BATCH):
            mb_idx = batch_idx[ms:ms + MICRO_BATCH]
            if len(mb_idx) == 0: continue
            xb = add_static_fn(X[mb_idx].to(DEVICE), static_tensor)
            yb = y[mb_idx].to(DEVICE)
            with torch.set_grad_enabled(train):
                pred = model(xb)
                loss = pinball_loss(pred, yb, QUANTILES)
            if train: (loss * len(mb_idx) / len(batch_idx)).backward()
            total_loss += loss.item() * len(mb_idx); total_n += len(mb_idx)
        if train: optimizer.step()
    return total_loss / total_n

def run_p2_epoch_gru(model, X, y_traj, optimizer, train):
    n = X.shape[0]
    idx = torch.randperm(n) if train else torch.arange(n)
    model.train(train)
    total_loss, total_n = 0.0, 0
    eff_batch = MICRO_BATCH * ACCUM_STEPS
    for start in range(0, n, eff_batch):
        if train: optimizer.zero_grad()
        batch_idx = idx[start:start + eff_batch]
        for ms in range(0, len(batch_idx), MICRO_BATCH):
            mb_idx = batch_idx[ms:ms + MICRO_BATCH]
            if len(mb_idx) == 0: continue
            xb = add_static_fn(X[mb_idx].to(DEVICE), static_tensor)
            yb = y_traj[mb_idx].to(DEVICE)
            with torch.set_grad_enabled(train):
                logits = model(xb, n_regions)
                loss = region_criterion(logits, yb)
            if train: loss.backward()
            total_loss += loss.item() * len(mb_idx); total_n += len(mb_idx)
        if train: optimizer.step()
    return total_loss / max(total_n, 1)

def predict_probs_gru(model, X, batch_size=64):
    model.eval()
    n = X.shape[0]
    out = np.zeros((n, n_regions, HORIZON), dtype=np.float32)
    with torch.no_grad():
        for start in range(0, n, batch_size):
            idx = torch.arange(start, min(start + batch_size, n))
            xb = add_static_fn(X[idx].to(DEVICE), static_tensor)
            logits = model(xb, n_regions)
            out[idx.numpy()] = torch.sigmoid(logits).cpu().numpy()
    return out

def train_one_config_graph(seed, lr, dropout, weight_decay):
    torch.manual_seed(seed); np.random.seed(seed)
    p1 = StationQuantileGCN(n_feats, dropout).to(DEVICE)
    opt1 = torch.optim.Adam(p1.parameters(), lr=lr, weight_decay=weight_decay)
    best_p1_val, best_p1_state = float("inf"), None
    for epoch in range(1, MAX_EPOCHS_P1 + 1):
        run_p1_epoch_graph(p1, wind_ewt_s, X0_t, yreg0_t, starts0_t, opt1, True)
        vl = run_p1_epoch_graph(p1, wind_ewt_s, X1_t, yreg1_t, starts1_t, opt1, False)
        if vl < best_p1_val:
            best_p1_val, best_p1_state = vl, copy.deepcopy(p1.state_dict())
    p1.load_state_dict(best_p1_state)
    encoder_copy = copy.deepcopy(p1.station_conv)
    p2 = RegionFromTrajectoryGCN(encoder_copy, region_membership_t, dropout).to(DEVICE)
    del p1; gc.collect()
    if DEVICE.type == "mps": torch.mps.empty_cache()
    opt2 = torch.optim.Adam(p2.parameters(), lr=lr, weight_decay=weight_decay)
    y_val_agg = ytraj1_t.numpy().max(axis=-1).reshape(-1)
    best_val_aucpr, best_state = -1.0, None
    for epoch in range(1, MAX_EPOCHS_P2 + 1):
        run_p2_epoch_graph(p2, wind_ewt_s, wind_ewt_r, X0_t, ytraj0_t, starts0_t, opt2, True)
        run_p2_epoch_graph(p2, wind_ewt_s, wind_ewt_r, X1_t, ytraj1_t, starts1_t, opt2, False)
        val_probs = predict_probs_graph(p2, wind_ewt_s, wind_ewt_r, X1_t, starts1_t)
        va = average_precision_score(y_val_agg, val_probs.max(axis=-1).reshape(-1))
        if va > best_val_aucpr:
            best_val_aucpr, best_state = va, copy.deepcopy(p2.state_dict())
    p2.load_state_dict(best_state)
    p2.eval()
    return p2, best_val_aucpr

def train_one_seed_gru(seed, lr, dropout, weight_decay):
    torch.manual_seed(seed); np.random.seed(seed)
    p1 = StationGRU(n_feats, dropout).to(DEVICE)
    opt1 = torch.optim.Adam(p1.parameters(), lr=lr, weight_decay=weight_decay)
    best_p1_val, best_p1_state = float("inf"), None
    for epoch in range(1, MAX_EPOCHS_P1 + 1):
        run_p1_epoch_gru(p1, X0_t, yreg0_t, opt1, True)
        vl = run_p1_epoch_gru(p1, X1_t, yreg1_t, opt1, False)
        if vl < best_p1_val:
            best_p1_val, best_p1_state = vl, copy.deepcopy(p1.state_dict())
    p1.load_state_dict(best_p1_state)
    encoder_copy = copy.deepcopy(p1.station_lin)
    p2 = RegionFromTrajectoryGRU(encoder_copy, region_membership_t, dropout).to(DEVICE)
    del p1; gc.collect()
    if DEVICE.type == "mps": torch.mps.empty_cache()
    opt2 = torch.optim.Adam(p2.parameters(), lr=lr, weight_decay=weight_decay)
    y_val_agg = ytraj1_t.numpy().max(axis=-1).reshape(-1)
    best_val_aucpr, best_state = -1.0, None
    for epoch in range(1, MAX_EPOCHS_P2 + 1):
        run_p2_epoch_gru(p2, X0_t, ytraj0_t, opt2, True)
        run_p2_epoch_gru(p2, X1_t, ytraj1_t, opt2, False)
        val_probs = predict_probs_gru(p2, X1_t)
        va = average_precision_score(y_val_agg, val_probs.max(axis=-1).reshape(-1))
        if va > best_val_aucpr:
            best_val_aucpr, best_state = va, copy.deepcopy(p2.state_dict())
    p2.load_state_dict(best_state)
    p2.eval()
    return p2, best_val_aucpr

# ================================================================
# STEP 1: hyperparameter grid search (wind_graph, on 2016/2017 only)
# ================================================================
GRID = [
    {"lr": 0.003, "dropout": 0.5, "weight_decay": 0.0005},
    {"lr": 0.001, "dropout": 0.5, "weight_decay": 0.0005},
    {"lr": 0.003, "dropout": 0.3, "weight_decay": 0.0005},
    {"lr": 0.003, "dropout": 0.5, "weight_decay": 0.001},
]
print(f"\n{'#'*15} HYPERPARAMETER SEARCH on 2016/2017 only (wind_graph, 4 configs) {'#'*15}")
grid_results = []
for cfg in GRID:
    t0 = time.time()
    p2, val_aucpr = train_one_config_graph(seed=0, **cfg)
    print(f"  {cfg}  ->  val_AUCPR={val_aucpr:.4f}  ({time.time()-t0:.0f}s)")
    grid_results.append({**cfg, "val_aucpr": float(val_aucpr)})
    del p2; gc.collect()
    if DEVICE.type == "mps": torch.mps.empty_cache()

best_cfg = max(grid_results, key=lambda r: r["val_aucpr"])
print(f"\nbest config (selected using ONLY 2016/2017): {best_cfg}")

# ================================================================
# STEP 2: 5 seeds x {wind_graph, minimal_gru}, same hyperparameters
# ================================================================
N_SEEDS = 5
results = {"wind_graph": [], "minimal_gru": []}

for seed in range(N_SEEDS):
    print(f"\n{'='*10} seed={seed}  model=wind_graph {'='*10}")
    t0 = time.time()
    p2, val_aucpr = train_one_config_graph(seed, best_cfg["lr"], best_cfg["dropout"], best_cfg["weight_decay"])
    test_probs = predict_probs_graph(p2, wind_ewt_s, wind_ewt_r, X2_t, starts2_t)
    y_test_agg = ytraj2.max(axis=-1).reshape(-1)
    test_agg_score = test_probs.max(axis=-1).reshape(-1)
    test_aucroc = roc_auc_score(y_test_agg, test_agg_score)
    test_aucpr = average_precision_score(y_test_agg, test_agg_score)
    print(f"  trained in {time.time()-t0:.0f}s  val_AUCPR={val_aucpr:.4f}  test_AUCROC={test_aucroc:.4f}  test_AUCPR={test_aucpr:.4f}")
    results["wind_graph"].append({"seed": seed, "val_aucpr": float(val_aucpr), "test_aucroc": float(test_aucroc), "test_aucpr": float(test_aucpr)})
    del p2; gc.collect()
    if DEVICE.type == "mps": torch.mps.empty_cache()

    print(f"\n{'='*10} seed={seed}  model=minimal_gru {'='*10}")
    t0 = time.time()
    p2, val_aucpr = train_one_seed_gru(seed, best_cfg["lr"], best_cfg["dropout"], best_cfg["weight_decay"])
    test_probs = predict_probs_gru(p2, X2_t)
    test_agg_score = test_probs.max(axis=-1).reshape(-1)
    test_aucroc = roc_auc_score(y_test_agg, test_agg_score)
    test_aucpr = average_precision_score(y_test_agg, test_agg_score)
    print(f"  trained in {time.time()-t0:.0f}s  val_AUCPR={val_aucpr:.4f}  test_AUCROC={test_aucroc:.4f}  test_AUCPR={test_aucpr:.4f}")
    results["minimal_gru"].append({"seed": seed, "val_aucpr": float(val_aucpr), "test_aucroc": float(test_aucroc), "test_aucpr": float(test_aucpr)})
    del p2; gc.collect()
    if DEVICE.type == "mps": torch.mps.empty_cache()

wind_aucpr = np.array([r["test_aucpr"] for r in results["wind_graph"]])
gru_aucpr = np.array([r["test_aucpr"] for r in results["minimal_gru"]])
wind_aucroc = np.array([r["test_aucroc"] for r in results["wind_graph"]])
gru_aucroc = np.array([r["test_aucroc"] for r in results["minimal_gru"]])

print(f"\n{'='*20} SUMMARY: train=2016/val=2017/test=2018, {N_SEEDS} seeds {'='*20}")
print(f"wind_graph   AUC-PR:  mean={wind_aucpr.mean():.4f}  values={wind_aucpr.round(4)}")
print(f"minimal_gru  AUC-PR:  mean={gru_aucpr.mean():.4f}  values={gru_aucpr.round(4)}")
print(f"wind_graph   AUC-ROC: mean={wind_aucroc.mean():.4f}  values={wind_aucroc.round(4)}")
print(f"minimal_gru  AUC-ROC: mean={gru_aucroc.mean():.4f}  values={gru_aucroc.round(4)}")

t_stat_pr, p_ttest_pr = stats.ttest_rel(wind_aucpr, gru_aucpr)
w_stat_pr, p_wilcoxon_pr = stats.wilcoxon(wind_aucpr, gru_aucpr)
n_wins_pr = int((wind_aucpr > gru_aucpr).sum())
print(f"\nAUC-PR: paired t-test t={t_stat_pr:.3f} p={p_ttest_pr:.4f}  |  Wilcoxon p={p_wilcoxon_pr:.4f}  |  wind_graph wins {n_wins_pr}/{N_SEEDS} seeds")

with open(f"{BASE}/wind_vs_gru_2016_2017_2018_split_5seed.json", "w") as f:
    json.dump({"split": "train=2016, val=2017, test=2018", "grid_search": grid_results, "best_config": best_cfg,
               "wind_graph": results["wind_graph"], "minimal_gru": results["minimal_gru"],
               "aucpr_paired_ttest": {"t": float(t_stat_pr), "p": float(p_ttest_pr)},
               "aucpr_wilcoxon": {"stat": float(w_stat_pr), "p": float(p_wilcoxon_pr)}}, f, indent=2)
print(f"\nsaved to {BASE}/wind_vs_gru_2016_2017_2018_split_5seed.json")


device=mps
train hours (2016): 8784  val hours (2017): 8760  test hours (2018): 8760
windows: train=8713 val=8689 test=8689
pos_weight (per-hour trajectory): 50.00

############### HYPERPARAMETER SEARCH on 2016/2017 only (wind_graph, 4 configs) ###############
  {'lr': 0.003, 'dropout': 0.5, 'weight_decay': 0.0005}  ->  val_AUCPR=0.4163  (417s)
  {'lr': 0.001, 'dropout': 0.5, 'weight_decay': 0.0005}  ->  val_AUCPR=0.4242  (414s)
  {'lr': 0.003, 'dropout': 0.3, 'weight_decay': 0.0005}  ->  val_AUCPR=0.4026  (410s)
  {'lr': 0.003, 'dropout': 0.5, 'weight_decay': 0.001}  ->  val_AUCPR=0.4216  (410s)

best config (selected using ONLY 2016/2017): {'lr': 0.001, 'dropout': 0.5, 'weight_decay': 0.0005, 'val_aucpr': 0.42421804918960565}

========== seed=0  model=wind_graph ==========
  trained in 455s  val_AUCPR=0.4244  test_AUCROC=0.9318  test_AUCPR=0.4546

========== seed=0  model=minimal_gru ==========
  trained in 64s  val_AUCPR=0.3851  test_AUCROC=0.9177  test_AUCPR=0.4012

========== seed

In [1]:
#rebuild the final model using 2019 as test year. Save model. 

# ================================================================
# BUILD + SAVE THE HEADLINE wind_graph MODEL (train=2016-2017, val=2018, test=2019)
# 1) Trains once (seed=0, established best hyperparameters) and saves everything
#    needed to reload/reuse it later for plots/figures without retraining.
# 2) Computes per-region precision/recall/MCC/lead-time (all 17 regions).
# ================================================================
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import os, time, copy, gc, json
from sklearn.metrics import (average_precision_score, roc_auc_score,
                              precision_score, recall_score, matthews_corrcoef, roc_curve)

BASE = "/Users/drewbaldwin/PM2_5 Research"
SAVE_DIR = f"{BASE}/saved_wind_graph_2019"
os.makedirs(SAVE_DIR, exist_ok=True)

df = pd.read_pickle(f"{BASE}/air_korea_final_imputed_with_blh.pkl")
station_order = sorted(df["Station_ID"].unique())
n_stations = len(station_order)

stations = df[["Station_ID", "lat", "lon"]].drop_duplicates("Station_ID").set_index("Station_ID").loc[station_order]
lats, lons = stations["lat"].to_numpy(), stations["lon"].to_numpy()

def haversine_km(lat1, lon1, lat2, lon2):
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
    dlat, dlon = lat2 - lat1, lon2 - lon1
    a = np.sin(dlat/2)**2 + np.cos(lat1)*np.cos(lat2)*np.sin(dlon/2)**2
    return 2 * 6371.0 * np.arcsin(np.sqrt(a))

def bearing_matrix(lat, lon):
    lat_r, lon_r = np.radians(lat), np.radians(lon)
    lat1, lat2 = lat_r[:, None], lat_r[None, :]
    dlon = lon_r[None, :] - lon_r[:, None]
    x = np.sin(dlon) * np.cos(lat2)
    y = np.cos(lat1) * np.sin(lat2) - np.sin(lat1) * np.cos(lat2) * np.cos(dlon)
    return (np.degrees(np.arctan2(x, y)) + 360) % 360

dist_km = haversine_km(lats[:, None], lons[:, None], lats[None, :], lons[None, :])
DIST_CUTOFF, RHO_KM = 250.0, 250.0
HYBRID_THRESHOLD_KM = 20.0
dist_edges = (dist_km <= DIST_CUTOFF) & (dist_km > 0)
bearing_from = bearing_matrix(lats, lons)

src_idx, dst_idx = np.nonzero(dist_edges)
edge_index_np = np.stack([src_idx, dst_idx])
decay_edge = np.exp(-dist_km[src_idx, dst_idx] / RHO_KM).astype(np.float32)
bearing_edge = bearing_from[src_idx, dst_idx].astype(np.float32)
close_edge_mask = (dist_km[src_idx, dst_idx] <= HYBRID_THRESHOLD_KM)

REGION_CENTROIDS = {
    "Seoul": (37.566, 126.978), "Busan": (35.180, 129.075), "Daegu": (35.872, 128.602),
    "Incheon": (37.483, 126.633), "Gwangju": (35.155, 126.916), "Daejeon": (36.350, 127.385),
    "Ulsan": (35.550, 129.317), "Sejong": (36.487, 127.282), "Gyeonggi": (37.500, 127.250),
    "Gangwon": (37.867, 127.733), "Chungbuk": (36.633, 127.483), "Chungnam": (36.500, 126.750),
    "Jeonbuk": (35.824, 127.148), "Jeonnam": (34.750, 127.000), "Gyeongbuk": (36.559, 128.729),
    "Gyeongnam": (35.271, 128.663), "Jeju": (33.513, 126.523),
}
region_names = list(REGION_CENTROIDS.keys())
n_regions = len(region_names)
region_lats = np.array([REGION_CENTROIDS[r][0] for r in region_names])
region_lons = np.array([REGION_CENTROIDS[r][1] for r in region_names])
dist_to_region = haversine_km(lats[:, None], lons[:, None], region_lats[None, :], region_lons[None, :])
station_region_idx = dist_to_region.argmin(axis=1)

region_membership = np.zeros((n_stations, n_regions), dtype=np.float32)
region_membership[np.arange(n_stations), station_region_idx] = 1.0
region_membership_t = torch.tensor(region_membership)

region_dist_km = haversine_km(region_lats[:, None], region_lons[:, None], region_lats[None, :], region_lons[None, :])
region_bearing = bearing_matrix(region_lats, region_lons)
r_src_idx, r_dst_idx = np.nonzero(~np.eye(n_regions, dtype=bool))
region_edge_index_np = np.stack([r_src_idx, r_dst_idx])
region_decay_edge = np.exp(-region_dist_km[r_src_idx, r_dst_idx] / RHO_KM).astype(np.float32)
region_bearing_edge = region_bearing[r_src_idx, r_dst_idx].astype(np.float32)

WINDOW, HORIZON = 36, 36
GRAPH_RECENT_HOURS = 18
EVENT_THRESHOLD = 75.0
SUSTAIN_HOURS = 2
QUANTILES = [0.50, 0.75, 0.90, 0.95, 0.99]
N_QUANTILES = len(QUANTILES)
TIME_FEATS = ["SO2", "CO", "NO2", "O3", "PM10", "PM25"]
STATIC_COLS = ["elevation_m", "urban_landuse_area_m2_3km", "green_space_area_3km",
               "building_footprint_area_3km", "railway_length_3km", "dist_to_coast_km",
               "dist_to_major_road_km", "industrial_area_m2_3km", "traffic_points_count_3km",
               "major_roads_count_3km", "total_road_length_3km"]

time_panels = {c: df.pivot(index="Datetime", columns="Station_ID", values=c)[station_order] for c in TIME_FEATS}
dt_index = time_panels["PM25"].index
n_time = len(dt_index)
years = dt_index.year.to_numpy()
doy = dt_index.dayofyear.to_numpy().astype(float)

split_id_per_hour = np.where((years == 2016) | (years == 2017), 0, np.where(years == 2018, 1, np.where(years == 2019, 2, -1)))
TRAIN_MASK = split_id_per_hour == 0

wind_dir_arr = df.pivot(index="Datetime", columns="Station_ID", values="winddirection_10m")[station_order].reindex(dt_index).to_numpy().astype(float)
wind_speed_arr = df.pivot(index="Datetime", columns="Station_ID", values="windspeed_10m")[station_order].reindex(dt_index).to_numpy().astype(float)
blh_arr = df.pivot(index="Datetime", columns="Station_ID", values="boundary_layer_height")[station_order].reindex(dt_index).to_numpy().astype(float)
pm25_raw_arr = time_panels["PM25"].to_numpy()

def regional_flat_mean(arr):
    out = np.zeros((n_time, n_regions), dtype=np.float32)
    for r in range(n_regions):
        cols = station_region_idx == r
        out[:, r] = np.nanmean(arr[:, cols], axis=1)
    return out

region_pm25 = regional_flat_mean(pm25_raw_arr)
wdir_sin_station = np.sin(np.radians(wind_dir_arr))
wdir_cos_station = np.cos(np.radians(wind_dir_arr))
region_windspeed = regional_flat_mean(wind_speed_arr)
region_wdir_sin = regional_flat_mean(wdir_sin_station)
region_wdir_cos = regional_flat_mean(wdir_cos_station)
region_wind_dir_deg = (np.degrees(np.arctan2(region_wdir_sin, region_wdir_cos)) + 360) % 360
region_wind_blows_toward = (region_wind_dir_deg + 180) % 360

season_sin_1d = np.sin(2 * np.pi * doy / 365.25)
season_cos_1d = np.cos(2 * np.pi * doy / 365.25)
season_sin = np.tile(season_sin_1d[:, None], (1, n_stations))
season_cos = np.tile(season_cos_1d[:, None], (1, n_stations))

TIME_FEATS_FULL = TIME_FEATS + ["windspeed_10m", "wdir_sin", "wdir_cos", "boundary_layer_height", "season_sin", "season_cos"]
n_time_feats, n_static_feats = len(TIME_FEATS_FULL), len(STATIC_COLS)
n_feats = n_time_feats + n_static_feats
pm25_col_idx = TIME_FEATS_FULL.index("PM25")

time_arr_raw = np.stack([time_panels[c].to_numpy() for c in TIME_FEATS] +
                         [wind_speed_arr, wdir_sin_station, wdir_cos_station, blh_arr, season_sin, season_cos], axis=-1)
del time_panels, season_sin, season_cos, blh_arr, pm25_raw_arr
gc.collect()

static_df = df[["Station_ID"] + STATIC_COLS].drop_duplicates("Station_ID").set_index("Station_ID").loc[station_order]
static_arr = static_df[STATIC_COLS].to_numpy()
del df
gc.collect()

rev = region_pm25[::-1]
roll_min_rev = pd.DataFrame(rev).rolling(window=SUSTAIN_HOURS, min_periods=SUSTAIN_HOURS).min().to_numpy()
region_episode_label = (roll_min_rev[::-1] >= EVENT_THRESHOLD).astype(np.float32)
del rev, roll_min_rev
gc.collect()

def pinball_loss(preds, target, quantiles):
    target_exp = target.unsqueeze(-1)
    diff = target_exp - preds
    q_tensor = torch.tensor(quantiles, device=preds.device, dtype=preds.dtype).view(*([1] * (preds.dim() - 1)), -1)
    return torch.max(q_tensor * diff, (q_tensor - 1) * diff).mean()

def monotonic_quantiles(raw):
    first = raw[..., :1]
    deltas = F.softplus(raw[..., 1:])
    return torch.cat([first, first + torch.cumsum(deltas, dim=-1)], dim=-1)

class WindConvLayer(nn.Module):
    def __init__(self, in_dim, out_dim):
        super().__init__()
        self.lin_self = nn.Linear(in_dim, out_dim)
        self.lin_neigh = nn.Linear(in_dim, out_dim)
        self.lin_connectivity = nn.Linear(1, out_dim)

    def forward(self, x, edge_index, edge_weight, num_nodes):
        src, dst = edge_index[0], edge_index[1]
        messages = x[src] * edge_weight.unsqueeze(-1)
        agg_sum = x.new_zeros(num_nodes, x.size(-1))
        agg_sum.index_add_(0, dst, messages)
        weight_sum = x.new_zeros(num_nodes)
        weight_sum.index_add_(0, dst, edge_weight)
        agg_mean = agg_sum / (weight_sum.unsqueeze(-1) + 1e-8)
        connectivity = torch.log1p(weight_sum.clamp(min=0)).unsqueeze(-1)
        return self.lin_self(x) + self.lin_neigh(agg_mean) + self.lin_connectivity(connectivity)

class AttentionPool(nn.Module):
    def __init__(self, hidden):
        super().__init__()
        self.attn_score = nn.Linear(hidden, 1)

    def forward(self, h_station, region_membership_t_local):
        B, N, H = h_station.shape
        scores = self.attn_score(h_station).squeeze(-1)
        scores = scores - scores.max(dim=1, keepdim=True).values
        exp_scores = torch.exp(scores)
        weighted_exp = exp_scores.unsqueeze(-1) * region_membership_t_local.unsqueeze(0)
        region_denom = weighted_exp.sum(dim=1)
        region_numer = torch.einsum('bnr,bnh->brh', weighted_exp, h_station)
        return region_numer / (region_denom.unsqueeze(-1) + 1e-8)

class StationQuantileGCN(nn.Module):
    def __init__(self, in_dim, dropout, hidden=32, gru_hidden=32):
        super().__init__()
        self.station_conv = WindConvLayer(in_dim, hidden)
        self.drop = nn.Dropout(dropout)
        self.gru = nn.GRU(hidden, gru_hidden, batch_first=True)
        self.head = nn.Linear(gru_hidden, N_QUANTILES)

    def forward(self, x_window, edge_index, edge_weight_seq):
        B, W, N, Fin = x_window.shape
        ei_b = torch.cat([edge_index + i * N for i in range(B)], dim=1)
        num_nodes = B * N
        h_seq = []
        for w in range(W):
            xt = x_window[:, w].reshape(B * N, Fin)
            ew_b = edge_weight_seq[:, w].reshape(-1)
            h = torch.relu(self.station_conv(xt, ei_b, ew_b, num_nodes))
            h = self.drop(h)
            h_seq.append(h.reshape(B, N, -1))
        h_seq = torch.stack(h_seq, dim=1).permute(0, 2, 1, 3).reshape(B * N, W, -1)
        _, h_final = self.gru(h_seq)
        embed = self.drop(h_final.squeeze(0).reshape(B, N, -1))
        return monotonic_quantiles(self.head(embed))

class RegionFromTrajectoryGCN(nn.Module):
    def __init__(self, station_conv, region_membership_t_local, dropout, hidden=32, gru_hidden=32):
        super().__init__()
        self.station_conv = station_conv
        self.attn_pool = AttentionPool(hidden)
        self.region_conv = WindConvLayer(hidden, hidden)
        self.drop = nn.Dropout(dropout)
        self.region_gru = nn.GRU(hidden, gru_hidden, batch_first=True)
        self.region_head = nn.Linear(gru_hidden, HORIZON)
        self.rmem = region_membership_t_local

    def forward(self, x_window, station_edge_index, station_edge_weight_seq, region_edge_index, region_edge_weight_seq, n_reg):
        B, W, N, Fin = x_window.shape
        station_ei_b = torch.cat([station_edge_index + i * N for i in range(B)], dim=1)
        region_ei_b = torch.cat([region_edge_index + i * n_reg for i in range(B)], dim=1)
        num_station_nodes = B * N
        num_region_nodes = B * n_reg
        h_region_seq = []
        for w in range(W):
            xt = x_window[:, w].reshape(B * N, Fin)
            ew_station_b = station_edge_weight_seq[:, w].reshape(-1)
            h_station = torch.relu(self.station_conv(xt, station_ei_b, ew_station_b, num_station_nodes))
            h_station = self.drop(h_station).reshape(B, N, -1)
            h_region_pooled = self.attn_pool(h_station, self.rmem).reshape(B * n_reg, -1)
            ew_region_b = region_edge_weight_seq[:, w].reshape(-1)
            h_region = torch.relu(self.region_conv(h_region_pooled, region_ei_b, ew_region_b, num_region_nodes))
            h_region = self.drop(h_region)
            h_region_seq.append(h_region.reshape(B, n_reg, -1))
        h_region_seq = torch.stack(h_region_seq, dim=1).permute(0, 2, 1, 3).reshape(B * n_reg, W, -1)
        _, h_final = self.region_gru(h_region_seq)
        embed = self.drop(h_final.squeeze(0).reshape(B, n_reg, -1))
        return self.region_head(embed)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else ("mps" if torch.backends.mps.is_available() else "cpu"))
MICRO_BATCH, ACCUM_STEPS = 16, 4
MAX_EPOCHS_P1, MAX_EPOCHS_P2 = 2, 2
print(f"device={DEVICE}")

region_membership_t = region_membership_t.to(DEVICE)
edge_index = torch.tensor(edge_index_np, dtype=torch.long).to(DEVICE)
region_edge_index = torch.tensor(region_edge_index_np, dtype=torch.long).to(DEVICE)

def add_static_fn(x_time_batch, static_tensor_local):
    B, W, N, _ = x_time_batch.shape
    static_b = static_tensor_local.unsqueeze(0).unsqueeze(0).expand(B, W, N, n_static_feats)
    return torch.cat([x_time_batch, static_b], dim=-1)

def gather_seq(ew_by_hour_t, starts_subset):
    idx = starts_subset.unsqueeze(1) + torch.arange(WINDOW).unsqueeze(0)
    ew = ew_by_hour_t[idx]
    ew = ew.clone()
    ew[:, :WINDOW - GRAPH_RECENT_HOURS, :] = 0.0
    return ew

print(f"train hours (2016-2017): {TRAIN_MASK.sum()}  val hours (2018): {(split_id_per_hour==1).sum()}  test hours (2019): {(split_id_per_hour==2).sum()}")

wind_blows_toward = (wind_dir_arr + 180) % 360
wbt_src = wind_blows_toward[:, src_idx]
cos_align = np.maximum(np.cos(np.radians(wbt_src - bearing_edge[None, :])), 0.0)
speed_src = wind_speed_arr[:, src_idx]
wind_component = (cos_align * speed_src).astype(np.float32)
del wbt_src, cos_align, speed_src
gc.collect()
ref_speed = np.float32(np.nanmean(wind_speed_arr[TRAIN_MASK]))
component = np.where(close_edge_mask[None, :], ref_speed, wind_component)
station_wind_raw = np.nan_to_num(decay_edge[None, :] * component, nan=0.0).astype(np.float32)
del component
gc.collect()

r_wbt_src = region_wind_blows_toward[:, r_src_idx]
r_cos_align = np.maximum(np.cos(np.radians(r_wbt_src - region_bearing_edge[None, :])), 0.0)
r_speed_src = region_windspeed[:, r_src_idx]
region_wind_raw = np.nan_to_num(region_decay_edge[None, :] * r_cos_align * r_speed_src, nan=0.0).astype(np.float32)
del r_wbt_src, r_cos_align, r_speed_src
gc.collect()

train_nonzero = station_wind_raw[TRAIN_MASK][station_wind_raw[TRAIN_MASK] > 0]
wind_scale_s = train_nonzero.std()
station_wind_edge_weight = (station_wind_raw / wind_scale_s).astype(np.float32)
region_train_nonzero = region_wind_raw[TRAIN_MASK][region_wind_raw[TRAIN_MASK] > 0]
wind_scale_r = region_train_nonzero.std()
region_wind_edge_weight = (region_wind_raw / wind_scale_r).astype(np.float32)
del train_nonzero, region_train_nonzero, station_wind_raw, region_wind_raw
gc.collect()

t_mean = np.nanmean(time_arr_raw[TRAIN_MASK], axis=(0, 1), keepdims=True)
t_std = np.nanstd(time_arr_raw[TRAIN_MASK], axis=(0, 1), keepdims=True) + 1e-6
time_arr_std = np.nan_to_num((time_arr_raw - t_mean) / t_std, nan=0.0)
s_mean, s_std = static_arr.mean(axis=0, keepdims=True), static_arr.std(axis=0, keepdims=True) + 1e-6
static_tensor = torch.tensor((static_arr - s_mean) / s_std, dtype=torch.float32).to(DEVICE)

p_buckets = {0: ([], [], [], []), 1: ([], [], [], []), 2: ([], [], [], [])}
for t in range(0, n_time - WINDOW - HORIZON + 1):
    target_t = t + WINDOW + HORIZON - 1
    s_start, s_target = split_id_per_hour[t], split_id_per_hour[target_t]
    if s_start != s_target or s_start == -1:
        continue
    x_win = time_arr_std[t:t + WINDOW]
    traj = region_episode_label[t + WINDOW: t + WINDOW + HORIZON]
    Xl, yregl, ytrajl, sl = p_buckets[s_start]
    Xl.append(x_win)
    yregl.append(time_arr_std[target_t, :, pm25_col_idx])
    ytrajl.append(traj.T)
    sl.append(t)

X0, yreg0, ytraj0, starts0 = (np.stack(v) for v in p_buckets[0])
X1, yreg1, ytraj1, starts1 = (np.stack(v) for v in p_buckets[1])
X2, yreg2, ytraj2, starts2 = (np.stack(v) for v in p_buckets[2])
del p_buckets, time_arr_std
gc.collect()
print(f"windows: train={len(X0)} val={len(X1)} test={len(X2)}")

X0_t = torch.tensor(X0, dtype=torch.float32); del X0
X1_t = torch.tensor(X1, dtype=torch.float32); del X1
X2_t = torch.tensor(X2, dtype=torch.float32); del X2
gc.collect()

yreg0_t, yreg1_t = torch.tensor(yreg0, dtype=torch.float32), torch.tensor(yreg1, dtype=torch.float32)
ytraj0_t = torch.tensor(ytraj0, dtype=torch.float32)
ytraj1_t = torch.tensor(ytraj1, dtype=torch.float32)
starts0_t, starts1_t, starts2_t = (torch.tensor(a, dtype=torch.long) for a in (starts0, starts1, starts2))
del yreg0, yreg1
gc.collect()

POS_WEIGHT = min(float((ytraj0_t.numel() - ytraj0_t.sum()) / ytraj0_t.sum().clamp(min=1)), 50.0)
print(f"pos_weight (per-hour trajectory): {POS_WEIGHT:.2f}")

wind_ewt_s = torch.tensor(station_wind_edge_weight)
wind_ewt_r = torch.tensor(region_wind_edge_weight)

region_criterion = nn.BCEWithLogitsLoss(pos_weight=torch.tensor(POS_WEIGHT))

def run_p1_epoch(model, ewt, X, y, starts, optimizer, train):
    n = X.shape[0]
    idx = torch.randperm(n) if train else torch.arange(n)
    model.train(train)
    total_loss, total_n = 0.0, 0
    eff_batch = MICRO_BATCH * ACCUM_STEPS
    for start in range(0, n, eff_batch):
        if train: optimizer.zero_grad()
        batch_idx = idx[start:start + eff_batch]
        for ms in range(0, len(batch_idx), MICRO_BATCH):
            mb_idx = batch_idx[ms:ms + MICRO_BATCH]
            if len(mb_idx) == 0: continue
            xb = add_static_fn(X[mb_idx].to(DEVICE), static_tensor)
            yb = y[mb_idx].to(DEVICE)
            ew_seq = gather_seq(ewt, starts[mb_idx]).to(DEVICE)
            with torch.set_grad_enabled(train):
                pred = model(xb, edge_index, ew_seq)
                loss = pinball_loss(pred, yb, QUANTILES)
            if train: (loss * len(mb_idx) / len(batch_idx)).backward()
            total_loss += loss.item() * len(mb_idx); total_n += len(mb_idx)
        if train: optimizer.step()
    return total_loss / total_n

def run_p2_epoch(model, ewt_s, ewt_r, X, y_traj, starts, optimizer, train):
    n = X.shape[0]
    idx = torch.randperm(n) if train else torch.arange(n)
    model.train(train)
    total_loss, total_n = 0.0, 0
    eff_batch = MICRO_BATCH * ACCUM_STEPS
    for start in range(0, n, eff_batch):
        if train: optimizer.zero_grad()
        batch_idx = idx[start:start + eff_batch]
        for ms in range(0, len(batch_idx), MICRO_BATCH):
            mb_idx = batch_idx[ms:ms + MICRO_BATCH]
            if len(mb_idx) == 0: continue
            xb = add_static_fn(X[mb_idx].to(DEVICE), static_tensor)
            yb = y_traj[mb_idx].to(DEVICE)
            ew_s = gather_seq(ewt_s, starts[mb_idx]).to(DEVICE)
            ew_r = gather_seq(ewt_r, starts[mb_idx]).to(DEVICE)
            with torch.set_grad_enabled(train):
                logits = model(xb, edge_index, ew_s, region_edge_index, ew_r, n_regions)
                loss = region_criterion(logits, yb)
            if train: loss.backward()
            total_loss += loss.item() * len(mb_idx); total_n += len(mb_idx)
        if train: optimizer.step()
    return total_loss / max(total_n, 1)

def predict_traj_probs(model, ewt_s, ewt_r, X, starts, batch_size=64):
    model.eval()
    n = X.shape[0]
    out = np.zeros((n, n_regions, HORIZON), dtype=np.float32)
    with torch.no_grad():
        for start in range(0, n, batch_size):
            idx = torch.arange(start, min(start + batch_size, n))
            xb = add_static_fn(X[idx].to(DEVICE), static_tensor)
            ew_s = gather_seq(ewt_s, starts[idx]).to(DEVICE)
            ew_r = gather_seq(ewt_r, starts[idx]).to(DEVICE)
            logits = model(xb, edge_index, ew_s, region_edge_index, ew_r, n_regions)
            out[idx.numpy()] = torch.sigmoid(logits).cpu().numpy()
    return out

def contiguous_lead_hours(prob_traj, threshold):
    if prob_traj[0] < threshold:
        return 0
    o = 1
    while o < HORIZON and prob_traj[o] >= threshold:
        o += 1
    return o

# ================================================================
# TRAIN (single run, seed=0, established best hyperparameters)
# ================================================================
LR, DROPOUT, WEIGHT_DECAY = 0.003, 0.5, 0.0005
SEED = 0
torch.manual_seed(SEED); np.random.seed(SEED)

t0 = time.time()
p1 = StationQuantileGCN(n_feats, DROPOUT).to(DEVICE)
opt1 = torch.optim.Adam(p1.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
best_p1_val, best_p1_state = float("inf"), None
for epoch in range(1, MAX_EPOCHS_P1 + 1):
    run_p1_epoch(p1, wind_ewt_s, X0_t, yreg0_t, starts0_t, opt1, True)
    vl = run_p1_epoch(p1, wind_ewt_s, X1_t, yreg1_t, starts1_t, opt1, False)
    if vl < best_p1_val:
        best_p1_val, best_p1_state = vl, copy.deepcopy(p1.state_dict())
p1.load_state_dict(best_p1_state)
encoder_copy = copy.deepcopy(p1.station_conv)
p2 = RegionFromTrajectoryGCN(encoder_copy, region_membership_t, DROPOUT).to(DEVICE)
del p1; gc.collect()
if DEVICE.type == "mps": torch.mps.empty_cache()

opt2 = torch.optim.Adam(p2.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
y_val_agg = ytraj1_t.numpy().max(axis=-1).reshape(-1)
best_val_aucpr, best_state = -1.0, None
for epoch in range(1, MAX_EPOCHS_P2 + 1):
    run_p2_epoch(p2, wind_ewt_s, wind_ewt_r, X0_t, ytraj0_t, starts0_t, opt2, True)
    run_p2_epoch(p2, wind_ewt_s, wind_ewt_r, X1_t, ytraj1_t, starts1_t, opt2, False)
    val_probs = predict_traj_probs(p2, wind_ewt_s, wind_ewt_r, X1_t, starts1_t)
    va = average_precision_score(y_val_agg, val_probs.max(axis=-1).reshape(-1))
    if va > best_val_aucpr:
        best_val_aucpr, best_state = va, copy.deepcopy(p2.state_dict())
p2.load_state_dict(best_state)
p2.eval()
print(f"trained in {time.time()-t0:.0f}s  val_AUCPR={best_val_aucpr:.4f}")

# ================================================================
# STEP 1: SAVE EVERYTHING NEEDED TO RELOAD THIS MODEL LATER
# ================================================================
torch.save(p2.state_dict(), f"{SAVE_DIR}/phase2_model.pt")
torch.save(encoder_copy.state_dict(), f"{SAVE_DIR}/phase1_station_encoder.pt")

np.savez(f"{SAVE_DIR}/normalization.npz", t_mean=t_mean, t_std=t_std, s_mean=s_mean, s_std=s_std)
np.savez(f"{SAVE_DIR}/edge_weights.npz",
         station_wind_edge_weight=wind_ewt_s.numpy(), region_wind_edge_weight=wind_ewt_r.numpy(),
         edge_index=edge_index_np, region_edge_index=region_edge_index_np,
         station_region_idx=station_region_idx, region_membership=region_membership)

metadata = {
    "station_order": [str(s) for s in station_order], "region_names": region_names,
    "n_feats": n_feats, "n_static_feats": n_static_feats, "n_time_feats": n_time_feats,
    "hidden": 32, "gru_hidden": 32, "dropout": DROPOUT, "lr": LR, "weight_decay": WEIGHT_DECAY,
    "seed": SEED, "WINDOW": WINDOW, "HORIZON": HORIZON, "GRAPH_RECENT_HOURS": GRAPH_RECENT_HOURS,
    "EVENT_THRESHOLD": EVENT_THRESHOLD, "SUSTAIN_HOURS": SUSTAIN_HOURS, "QUANTILES": QUANTILES,
    "POS_WEIGHT": POS_WEIGHT, "val_aucpr": float(best_val_aucpr),
    "split": "train=2016-2017, val=2018, test=2019",
    "TIME_FEATS_FULL": TIME_FEATS_FULL, "STATIC_COLS": STATIC_COLS,
}
with open(f"{SAVE_DIR}/metadata.json", "w") as f:
    json.dump(metadata, f, indent=2)
print(f"saved model + normalization + edge weights + metadata to {SAVE_DIR}")

# ================================================================
# STEP 2: PER-REGION PERFORMANCE (precision/recall/MCC/lead-time, all 17 regions)
# ================================================================
val_probs = predict_traj_probs(p2, wind_ewt_s, wind_ewt_r, X1_t, starts1_t)
test_probs = predict_traj_probs(p2, wind_ewt_s, wind_ewt_r, X2_t, starts2_t)

y_val_agg = ytraj1_t.numpy().max(axis=-1).reshape(-1)
val_agg_score = val_probs.max(axis=-1).reshape(-1)

# global F1-optimal threshold, selected on val (flattened across all regions) -- same convention as headline result
fpr, tpr, thresholds = roc_curve(y_val_agg, val_agg_score)
f1_scores = []
for th in thresholds:
    pred = (val_agg_score >= th).astype(int)
    p = precision_score(y_val_agg, pred, zero_division=0)
    r = recall_score(y_val_agg, pred, zero_division=0)
    f1 = 2 * p * r / (p + r) if (p + r) > 0 else 0.0
    f1_scores.append(f1)
GLOBAL_THRESH = thresholds[np.argmax(f1_scores)]
print(f"\nglobal F1-optimal threshold (selected on val): {GLOBAL_THRESH:.4f}")

per_region_results = []
print(f"\n{'region':<10} {'AUC-PR':>8} {'AUC-ROC':>8} {'precision':>10} {'recall':>8} {'MCC':>7} {'n_pos':>6} {'mean_lead':>10} {'median_lead':>12}")
for r in range(n_regions):
    y_true_r = ytraj2[:, r].max(axis=-1)  # (n_windows,)
    score_r = test_probs[:, r].max(axis=-1)  # (n_windows,)
    n_pos = int(y_true_r.sum())
    if n_pos == 0:
        print(f"{region_names[r]:<10}  -- no positive test windows, skipping --")
        continue
    aucpr_r = average_precision_score(y_true_r, score_r)
    aucroc_r = roc_auc_score(y_true_r, score_r) if len(np.unique(y_true_r)) > 1 else float("nan")
    pred_r = (score_r >= GLOBAL_THRESH).astype(int)
    precision_r = precision_score(y_true_r, pred_r, zero_division=0)
    recall_r = recall_score(y_true_r, pred_r, zero_division=0)
    mcc_r = matthews_corrcoef(y_true_r, pred_r)

    lead_list = []
    for i in range(len(y_true_r)):
        if ytraj2[i, r].max() < 1:
            continue
        lead_list.append(contiguous_lead_hours(test_probs[i, r], GLOBAL_THRESH))
    lead_arr = np.array(lead_list)

    print(f"{region_names[r]:<10} {aucpr_r:>8.4f} {aucroc_r:>8.4f} {precision_r:>10.4f} {recall_r:>8.4f} {mcc_r:>7.4f} "
          f"{n_pos:>6} {lead_arr.mean():>10.2f} {np.median(lead_arr):>12.1f}")

    per_region_results.append({
        "region": region_names[r], "aucpr": float(aucpr_r), "aucroc": float(aucroc_r),
        "precision": float(precision_r), "recall": float(recall_r), "mcc": float(mcc_r),
        "n_positive_windows": n_pos, "mean_lead_hours": float(lead_arr.mean()),
        "median_lead_hours": float(np.median(lead_arr)),
    })

with open(f"{BASE}/per_region_performance_2019.json", "w") as f:
    json.dump({"global_threshold": float(GLOBAL_THRESH), "per_region": per_region_results}, f, indent=2)
print(f"\nsaved per-region breakdown to {BASE}/per_region_performance_2019.json")


device=mps
train hours (2016-2017): 17544  val hours (2018): 8760  test hours (2019): 8760
windows: train=17473 val=8689 test=8689
pos_weight (per-hour trajectory): 50.00
trained in 617s  val_AUCPR=0.5063
saved model + normalization + edge weights + metadata to /Users/drewbaldwin/PM2_5 Research/saved_wind_graph_2019

global F1-optimal threshold (selected on val): 0.8469

region       AUC-PR  AUC-ROC  precision   recall     MCC  n_pos  mean_lead  median_lead
Seoul        0.7717   0.9682     0.7868   0.5605  0.6387    744      14.66          0.0
Busan        0.4710   0.9545     0.6115   0.4103  0.4898    234       4.78          0.0
Daegu        0.7195   0.9703     0.7639   0.5457  0.6307    427       9.48          0.0
Incheon      0.7390   0.9718     0.6157   0.6453  0.6103    437      18.05         22.0
Gwangju      0.6769   0.9654     0.5663   0.6416  0.5880    293      14.54         16.0
Daejeon      0.8331   0.9851     0.7970   0.7521  0.7649    355      16.55         19.0
Ulsan     

In [2]:
# ================================================================
# PER-REGION THRESHOLD COMPARISON (using the saved 2019 model, no retraining)
# Compares: global threshold (one cutoff for all regions) vs.
#           per-region threshold (each region's own F1-optimal cutoff on val)
# ================================================================
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import os, gc, json
from sklearn.metrics import (average_precision_score, roc_auc_score,
                              precision_score, recall_score, matthews_corrcoef, roc_curve)

BASE = "/Users/drewbaldwin/PM2_5 Research"
SAVE_DIR = f"{BASE}/saved_wind_graph_2019"

with open(f"{SAVE_DIR}/metadata.json") as f:
    metadata = json.load(f)

df = pd.read_pickle(f"{BASE}/air_korea_final_imputed_with_blh.pkl")
station_order = sorted(df["Station_ID"].unique())
n_stations = len(station_order)

stations = df[["Station_ID", "lat", "lon"]].drop_duplicates("Station_ID").set_index("Station_ID").loc[station_order]
lats, lons = stations["lat"].to_numpy(), stations["lon"].to_numpy()

def haversine_km(lat1, lon1, lat2, lon2):
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
    dlat, dlon = lat2 - lat1, lon2 - lon1
    a = np.sin(dlat/2)**2 + np.cos(lat1)*np.cos(lat2)*np.sin(dlon/2)**2
    return 2 * 6371.0 * np.arcsin(np.sqrt(a))

def bearing_matrix(lat, lon):
    lat_r, lon_r = np.radians(lat), np.radians(lon)
    lat1, lat2 = lat_r[:, None], lat_r[None, :]
    dlon = lon_r[None, :] - lon_r[:, None]
    x = np.sin(dlon) * np.cos(lat2)
    y = np.cos(lat1) * np.sin(lat2) - np.sin(lat1) * np.cos(lat2) * np.cos(dlon)
    return (np.degrees(np.arctan2(x, y)) + 360) % 360

dist_km = haversine_km(lats[:, None], lons[:, None], lats[None, :], lons[None, :])
DIST_CUTOFF, RHO_KM = 250.0, 250.0
HYBRID_THRESHOLD_KM = 20.0
dist_edges = (dist_km <= DIST_CUTOFF) & (dist_km > 0)
bearing_from = bearing_matrix(lats, lons)

src_idx, dst_idx = np.nonzero(dist_edges)
edge_index_np = np.stack([src_idx, dst_idx])
decay_edge = np.exp(-dist_km[src_idx, dst_idx] / RHO_KM).astype(np.float32)
bearing_edge = bearing_from[src_idx, dst_idx].astype(np.float32)
close_edge_mask = (dist_km[src_idx, dst_idx] <= HYBRID_THRESHOLD_KM)

REGION_CENTROIDS = {
    "Seoul": (37.566, 126.978), "Busan": (35.180, 129.075), "Daegu": (35.872, 128.602),
    "Incheon": (37.483, 126.633), "Gwangju": (35.155, 126.916), "Daejeon": (36.350, 127.385),
    "Ulsan": (35.550, 129.317), "Sejong": (36.487, 127.282), "Gyeonggi": (37.500, 127.250),
    "Gangwon": (37.867, 127.733), "Chungbuk": (36.633, 127.483), "Chungnam": (36.500, 126.750),
    "Jeonbuk": (35.824, 127.148), "Jeonnam": (34.750, 127.000), "Gyeongbuk": (36.559, 128.729),
    "Gyeongnam": (35.271, 128.663), "Jeju": (33.513, 126.523),
}
region_names = list(REGION_CENTROIDS.keys())
n_regions = len(region_names)
region_lats = np.array([REGION_CENTROIDS[r][0] for r in region_names])
region_lons = np.array([REGION_CENTROIDS[r][1] for r in region_names])
dist_to_region = haversine_km(lats[:, None], lons[:, None], region_lats[None, :], region_lons[None, :])
station_region_idx = dist_to_region.argmin(axis=1)

region_membership = np.zeros((n_stations, n_regions), dtype=np.float32)
region_membership[np.arange(n_stations), station_region_idx] = 1.0
region_membership_t = torch.tensor(region_membership)

region_dist_km = haversine_km(region_lats[:, None], region_lons[:, None], region_lats[None, :], region_lons[None, :])
region_bearing = bearing_matrix(region_lats, region_lons)
r_src_idx, r_dst_idx = np.nonzero(~np.eye(n_regions, dtype=bool))
region_edge_index_np = np.stack([r_src_idx, r_dst_idx])
region_decay_edge = np.exp(-region_dist_km[r_src_idx, r_dst_idx] / RHO_KM).astype(np.float32)
region_bearing_edge = region_bearing[r_src_idx, r_dst_idx].astype(np.float32)

WINDOW, HORIZON = metadata["WINDOW"], metadata["HORIZON"]
GRAPH_RECENT_HOURS = metadata["GRAPH_RECENT_HOURS"]
EVENT_THRESHOLD = metadata["EVENT_THRESHOLD"]
SUSTAIN_HOURS = metadata["SUSTAIN_HOURS"]
QUANTILES = metadata["QUANTILES"]
N_QUANTILES = len(QUANTILES)
TIME_FEATS = ["SO2", "CO", "NO2", "O3", "PM10", "PM25"]
STATIC_COLS = metadata["STATIC_COLS"]

time_panels = {c: df.pivot(index="Datetime", columns="Station_ID", values=c)[station_order] for c in TIME_FEATS}
dt_index = time_panels["PM25"].index
n_time = len(dt_index)
years = dt_index.year.to_numpy()
doy = dt_index.dayofyear.to_numpy().astype(float)

split_id_per_hour = np.where((years == 2016) | (years == 2017), 0, np.where(years == 2018, 1, np.where(years == 2019, 2, -1)))
TRAIN_MASK = split_id_per_hour == 0

wind_dir_arr = df.pivot(index="Datetime", columns="Station_ID", values="winddirection_10m")[station_order].reindex(dt_index).to_numpy().astype(float)
wind_speed_arr = df.pivot(index="Datetime", columns="Station_ID", values="windspeed_10m")[station_order].reindex(dt_index).to_numpy().astype(float)
blh_arr = df.pivot(index="Datetime", columns="Station_ID", values="boundary_layer_height")[station_order].reindex(dt_index).to_numpy().astype(float)
pm25_raw_arr = time_panels["PM25"].to_numpy()

def regional_flat_mean(arr):
    out = np.zeros((n_time, n_regions), dtype=np.float32)
    for r in range(n_regions):
        cols = station_region_idx == r
        out[:, r] = np.nanmean(arr[:, cols], axis=1)
    return out

region_pm25 = regional_flat_mean(pm25_raw_arr)
wdir_sin_station = np.sin(np.radians(wind_dir_arr))
wdir_cos_station = np.cos(np.radians(wind_dir_arr))
region_windspeed = regional_flat_mean(wind_speed_arr)
region_wdir_sin = regional_flat_mean(wdir_sin_station)
region_wdir_cos = regional_flat_mean(wdir_cos_station)
region_wind_dir_deg = (np.degrees(np.arctan2(region_wdir_sin, region_wdir_cos)) + 360) % 360
region_wind_blows_toward = (region_wind_dir_deg + 180) % 360

season_sin_1d = np.sin(2 * np.pi * doy / 365.25)
season_cos_1d = np.cos(2 * np.pi * doy / 365.25)
season_sin = np.tile(season_sin_1d[:, None], (1, n_stations))
season_cos = np.tile(season_cos_1d[:, None], (1, n_stations))

TIME_FEATS_FULL = metadata["TIME_FEATS_FULL"]
n_time_feats, n_static_feats = metadata["n_time_feats"], metadata["n_static_feats"]
n_feats = metadata["n_feats"]
pm25_col_idx = TIME_FEATS_FULL.index("PM25")

time_arr_raw = np.stack([time_panels[c].to_numpy() for c in TIME_FEATS] +
                         [wind_speed_arr, wdir_sin_station, wdir_cos_station, blh_arr, season_sin, season_cos], axis=-1)
del time_panels, season_sin, season_cos, blh_arr, pm25_raw_arr
gc.collect()

static_df = df[["Station_ID"] + STATIC_COLS].drop_duplicates("Station_ID").set_index("Station_ID").loc[station_order]
static_arr = static_df[STATIC_COLS].to_numpy()
del df
gc.collect()

rev = region_pm25[::-1]
roll_min_rev = pd.DataFrame(rev).rolling(window=SUSTAIN_HOURS, min_periods=SUSTAIN_HOURS).min().to_numpy()
region_episode_label = (roll_min_rev[::-1] >= EVENT_THRESHOLD).astype(np.float32)
del rev, roll_min_rev
gc.collect()

def monotonic_quantiles(raw):
    first = raw[..., :1]
    deltas = F.softplus(raw[..., 1:])
    return torch.cat([first, first + torch.cumsum(deltas, dim=-1)], dim=-1)

class WindConvLayer(nn.Module):
    def __init__(self, in_dim, out_dim):
        super().__init__()
        self.lin_self = nn.Linear(in_dim, out_dim)
        self.lin_neigh = nn.Linear(in_dim, out_dim)
        self.lin_connectivity = nn.Linear(1, out_dim)

    def forward(self, x, edge_index, edge_weight, num_nodes):
        src, dst = edge_index[0], edge_index[1]
        messages = x[src] * edge_weight.unsqueeze(-1)
        agg_sum = x.new_zeros(num_nodes, x.size(-1))
        agg_sum.index_add_(0, dst, messages)
        weight_sum = x.new_zeros(num_nodes)
        weight_sum.index_add_(0, dst, edge_weight)
        agg_mean = agg_sum / (weight_sum.unsqueeze(-1) + 1e-8)
        connectivity = torch.log1p(weight_sum.clamp(min=0)).unsqueeze(-1)
        return self.lin_self(x) + self.lin_neigh(agg_mean) + self.lin_connectivity(connectivity)

class AttentionPool(nn.Module):
    def __init__(self, hidden):
        super().__init__()
        self.attn_score = nn.Linear(hidden, 1)

    def forward(self, h_station, region_membership_t_local):
        B, N, H = h_station.shape
        scores = self.attn_score(h_station).squeeze(-1)
        scores = scores - scores.max(dim=1, keepdim=True).values
        exp_scores = torch.exp(scores)
        weighted_exp = exp_scores.unsqueeze(-1) * region_membership_t_local.unsqueeze(0)
        region_denom = weighted_exp.sum(dim=1)
        region_numer = torch.einsum('bnr,bnh->brh', weighted_exp, h_station)
        return region_numer / (region_denom.unsqueeze(-1) + 1e-8)

class WindConvEncoder(nn.Module):
    """Matches the station_conv submodule structure saved from StationQuantileGCN."""
    def __init__(self, in_dim, hidden=32):
        super().__init__()
        self.station_conv = WindConvLayer(in_dim, hidden)

class RegionFromTrajectoryGCN(nn.Module):
    def __init__(self, station_conv, region_membership_t_local, dropout, hidden=32, gru_hidden=32):
        super().__init__()
        self.station_conv = station_conv
        self.attn_pool = AttentionPool(hidden)
        self.region_conv = WindConvLayer(hidden, hidden)
        self.drop = nn.Dropout(dropout)
        self.region_gru = nn.GRU(hidden, gru_hidden, batch_first=True)
        self.region_head = nn.Linear(gru_hidden, HORIZON)
        self.rmem = region_membership_t_local

    def forward(self, x_window, station_edge_index, station_edge_weight_seq, region_edge_index, region_edge_weight_seq, n_reg):
        B, W, N, Fin = x_window.shape
        station_ei_b = torch.cat([station_edge_index + i * N for i in range(B)], dim=1)
        region_ei_b = torch.cat([region_edge_index + i * n_reg for i in range(B)], dim=1)
        num_station_nodes = B * N
        num_region_nodes = B * n_reg
        h_region_seq = []
        for w in range(W):
            xt = x_window[:, w].reshape(B * N, Fin)
            ew_station_b = station_edge_weight_seq[:, w].reshape(-1)
            h_station = torch.relu(self.station_conv(xt, station_ei_b, ew_station_b, num_station_nodes))
            h_station = self.drop(h_station).reshape(B, N, -1)
            h_region_pooled = self.attn_pool(h_station, self.rmem).reshape(B * n_reg, -1)
            ew_region_b = region_edge_weight_seq[:, w].reshape(-1)
            h_region = torch.relu(self.region_conv(h_region_pooled, region_ei_b, ew_region_b, num_region_nodes))
            h_region = self.drop(h_region)
            h_region_seq.append(h_region.reshape(B, n_reg, -1))
        h_region_seq = torch.stack(h_region_seq, dim=1).permute(0, 2, 1, 3).reshape(B * n_reg, W, -1)
        _, h_final = self.region_gru(h_region_seq)
        embed = self.drop(h_final.squeeze(0).reshape(B, n_reg, -1))
        return self.region_head(embed)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else ("mps" if torch.backends.mps.is_available() else "cpu"))
print(f"device={DEVICE}")

region_membership_t = region_membership_t.to(DEVICE)
edge_index = torch.tensor(edge_index_np, dtype=torch.long).to(DEVICE)
region_edge_index = torch.tensor(region_edge_index_np, dtype=torch.long).to(DEVICE)

def add_static_fn(x_time_batch, static_tensor_local):
    B, W, N, _ = x_time_batch.shape
    static_b = static_tensor_local.unsqueeze(0).unsqueeze(0).expand(B, W, N, n_static_feats)
    return torch.cat([x_time_batch, static_b], dim=-1)

def gather_seq(ew_by_hour_t, starts_subset):
    idx = starts_subset.unsqueeze(1) + torch.arange(WINDOW).unsqueeze(0)
    ew = ew_by_hour_t[idx]
    ew = ew.clone()
    ew[:, :WINDOW - GRAPH_RECENT_HOURS, :] = 0.0
    return ew

# ---- rebuild wind edges (deterministic, matches what was saved) ----
wind_blows_toward = (wind_dir_arr + 180) % 360
wbt_src = wind_blows_toward[:, src_idx]
cos_align = np.maximum(np.cos(np.radians(wbt_src - bearing_edge[None, :])), 0.0)
speed_src = wind_speed_arr[:, src_idx]
wind_component = (cos_align * speed_src).astype(np.float32)
del wbt_src, cos_align, speed_src
gc.collect()
ref_speed = np.float32(np.nanmean(wind_speed_arr[TRAIN_MASK]))
component = np.where(close_edge_mask[None, :], ref_speed, wind_component)
station_wind_raw = np.nan_to_num(decay_edge[None, :] * component, nan=0.0).astype(np.float32)
del component
gc.collect()

r_wbt_src = region_wind_blows_toward[:, r_src_idx]
r_cos_align = np.maximum(np.cos(np.radians(r_wbt_src - region_bearing_edge[None, :])), 0.0)
r_speed_src = region_windspeed[:, r_src_idx]
region_wind_raw = np.nan_to_num(region_decay_edge[None, :] * r_cos_align * r_speed_src, nan=0.0).astype(np.float32)
del r_wbt_src, r_cos_align, r_speed_src
gc.collect()

train_nonzero = station_wind_raw[TRAIN_MASK][station_wind_raw[TRAIN_MASK] > 0]
station_wind_edge_weight = (station_wind_raw / train_nonzero.std()).astype(np.float32)
region_train_nonzero = region_wind_raw[TRAIN_MASK][region_wind_raw[TRAIN_MASK] > 0]
region_wind_edge_weight = (region_wind_raw / region_train_nonzero.std()).astype(np.float32)
del train_nonzero, region_train_nonzero, station_wind_raw, region_wind_raw
gc.collect()

wind_ewt_s = torch.tensor(station_wind_edge_weight)
wind_ewt_r = torch.tensor(region_wind_edge_weight)

# ---- normalization (must match training exactly) ----
t_mean = np.nanmean(time_arr_raw[TRAIN_MASK], axis=(0, 1), keepdims=True)
t_std = np.nanstd(time_arr_raw[TRAIN_MASK], axis=(0, 1), keepdims=True) + 1e-6
time_arr_std = np.nan_to_num((time_arr_raw - t_mean) / t_std, nan=0.0)
s_mean, s_std = static_arr.mean(axis=0, keepdims=True), static_arr.std(axis=0, keepdims=True) + 1e-6
static_tensor = torch.tensor((static_arr - s_mean) / s_std, dtype=torch.float32).to(DEVICE)

p_buckets = {0: ([], [], []), 1: ([], [], []), 2: ([], [], [])}
for t in range(0, n_time - WINDOW - HORIZON + 1):
    target_t = t + WINDOW + HORIZON - 1
    s_start, s_target = split_id_per_hour[t], split_id_per_hour[target_t]
    if s_start != s_target or s_start == -1:
        continue
    x_win = time_arr_std[t:t + WINDOW]
    traj = region_episode_label[t + WINDOW: t + WINDOW + HORIZON]
    Xl, ytrajl, sl = p_buckets[s_start]
    Xl.append(x_win)
    ytrajl.append(traj.T)
    sl.append(t)

X1, ytraj1, starts1 = (np.stack(v) for v in p_buckets[1])
X2, ytraj2, starts2 = (np.stack(v) for v in p_buckets[2])
del p_buckets, time_arr_std
gc.collect()
print(f"val windows={len(X1)}  test windows={len(X2)}")

X1_t = torch.tensor(X1, dtype=torch.float32)
X2_t = torch.tensor(X2, dtype=torch.float32)
starts1_t = torch.tensor(starts1, dtype=torch.long)
starts2_t = torch.tensor(starts2, dtype=torch.long)

# ---- load saved model ----
encoder = WindConvEncoder(n_feats).station_conv
encoder.load_state_dict(torch.load(f"{SAVE_DIR}/phase1_station_encoder.pt", map_location=DEVICE))
p2 = RegionFromTrajectoryGCN(encoder, region_membership_t, metadata["dropout"]).to(DEVICE)
p2.load_state_dict(torch.load(f"{SAVE_DIR}/phase2_model.pt", map_location=DEVICE))
p2.eval()
print("loaded saved model")

def predict_traj_probs(model, ewt_s, ewt_r, X, starts, batch_size=64):
    model.eval()
    n = X.shape[0]
    out = np.zeros((n, n_regions, HORIZON), dtype=np.float32)
    with torch.no_grad():
        for start in range(0, n, batch_size):
            idx = torch.arange(start, min(start + batch_size, n))
            xb = add_static_fn(X[idx].to(DEVICE), static_tensor)
            ew_s = gather_seq(ewt_s, starts[idx]).to(DEVICE)
            ew_r = gather_seq(ewt_r, starts[idx]).to(DEVICE)
            logits = model(xb, edge_index, ew_s, region_edge_index, ew_r, n_regions)
            out[idx.numpy()] = torch.sigmoid(logits).cpu().numpy()
    return out

def contiguous_lead_hours(prob_traj, threshold):
    if prob_traj[0] < threshold:
        return 0
    o = 1
    while o < HORIZON and prob_traj[o] >= threshold:
        o += 1
    return o

val_probs = predict_traj_probs(p2, wind_ewt_s, wind_ewt_r, X1_t, starts1_t)
test_probs = predict_traj_probs(p2, wind_ewt_s, wind_ewt_r, X2_t, starts2_t)

# ---- global threshold (reused from before) ----
y_val_agg = ytraj1.max(axis=-1).reshape(-1)
val_agg_score = val_probs.max(axis=-1).reshape(-1)
fpr, tpr, thresholds = roc_curve(y_val_agg, val_agg_score)
f1_scores = []
for th in thresholds:
    pred = (val_agg_score >= th).astype(int)
    p = precision_score(y_val_agg, pred, zero_division=0)
    r = recall_score(y_val_agg, pred, zero_division=0)
    f1 = 2 * p * r / (p + r) if (p + r) > 0 else 0.0
    f1_scores.append(f1)
GLOBAL_THRESH = thresholds[np.argmax(f1_scores)]
print(f"global threshold: {GLOBAL_THRESH:.4f}")

# ================================================================
# PER-REGION THRESHOLD COMPARISON: global vs. region-specific
# ================================================================
print(f"\n{'region':<10} {'n_val_pos':>10} {'region_thr':>11} {'--- GLOBAL ---':>0}")
comparison = []
for r in range(n_regions):
    y_val_r = ytraj1[:, r].max(axis=-1)
    score_val_r = val_probs[:, r].max(axis=-1)
    n_val_pos = int(y_val_r.sum())

    y_test_r = ytraj2[:, r].max(axis=-1)
    score_test_r = test_probs[:, r].max(axis=-1)
    n_test_pos = int(y_test_r.sum())
    if n_test_pos == 0:
        continue

    def eval_at(th):
        pred = (score_test_r >= th).astype(int)
        precision = precision_score(y_test_r, pred, zero_division=0)
        recall = recall_score(y_test_r, pred, zero_division=0)
        mcc = matthews_corrcoef(y_test_r, pred)
        lead_list = [contiguous_lead_hours(test_probs[i, r], th) for i in range(len(y_test_r)) if ytraj2[i, r].max() >= 1]
        lead_arr = np.array(lead_list)
        return precision, recall, mcc, lead_arr.mean(), np.median(lead_arr)

    # region-specific F1-optimal threshold, selected from THIS region's own val data
    if n_val_pos >= 3:  # need at least a few positives to select a meaningful threshold
        fpr_r, tpr_r, thr_r = roc_curve(y_val_r, score_val_r)
        f1_r = []
        for th in thr_r:
            pred = (score_val_r >= th).astype(int)
            p = precision_score(y_val_r, pred, zero_division=0)
            rc = recall_score(y_val_r, pred, zero_division=0)
            f1_r.append(2 * p * rc / (p + rc) if (p + rc) > 0 else 0.0)
        region_thresh = thr_r[np.argmax(f1_r)]
    else:
        region_thresh = GLOBAL_THRESH  # too few val positives to trust a region-specific threshold

    prec_g, rec_g, mcc_g, mean_lead_g, med_lead_g = eval_at(GLOBAL_THRESH)
    prec_r, rec_r, mcc_r, mean_lead_r, med_lead_r = eval_at(region_thresh)

    print(f"{region_names[r]:<10} n_val_pos={n_val_pos:<6} region_thr={region_thresh:.4f}  "
          f"GLOBAL: prec={prec_g:.3f} rec={rec_g:.3f} mcc={mcc_g:.3f} lead(mean/med)={mean_lead_g:.1f}/{med_lead_g:.1f}   "
          f"REGION: prec={prec_r:.3f} rec={rec_r:.3f} mcc={mcc_r:.3f} lead(mean/med)={mean_lead_r:.1f}/{med_lead_r:.1f}")

    comparison.append({
        "region": region_names[r], "n_val_pos": n_val_pos, "n_test_pos": n_test_pos,
        "region_threshold": float(region_thresh),
        "global": {"precision": float(prec_g), "recall": float(rec_g), "mcc": float(mcc_g),
                   "mean_lead": float(mean_lead_g), "median_lead": float(med_lead_g)},
        "region_specific": {"precision": float(prec_r), "recall": float(rec_r), "mcc": float(mcc_r),
                             "mean_lead": float(mean_lead_r), "median_lead": float(med_lead_r)},
    })

with open(f"{BASE}/per_region_threshold_comparison.json", "w") as f:
    json.dump({"global_threshold": float(GLOBAL_THRESH), "comparison": comparison}, f, indent=2)
print(f"\nsaved to {BASE}/per_region_threshold_comparison.json")


device=mps
val windows=8689  test windows=8689
loaded saved model
global threshold: 0.8469

region      n_val_pos  region_thr --- GLOBAL ---
Seoul      n_val_pos=499    region_thr=0.8499  GLOBAL: prec=0.787 rec=0.560 mcc=0.639 lead(mean/med)=14.7/0.0   REGION: prec=0.794 rec=0.560 mcc=0.642 lead(mean/med)=14.5/0.0
Busan      n_val_pos=133    region_thr=0.8679  GLOBAL: prec=0.611 rec=0.410 mcc=0.490 lead(mean/med)=4.8/0.0   REGION: prec=0.645 rec=0.389 mcc=0.491 lead(mean/med)=4.0/0.0
Daegu      n_val_pos=304    region_thr=0.6828  GLOBAL: prec=0.764 rec=0.546 mcc=0.631 lead(mean/med)=9.5/0.0   REGION: prec=0.694 rec=0.642 mcc=0.651 lead(mean/med)=17.1/19.0
Incheon    n_val_pos=438    region_thr=0.8229  GLOBAL: prec=0.616 rec=0.645 mcc=0.610 lead(mean/med)=18.0/22.0   REGION: prec=0.602 rec=0.666 mcc=0.613 lead(mean/med)=19.2/27.0
Gwangju    n_val_pos=368    region_thr=0.4975  GLOBAL: prec=0.566 rec=0.642 mcc=0.588 lead(mean/med)=14.5/16.0   REGION: prec=0.375 rec=0.867 mcc=0.550 lead(me

In [2]:
#obtain the advanced warning time of the graph method vs no graph method. 
# ================================================================
# ALTERNATIVE LEAD-TIME METRIC: "alarm-to-spike" gap, per real event
# Reconstructs a continuous hourly alert timeline (using each window's
# nearest-hour/offset-1 prediction, exactly as a real deployed system
# would query the model once per hour), then for each real onset event,
# finds how many CONTINUOUS hours before it the alarm was already sounding.
# Uses the saved 2019 headline model -- inference only, no retraining.
# ================================================================
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import os, gc, json
from sklearn.metrics import roc_curve, precision_score, recall_score, average_precision_score

BASE = "/Users/drewbaldwin/PM2_5 Research"
SAVE_DIR = f"{BASE}/saved_wind_graph_2019"

with open(f"{SAVE_DIR}/metadata.json") as f:
    metadata = json.load(f)

df = pd.read_pickle(f"{BASE}/air_korea_final_imputed_with_blh.pkl")
station_order = sorted(df["Station_ID"].unique())
n_stations = len(station_order)

stations = df[["Station_ID", "lat", "lon"]].drop_duplicates("Station_ID").set_index("Station_ID").loc[station_order]
lats, lons = stations["lat"].to_numpy(), stations["lon"].to_numpy()

def haversine_km(lat1, lon1, lat2, lon2):
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
    dlat, dlon = lat2 - lat1, lon2 - lon1
    a = np.sin(dlat/2)**2 + np.cos(lat1)*np.cos(lat2)*np.sin(dlon/2)**2
    return 2 * 6371.0 * np.arcsin(np.sqrt(a))

def bearing_matrix(lat, lon):
    lat_r, lon_r = np.radians(lat), np.radians(lon)
    lat1, lat2 = lat_r[:, None], lat_r[None, :]
    dlon = lon_r[None, :] - lon_r[:, None]
    x = np.sin(dlon) * np.cos(lat2)
    y = np.cos(lat1) * np.sin(lat2) - np.sin(lat1) * np.cos(lat2) * np.cos(dlon)
    return (np.degrees(np.arctan2(x, y)) + 360) % 360

dist_km = haversine_km(lats[:, None], lons[:, None], lats[None, :], lons[None, :])
DIST_CUTOFF, RHO_KM = 250.0, 250.0
HYBRID_THRESHOLD_KM = 20.0
dist_edges = (dist_km <= DIST_CUTOFF) & (dist_km > 0)
bearing_from = bearing_matrix(lats, lons)

src_idx, dst_idx = np.nonzero(dist_edges)
edge_index_np = np.stack([src_idx, dst_idx])
decay_edge = np.exp(-dist_km[src_idx, dst_idx] / RHO_KM).astype(np.float32)
bearing_edge = bearing_from[src_idx, dst_idx].astype(np.float32)
close_edge_mask = (dist_km[src_idx, dst_idx] <= HYBRID_THRESHOLD_KM)

REGION_CENTROIDS = {
    "Seoul": (37.566, 126.978), "Busan": (35.180, 129.075), "Daegu": (35.872, 128.602),
    "Incheon": (37.483, 126.633), "Gwangju": (35.155, 126.916), "Daejeon": (36.350, 127.385),
    "Ulsan": (35.550, 129.317), "Sejong": (36.487, 127.282), "Gyeonggi": (37.500, 127.250),
    "Gangwon": (37.867, 127.733), "Chungbuk": (36.633, 127.483), "Chungnam": (36.500, 126.750),
    "Jeonbuk": (35.824, 127.148), "Jeonnam": (34.750, 127.000), "Gyeongbuk": (36.559, 128.729),
    "Gyeongnam": (35.271, 128.663), "Jeju": (33.513, 126.523),
}
region_names = list(REGION_CENTROIDS.keys())
n_regions = len(region_names)
region_lats = np.array([REGION_CENTROIDS[r][0] for r in region_names])
region_lons = np.array([REGION_CENTROIDS[r][1] for r in region_names])
dist_to_region = haversine_km(lats[:, None], lons[:, None], region_lats[None, :], region_lons[None, :])
station_region_idx = dist_to_region.argmin(axis=1)

region_membership = np.zeros((n_stations, n_regions), dtype=np.float32)
region_membership[np.arange(n_stations), station_region_idx] = 1.0
region_membership_t = torch.tensor(region_membership)

region_dist_km = haversine_km(region_lats[:, None], region_lons[:, None], region_lats[None, :], region_lons[None, :])
region_bearing = bearing_matrix(region_lats, region_lons)
r_src_idx, r_dst_idx = np.nonzero(~np.eye(n_regions, dtype=bool))
region_edge_index_np = np.stack([r_src_idx, r_dst_idx])
region_decay_edge = np.exp(-region_dist_km[r_src_idx, r_dst_idx] / RHO_KM).astype(np.float32)
region_bearing_edge = region_bearing[r_src_idx, r_dst_idx].astype(np.float32)

WINDOW, HORIZON = metadata["WINDOW"], metadata["HORIZON"]
GRAPH_RECENT_HOURS = metadata["GRAPH_RECENT_HOURS"]
EVENT_THRESHOLD = metadata["EVENT_THRESHOLD"]
SUSTAIN_HOURS = metadata["SUSTAIN_HOURS"]
QUANTILES = metadata["QUANTILES"]
TIME_FEATS = ["SO2", "CO", "NO2", "O3", "PM10", "PM25"]
STATIC_COLS = metadata["STATIC_COLS"]

time_panels = {c: df.pivot(index="Datetime", columns="Station_ID", values=c)[station_order] for c in TIME_FEATS}
dt_index = time_panels["PM25"].index
n_time = len(dt_index)
years = dt_index.year.to_numpy()
doy = dt_index.dayofyear.to_numpy().astype(float)

split_id_per_hour = np.where((years == 2016) | (years == 2017), 0, np.where(years == 2018, 1, np.where(years == 2019, 2, -1)))
TRAIN_MASK = split_id_per_hour == 0

wind_dir_arr = df.pivot(index="Datetime", columns="Station_ID", values="winddirection_10m")[station_order].reindex(dt_index).to_numpy().astype(float)
wind_speed_arr = df.pivot(index="Datetime", columns="Station_ID", values="windspeed_10m")[station_order].reindex(dt_index).to_numpy().astype(float)
blh_arr = df.pivot(index="Datetime", columns="Station_ID", values="boundary_layer_height")[station_order].reindex(dt_index).to_numpy().astype(float)
pm25_raw_arr = time_panels["PM25"].to_numpy()

def regional_flat_mean(arr):
    out = np.zeros((n_time, n_regions), dtype=np.float32)
    for r in range(n_regions):
        cols = station_region_idx == r
        out[:, r] = np.nanmean(arr[:, cols], axis=1)
    return out

region_pm25 = regional_flat_mean(pm25_raw_arr)
wdir_sin_station = np.sin(np.radians(wind_dir_arr))
wdir_cos_station = np.cos(np.radians(wind_dir_arr))
region_windspeed = regional_flat_mean(wind_speed_arr)
region_wdir_sin = regional_flat_mean(wdir_sin_station)
region_wdir_cos = regional_flat_mean(wdir_cos_station)
region_wind_dir_deg = (np.degrees(np.arctan2(region_wdir_sin, region_wdir_cos)) + 360) % 360
region_wind_blows_toward = (region_wind_dir_deg + 180) % 360

season_sin_1d = np.sin(2 * np.pi * doy / 365.25)
season_cos_1d = np.cos(2 * np.pi * doy / 365.25)
season_sin = np.tile(season_sin_1d[:, None], (1, n_stations))
season_cos = np.tile(season_cos_1d[:, None], (1, n_stations))

TIME_FEATS_FULL = metadata["TIME_FEATS_FULL"]
n_time_feats, n_static_feats = metadata["n_time_feats"], metadata["n_static_feats"]
n_feats = metadata["n_feats"]
pm25_col_idx = TIME_FEATS_FULL.index("PM25")

time_arr_raw = np.stack([time_panels[c].to_numpy() for c in TIME_FEATS] +
                         [wind_speed_arr, wdir_sin_station, wdir_cos_station, blh_arr, season_sin, season_cos], axis=-1)
del time_panels, season_sin, season_cos, blh_arr, pm25_raw_arr
gc.collect()

static_df = df[["Station_ID"] + STATIC_COLS].drop_duplicates("Station_ID").set_index("Station_ID").loc[station_order]
static_arr = static_df[STATIC_COLS].to_numpy()
del df
gc.collect()

rev = region_pm25[::-1]
roll_min_rev = pd.DataFrame(rev).rolling(window=SUSTAIN_HOURS, min_periods=SUSTAIN_HOURS).min().to_numpy()
region_episode_label = (roll_min_rev[::-1] >= EVENT_THRESHOLD).astype(np.float32)
del rev, roll_min_rev
gc.collect()

def monotonic_quantiles(raw):
    first = raw[..., :1]
    deltas = F.softplus(raw[..., 1:])
    return torch.cat([first, first + torch.cumsum(deltas, dim=-1)], dim=-1)

class WindConvLayer(nn.Module):
    def __init__(self, in_dim, out_dim):
        super().__init__()
        self.lin_self = nn.Linear(in_dim, out_dim)
        self.lin_neigh = nn.Linear(in_dim, out_dim)
        self.lin_connectivity = nn.Linear(1, out_dim)

    def forward(self, x, edge_index, edge_weight, num_nodes):
        src, dst = edge_index[0], edge_index[1]
        messages = x[src] * edge_weight.unsqueeze(-1)
        agg_sum = x.new_zeros(num_nodes, x.size(-1))
        agg_sum.index_add_(0, dst, messages)
        weight_sum = x.new_zeros(num_nodes)
        weight_sum.index_add_(0, dst, edge_weight)
        agg_mean = agg_sum / (weight_sum.unsqueeze(-1) + 1e-8)
        connectivity = torch.log1p(weight_sum.clamp(min=0)).unsqueeze(-1)
        return self.lin_self(x) + self.lin_neigh(agg_mean) + self.lin_connectivity(connectivity)

class AttentionPool(nn.Module):
    def __init__(self, hidden):
        super().__init__()
        self.attn_score = nn.Linear(hidden, 1)

    def forward(self, h_station, region_membership_t_local):
        B, N, H = h_station.shape
        scores = self.attn_score(h_station).squeeze(-1)
        scores = scores - scores.max(dim=1, keepdim=True).values
        exp_scores = torch.exp(scores)
        weighted_exp = exp_scores.unsqueeze(-1) * region_membership_t_local.unsqueeze(0)
        region_denom = weighted_exp.sum(dim=1)
        region_numer = torch.einsum('bnr,bnh->brh', weighted_exp, h_station)
        return region_numer / (region_denom.unsqueeze(-1) + 1e-8)

class WindConvEncoder(nn.Module):
    def __init__(self, in_dim, hidden=32):
        super().__init__()
        self.station_conv = WindConvLayer(in_dim, hidden)

class RegionFromTrajectoryGCN(nn.Module):
    def __init__(self, station_conv, region_membership_t_local, dropout, hidden=32, gru_hidden=32):
        super().__init__()
        self.station_conv = station_conv
        self.attn_pool = AttentionPool(hidden)
        self.region_conv = WindConvLayer(hidden, hidden)
        self.drop = nn.Dropout(dropout)
        self.region_gru = nn.GRU(hidden, gru_hidden, batch_first=True)
        self.region_head = nn.Linear(gru_hidden, HORIZON)
        self.rmem = region_membership_t_local

    def forward(self, x_window, station_edge_index, station_edge_weight_seq, region_edge_index, region_edge_weight_seq, n_reg):
        B, W, N, Fin = x_window.shape
        station_ei_b = torch.cat([station_edge_index + i * N for i in range(B)], dim=1)
        region_ei_b = torch.cat([region_edge_index + i * n_reg for i in range(B)], dim=1)
        num_station_nodes = B * N
        num_region_nodes = B * n_reg
        h_region_seq = []
        for w in range(W):
            xt = x_window[:, w].reshape(B * N, Fin)
            ew_station_b = station_edge_weight_seq[:, w].reshape(-1)
            h_station = torch.relu(self.station_conv(xt, station_ei_b, ew_station_b, num_station_nodes))
            h_station = self.drop(h_station).reshape(B, N, -1)
            h_region_pooled = self.attn_pool(h_station, self.rmem).reshape(B * n_reg, -1)
            ew_region_b = region_edge_weight_seq[:, w].reshape(-1)
            h_region = torch.relu(self.region_conv(h_region_pooled, region_ei_b, ew_region_b, num_region_nodes))
            h_region = self.drop(h_region)
            h_region_seq.append(h_region.reshape(B, n_reg, -1))
        h_region_seq = torch.stack(h_region_seq, dim=1).permute(0, 2, 1, 3).reshape(B * n_reg, W, -1)
        _, h_final = self.region_gru(h_region_seq)
        embed = self.drop(h_final.squeeze(0).reshape(B, n_reg, -1))
        return self.region_head(embed)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else ("mps" if torch.backends.mps.is_available() else "cpu"))
print(f"device={DEVICE}")

region_membership_t = region_membership_t.to(DEVICE)
edge_index = torch.tensor(edge_index_np, dtype=torch.long).to(DEVICE)
region_edge_index = torch.tensor(region_edge_index_np, dtype=torch.long).to(DEVICE)

def add_static_fn(x_time_batch, static_tensor_local):
    B, W, N, _ = x_time_batch.shape
    static_b = static_tensor_local.unsqueeze(0).unsqueeze(0).expand(B, W, N, n_static_feats)
    return torch.cat([x_time_batch, static_b], dim=-1)

def gather_seq(ew_by_hour_t, starts_subset):
    idx = starts_subset.unsqueeze(1) + torch.arange(WINDOW).unsqueeze(0)
    ew = ew_by_hour_t[idx]
    ew = ew.clone()
    ew[:, :WINDOW - GRAPH_RECENT_HOURS, :] = 0.0
    return ew

wind_blows_toward = (wind_dir_arr + 180) % 360
wbt_src = wind_blows_toward[:, src_idx]
cos_align = np.maximum(np.cos(np.radians(wbt_src - bearing_edge[None, :])), 0.0)
speed_src = wind_speed_arr[:, src_idx]
wind_component = (cos_align * speed_src).astype(np.float32)
del wbt_src, cos_align, speed_src
gc.collect()
ref_speed = np.float32(np.nanmean(wind_speed_arr[TRAIN_MASK]))
component = np.where(close_edge_mask[None, :], ref_speed, wind_component)
station_wind_raw = np.nan_to_num(decay_edge[None, :] * component, nan=0.0).astype(np.float32)
del component
gc.collect()

r_wbt_src = region_wind_blows_toward[:, r_src_idx]
r_cos_align = np.maximum(np.cos(np.radians(r_wbt_src - region_bearing_edge[None, :])), 0.0)
r_speed_src = region_windspeed[:, r_src_idx]
region_wind_raw = np.nan_to_num(region_decay_edge[None, :] * r_cos_align * r_speed_src, nan=0.0).astype(np.float32)
del r_wbt_src, r_cos_align, r_speed_src
gc.collect()

train_nonzero = station_wind_raw[TRAIN_MASK][station_wind_raw[TRAIN_MASK] > 0]
station_wind_edge_weight = (station_wind_raw / train_nonzero.std()).astype(np.float32)
region_train_nonzero = region_wind_raw[TRAIN_MASK][region_wind_raw[TRAIN_MASK] > 0]
region_wind_edge_weight = (region_wind_raw / region_train_nonzero.std()).astype(np.float32)
del train_nonzero, region_train_nonzero, station_wind_raw, region_wind_raw
gc.collect()

t_mean = np.nanmean(time_arr_raw[TRAIN_MASK], axis=(0, 1), keepdims=True)
t_std = np.nanstd(time_arr_raw[TRAIN_MASK], axis=(0, 1), keepdims=True) + 1e-6
time_arr_std = np.nan_to_num((time_arr_raw - t_mean) / t_std, nan=0.0)
s_mean, s_std = static_arr.mean(axis=0, keepdims=True), static_arr.std(axis=0, keepdims=True) + 1e-6
static_tensor = torch.tensor((static_arr - s_mean) / s_std, dtype=torch.float32).to(DEVICE)

p_buckets = {1: ([], [], []), 2: ([], [], [])}
for t in range(0, n_time - WINDOW - HORIZON + 1):
    target_t = t + WINDOW + HORIZON - 1
    s_start, s_target = split_id_per_hour[t], split_id_per_hour[target_t]
    if s_start != s_target or s_start not in (1, 2):
        continue
    x_win = time_arr_std[t:t + WINDOW]
    traj = region_episode_label[t + WINDOW: t + WINDOW + HORIZON]
    Xl, ytrajl, sl = p_buckets[s_start]
    Xl.append(x_win)
    ytrajl.append(traj.T)
    sl.append(t)

X1, ytraj1, starts1 = (np.stack(v) for v in p_buckets[1])
X2, ytraj2, starts2 = (np.stack(v) for v in p_buckets[2])
del p_buckets, time_arr_std
gc.collect()
print(f"val windows={len(X1)}  test windows={len(X2)}")

X1_t = torch.tensor(X1, dtype=torch.float32)
X2_t = torch.tensor(X2, dtype=torch.float32)
starts1_t = torch.tensor(starts1, dtype=torch.long)
starts2_t = torch.tensor(starts2, dtype=torch.long)

wind_ewt_s = torch.tensor(station_wind_edge_weight)
wind_ewt_r = torch.tensor(region_wind_edge_weight)

encoder = WindConvEncoder(n_feats).station_conv
encoder.load_state_dict(torch.load(f"{SAVE_DIR}/phase1_station_encoder.pt", map_location=DEVICE))
p2 = RegionFromTrajectoryGCN(encoder, region_membership_t, metadata["dropout"]).to(DEVICE)
p2.load_state_dict(torch.load(f"{SAVE_DIR}/phase2_model.pt", map_location=DEVICE))
p2.eval()
print("loaded saved model")

def predict_traj_probs(model, ewt_s, ewt_r, X, starts, batch_size=64):
    model.eval()
    n = X.shape[0]
    out = np.zeros((n, n_regions, HORIZON), dtype=np.float32)
    with torch.no_grad():
        for start in range(0, n, batch_size):
            idx = torch.arange(start, min(start + batch_size, n))
            xb = add_static_fn(X[idx].to(DEVICE), static_tensor)
            ew_s = gather_seq(ewt_s, starts[idx]).to(DEVICE)
            ew_r = gather_seq(ewt_r, starts[idx]).to(DEVICE)
            logits = model(xb, edge_index, ew_s, region_edge_index, ew_r, n_regions)
            out[idx.numpy()] = torch.sigmoid(logits).cpu().numpy()
    return out

val_probs = predict_traj_probs(p2, wind_ewt_s, wind_ewt_r, X1_t, starts1_t)
test_probs = predict_traj_probs(p2, wind_ewt_s, wind_ewt_r, X2_t, starts2_t)

y_val_agg = ytraj1.max(axis=-1).reshape(-1)
val_agg_score = val_probs.max(axis=-1).reshape(-1)
fpr, tpr, thresholds = roc_curve(y_val_agg, val_agg_score)
f1_scores = []
for th in thresholds:
    pred = (val_agg_score >= th).astype(int)
    p = precision_score(y_val_agg, pred, zero_division=0)
    r = recall_score(y_val_agg, pred, zero_division=0)
    f1_scores.append(2 * p * r / (p + r) if (p + r) > 0 else 0.0)
GLOBAL_THRESH = thresholds[np.argmax(f1_scores)]
print(f"threshold: {GLOBAL_THRESH:.4f}")

# ================================================================
# Build the continuous "nearest-hour" alert timeline: for each absolute
# test-period hour h, use the offset-1 (very-next-hour) prediction from
# the window queried exactly 1 hour before h -- this is how a real
# deployed system would operate (re-run every hour, act on the nearest forecast).
# ================================================================
test_start_offset = starts2.min()  # first window's t
n_test_hours = starts2.max() - test_start_offset + WINDOW + 2  # enough room for all offset-1 predictions
alert_timeline = np.full((n_test_hours, n_regions), np.nan, dtype=np.float32)  # prob, indexed by (absolute_hour - test_start_offset)
for i, t in enumerate(starts2):
    predicted_hour = t + WINDOW + 1 - test_start_offset  # the hour this window's offset-1 prediction is FOR
    if 0 <= predicted_hour < n_test_hours:
        alert_timeline[predicted_hour] = test_probs[i, :, 0]  # offset-1 (index 0) prediction, all regions

alert_flag = (alert_timeline >= GLOBAL_THRESH)  # (n_test_hours, n_regions), NaN-safe since NaN>=x is False

# ================================================================
# Per real-event "alarm-to-spike" lead time
# ================================================================
print(f"\n{'region':<10} {'n_events':>9} {'mean_alarm_lead':>16} {'median_alarm_lead':>18} {'pct_caught_at_all':>18}")
per_region_alarm_results = []
for r in range(n_regions):
    label_r = region_episode_label[test_start_offset: test_start_offset + n_test_hours, r]
    onsets = [h for h in range(1, n_test_hours) if label_r[h] == 1 and label_r[h-1] == 0]
    if len(onsets) == 0:
        continue
    alarm_leads = []
    for onset_h in onsets:
        # walk backward from onset_h, counting continuous alert hours
        lead = 0
        h = onset_h - 1
        while h >= 0 and alert_flag[h, r]:
            lead += 1
            h -= 1
        alarm_leads.append(lead)
    alarm_leads = np.array(alarm_leads)
    pct_caught = 100 * (alarm_leads > 0).mean()
    print(f"{region_names[r]:<10} {len(onsets):>9} {alarm_leads.mean():>16.2f} {np.median(alarm_leads):>18.1f} {pct_caught:>17.1f}%")
    per_region_alarm_results.append({
        "region": region_names[r], "n_events": len(onsets),
        "mean_alarm_lead_hours": float(alarm_leads.mean()), "median_alarm_lead_hours": float(np.median(alarm_leads)),
        "pct_caught_at_all": float(pct_caught), "all_leads": alarm_leads.tolist(),
    })

with open(f"{BASE}/alarm_to_spike_lead_time.json", "w") as f:
    json.dump({"threshold": float(GLOBAL_THRESH), "per_region": per_region_alarm_results}, f, indent=2)
print(f"\nsaved to {BASE}/alarm_to_spike_lead_time.json")



device=mps
val windows=8689  test windows=8689
loaded saved model
threshold: 0.8469

region      n_events  mean_alarm_lead  median_alarm_lead  pct_caught_at_all
Seoul             21            14.95                4.0              61.9%
Busan              6             7.83                2.0              50.0%
Daegu             13            13.85                7.0              76.9%
Incheon           10            12.10                6.0              70.0%
Gwangju            9            28.44               16.0              66.7%
Daejeon           10            16.00                6.5              70.0%
Ulsan              8             4.50                2.5              62.5%
Sejong            26            18.58               10.0              76.9%
Gyeonggi          20            10.65                5.0              65.0%
Gangwon           12            19.58               22.5              91.7%
Chungbuk          23            16.91               10.0              87.0%
Chu

In [ ]:
#get the final numbers for publication (auc, auc pr, recall etc.)

In [ ]:
#make plots, do exploratory analysis etc. 



In [5]:
# ================================================================
# INTERACTIVE FOLIUM MAP (v3): back to CartoDB positron (clean background
# so the wind-graph lines are actually visible), keeping the per-region
# catch-rate/lead-time marker changes from v2.
# ================================================================
import folium
from folium.plugins import AntPath
import pandas as pd
import numpy as np

BASE = "/Users/drewbaldwin/PM2_5 Research"

REGION_CENTROIDS = {
    "Seoul": (37.566, 126.978), "Busan": (35.180, 129.075), "Daegu": (35.872, 128.602),
    "Incheon": (37.483, 126.633), "Gwangju": (35.155, 126.916), "Daejeon": (36.350, 127.385),
    "Ulsan": (35.550, 129.317), "Sejong": (36.487, 127.282), "Gyeonggi": (37.500, 127.250),
    "Gangwon": (37.867, 127.733), "Chungbuk": (36.633, 127.483), "Chungnam": (36.500, 126.750),
    "Jeonbuk": (35.824, 127.148), "Jeonnam": (34.750, 127.000), "Gyeongbuk": (36.559, 128.729),
    "Gyeongnam": (35.271, 128.663), "Jeju": (33.513, 126.523),
}

# per-region "alarm-to-spike" results: % of real events caught with any continuous
# advance warning, and the average lead time (hours) when caught
PER_REGION_ALARM = {
    "Seoul":     {"n_events": 21, "mean_lead": 14.95, "median_lead": 4.0,  "pct_caught": 61.9},
    "Busan":     {"n_events": 6,  "mean_lead": 7.83,  "median_lead": 2.0,  "pct_caught": 50.0},
    "Daegu":     {"n_events": 13, "mean_lead": 13.85, "median_lead": 7.0,  "pct_caught": 76.9},
    "Incheon":   {"n_events": 10, "mean_lead": 12.10, "median_lead": 6.0,  "pct_caught": 70.0},
    "Gwangju":   {"n_events": 9,  "mean_lead": 28.44, "median_lead": 16.0, "pct_caught": 66.7},
    "Daejeon":   {"n_events": 10, "mean_lead": 16.00, "median_lead": 6.5,  "pct_caught": 70.0},
    "Ulsan":     {"n_events": 8,  "mean_lead": 4.50,  "median_lead": 2.5,  "pct_caught": 62.5},
    "Sejong":    {"n_events": 26, "mean_lead": 18.58, "median_lead": 10.0, "pct_caught": 76.9},
    "Gyeonggi":  {"n_events": 20, "mean_lead": 10.65, "median_lead": 5.0,  "pct_caught": 65.0},
    "Gangwon":   {"n_events": 12, "mean_lead": 19.58, "median_lead": 22.5, "pct_caught": 91.7},
    "Chungbuk":  {"n_events": 23, "mean_lead": 16.91, "median_lead": 10.0, "pct_caught": 87.0},
    "Chungnam":  {"n_events": 18, "mean_lead": 8.61,  "median_lead": 1.5,  "pct_caught": 66.7},
    "Jeonbuk":   {"n_events": 23, "mean_lead": 25.61, "median_lead": 16.0, "pct_caught": 87.0},
    "Jeonnam":   {"n_events": 3,  "mean_lead": 6.00,  "median_lead": 0.0,  "pct_caught": 33.3},
    "Gyeongbuk": {"n_events": 9,  "mean_lead": 5.89,  "median_lead": 0.0,  "pct_caught": 44.4},
    "Gyeongnam": {"n_events": 3,  "mean_lead": 2.33,  "median_lead": 0.0,  "pct_caught": 33.3},
    "Jeju":      {"n_events": 6,  "mean_lead": 3.17,  "median_lead": 0.0,  "pct_caught": 16.7},
}

ABLATION_RESULTS = [
    {"source": "Daegu", "target": "Busan", "verdict": "validated", "mean_delta": 0.0177,
     "note": "Passes all 3 gates: seed-consistent, survives wind-stripped control, climatology confirmed (p=0.0061)."},
    {"source": "Incheon", "target": "Seoul", "verdict": "validated", "mean_delta": 0.0165,
     "note": "Passes all 3 gates: seed-consistent, survives control, climatology confirmed (p=0.0049)."},
    {"source": "Jeonnam", "target": "Jeju", "verdict": "unconfirmed", "mean_delta": 0.0410,
     "note": "Wind-specific (survives control) but climatology test does not confirm directional transport -- likely correlated regional buildup."},
    {"source": "Incheon", "target": "Jeonbuk", "verdict": "unconfirmed", "mean_delta": 0.0292,
     "note": "Wind-specific but climatology unconfirmed."},
    {"source": "Incheon", "target": "Chungnam", "verdict": "unconfirmed", "mean_delta": 0.0269,
     "note": "Wind-specific but climatology unconfirmed."},
    {"source": "Incheon", "target": "Sejong", "verdict": "unconfirmed", "mean_delta": 0.0206,
     "note": "Wind-specific but climatology unconfirmed."},
    {"source": "Gwangju", "target": "Jeonnam", "verdict": "unconfirmed", "mean_delta": 0.0203,
     "note": "Wind-specific but climatology unconfirmed."},
    {"source": "Chungbuk", "target": "Ulsan", "verdict": "structural", "mean_delta": 0.0162,
     "note": "Does NOT survive wind-stripped control -- structural/correlational, not wind-specific."},
]

VERDICT_STYLE = {
    "validated": {"color": "#1a9850", "weight": 5, "delay": 400},
    "unconfirmed": {"color": "#fc8d59", "weight": 3, "delay": 800},
    "structural": {"color": "#999999", "weight": 2, "delay": 1500},
}

def catch_rate_to_color(pct):
    if pct < 30: return "#d73027"
    elif pct < 50: return "#fc8d59"
    elif pct < 65: return "#fee08b"
    elif pct < 80: return "#91cf60"
    else: return "#1a9850"

# ================================================================
# Build the map
# ================================================================
m = folium.Map(location=[36.3, 127.8], zoom_start=7, tiles="CartoDB positron")

# --- background: climatological wind-graph structure ---
df = pd.read_pickle(f"{BASE}/air_korea_final_imputed_with_blh.pkl")
station_order = sorted(df["Station_ID"].unique())
stations = df[["Station_ID", "lat", "lon"]].drop_duplicates("Station_ID").set_index("Station_ID").loc[station_order]
lats, lons = stations["lat"].to_numpy(), stations["lon"].to_numpy()

def haversine_km(lat1, lon1, lat2, lon2):
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
    dlat, dlon = lat2 - lat1, lon2 - lon1
    a = np.sin(dlat/2)**2 + np.cos(lat1)*np.cos(lat2)*np.sin(dlon/2)**2
    return 2 * 6371.0 * np.arcsin(np.sqrt(a))

def bearing_matrix(lat, lon):
    lat_r, lon_r = np.radians(lat), np.radians(lon)
    lat1, lat2 = lat_r[:, None], lat_r[None, :]
    dlon = lon_r[None, :] - lon_r[:, None]
    x = np.sin(dlon) * np.cos(lat2)
    y = np.cos(lat1) * np.sin(lat2) - np.sin(lat1) * np.cos(lat2) * np.cos(dlon)
    return (np.degrees(np.arctan2(x, y)) + 360) % 360

region_names = list(REGION_CENTROIDS.keys())
n_regions = len(region_names)
region_lats = np.array([REGION_CENTROIDS[r][0] for r in region_names])
region_lons = np.array([REGION_CENTROIDS[r][1] for r in region_names])
dist_to_region = haversine_km(lats[:, None], lons[:, None], region_lats[None, :], region_lons[None, :])
station_region_idx = dist_to_region.argmin(axis=1)

df["year"] = df["Datetime"].dt.year
train_df = df[df["year"].isin([2016, 2017])]

def regional_mean(sub_df, col):
    piv = sub_df.pivot(index="Datetime", columns="Station_ID", values=col)[station_order]
    arr = piv.to_numpy()
    out = np.zeros((arr.shape[0], n_regions))
    for r in range(n_regions):
        cols = station_region_idx == r
        out[:, r] = np.nanmean(arr[:, cols], axis=1)
    return out

region_windspeed = regional_mean(train_df, "windspeed_10m")
wdir = regional_mean(train_df, "winddirection_10m")
wdir_sin = np.sin(np.radians(wdir)); wdir_cos = np.cos(np.radians(wdir))
region_wind_dir = (np.degrees(np.arctan2(np.nanmean(wdir_sin, axis=0, keepdims=True), np.nanmean(wdir_cos, axis=0, keepdims=True))) + 360) % 360
region_wind_blows_toward = (region_wind_dir[0] + 180) % 360

region_dist_km = haversine_km(region_lats[:, None], region_lons[:, None], region_lats[None, :], region_lons[None, :])
region_bearing = bearing_matrix(region_lats, region_lons)
mean_windspeed = np.nanmean(region_windspeed, axis=0)

RHO_KM = 250.0
climatological_edges = []
for i in range(n_regions):
    for j in range(n_regions):
        if i == j: continue
        decay = np.exp(-region_dist_km[i, j] / RHO_KM)
        align = max(np.cos(np.radians(region_wind_blows_toward[i] - region_bearing[i, j])), 0.0)
        strength = decay * align * mean_windspeed[i]
        climatological_edges.append((region_names[i], region_names[j], strength))

climatological_edges.sort(key=lambda e: -e[2])
top_climatological = climatological_edges[:20]
max_strength = top_climatological[0][2]

bg_layer = folium.FeatureGroup(name="Climatological wind-graph structure (background)", show=True)
for src, tgt, strength in top_climatological:
    opacity = 0.15 + 0.35 * (strength / max_strength)
    folium.PolyLine([REGION_CENTROIDS[src], REGION_CENTROIDS[tgt]], color="#4575b4", weight=1.5, opacity=opacity,
                     tooltip=f"{src} → {tgt} (climatological strength: {strength:.3f})").add_to(bg_layer)
bg_layer.add_to(m)

# --- ablation-validated / unconfirmed / structural dependencies ---
ablation_layer = folium.FeatureGroup(name="Ablation-tested dependencies (animated = direction)", show=True)
for pair in ABLATION_RESULTS:
    src, tgt, verdict = pair["source"], pair["target"], pair["verdict"]
    style = VERDICT_STYLE[verdict]
    popup_html = (f"<b>{src} → {tgt}</b><br>"
                  f"Verdict: <b>{verdict.upper()}</b><br>"
                  f"Ablation mean ΔAUC-PR: {pair['mean_delta']:.4f}<br>"
                  f"{pair['note']}")
    AntPath(
        locations=[REGION_CENTROIDS[src], REGION_CENTROIDS[tgt]],
        color=style["color"], weight=style["weight"], delay=style["delay"], dash_array=[10, 20],
        popup=folium.Popup(popup_html, max_width=300),
        tooltip=f"{src} → {tgt} ({verdict})",
    ).add_to(ablation_layer)
ablation_layer.add_to(m)

# --- region markers: color = % of real events caught, size = average lead time ---
marker_layer = folium.FeatureGroup(name="Regions (color = event catch rate, size = avg lead time)", show=True)
for region, (lat, lon) in REGION_CENTROIDS.items():
    a = PER_REGION_ALARM[region]
    color = catch_rate_to_color(a["pct_caught"])
    radius = 8 + a["mean_lead"] * 0.8
    popup_html = (f"<b>{region}</b><br>"
                  f"<b>Catches {a['pct_caught']:.0f}% of real events</b> with advance warning<br>"
                  f"<b>Average {a['mean_lead']:.1f} hours</b> of advance warning when caught<br>"
                  f"(median: {a['median_lead']:.1f} hours)<br>"
                  f"Based on {a['n_events']} real sustained-episode events in 2019")
    folium.CircleMarker(
        location=[lat, lon], radius=radius, color="black", weight=1.2,
        fill=True, fill_color=color, fill_opacity=0.8,
        popup=folium.Popup(popup_html, max_width=280),
        tooltip=f"{region}: {a['pct_caught']:.0f}% caught, {a['mean_lead']:.1f}h avg lead",
    ).add_to(marker_layer)
    folium.Marker(
        location=[lat, lon],
        icon=folium.DivIcon(html=f'<div style="font-size:10px;font-weight:bold;color:#222;text-shadow:1px 1px 2px white;">{region}</div>',
                             icon_size=(0, 0), icon_anchor=(-8, 8)),
    ).add_to(marker_layer)
marker_layer.add_to(m)

legend_html = """
<div style="position: fixed; bottom: 30px; left: 30px; z-index: 9999; background: white;
            padding: 12px; border: 2px solid #444; border-radius: 6px; font-size: 13px; line-height: 1.5; max-width: 320px;">
<b>Wind-graph structure, ablation results & regional early-warning performance</b><br><br>
<span style="color:#1a9850;">&#9644;&#9644;</span> Validated wind-transport dependency<br>
<span style="color:#fc8d59;">&#9644;&#9644;</span> Wind-specific, mechanism unconfirmed<br>
<span style="color:#999999;">&#9644;&#9644;</span> Structural only (not wind-specific)<br>
<span style="color:#4575b4;">&#9644;</span> Climatological background connectivity<br><br>
<b>Region markers:</b><br>
Color = % of real 2019 events caught with any advance warning (red=rarely, green=usually)<br>
Size = average lead-time hours when caught
</div>
"""
m.get_root().html.add_child(folium.Element(legend_html))

folium.LayerControl(collapsed=False).add_to(m)

OUT_PATH = f"{BASE}/wind_graph_map_v3.html"
m.save(OUT_PATH)
print(f"saved interactive map to {OUT_PATH}")
m


saved interactive map to /Users/drewbaldwin/PM2_5 Research/wind_graph_map_v3.html


In [6]:
# ================================================================
# FOUR SUPPLEMENTARY FIGURES/TABLES using the saved 2019 headline model
# and the F1-optimal threshold (selected on val, same convention as
# every other result in this paper). Inference only -- no retraining.
#
# 1. Per-region scatter: AUC-PR vs. number of real episodes
# 2. Precision-recall curve, with Youden's J and F1-optimal marked
# 3. Lead-time distribution histogram
# 4. Confusion matrix table (aggregate, at F1-optimal threshold)
# ================================================================
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import os, gc, json
from sklearn.metrics import (average_precision_score, roc_auc_score, precision_recall_curve,
                              precision_score, recall_score, matthews_corrcoef, roc_curve, confusion_matrix)

BASE = "/Users/drewbaldwin/PM2_5 Research"
SAVE_DIR = f"{BASE}/saved_wind_graph_2019"

with open(f"{SAVE_DIR}/metadata.json") as f:
    metadata = json.load(f)

df = pd.read_pickle(f"{BASE}/air_korea_final_imputed_with_blh.pkl")
station_order = sorted(df["Station_ID"].unique())
n_stations = len(station_order)

stations = df[["Station_ID", "lat", "lon"]].drop_duplicates("Station_ID").set_index("Station_ID").loc[station_order]
lats, lons = stations["lat"].to_numpy(), stations["lon"].to_numpy()

def haversine_km(lat1, lon1, lat2, lon2):
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
    dlat, dlon = lat2 - lat1, lon2 - lon1
    a = np.sin(dlat/2)**2 + np.cos(lat1)*np.cos(lat2)*np.sin(dlon/2)**2
    return 2 * 6371.0 * np.arcsin(np.sqrt(a))

def bearing_matrix(lat, lon):
    lat_r, lon_r = np.radians(lat), np.radians(lon)
    lat1, lat2 = lat_r[:, None], lat_r[None, :]
    dlon = lon_r[None, :] - lon_r[:, None]
    x = np.sin(dlon) * np.cos(lat2)
    y = np.cos(lat1) * np.sin(lat2) - np.sin(lat1) * np.cos(lat2) * np.cos(dlon)
    return (np.degrees(np.arctan2(x, y)) + 360) % 360

dist_km = haversine_km(lats[:, None], lons[:, None], lats[None, :], lons[None, :])
DIST_CUTOFF, RHO_KM = 250.0, 250.0
HYBRID_THRESHOLD_KM = 20.0
dist_edges = (dist_km <= DIST_CUTOFF) & (dist_km > 0)
bearing_from = bearing_matrix(lats, lons)

src_idx, dst_idx = np.nonzero(dist_edges)
edge_index_np = np.stack([src_idx, dst_idx])
decay_edge = np.exp(-dist_km[src_idx, dst_idx] / RHO_KM).astype(np.float32)
bearing_edge = bearing_from[src_idx, dst_idx].astype(np.float32)
close_edge_mask = (dist_km[src_idx, dst_idx] <= HYBRID_THRESHOLD_KM)

REGION_CENTROIDS = {
    "Seoul": (37.566, 126.978), "Busan": (35.180, 129.075), "Daegu": (35.872, 128.602),
    "Incheon": (37.483, 126.633), "Gwangju": (35.155, 126.916), "Daejeon": (36.350, 127.385),
    "Ulsan": (35.550, 129.317), "Sejong": (36.487, 127.282), "Gyeonggi": (37.500, 127.250),
    "Gangwon": (37.867, 127.733), "Chungbuk": (36.633, 127.483), "Chungnam": (36.500, 126.750),
    "Jeonbuk": (35.824, 127.148), "Jeonnam": (34.750, 127.000), "Gyeongbuk": (36.559, 128.729),
    "Gyeongnam": (35.271, 128.663), "Jeju": (33.513, 126.523),
}
region_names = list(REGION_CENTROIDS.keys())
n_regions = len(region_names)
region_lats = np.array([REGION_CENTROIDS[r][0] for r in region_names])
region_lons = np.array([REGION_CENTROIDS[r][1] for r in region_names])
dist_to_region = haversine_km(lats[:, None], lons[:, None], region_lats[None, :], region_lons[None, :])
station_region_idx = dist_to_region.argmin(axis=1)

region_membership = np.zeros((n_stations, n_regions), dtype=np.float32)
region_membership[np.arange(n_stations), station_region_idx] = 1.0
region_membership_t = torch.tensor(region_membership)

region_dist_km = haversine_km(region_lats[:, None], region_lons[:, None], region_lats[None, :], region_lons[None, :])
region_bearing = bearing_matrix(region_lats, region_lons)
r_src_idx, r_dst_idx = np.nonzero(~np.eye(n_regions, dtype=bool))
region_edge_index_np = np.stack([r_src_idx, r_dst_idx])
region_decay_edge = np.exp(-region_dist_km[r_src_idx, r_dst_idx] / RHO_KM).astype(np.float32)
region_bearing_edge = region_bearing[r_src_idx, r_dst_idx].astype(np.float32)

WINDOW, HORIZON = metadata["WINDOW"], metadata["HORIZON"]
GRAPH_RECENT_HOURS = metadata["GRAPH_RECENT_HOURS"]
EVENT_THRESHOLD = metadata["EVENT_THRESHOLD"]
SUSTAIN_HOURS = metadata["SUSTAIN_HOURS"]
TIME_FEATS = ["SO2", "CO", "NO2", "O3", "PM10", "PM25"]
STATIC_COLS = metadata["STATIC_COLS"]

time_panels = {c: df.pivot(index="Datetime", columns="Station_ID", values=c)[station_order] for c in TIME_FEATS}
dt_index = time_panels["PM25"].index
n_time = len(dt_index)
years = dt_index.year.to_numpy()
doy = dt_index.dayofyear.to_numpy().astype(float)

split_id_per_hour = np.where((years == 2016) | (years == 2017), 0, np.where(years == 2018, 1, np.where(years == 2019, 2, -1)))
TRAIN_MASK = split_id_per_hour == 0

wind_dir_arr = df.pivot(index="Datetime", columns="Station_ID", values="winddirection_10m")[station_order].reindex(dt_index).to_numpy().astype(float)
wind_speed_arr = df.pivot(index="Datetime", columns="Station_ID", values="windspeed_10m")[station_order].reindex(dt_index).to_numpy().astype(float)
blh_arr = df.pivot(index="Datetime", columns="Station_ID", values="boundary_layer_height")[station_order].reindex(dt_index).to_numpy().astype(float)
pm25_raw_arr = time_panels["PM25"].to_numpy()

def regional_flat_mean(arr):
    out = np.zeros((n_time, n_regions), dtype=np.float32)
    for r in range(n_regions):
        cols = station_region_idx == r
        out[:, r] = np.nanmean(arr[:, cols], axis=1)
    return out

region_pm25 = regional_flat_mean(pm25_raw_arr)
wdir_sin_station = np.sin(np.radians(wind_dir_arr))
wdir_cos_station = np.cos(np.radians(wind_dir_arr))
region_windspeed = regional_flat_mean(wind_speed_arr)
region_wdir_sin = regional_flat_mean(wdir_sin_station)
region_wdir_cos = regional_flat_mean(wdir_cos_station)
region_wind_dir_deg = (np.degrees(np.arctan2(region_wdir_sin, region_wdir_cos)) + 360) % 360
region_wind_blows_toward = (region_wind_dir_deg + 180) % 360

season_sin_1d = np.sin(2 * np.pi * doy / 365.25)
season_cos_1d = np.cos(2 * np.pi * doy / 365.25)
season_sin = np.tile(season_sin_1d[:, None], (1, n_stations))
season_cos = np.tile(season_cos_1d[:, None], (1, n_stations))

TIME_FEATS_FULL = metadata["TIME_FEATS_FULL"]
n_time_feats, n_static_feats = metadata["n_time_feats"], metadata["n_static_feats"]
n_feats = metadata["n_feats"]
pm25_col_idx = TIME_FEATS_FULL.index("PM25")

time_arr_raw = np.stack([time_panels[c].to_numpy() for c in TIME_FEATS] +
                         [wind_speed_arr, wdir_sin_station, wdir_cos_station, blh_arr, season_sin, season_cos], axis=-1)
del time_panels, season_sin, season_cos, blh_arr, pm25_raw_arr
gc.collect()

static_df = df[["Station_ID"] + STATIC_COLS].drop_duplicates("Station_ID").set_index("Station_ID").loc[station_order]
static_arr = static_df[STATIC_COLS].to_numpy()
del df
gc.collect()

rev = region_pm25[::-1]
roll_min_rev = pd.DataFrame(rev).rolling(window=SUSTAIN_HOURS, min_periods=SUSTAIN_HOURS).min().to_numpy()
region_episode_label = (roll_min_rev[::-1] >= EVENT_THRESHOLD).astype(np.float32)
del rev, roll_min_rev
gc.collect()

def monotonic_quantiles(raw):
    first = raw[..., :1]
    deltas = F.softplus(raw[..., 1:])
    return torch.cat([first, first + torch.cumsum(deltas, dim=-1)], dim=-1)

class WindConvLayer(nn.Module):
    def __init__(self, in_dim, out_dim):
        super().__init__()
        self.lin_self = nn.Linear(in_dim, out_dim)
        self.lin_neigh = nn.Linear(in_dim, out_dim)
        self.lin_connectivity = nn.Linear(1, out_dim)

    def forward(self, x, edge_index, edge_weight, num_nodes):
        src, dst = edge_index[0], edge_index[1]
        messages = x[src] * edge_weight.unsqueeze(-1)
        agg_sum = x.new_zeros(num_nodes, x.size(-1))
        agg_sum.index_add_(0, dst, messages)
        weight_sum = x.new_zeros(num_nodes)
        weight_sum.index_add_(0, dst, edge_weight)
        agg_mean = agg_sum / (weight_sum.unsqueeze(-1) + 1e-8)
        connectivity = torch.log1p(weight_sum.clamp(min=0)).unsqueeze(-1)
        return self.lin_self(x) + self.lin_neigh(agg_mean) + self.lin_connectivity(connectivity)

class AttentionPool(nn.Module):
    def __init__(self, hidden):
        super().__init__()
        self.attn_score = nn.Linear(hidden, 1)

    def forward(self, h_station, region_membership_t_local):
        B, N, H = h_station.shape
        scores = self.attn_score(h_station).squeeze(-1)
        scores = scores - scores.max(dim=1, keepdim=True).values
        exp_scores = torch.exp(scores)
        weighted_exp = exp_scores.unsqueeze(-1) * region_membership_t_local.unsqueeze(0)
        region_denom = weighted_exp.sum(dim=1)
        region_numer = torch.einsum('bnr,bnh->brh', weighted_exp, h_station)
        return region_numer / (region_denom.unsqueeze(-1) + 1e-8)

class WindConvEncoder(nn.Module):
    def __init__(self, in_dim, hidden=32):
        super().__init__()
        self.station_conv = WindConvLayer(in_dim, hidden)

class RegionFromTrajectoryGCN(nn.Module):
    def __init__(self, station_conv, region_membership_t_local, dropout, hidden=32, gru_hidden=32):
        super().__init__()
        self.station_conv = station_conv
        self.attn_pool = AttentionPool(hidden)
        self.region_conv = WindConvLayer(hidden, hidden)
        self.drop = nn.Dropout(dropout)
        self.region_gru = nn.GRU(hidden, gru_hidden, batch_first=True)
        self.region_head = nn.Linear(gru_hidden, HORIZON)
        self.rmem = region_membership_t_local

    def forward(self, x_window, station_edge_index, station_edge_weight_seq, region_edge_index, region_edge_weight_seq, n_reg):
        B, W, N, Fin = x_window.shape
        station_ei_b = torch.cat([station_edge_index + i * N for i in range(B)], dim=1)
        region_ei_b = torch.cat([region_edge_index + i * n_reg for i in range(B)], dim=1)
        num_station_nodes = B * N
        num_region_nodes = B * n_reg
        h_region_seq = []
        for w in range(W):
            xt = x_window[:, w].reshape(B * N, Fin)
            ew_station_b = station_edge_weight_seq[:, w].reshape(-1)
            h_station = torch.relu(self.station_conv(xt, station_ei_b, ew_station_b, num_station_nodes))
            h_station = self.drop(h_station).reshape(B, N, -1)
            h_region_pooled = self.attn_pool(h_station, self.rmem).reshape(B * n_reg, -1)
            ew_region_b = region_edge_weight_seq[:, w].reshape(-1)
            h_region = torch.relu(self.region_conv(h_region_pooled, region_ei_b, ew_region_b, num_region_nodes))
            h_region = self.drop(h_region)
            h_region_seq.append(h_region.reshape(B, n_reg, -1))
        h_region_seq = torch.stack(h_region_seq, dim=1).permute(0, 2, 1, 3).reshape(B * n_reg, W, -1)
        _, h_final = self.region_gru(h_region_seq)
        embed = self.drop(h_final.squeeze(0).reshape(B, n_reg, -1))
        return self.region_head(embed)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else ("mps" if torch.backends.mps.is_available() else "cpu"))
print(f"device={DEVICE}")

region_membership_t = region_membership_t.to(DEVICE)
edge_index = torch.tensor(edge_index_np, dtype=torch.long).to(DEVICE)
region_edge_index = torch.tensor(region_edge_index_np, dtype=torch.long).to(DEVICE)

def add_static_fn(x_time_batch, static_tensor_local):
    B, W, N, _ = x_time_batch.shape
    static_b = static_tensor_local.unsqueeze(0).unsqueeze(0).expand(B, W, N, n_static_feats)
    return torch.cat([x_time_batch, static_b], dim=-1)

def gather_seq(ew_by_hour_t, starts_subset):
    idx = starts_subset.unsqueeze(1) + torch.arange(WINDOW).unsqueeze(0)
    ew = ew_by_hour_t[idx]
    ew = ew.clone()
    ew[:, :WINDOW - GRAPH_RECENT_HOURS, :] = 0.0
    return ew

wind_blows_toward = (wind_dir_arr + 180) % 360
wbt_src = wind_blows_toward[:, src_idx]
cos_align = np.maximum(np.cos(np.radians(wbt_src - bearing_edge[None, :])), 0.0)
speed_src = wind_speed_arr[:, src_idx]
wind_component = (cos_align * speed_src).astype(np.float32)
del wbt_src, cos_align, speed_src
gc.collect()
ref_speed = np.float32(np.nanmean(wind_speed_arr[TRAIN_MASK]))
component = np.where(close_edge_mask[None, :], ref_speed, wind_component)
station_wind_raw = np.nan_to_num(decay_edge[None, :] * component, nan=0.0).astype(np.float32)
del component
gc.collect()

r_wbt_src = region_wind_blows_toward[:, r_src_idx]
r_cos_align = np.maximum(np.cos(np.radians(r_wbt_src - region_bearing_edge[None, :])), 0.0)
r_speed_src = region_windspeed[:, r_src_idx]
region_wind_raw = np.nan_to_num(region_decay_edge[None, :] * r_cos_align * r_speed_src, nan=0.0).astype(np.float32)
del r_wbt_src, r_cos_align, r_speed_src
gc.collect()

train_nonzero = station_wind_raw[TRAIN_MASK][station_wind_raw[TRAIN_MASK] > 0]
station_wind_edge_weight = (station_wind_raw / train_nonzero.std()).astype(np.float32)
region_train_nonzero = region_wind_raw[TRAIN_MASK][region_wind_raw[TRAIN_MASK] > 0]
region_wind_edge_weight = (region_wind_raw / region_train_nonzero.std()).astype(np.float32)
del train_nonzero, region_train_nonzero, station_wind_raw, region_wind_raw
gc.collect()

t_mean = np.nanmean(time_arr_raw[TRAIN_MASK], axis=(0, 1), keepdims=True)
t_std = np.nanstd(time_arr_raw[TRAIN_MASK], axis=(0, 1), keepdims=True) + 1e-6
time_arr_std = np.nan_to_num((time_arr_raw - t_mean) / t_std, nan=0.0)
s_mean, s_std = static_arr.mean(axis=0, keepdims=True), static_arr.std(axis=0, keepdims=True) + 1e-6
static_tensor = torch.tensor((static_arr - s_mean) / s_std, dtype=torch.float32).to(DEVICE)

p_buckets = {1: ([], [], []), 2: ([], [], [])}
for t in range(0, n_time - WINDOW - HORIZON + 1):
    target_t = t + WINDOW + HORIZON - 1
    s_start, s_target = split_id_per_hour[t], split_id_per_hour[target_t]
    if s_start != s_target or s_start not in (1, 2):
        continue
    x_win = time_arr_std[t:t + WINDOW]
    traj = region_episode_label[t + WINDOW: t + WINDOW + HORIZON]
    Xl, ytrajl, sl = p_buckets[s_start]
    Xl.append(x_win)
    ytrajl.append(traj.T)
    sl.append(t)

X1, ytraj1, starts1 = (np.stack(v) for v in p_buckets[1])
X2, ytraj2, starts2 = (np.stack(v) for v in p_buckets[2])
del p_buckets, time_arr_std
gc.collect()
print(f"val windows={len(X1)}  test windows={len(X2)}")

X1_t = torch.tensor(X1, dtype=torch.float32)
X2_t = torch.tensor(X2, dtype=torch.float32)
starts1_t = torch.tensor(starts1, dtype=torch.long)
starts2_t = torch.tensor(starts2, dtype=torch.long)

wind_ewt_s = torch.tensor(station_wind_edge_weight)
wind_ewt_r = torch.tensor(region_wind_edge_weight)

encoder = WindConvEncoder(n_feats).station_conv
encoder.load_state_dict(torch.load(f"{SAVE_DIR}/phase1_station_encoder.pt", map_location=DEVICE))
p2 = RegionFromTrajectoryGCN(encoder, region_membership_t, metadata["dropout"]).to(DEVICE)
p2.load_state_dict(torch.load(f"{SAVE_DIR}/phase2_model.pt", map_location=DEVICE))
p2.eval()
print("loaded saved model")

def predict_traj_probs(model, ewt_s, ewt_r, X, starts, batch_size=64):
    model.eval()
    n = X.shape[0]
    out = np.zeros((n, n_regions, HORIZON), dtype=np.float32)
    with torch.no_grad():
        for start in range(0, n, batch_size):
            idx = torch.arange(start, min(start + batch_size, n))
            xb = add_static_fn(X[idx].to(DEVICE), static_tensor)
            ew_s = gather_seq(ewt_s, starts[idx]).to(DEVICE)
            ew_r = gather_seq(ewt_r, starts[idx]).to(DEVICE)
            logits = model(xb, edge_index, ew_s, region_edge_index, ew_r, n_regions)
            out[idx.numpy()] = torch.sigmoid(logits).cpu().numpy()
    return out

def contiguous_lead_hours(prob_traj, threshold):
    if prob_traj[0] < threshold:
        return 0
    o = 1
    while o < HORIZON and prob_traj[o] >= threshold:
        o += 1
    return o

val_probs = predict_traj_probs(p2, wind_ewt_s, wind_ewt_r, X1_t, starts1_t)
test_probs = predict_traj_probs(p2, wind_ewt_s, wind_ewt_r, X2_t, starts2_t)

y_val_agg = ytraj1.max(axis=-1).reshape(-1)
y_test_agg = ytraj2.max(axis=-1).reshape(-1)
val_agg_score = val_probs.max(axis=-1).reshape(-1)
test_agg_score = test_probs.max(axis=-1).reshape(-1)

# F1-optimal threshold, selected on val (same convention as every other result in this paper)
fpr, tpr, roc_thresholds = roc_curve(y_val_agg, val_agg_score)
thresh_youden = roc_thresholds[np.argmax(tpr - fpr)]

prec_curve_val, rec_curve_val, pr_thresholds = precision_recall_curve(y_val_agg, val_agg_score)
f1_curve_val = 2 * prec_curve_val * rec_curve_val / (prec_curve_val + rec_curve_val + 1e-12)
best_f1_idx = np.argmax(f1_curve_val[:-1])  # last point has no corresponding threshold
GLOBAL_THRESH = pr_thresholds[best_f1_idx]
print(f"F1-optimal threshold (selected on val): {GLOBAL_THRESH:.4f}")
print(f"Youden's J threshold (selected on val): {thresh_youden:.4f}")

# ================================================================
# FIGURE 1: per-region AUC-PR vs. number of real episodes
# ================================================================
fig1, ax1 = plt.subplots(figsize=(8, 6))
region_stats = []
for r in range(n_regions):
    y_true_r = ytraj2[:, r].max(axis=-1)
    score_r = test_probs[:, r].max(axis=-1)
    if y_true_r.sum() == 0:
        continue
    aucpr_r = average_precision_score(y_true_r, score_r)
    label_r = region_episode_label[:, r]
    # count real onset events over the full 2016-2019 period for a stable per-region "n_events" figure
    onsets = int(((label_r[1:] == 1) & (label_r[:-1] == 0)).sum() + (1 if label_r[0] == 1 else 0))
    region_stats.append({"region": region_names[r], "aucpr": aucpr_r, "n_events_full_period": onsets})
    ax1.scatter(onsets, aucpr_r, s=60, color="#4575b4", edgecolor="black", zorder=3)
    ax1.annotate(region_names[r], (onsets, aucpr_r), textcoords="offset points", xytext=(6, 4), fontsize=9)

ax1.set_xlabel("Number of real sustained-episode events (2016-2019)")
ax1.set_ylabel("Test-year (2019) AUC-PR")
ax1.set_title("Per-region model performance vs. data availability")
ax1.grid(alpha=0.3)
fig1.tight_layout()
fig1.savefig(f"{BASE}/fig1_aucpr_vs_nevents.png", dpi=150)
print("saved fig1_aucpr_vs_nevents.png")

corr = np.corrcoef([s["n_events_full_period"] for s in region_stats], [s["aucpr"] for s in region_stats])[0, 1]
print(f"correlation between event count and AUC-PR: r={corr:.3f}")

# ================================================================
# FIGURE 2: precision-recall curve, with Youden's J and F1-optimal marked
# ================================================================
prec_curve_test, rec_curve_test, _ = precision_recall_curve(y_test_agg, test_agg_score)
test_aucpr = average_precision_score(y_test_agg, test_agg_score)

pred_f1 = (test_agg_score >= GLOBAL_THRESH).astype(int)
prec_f1_pt = precision_score(y_test_agg, pred_f1, zero_division=0)
rec_f1_pt = recall_score(y_test_agg, pred_f1, zero_division=0)

pred_youden = (test_agg_score >= thresh_youden).astype(int)
prec_youden_pt = precision_score(y_test_agg, pred_youden, zero_division=0)
rec_youden_pt = recall_score(y_test_agg, pred_youden, zero_division=0)

fig2, ax2 = plt.subplots(figsize=(7, 6))
ax2.plot(rec_curve_test, prec_curve_test, color="#4575b4", label=f"Test PR curve (AUC-PR={test_aucpr:.3f})")
ax2.scatter([rec_f1_pt], [prec_f1_pt], color="#1a9850", s=100, zorder=5,
            label=f"F1-optimal (thresh={GLOBAL_THRESH:.3f}): P={prec_f1_pt:.2f}, R={rec_f1_pt:.2f}")
ax2.scatter([rec_youden_pt], [prec_youden_pt], color="#d73027", s=100, zorder=5, marker="^",
            label=f"Youden's J (thresh={thresh_youden:.3f}): P={prec_youden_pt:.2f}, R={rec_youden_pt:.2f}")
ax2.set_xlabel("Recall")
ax2.set_ylabel("Precision")
ax2.set_title("Precision-Recall Curve (2019 test set)")
ax2.legend(loc="lower left", fontsize=9)
ax2.grid(alpha=0.3)
fig2.tight_layout()
fig2.savefig(f"{BASE}/fig2_precision_recall_curve.png", dpi=150)
print("saved fig2_precision_recall_curve.png")

# ================================================================
# FIGURE 3: lead-time distribution histogram (at F1-optimal threshold)
# ================================================================
lead_hours_list = []
n_windows, n_reg = ytraj2.shape[0], ytraj2.shape[1]
for i in range(n_windows):
    for r in range(n_reg):
        if ytraj2[i, r].max() < 1:
            continue
        lead_hours_list.append(contiguous_lead_hours(test_probs[i, r], GLOBAL_THRESH))
lead_hours_arr = np.array(lead_hours_list)

fig3, ax3 = plt.subplots(figsize=(8, 5))
ax3.hist(lead_hours_arr, bins=np.arange(0, HORIZON + 2) - 0.5, color="#91bfdb", edgecolor="black")
ax3.axvline(lead_hours_arr.mean(), color="#d73027", linestyle="--", linewidth=2, label=f"Mean = {lead_hours_arr.mean():.1f}h")
ax3.axvline(np.median(lead_hours_arr), color="#1a9850", linestyle="--", linewidth=2, label=f"Median = {np.median(lead_hours_arr):.1f}h")
ax3.set_xlabel("Contiguous lead time (hours)")
ax3.set_ylabel("Number of true-positive windows")
ax3.set_title(f"Lead-time distribution at F1-optimal threshold ({GLOBAL_THRESH:.3f})")
ax3.legend()
ax3.grid(alpha=0.3, axis="y")
fig3.tight_layout()
fig3.savefig(f"{BASE}/fig3_lead_time_histogram.png", dpi=150)
print("saved fig3_lead_time_histogram.png")

# ================================================================
# TABLE 4: confusion matrix at F1-optimal threshold
# ================================================================
cm = confusion_matrix(y_test_agg, pred_f1)
tn, fp, fn, tp = cm.ravel()
print(f"\n{'='*20} CONFUSION MATRIX (F1-optimal threshold={GLOBAL_THRESH:.4f}) {'='*20}")
print(f"                 Predicted: No Episode   Predicted: Episode")
print(f"Actual: No Episode      {tn:>12,}          {fp:>12,}")
print(f"Actual: Episode         {fn:>12,}          {tp:>12,}")
print(f"\nTotal: {tn+fp+fn+tp:,}  |  Precision={tp/(tp+fp):.4f}  Recall={tp/(tp+fn):.4f}  "
      f"Accuracy={(tp+tn)/(tn+fp+fn+tp):.4f}  MCC={matthews_corrcoef(y_test_agg, pred_f1):.4f}")

with open(f"{BASE}/supplementary_figures_data.json", "w") as f:
    json.dump({
        "f1_threshold": float(GLOBAL_THRESH), "youden_threshold": float(thresh_youden),
        "region_stats": region_stats, "aucpr_vs_nevents_corr": float(corr),
        "test_aucpr": float(test_aucpr),
        "lead_time": {"mean": float(lead_hours_arr.mean()), "median": float(np.median(lead_hours_arr)),
                       "values": lead_hours_arr.tolist()},
        "confusion_matrix": {"tn": int(tn), "fp": int(fp), "fn": int(fn), "tp": int(tp)},
    }, f, indent=2)
print(f"\nsaved all underlying data to {BASE}/supplementary_figures_data.json")


device=mps
val windows=8689  test windows=8689
loaded saved model
F1-optimal threshold (selected on val): 0.8469
Youden's J threshold (selected on val): 0.0684
saved fig1_aucpr_vs_nevents.png
correlation between event count and AUC-PR: r=0.668
saved fig2_precision_recall_curve.png
saved fig3_lead_time_histogram.png

==================== CONFUSION MATRIX (F1-optimal threshold=0.8469) ====================
                 Predicted: No Episode   Predicted: Episode
Actual: No Episode           137,881                 1,903
Actual: Episode                3,203                 4,726

Total: 147,713  |  Precision=0.7129  Recall=0.5960  Accuracy=0.9654  MCC=0.6340

saved all underlying data to /Users/drewbaldwin/PM2_5 Research/supplementary_figures_data.json


In [ ]:
#testing against baseline models more

In [1]:
#simpler tree based method. 

# ================================================================
# GRADIENT-BOOSTED TREE BASELINES using sklearn's HistGradientBoostingClassifier
# (avoids XGBoost's macOS OpenMP dependency issue -- no new installs needed)
# Two versions: A) single-region-only features, B) + nearest-3-region features
# Same train=2016-2017/val=2018/test=2019 split, same F1-optimal threshold convention.
# ================================================================
import numpy as np
import pandas as pd
import os, gc, json
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import (average_precision_score, roc_auc_score, precision_score,
                              recall_score, matthews_corrcoef, roc_curve)

BASE = "/Users/drewbaldwin/PM2_5 Research"
df = pd.read_pickle(f"{BASE}/air_korea_final_imputed_with_blh.pkl")
station_order = sorted(df["Station_ID"].unique())
n_stations = len(station_order)

stations = df[["Station_ID", "lat", "lon"]].drop_duplicates("Station_ID").set_index("Station_ID").loc[station_order]
lats, lons = stations["lat"].to_numpy(), stations["lon"].to_numpy()

def haversine_km(lat1, lon1, lat2, lon2):
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
    dlat, dlon = lat2 - lat1, lon2 - lon1
    a = np.sin(dlat/2)**2 + np.cos(lat1)*np.cos(lat2)*np.sin(dlon/2)**2
    return 2 * 6371.0 * np.arcsin(np.sqrt(a))

REGION_CENTROIDS = {
    "Seoul": (37.566, 126.978), "Busan": (35.180, 129.075), "Daegu": (35.872, 128.602),
    "Incheon": (37.483, 126.633), "Gwangju": (35.155, 126.916), "Daejeon": (36.350, 127.385),
    "Ulsan": (35.550, 129.317), "Sejong": (36.487, 127.282), "Gyeonggi": (37.500, 127.250),
    "Gangwon": (37.867, 127.733), "Chungbuk": (36.633, 127.483), "Chungnam": (36.500, 126.750),
    "Jeonbuk": (35.824, 127.148), "Jeonnam": (34.750, 127.000), "Gyeongbuk": (36.559, 128.729),
    "Gyeongnam": (35.271, 128.663), "Jeju": (33.513, 126.523),
}
region_names = list(REGION_CENTROIDS.keys())
n_regions = len(region_names)
region_lats = np.array([REGION_CENTROIDS[r][0] for r in region_names])
region_lons = np.array([REGION_CENTROIDS[r][1] for r in region_names])
dist_to_region = haversine_km(lats[:, None], lons[:, None], region_lats[None, :], region_lons[None, :])
station_region_idx = dist_to_region.argmin(axis=1)

region_dist_km = haversine_km(region_lats[:, None], region_lons[:, None], region_lats[None, :], region_lons[None, :])
NEAREST_K = 3
nearest_neighbors = np.argsort(region_dist_km, axis=1)[:, 1:NEAREST_K + 1]  # exclude self (col 0)

WINDOW, HORIZON = 36, 36
EVENT_THRESHOLD = 75.0
SUSTAIN_HOURS = 2
TIME_FEATS = ["SO2", "CO", "NO2", "O3", "PM10", "PM25"]
STATIC_COLS = ["elevation_m", "urban_landuse_area_m2_3km", "green_space_area_3km",
               "building_footprint_area_3km", "railway_length_3km", "dist_to_coast_km",
               "dist_to_major_road_km", "industrial_area_m2_3km", "traffic_points_count_3km",
               "major_roads_count_3km", "total_road_length_3km"]

time_panels = {c: df.pivot(index="Datetime", columns="Station_ID", values=c)[station_order] for c in TIME_FEATS}
dt_index = time_panels["PM25"].index
n_time = len(dt_index)
years = dt_index.year.to_numpy()
doy = dt_index.dayofyear.to_numpy().astype(float)

split_id_per_hour = np.where((years == 2016) | (years == 2017), 0, np.where(years == 2018, 1, np.where(years == 2019, 2, -1)))
TRAIN_MASK = split_id_per_hour == 0

wind_speed_arr = df.pivot(index="Datetime", columns="Station_ID", values="windspeed_10m")[station_order].reindex(dt_index).to_numpy().astype(float)
wind_dir_arr = df.pivot(index="Datetime", columns="Station_ID", values="winddirection_10m")[station_order].reindex(dt_index).to_numpy().astype(float)
blh_arr = df.pivot(index="Datetime", columns="Station_ID", values="boundary_layer_height")[station_order].reindex(dt_index).to_numpy().astype(float)
pm25_raw_arr = time_panels["PM25"].to_numpy()

def regional_flat_mean(arr):
    out = np.zeros((n_time, n_regions), dtype=np.float32)
    for r in range(n_regions):
        cols = station_region_idx == r
        out[:, r] = np.nanmean(arr[:, cols], axis=1)
    return out

region_pm25 = regional_flat_mean(pm25_raw_arr)
wdir_sin_station = np.sin(np.radians(wind_dir_arr))
wdir_cos_station = np.cos(np.radians(wind_dir_arr))
season_sin_1d = np.sin(2 * np.pi * doy / 365.25)
season_cos_1d = np.cos(2 * np.pi * doy / 365.25)
season_sin = np.tile(season_sin_1d[:, None], (1, n_stations))
season_cos = np.tile(season_cos_1d[:, None], (1, n_stations))

TIME_FEATS_FULL = TIME_FEATS + ["windspeed_10m", "wdir_sin", "wdir_cos", "boundary_layer_height", "season_sin", "season_cos"]
n_channels = len(TIME_FEATS_FULL)

time_arr_raw = np.stack([time_panels[c].to_numpy() for c in TIME_FEATS] +
                         [wind_speed_arr, wdir_sin_station, wdir_cos_station, blh_arr, season_sin, season_cos], axis=-1)
del time_panels, season_sin, season_cos, blh_arr
gc.collect()

region_channels = np.zeros((n_time, n_regions, n_channels), dtype=np.float32)
for r in range(n_regions):
    cols = station_region_idx == r
    for c in range(n_channels):
        region_channels[:, r, c] = np.nanmean(time_arr_raw[:, cols, c], axis=1)
del time_arr_raw
gc.collect()

static_df = df[["Station_ID"] + STATIC_COLS].drop_duplicates("Station_ID").set_index("Station_ID").loc[station_order]
static_arr = static_df[STATIC_COLS].to_numpy()
region_static = np.zeros((n_regions, len(STATIC_COLS)), dtype=np.float32)
for r in range(n_regions):
    cols = station_region_idx == r
    region_static[r] = static_arr[cols].mean(axis=0)
del df, static_arr
gc.collect()

rev = region_pm25[::-1]
roll_min_rev = pd.DataFrame(rev).rolling(window=SUSTAIN_HOURS, min_periods=SUSTAIN_HOURS).min().to_numpy()
region_episode_label = (roll_min_rev[::-1] >= EVENT_THRESHOLD).astype(np.float32)
del rev, roll_min_rev, pm25_raw_arr
gc.collect()

print(f"train hours: {TRAIN_MASK.sum()}  val hours: {(split_id_per_hour==1).sum()}  test hours: {(split_id_per_hour==2).sum()}")

# ================================================================
# Build flattened tabular features
# ================================================================
FEAT_NAMES_A = ([f"{c}_mean" for c in TIME_FEATS_FULL] + [f"{c}_last" for c in TIME_FEATS_FULL] + STATIC_COLS)
FEAT_NAMES_B = FEAT_NAMES_A + [f"neighbor{k}_PM25_mean" for k in range(NEAREST_K)] + [f"neighbor{k}_PM25_last" for k in range(NEAREST_K)]

buckets = {0: {"A": [], "B": [], "y": []}, 1: {"A": [], "B": [], "y": []}, 2: {"A": [], "B": [], "y": []}}
pm25_idx = TIME_FEATS_FULL.index("PM25")

for t in range(0, n_time - WINDOW - HORIZON + 1):
    target_t = t + WINDOW + HORIZON - 1
    s_start, s_target = split_id_per_hour[t], split_id_per_hour[target_t]
    if s_start != s_target or s_start == -1:
        continue
    window = region_channels[t:t + WINDOW]
    window_mean = window.mean(axis=0)
    window_last = window[-1]
    labels = region_episode_label[t + WINDOW: t + WINDOW + HORIZON].max(axis=0)

    for r in range(n_regions):
        feat_A = np.concatenate([window_mean[r], window_last[r], region_static[r]])
        neighbor_idx = nearest_neighbors[r]
        neigh_mean = window_mean[neighbor_idx, pm25_idx]
        neigh_last = window_last[neighbor_idx, pm25_idx]
        feat_B = np.concatenate([feat_A, neigh_mean, neigh_last])
        buckets[s_start]["A"].append(feat_A)
        buckets[s_start]["B"].append(feat_B)
        buckets[s_start]["y"].append(labels[r])

X0_A = np.stack(buckets[0]["A"]); X1_A = np.stack(buckets[1]["A"]); X2_A = np.stack(buckets[2]["A"])
X0_B = np.stack(buckets[0]["B"]); X1_B = np.stack(buckets[1]["B"]); X2_B = np.stack(buckets[2]["B"])
y0 = np.array(buckets[0]["y"]); y1 = np.array(buckets[1]["y"]); y2 = np.array(buckets[2]["y"])
del buckets, region_channels
gc.collect()
print(f"rows: train={len(y0)} val={len(y1)} test={len(y2)}  (positive rate train={y0.mean():.4f})")

POS_WEIGHT = min(float((len(y0) - y0.sum()) / max(y0.sum(), 1)), 50.0)
print(f"pos_weight (capped at 50, same convention as wind_graph): {POS_WEIGHT:.2f}")
sample_weight_0 = np.where(y0 == 1, POS_WEIGHT, 1.0)

def train_and_eval_hgb(X0, X1, X2, label):
    model = HistGradientBoostingClassifier(
        max_iter=300, learning_rate=0.05, max_depth=6,
        early_stopping=True, validation_fraction=0.1, n_iter_no_change=20,
        random_state=0,
    )
    model.fit(X0, y0, sample_weight=sample_weight_0)

    val_score = model.predict_proba(X1)[:, 1]
    test_score = model.predict_proba(X2)[:, 1]

    test_aucroc = roc_auc_score(y2, test_score)
    test_aucpr = average_precision_score(y2, test_score)

    fpr, tpr, thresholds = roc_curve(y1, val_score)
    f1_scores = []
    for th in thresholds:
        pred = (val_score >= th).astype(int)
        p = precision_score(y1, pred, zero_division=0)
        r = recall_score(y1, pred, zero_division=0)
        f1_scores.append(2 * p * r / (p + r) if (p + r) > 0 else 0.0)
    best_thresh = thresholds[np.argmax(f1_scores)]

    pred_test = (test_score >= best_thresh).astype(int)
    precision = precision_score(y2, pred_test, zero_division=0)
    recall = recall_score(y2, pred_test, zero_division=0)
    mcc = matthews_corrcoef(y2, pred_test)
    far = 1 - precision

    print(f"\n--- {label} ---")
    print(f"  n_iterations used: {model.n_iter_}")
    print(f"  test AUC-ROC={test_aucroc:.4f}  AUC-PR={test_aucpr:.4f}")
    print(f"  F1-optimal thresh={best_thresh:.4f}  precision={precision:.4f}  recall={recall:.4f}  FAR={far:.4f}  MCC={mcc:.4f}")

    return {"label": label, "test_aucroc": float(test_aucroc), "test_aucpr": float(test_aucpr),
            "threshold": float(best_thresh), "precision": float(precision), "recall": float(recall),
            "far": float(far), "mcc": float(mcc)}

print(f"\n{'='*20} MODEL A: single-region features only {'='*20}")
result_A = train_and_eval_hgb(X0_A, X1_A, X2_A, "HistGB (single-region only)")

print(f"\n{'='*20} MODEL B: + nearest-3-regions' PM2.5 (hand-engineered cross-region features) {'='*20}")
result_B = train_and_eval_hgb(X0_B, X1_B, X2_B, "HistGB (+ nearest-3-region features)")

print(f"\n{'='*20} SUMMARY: compare to wind_graph headline {'='*20}")
print(f"wind_graph (headline, 3-seed mean):    AUC-ROC=0.9594  AUC-PR=0.7124  precision=0.6737  recall=0.6216  FAR=0.3263  MCC=0.6276")
print(f"HistGB (single-region only):            AUC-ROC={result_A['test_aucroc']:.4f}  AUC-PR={result_A['test_aucpr']:.4f}  "
      f"precision={result_A['precision']:.4f}  recall={result_A['recall']:.4f}  FAR={result_A['far']:.4f}  MCC={result_A['mcc']:.4f}")
print(f"HistGB (+ nearest-3-region features):   AUC-ROC={result_B['test_aucroc']:.4f}  AUC-PR={result_B['test_aucpr']:.4f}  "
      f"precision={result_B['precision']:.4f}  recall={result_B['recall']:.4f}  FAR={result_B['far']:.4f}  MCC={result_B['mcc']:.4f}")

with open(f"{BASE}/histgb_baselines.json", "w") as f:
    json.dump({"single_region": result_A, "cross_region": result_B,
               "n_train": int(len(y0)), "n_val": int(len(y1)), "n_test": int(len(y2)),
               "train_positive_rate": float(y0.mean())}, f, indent=2)
print(f"\nsaved to {BASE}/histgb_baselines.json")



train hours: 17544  val hours: 8760  test hours: 8760
rows: train=297041 val=147713 test=147713  (positive rate train=0.0370)
pos_weight (capped at 50, same convention as wind_graph): 26.01

==================== MODEL A: single-region features only ====================

--- HistGB (single-region only) ---
  n_iterations used: 300
  test AUC-ROC=0.9289  AUC-PR=0.5809
  F1-optimal thresh=0.6587  precision=0.5131  recall=0.5508  FAR=0.4869  MCC=0.5040

==================== MODEL B: + nearest-3-regions' PM2.5 (hand-engineered cross-region features) ====================

--- HistGB (+ nearest-3-region features) ---
  n_iterations used: 300
  test AUC-ROC=0.9263  AUC-PR=0.5712
  F1-optimal thresh=0.6617  precision=0.5406  recall=0.5272  FAR=0.4594  MCC=0.5078

==================== SUMMARY: compare to wind_graph headline ====================
wind_graph (headline, 3-seed mean):    AUC-ROC=0.9594  AUC-PR=0.7124  precision=0.6737  recall=0.6216  FAR=0.3263  MCC=0.6276
HistGB (single-region only)

In [1]:
# ================================================================
# EXTEND minimal_gru TO 10 SEEDS: load existing 5-seed results (0-4),
# train 5 more (5-9), combine into a full 10-seed comparison against
# the already-completed 10-seed wind_graph/no_graph results.
# ================================================================
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import os, time, copy, gc, json
from sklearn.metrics import average_precision_score, roc_auc_score
from scipy import stats

BASE = "/Users/drewbaldwin/PM2_5 Research"

# ---- load existing results ----
with open(f"{BASE}/wind_vs_minimal_gru_5seed.json") as f:
    prior_minimal = json.load(f)
with open(f"{BASE}/wind_vs_nograph_10seed.json") as f:
    prior_10seed = json.load(f)

existing_minimal_gru_results = prior_minimal["minimal_gru"]  # seeds 0-4
print(f"loaded {len(existing_minimal_gru_results)} existing minimal_gru results (seeds 0-4)")

df = pd.read_pickle(f"{BASE}/air_korea_final_imputed_with_blh.pkl")
station_order = sorted(df["Station_ID"].unique())
n_stations = len(station_order)

stations = df[["Station_ID", "lat", "lon"]].drop_duplicates("Station_ID").set_index("Station_ID").loc[station_order]
lats, lons = stations["lat"].to_numpy(), stations["lon"].to_numpy()

def haversine_km(lat1, lon1, lat2, lon2):
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
    dlat, dlon = lat2 - lat1, lon2 - lon1
    a = np.sin(dlat/2)**2 + np.cos(lat1)*np.cos(lat2)*np.sin(dlon/2)**2
    return 2 * 6371.0 * np.arcsin(np.sqrt(a))

REGION_CENTROIDS = {
    "Seoul": (37.566, 126.978), "Busan": (35.180, 129.075), "Daegu": (35.872, 128.602),
    "Incheon": (37.483, 126.633), "Gwangju": (35.155, 126.916), "Daejeon": (36.350, 127.385),
    "Ulsan": (35.550, 129.317), "Sejong": (36.487, 127.282), "Gyeonggi": (37.500, 127.250),
    "Gangwon": (37.867, 127.733), "Chungbuk": (36.633, 127.483), "Chungnam": (36.500, 126.750),
    "Jeonbuk": (35.824, 127.148), "Jeonnam": (34.750, 127.000), "Gyeongbuk": (36.559, 128.729),
    "Gyeongnam": (35.271, 128.663), "Jeju": (33.513, 126.523),
}
region_names = list(REGION_CENTROIDS.keys())
n_regions = len(region_names)
region_lats = np.array([REGION_CENTROIDS[r][0] for r in region_names])
region_lons = np.array([REGION_CENTROIDS[r][1] for r in region_names])
dist_to_region = haversine_km(lats[:, None], lons[:, None], region_lats[None, :], region_lons[None, :])
station_region_idx = dist_to_region.argmin(axis=1)

region_membership = np.zeros((n_stations, n_regions), dtype=np.float32)
region_membership[np.arange(n_stations), station_region_idx] = 1.0
region_membership_t = torch.tensor(region_membership)

WINDOW, HORIZON = 36, 36
EVENT_THRESHOLD = 75.0
SUSTAIN_HOURS = 2
QUANTILES = [0.50, 0.75, 0.90, 0.95, 0.99]
N_QUANTILES = len(QUANTILES)
TIME_FEATS = ["SO2", "CO", "NO2", "O3", "PM10", "PM25"]
STATIC_COLS = ["elevation_m", "urban_landuse_area_m2_3km", "green_space_area_3km",
               "building_footprint_area_3km", "railway_length_3km", "dist_to_coast_km",
               "dist_to_major_road_km", "industrial_area_m2_3km", "traffic_points_count_3km",
               "major_roads_count_3km", "total_road_length_3km"]

time_panels = {c: df.pivot(index="Datetime", columns="Station_ID", values=c)[station_order] for c in TIME_FEATS}
dt_index = time_panels["PM25"].index
n_time = len(dt_index)
years = dt_index.year.to_numpy()
doy = dt_index.dayofyear.to_numpy().astype(float)

split_id_per_hour = np.where((years == 2016) | (years == 2017), 0, np.where(years == 2018, 1, np.where(years == 2019, 2, -1)))
TRAIN_MASK = split_id_per_hour == 0

wind_speed_arr = df.pivot(index="Datetime", columns="Station_ID", values="windspeed_10m")[station_order].reindex(dt_index).to_numpy().astype(float)
wind_dir_arr = df.pivot(index="Datetime", columns="Station_ID", values="winddirection_10m")[station_order].reindex(dt_index).to_numpy().astype(float)
blh_arr = df.pivot(index="Datetime", columns="Station_ID", values="boundary_layer_height")[station_order].reindex(dt_index).to_numpy().astype(float)
pm25_raw_arr = time_panels["PM25"].to_numpy()

def regional_flat_mean(arr):
    out = np.zeros((n_time, n_regions), dtype=np.float32)
    for r in range(n_regions):
        cols = station_region_idx == r
        out[:, r] = np.nanmean(arr[:, cols], axis=1)
    return out

region_pm25 = regional_flat_mean(pm25_raw_arr)
wdir_sin_station = np.sin(np.radians(wind_dir_arr))
wdir_cos_station = np.cos(np.radians(wind_dir_arr))

season_sin_1d = np.sin(2 * np.pi * doy / 365.25)
season_cos_1d = np.cos(2 * np.pi * doy / 365.25)
season_sin = np.tile(season_sin_1d[:, None], (1, n_stations))
season_cos = np.tile(season_cos_1d[:, None], (1, n_stations))

TIME_FEATS_FULL = TIME_FEATS + ["windspeed_10m", "wdir_sin", "wdir_cos", "boundary_layer_height", "season_sin", "season_cos"]
n_time_feats, n_static_feats = len(TIME_FEATS_FULL), len(STATIC_COLS)
n_feats = n_time_feats + n_static_feats
pm25_col_idx = TIME_FEATS_FULL.index("PM25")

time_arr_raw = np.stack([time_panels[c].to_numpy() for c in TIME_FEATS] +
                         [wind_speed_arr, wdir_sin_station, wdir_cos_station, blh_arr, season_sin, season_cos], axis=-1)
del time_panels, season_sin, season_cos, blh_arr, pm25_raw_arr
gc.collect()

static_df = df[["Station_ID"] + STATIC_COLS].drop_duplicates("Station_ID").set_index("Station_ID").loc[station_order]
static_arr = static_df[STATIC_COLS].to_numpy()
del df
gc.collect()

rev = region_pm25[::-1]
roll_min_rev = pd.DataFrame(rev).rolling(window=SUSTAIN_HOURS, min_periods=SUSTAIN_HOURS).min().to_numpy()
region_episode_label = (roll_min_rev[::-1] >= EVENT_THRESHOLD).astype(np.float32)
del rev, roll_min_rev
gc.collect()

def pinball_loss(preds, target, quantiles):
    target_exp = target.unsqueeze(-1)
    diff = target_exp - preds
    q_tensor = torch.tensor(quantiles, device=preds.device, dtype=preds.dtype).view(*([1] * (preds.dim() - 1)), -1)
    return torch.max(q_tensor * diff, (q_tensor - 1) * diff).mean()

def monotonic_quantiles(raw):
    first = raw[..., :1]
    deltas = F.softplus(raw[..., 1:])
    return torch.cat([first, first + torch.cumsum(deltas, dim=-1)], dim=-1)

class AttentionPool(nn.Module):
    def __init__(self, hidden):
        super().__init__()
        self.attn_score = nn.Linear(hidden, 1)

    def forward(self, h_station, region_membership_t_local):
        B, N, H = h_station.shape
        scores = self.attn_score(h_station).squeeze(-1)
        scores = scores - scores.max(dim=1, keepdim=True).values
        exp_scores = torch.exp(scores)
        weighted_exp = exp_scores.unsqueeze(-1) * region_membership_t_local.unsqueeze(0)
        region_denom = weighted_exp.sum(dim=1)
        region_numer = torch.einsum('bnr,bnh->brh', weighted_exp, h_station)
        return region_numer / (region_denom.unsqueeze(-1) + 1e-8)

class StationGRU(nn.Module):
    def __init__(self, in_dim, dropout, hidden=32, gru_hidden=32):
        super().__init__()
        self.station_lin = nn.Linear(in_dim, hidden)
        self.drop = nn.Dropout(dropout)
        self.gru = nn.GRU(hidden, gru_hidden, batch_first=True)
        self.head = nn.Linear(gru_hidden, N_QUANTILES)

    def forward(self, x_window):
        B, W, N, Fin = x_window.shape
        h_seq = []
        for w in range(W):
            xt = x_window[:, w].reshape(B * N, Fin)
            h = torch.relu(self.station_lin(xt))
            h = self.drop(h)
            h_seq.append(h.reshape(B, N, -1))
        h_seq = torch.stack(h_seq, dim=1).permute(0, 2, 1, 3).reshape(B * N, W, -1)
        _, h_final = self.gru(h_seq)
        embed = self.drop(h_final.squeeze(0).reshape(B, N, -1))
        return monotonic_quantiles(self.head(embed))

class RegionFromTrajectoryGRU(nn.Module):
    def __init__(self, station_lin, region_membership_t_local, dropout, hidden=32, gru_hidden=32):
        super().__init__()
        self.station_lin = station_lin
        self.attn_pool = AttentionPool(hidden)
        self.drop = nn.Dropout(dropout)
        self.region_gru = nn.GRU(hidden, gru_hidden, batch_first=True)
        self.region_head = nn.Linear(gru_hidden, HORIZON)
        self.rmem = region_membership_t_local

    def forward(self, x_window, n_reg):
        B, W, N, Fin = x_window.shape
        h_region_seq = []
        for w in range(W):
            xt = x_window[:, w].reshape(B * N, Fin)
            h_station = torch.relu(self.station_lin(xt)).reshape(B, N, -1)
            h_station = self.drop(h_station)
            h_region_pooled = self.attn_pool(h_station, self.rmem)
            h_region_pooled = self.drop(h_region_pooled)
            h_region_seq.append(h_region_pooled)
        h_region_seq = torch.stack(h_region_seq, dim=1).permute(0, 2, 1, 3).reshape(B * n_reg, W, -1)
        _, h_final = self.region_gru(h_region_seq)
        embed = self.drop(h_final.squeeze(0).reshape(B, n_reg, -1))
        return self.region_head(embed)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else ("mps" if torch.backends.mps.is_available() else "cpu"))
MICRO_BATCH, ACCUM_STEPS = 16, 4
MAX_EPOCHS_P1, MAX_EPOCHS_P2 = 2, 2
print(f"device={DEVICE}")

region_membership_t = region_membership_t.to(DEVICE)

def add_static_fn(x_time_batch, static_tensor_local):
    B, W, N, _ = x_time_batch.shape
    static_b = static_tensor_local.unsqueeze(0).unsqueeze(0).expand(B, W, N, n_static_feats)
    return torch.cat([x_time_batch, static_b], dim=-1)

print(f"train hours (2016-2017): {TRAIN_MASK.sum()}  val hours (2018): {(split_id_per_hour==1).sum()}  test hours (2019): {(split_id_per_hour==2).sum()}")

t_mean = np.nanmean(time_arr_raw[TRAIN_MASK], axis=(0, 1), keepdims=True)
t_std = np.nanstd(time_arr_raw[TRAIN_MASK], axis=(0, 1), keepdims=True) + 1e-6
time_arr_std = np.nan_to_num((time_arr_raw - t_mean) / t_std, nan=0.0)
s_mean, s_std = static_arr.mean(axis=0, keepdims=True), static_arr.std(axis=0, keepdims=True) + 1e-6
static_tensor = torch.tensor((static_arr - s_mean) / s_std, dtype=torch.float32).to(DEVICE)

p_buckets = {0: ([], [], []), 1: ([], [], []), 2: ([], [], [])}
for t in range(0, n_time - WINDOW - HORIZON + 1):
    target_t = t + WINDOW + HORIZON - 1
    s_start, s_target = split_id_per_hour[t], split_id_per_hour[target_t]
    if s_start != s_target or s_start == -1:
        continue
    x_win = time_arr_std[t:t + WINDOW]
    traj = region_episode_label[t + WINDOW: t + WINDOW + HORIZON]
    Xl, yregl, ytrajl = p_buckets[s_start]
    Xl.append(x_win)
    yregl.append(time_arr_std[target_t, :, pm25_col_idx])
    ytrajl.append(traj.T)

X0, yreg0, ytraj0 = (np.stack(v) for v in p_buckets[0])
X1, yreg1, ytraj1 = (np.stack(v) for v in p_buckets[1])
X2, yreg2, ytraj2 = (np.stack(v) for v in p_buckets[2])
del p_buckets, time_arr_std
gc.collect()
print(f"windows: train={len(X0)} val={len(X1)} test={len(X2)}")

X0_t = torch.tensor(X0, dtype=torch.float32); del X0
X1_t = torch.tensor(X1, dtype=torch.float32); del X1
X2_t = torch.tensor(X2, dtype=torch.float32); del X2
gc.collect()

yreg0_t, yreg1_t = torch.tensor(yreg0, dtype=torch.float32), torch.tensor(yreg1, dtype=torch.float32)
ytraj0_t = torch.tensor(ytraj0, dtype=torch.float32)
ytraj1_t = torch.tensor(ytraj1, dtype=torch.float32)
del yreg0, yreg1
gc.collect()

POS_WEIGHT = min(float((ytraj0_t.numel() - ytraj0_t.sum()) / ytraj0_t.sum().clamp(min=1)), 50.0)
print(f"pos_weight (per-hour trajectory): {POS_WEIGHT:.2f}")

region_criterion = nn.BCEWithLogitsLoss(pos_weight=torch.tensor(POS_WEIGHT))

def run_p1_epoch(model, X, y, optimizer, train):
    n = X.shape[0]
    idx = torch.randperm(n) if train else torch.arange(n)
    model.train(train)
    total_loss, total_n = 0.0, 0
    eff_batch = MICRO_BATCH * ACCUM_STEPS
    for start in range(0, n, eff_batch):
        if train: optimizer.zero_grad()
        batch_idx = idx[start:start + eff_batch]
        for ms in range(0, len(batch_idx), MICRO_BATCH):
            mb_idx = batch_idx[ms:ms + MICRO_BATCH]
            if len(mb_idx) == 0: continue
            xb = add_static_fn(X[mb_idx].to(DEVICE), static_tensor)
            yb = y[mb_idx].to(DEVICE)
            with torch.set_grad_enabled(train):
                pred = model(xb)
                loss = pinball_loss(pred, yb, QUANTILES)
            if train: (loss * len(mb_idx) / len(batch_idx)).backward()
            total_loss += loss.item() * len(mb_idx); total_n += len(mb_idx)
        if train: optimizer.step()
    return total_loss / total_n

def run_p2_epoch(model, X, y_traj, optimizer, train):
    n = X.shape[0]
    idx = torch.randperm(n) if train else torch.arange(n)
    model.train(train)
    total_loss, total_n = 0.0, 0
    eff_batch = MICRO_BATCH * ACCUM_STEPS
    for start in range(0, n, eff_batch):
        if train: optimizer.zero_grad()
        batch_idx = idx[start:start + eff_batch]
        for ms in range(0, len(batch_idx), MICRO_BATCH):
            mb_idx = batch_idx[ms:ms + MICRO_BATCH]
            if len(mb_idx) == 0: continue
            xb = add_static_fn(X[mb_idx].to(DEVICE), static_tensor)
            yb = y_traj[mb_idx].to(DEVICE)
            with torch.set_grad_enabled(train):
                logits = model(xb, n_regions)
                loss = region_criterion(logits, yb)
            if train: loss.backward()
            total_loss += loss.item() * len(mb_idx); total_n += len(mb_idx)
        if train: optimizer.step()
    return total_loss / max(total_n, 1)

def predict_traj_probs(model, X, batch_size=64):
    model.eval()
    n = X.shape[0]
    out = np.zeros((n, n_regions, HORIZON), dtype=np.float32)
    with torch.no_grad():
        for start in range(0, n, batch_size):
            idx = torch.arange(start, min(start + batch_size, n))
            xb = add_static_fn(X[idx].to(DEVICE), static_tensor)
            logits = model(xb, n_regions)
            out[idx.numpy()] = torch.sigmoid(logits).cpu().numpy()
    return out

def train_one_seed_minimal(seed, lr=3e-3, dropout=0.5, weight_decay=5e-4):
    torch.manual_seed(seed); np.random.seed(seed)
    p1 = StationGRU(n_feats, dropout).to(DEVICE)
    opt1 = torch.optim.Adam(p1.parameters(), lr=lr, weight_decay=weight_decay)
    best_p1_val, best_p1_state = float("inf"), None
    for epoch in range(1, MAX_EPOCHS_P1 + 1):
        run_p1_epoch(p1, X0_t, yreg0_t, opt1, True)
        vl = run_p1_epoch(p1, X1_t, yreg1_t, opt1, False)
        if vl < best_p1_val:
            best_p1_val, best_p1_state = vl, copy.deepcopy(p1.state_dict())
    p1.load_state_dict(best_p1_state)
    encoder_copy = copy.deepcopy(p1.station_lin)
    p2 = RegionFromTrajectoryGRU(encoder_copy, region_membership_t, dropout).to(DEVICE)
    del p1; gc.collect()
    if DEVICE.type == "mps": torch.mps.empty_cache()

    opt2 = torch.optim.Adam(p2.parameters(), lr=lr, weight_decay=weight_decay)
    y_val_agg = ytraj1_t.numpy().max(axis=-1).reshape(-1)
    best_val_aucpr, best_state = -1.0, None
    for epoch in range(1, MAX_EPOCHS_P2 + 1):
        run_p2_epoch(p2, X0_t, ytraj0_t, opt2, True)
        run_p2_epoch(p2, X1_t, ytraj1_t, opt2, False)
        val_probs = predict_traj_probs(p2, X1_t)
        va = average_precision_score(y_val_agg, val_probs.max(axis=-1).reshape(-1))
        if va > best_val_aucpr:
            best_val_aucpr, best_state = va, copy.deepcopy(p2.state_dict())
    p2.load_state_dict(best_state)
    p2.eval()
    return p2, best_val_aucpr

# ================================================================
# RUN: 5 NEW seeds (5-9)
# ================================================================
new_results = []
for seed in range(5, 10):
    print(f"\n========== seed={seed}  model=minimal_gru ==========")
    t0 = time.time()
    p2, val_aucpr = train_one_seed_minimal(seed)
    test_probs = predict_traj_probs(p2, X2_t)
    y_test_agg = ytraj2.max(axis=-1).reshape(-1)
    test_agg_score = test_probs.max(axis=-1).reshape(-1)
    test_aucroc = roc_auc_score(y_test_agg, test_agg_score)
    test_aucpr = average_precision_score(y_test_agg, test_agg_score)
    print(f"  trained in {time.time()-t0:.0f}s  val_AUCPR={val_aucpr:.4f}  test_AUCROC={test_aucroc:.4f}  test_AUCPR={test_aucpr:.4f}")
    new_results.append({"seed": seed, "val_aucpr": float(val_aucpr),
                         "test_aucroc": float(test_aucroc), "test_aucpr": float(test_aucpr)})
    del p2; gc.collect()
    if DEVICE.type == "mps": torch.mps.empty_cache()

# ================================================================
# COMBINE: 5 existing (0-4) + 5 new (5-9) = 10 total minimal_gru seeds
# Compare to the full 10-seed wind_graph and no_graph results.
# ================================================================
all_minimal_gru = existing_minimal_gru_results + new_results
wind_10 = np.array([r["test_aucpr"] for r in prior_10seed["wind_graph"]])
nograph_10 = np.array([r["test_aucpr"] for r in prior_10seed["no_graph"]])
minimal_10 = np.array([r["test_aucpr"] for r in all_minimal_gru])
wind_10_roc = np.array([r["test_aucroc"] for r in prior_10seed["wind_graph"]])
nograph_10_roc = np.array([r["test_aucroc"] for r in prior_10seed["no_graph"]])
minimal_10_roc = np.array([r["test_aucroc"] for r in all_minimal_gru])

print(f"\n{'='*20} FULL 10-SEED COMPARISON: wind_graph vs no_graph vs minimal_gru {'='*20}")
print(f"wind_graph   AUC-PR:  mean={wind_10.mean():.4f}  std={wind_10.std():.4f}")
print(f"no_graph     AUC-PR:  mean={nograph_10.mean():.4f}  std={nograph_10.std():.4f}")
print(f"minimal_gru  AUC-PR:  mean={minimal_10.mean():.4f}  std={minimal_10.std():.4f}  values={minimal_10.round(4)}")
print(f"wind_graph   AUC-ROC: mean={wind_10_roc.mean():.4f}  std={wind_10_roc.std():.4f}")
print(f"no_graph     AUC-ROC: mean={nograph_10_roc.mean():.4f}  std={nograph_10_roc.std():.4f}")
print(f"minimal_gru  AUC-ROC: mean={minimal_10_roc.mean():.4f}  std={minimal_10_roc.std():.4f}  values={minimal_10_roc.round(4)}")

t_stat, p_ttest = stats.ttest_rel(wind_10, minimal_10)
w_stat, p_wilcoxon = stats.wilcoxon(wind_10, minimal_10)
n_wins = int((wind_10 > minimal_10).sum())
print(f"\nwind_graph vs minimal_gru (10 seeds): paired t-test t={t_stat:.3f} p={p_ttest:.6f}  |  Wilcoxon p={p_wilcoxon:.4f}  |  wind_graph wins {n_wins}/10")

with open(f"{BASE}/three_way_10seed_comparison.json", "w") as f:
    json.dump({
        "wind_graph": prior_10seed["wind_graph"], "no_graph": prior_10seed["no_graph"], "minimal_gru": all_minimal_gru,
        "minimal_gru_vs_wind_ttest": {"t": float(t_stat), "p": float(p_ttest)},
        "minimal_gru_vs_wind_wilcoxon": {"stat": float(w_stat), "p": float(p_wilcoxon)},
    }, f, indent=2)
print(f"\nsaved to {BASE}/three_way_10seed_comparison.json")


loaded 5 existing minimal_gru results (seeds 0-4)
device=mps
train hours (2016-2017): 17544  val hours (2018): 8760  test hours (2019): 8760
windows: train=17473 val=8689 test=8689
pos_weight (per-hour trajectory): 50.00

========== seed=5  model=minimal_gru ==========
  trained in 110s  val_AUCPR=0.4531  test_AUCROC=0.9456  test_AUCPR=0.6677

========== seed=6  model=minimal_gru ==========
  trained in 107s  val_AUCPR=0.4561  test_AUCROC=0.9495  test_AUCPR=0.6787

========== seed=7  model=minimal_gru ==========
  trained in 107s  val_AUCPR=0.4501  test_AUCROC=0.9487  test_AUCPR=0.6737

========== seed=8  model=minimal_gru ==========
  trained in 107s  val_AUCPR=0.4561  test_AUCROC=0.9479  test_AUCPR=0.6722

========== seed=9  model=minimal_gru ==========
  trained in 106s  val_AUCPR=0.4574  test_AUCROC=0.9455  test_AUCPR=0.6792

==================== FULL 10-SEED COMPARISON: wind_graph vs no_graph vs minimal_gru ====================
wind_graph   AUC-PR:  mean=0.7113  std=0.0080
no_grap

In [1]:
# ================================================================
# REGULARIZATION EXPERIMENT: does stronger dropout/weight_decay fix
# Phase 2's fast overfitting (found to peak at epoch 1 in the earlier
# early-stopping diagnostic)? Tests 3 configs, each with extended
# training + real patience-based early stopping, single seed=0,
# same train=2016-2017/val=2018/test=2019 split.
# ================================================================
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import os, time, copy, gc, json
from sklearn.metrics import average_precision_score, roc_auc_score

BASE = "/Users/drewbaldwin/PM2_5 Research"
df = pd.read_pickle(f"{BASE}/air_korea_final_imputed_with_blh.pkl")
station_order = sorted(df["Station_ID"].unique())
n_stations = len(station_order)

stations = df[["Station_ID", "lat", "lon"]].drop_duplicates("Station_ID").set_index("Station_ID").loc[station_order]
lats, lons = stations["lat"].to_numpy(), stations["lon"].to_numpy()

def haversine_km(lat1, lon1, lat2, lon2):
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
    dlat, dlon = lat2 - lat1, lon2 - lon1
    a = np.sin(dlat/2)**2 + np.cos(lat1)*np.cos(lat2)*np.sin(dlon/2)**2
    return 2 * 6371.0 * np.arcsin(np.sqrt(a))

def bearing_matrix(lat, lon):
    lat_r, lon_r = np.radians(lat), np.radians(lon)
    lat1, lat2 = lat_r[:, None], lat_r[None, :]
    dlon = lon_r[None, :] - lon_r[:, None]
    x = np.sin(dlon) * np.cos(lat2)
    y = np.cos(lat1) * np.sin(lat2) - np.sin(lat1) * np.cos(lat2) * np.cos(dlon)
    return (np.degrees(np.arctan2(x, y)) + 360) % 360

dist_km = haversine_km(lats[:, None], lons[:, None], lats[None, :], lons[None, :])
DIST_CUTOFF, RHO_KM = 250.0, 250.0
HYBRID_THRESHOLD_KM = 20.0
dist_edges = (dist_km <= DIST_CUTOFF) & (dist_km > 0)
bearing_from = bearing_matrix(lats, lons)

src_idx, dst_idx = np.nonzero(dist_edges)
edge_index_np = np.stack([src_idx, dst_idx])
decay_edge = np.exp(-dist_km[src_idx, dst_idx] / RHO_KM).astype(np.float32)
bearing_edge = bearing_from[src_idx, dst_idx].astype(np.float32)
close_edge_mask = (dist_km[src_idx, dst_idx] <= HYBRID_THRESHOLD_KM)

REGION_CENTROIDS = {
    "Seoul": (37.566, 126.978), "Busan": (35.180, 129.075), "Daegu": (35.872, 128.602),
    "Incheon": (37.483, 126.633), "Gwangju": (35.155, 126.916), "Daejeon": (36.350, 127.385),
    "Ulsan": (35.550, 129.317), "Sejong": (36.487, 127.282), "Gyeonggi": (37.500, 127.250),
    "Gangwon": (37.867, 127.733), "Chungbuk": (36.633, 127.483), "Chungnam": (36.500, 126.750),
    "Jeonbuk": (35.824, 127.148), "Jeonnam": (34.750, 127.000), "Gyeongbuk": (36.559, 128.729),
    "Gyeongnam": (35.271, 128.663), "Jeju": (33.513, 126.523),
}
region_names = list(REGION_CENTROIDS.keys())
n_regions = len(region_names)
region_lats = np.array([REGION_CENTROIDS[r][0] for r in region_names])
region_lons = np.array([REGION_CENTROIDS[r][1] for r in region_names])
dist_to_region = haversine_km(lats[:, None], lons[:, None], region_lats[None, :], region_lons[None, :])
station_region_idx = dist_to_region.argmin(axis=1)

region_membership = np.zeros((n_stations, n_regions), dtype=np.float32)
region_membership[np.arange(n_stations), station_region_idx] = 1.0
region_membership_t = torch.tensor(region_membership)

region_dist_km = haversine_km(region_lats[:, None], region_lons[:, None], region_lats[None, :], region_lons[None, :])
region_bearing = bearing_matrix(region_lats, region_lons)
r_src_idx, r_dst_idx = np.nonzero(~np.eye(n_regions, dtype=bool))
region_edge_index_np = np.stack([r_src_idx, r_dst_idx])
region_decay_edge = np.exp(-region_dist_km[r_src_idx, r_dst_idx] / RHO_KM).astype(np.float32)
region_bearing_edge = region_bearing[r_src_idx, r_dst_idx].astype(np.float32)

WINDOW, HORIZON = 36, 36
GRAPH_RECENT_HOURS = 18
EVENT_THRESHOLD = 75.0
SUSTAIN_HOURS = 2
QUANTILES = [0.50, 0.75, 0.90, 0.95, 0.99]
N_QUANTILES = len(QUANTILES)
TIME_FEATS = ["SO2", "CO", "NO2", "O3", "PM10", "PM25"]
STATIC_COLS = ["elevation_m", "urban_landuse_area_m2_3km", "green_space_area_3km",
               "building_footprint_area_3km", "railway_length_3km", "dist_to_coast_km",
               "dist_to_major_road_km", "industrial_area_m2_3km", "traffic_points_count_3km",
               "major_roads_count_3km", "total_road_length_3km"]

time_panels = {c: df.pivot(index="Datetime", columns="Station_ID", values=c)[station_order] for c in TIME_FEATS}
dt_index = time_panels["PM25"].index
n_time = len(dt_index)
years = dt_index.year.to_numpy()
doy = dt_index.dayofyear.to_numpy().astype(float)

split_id_per_hour = np.where((years == 2016) | (years == 2017), 0, np.where(years == 2018, 1, np.where(years == 2019, 2, -1)))
TRAIN_MASK = split_id_per_hour == 0

wind_dir_arr = df.pivot(index="Datetime", columns="Station_ID", values="winddirection_10m")[station_order].reindex(dt_index).to_numpy().astype(float)
wind_speed_arr = df.pivot(index="Datetime", columns="Station_ID", values="windspeed_10m")[station_order].reindex(dt_index).to_numpy().astype(float)
blh_arr = df.pivot(index="Datetime", columns="Station_ID", values="boundary_layer_height")[station_order].reindex(dt_index).to_numpy().astype(float)
pm25_raw_arr = time_panels["PM25"].to_numpy()

def regional_flat_mean(arr):
    out = np.zeros((n_time, n_regions), dtype=np.float32)
    for r in range(n_regions):
        cols = station_region_idx == r
        out[:, r] = np.nanmean(arr[:, cols], axis=1)
    return out

region_pm25 = regional_flat_mean(pm25_raw_arr)
wdir_sin_station = np.sin(np.radians(wind_dir_arr))
wdir_cos_station = np.cos(np.radians(wind_dir_arr))
region_windspeed = regional_flat_mean(wind_speed_arr)
region_wdir_sin = regional_flat_mean(wdir_sin_station)
region_wdir_cos = regional_flat_mean(wdir_cos_station)
region_wind_dir_deg = (np.degrees(np.arctan2(region_wdir_sin, region_wdir_cos)) + 360) % 360
region_wind_blows_toward = (region_wind_dir_deg + 180) % 360

season_sin_1d = np.sin(2 * np.pi * doy / 365.25)
season_cos_1d = np.cos(2 * np.pi * doy / 365.25)
season_sin = np.tile(season_sin_1d[:, None], (1, n_stations))
season_cos = np.tile(season_cos_1d[:, None], (1, n_stations))

TIME_FEATS_FULL = TIME_FEATS + ["windspeed_10m", "wdir_sin", "wdir_cos", "boundary_layer_height", "season_sin", "season_cos"]
n_time_feats, n_static_feats = len(TIME_FEATS_FULL), len(STATIC_COLS)
n_feats = n_time_feats + n_static_feats
pm25_col_idx = TIME_FEATS_FULL.index("PM25")

time_arr_raw = np.stack([time_panels[c].to_numpy() for c in TIME_FEATS] +
                         [wind_speed_arr, wdir_sin_station, wdir_cos_station, blh_arr, season_sin, season_cos], axis=-1)
del time_panels, season_sin, season_cos, blh_arr, pm25_raw_arr
gc.collect()

static_df = df[["Station_ID"] + STATIC_COLS].drop_duplicates("Station_ID").set_index("Station_ID").loc[station_order]
static_arr = static_df[STATIC_COLS].to_numpy()
del df
gc.collect()

rev = region_pm25[::-1]
roll_min_rev = pd.DataFrame(rev).rolling(window=SUSTAIN_HOURS, min_periods=SUSTAIN_HOURS).min().to_numpy()
region_episode_label = (roll_min_rev[::-1] >= EVENT_THRESHOLD).astype(np.float32)
del rev, roll_min_rev
gc.collect()

def pinball_loss(preds, target, quantiles):
    target_exp = target.unsqueeze(-1)
    diff = target_exp - preds
    q_tensor = torch.tensor(quantiles, device=preds.device, dtype=preds.dtype).view(*([1] * (preds.dim() - 1)), -1)
    return torch.max(q_tensor * diff, (q_tensor - 1) * diff).mean()

def monotonic_quantiles(raw):
    first = raw[..., :1]
    deltas = F.softplus(raw[..., 1:])
    return torch.cat([first, first + torch.cumsum(deltas, dim=-1)], dim=-1)

class WindConvLayer(nn.Module):
    def __init__(self, in_dim, out_dim):
        super().__init__()
        self.lin_self = nn.Linear(in_dim, out_dim)
        self.lin_neigh = nn.Linear(in_dim, out_dim)
        self.lin_connectivity = nn.Linear(1, out_dim)

    def forward(self, x, edge_index, edge_weight, num_nodes):
        src, dst = edge_index[0], edge_index[1]
        messages = x[src] * edge_weight.unsqueeze(-1)
        agg_sum = x.new_zeros(num_nodes, x.size(-1))
        agg_sum.index_add_(0, dst, messages)
        weight_sum = x.new_zeros(num_nodes)
        weight_sum.index_add_(0, dst, edge_weight)
        agg_mean = agg_sum / (weight_sum.unsqueeze(-1) + 1e-8)
        connectivity = torch.log1p(weight_sum.clamp(min=0)).unsqueeze(-1)
        return self.lin_self(x) + self.lin_neigh(agg_mean) + self.lin_connectivity(connectivity)

class AttentionPool(nn.Module):
    def __init__(self, hidden):
        super().__init__()
        self.attn_score = nn.Linear(hidden, 1)

    def forward(self, h_station, region_membership_t_local):
        B, N, H = h_station.shape
        scores = self.attn_score(h_station).squeeze(-1)
        scores = scores - scores.max(dim=1, keepdim=True).values
        exp_scores = torch.exp(scores)
        weighted_exp = exp_scores.unsqueeze(-1) * region_membership_t_local.unsqueeze(0)
        region_denom = weighted_exp.sum(dim=1)
        region_numer = torch.einsum('bnr,bnh->brh', weighted_exp, h_station)
        return region_numer / (region_denom.unsqueeze(-1) + 1e-8)

class StationQuantileGCN(nn.Module):
    def __init__(self, in_dim, dropout, hidden=32, gru_hidden=32):
        super().__init__()
        self.station_conv = WindConvLayer(in_dim, hidden)
        self.drop = nn.Dropout(dropout)
        self.gru = nn.GRU(hidden, gru_hidden, batch_first=True)
        self.head = nn.Linear(gru_hidden, N_QUANTILES)

    def forward(self, x_window, edge_index, edge_weight_seq):
        B, W, N, Fin = x_window.shape
        ei_b = torch.cat([edge_index + i * N for i in range(B)], dim=1)
        num_nodes = B * N
        h_seq = []
        for w in range(W):
            xt = x_window[:, w].reshape(B * N, Fin)
            ew_b = edge_weight_seq[:, w].reshape(-1)
            h = torch.relu(self.station_conv(xt, ei_b, ew_b, num_nodes))
            h = self.drop(h)
            h_seq.append(h.reshape(B, N, -1))
        h_seq = torch.stack(h_seq, dim=1).permute(0, 2, 1, 3).reshape(B * N, W, -1)
        _, h_final = self.gru(h_seq)
        embed = self.drop(h_final.squeeze(0).reshape(B, N, -1))
        return monotonic_quantiles(self.head(embed))

class RegionFromTrajectoryGCN(nn.Module):
    def __init__(self, station_conv, region_membership_t_local, dropout, hidden=32, gru_hidden=32):
        super().__init__()
        self.station_conv = station_conv
        self.attn_pool = AttentionPool(hidden)
        self.region_conv = WindConvLayer(hidden, hidden)
        self.drop = nn.Dropout(dropout)
        self.region_gru = nn.GRU(hidden, gru_hidden, batch_first=True)
        self.region_head = nn.Linear(gru_hidden, HORIZON)
        self.rmem = region_membership_t_local

    def forward(self, x_window, station_edge_index, station_edge_weight_seq, region_edge_index, region_edge_weight_seq, n_reg):
        B, W, N, Fin = x_window.shape
        station_ei_b = torch.cat([station_edge_index + i * N for i in range(B)], dim=1)
        region_ei_b = torch.cat([region_edge_index + i * n_reg for i in range(B)], dim=1)
        num_station_nodes = B * N
        num_region_nodes = B * n_reg
        h_region_seq = []
        for w in range(W):
            xt = x_window[:, w].reshape(B * N, Fin)
            ew_station_b = station_edge_weight_seq[:, w].reshape(-1)
            h_station = torch.relu(self.station_conv(xt, station_ei_b, ew_station_b, num_station_nodes))
            h_station = self.drop(h_station).reshape(B, N, -1)
            h_region_pooled = self.attn_pool(h_station, self.rmem).reshape(B * n_reg, -1)
            ew_region_b = region_edge_weight_seq[:, w].reshape(-1)
            h_region = torch.relu(self.region_conv(h_region_pooled, region_ei_b, ew_region_b, num_region_nodes))
            h_region = self.drop(h_region)
            h_region_seq.append(h_region.reshape(B, n_reg, -1))
        h_region_seq = torch.stack(h_region_seq, dim=1).permute(0, 2, 1, 3).reshape(B * n_reg, W, -1)
        _, h_final = self.region_gru(h_region_seq)
        embed = self.drop(h_final.squeeze(0).reshape(B, n_reg, -1))
        return self.region_head(embed)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else ("mps" if torch.backends.mps.is_available() else "cpu"))
MICRO_BATCH, ACCUM_STEPS = 16, 4
print(f"device={DEVICE}")

region_membership_t = region_membership_t.to(DEVICE)
edge_index = torch.tensor(edge_index_np, dtype=torch.long).to(DEVICE)
region_edge_index = torch.tensor(region_edge_index_np, dtype=torch.long).to(DEVICE)

def add_static_fn(x_time_batch, static_tensor_local):
    B, W, N, _ = x_time_batch.shape
    static_b = static_tensor_local.unsqueeze(0).unsqueeze(0).expand(B, W, N, n_static_feats)
    return torch.cat([x_time_batch, static_b], dim=-1)

def gather_seq(ew_by_hour_t, starts_subset):
    idx = starts_subset.unsqueeze(1) + torch.arange(WINDOW).unsqueeze(0)
    ew = ew_by_hour_t[idx]
    ew = ew.clone()
    ew[:, :WINDOW - GRAPH_RECENT_HOURS, :] = 0.0
    return ew

print(f"train hours (2016-2017): {TRAIN_MASK.sum()}  val hours (2018): {(split_id_per_hour==1).sum()}  test hours (2019): {(split_id_per_hour==2).sum()}")

wind_blows_toward = (wind_dir_arr + 180) % 360
wbt_src = wind_blows_toward[:, src_idx]
cos_align = np.maximum(np.cos(np.radians(wbt_src - bearing_edge[None, :])), 0.0)
speed_src = wind_speed_arr[:, src_idx]
wind_component = (cos_align * speed_src).astype(np.float32)
del wbt_src, cos_align, speed_src
gc.collect()
ref_speed = np.float32(np.nanmean(wind_speed_arr[TRAIN_MASK]))
component = np.where(close_edge_mask[None, :], ref_speed, wind_component)
station_wind_raw = np.nan_to_num(decay_edge[None, :] * component, nan=0.0).astype(np.float32)
del component
gc.collect()

r_wbt_src = region_wind_blows_toward[:, r_src_idx]
r_cos_align = np.maximum(np.cos(np.radians(r_wbt_src - region_bearing_edge[None, :])), 0.0)
r_speed_src = region_windspeed[:, r_src_idx]
region_wind_raw = np.nan_to_num(region_decay_edge[None, :] * r_cos_align * r_speed_src, nan=0.0).astype(np.float32)
del r_wbt_src, r_cos_align, r_speed_src
gc.collect()

train_nonzero = station_wind_raw[TRAIN_MASK][station_wind_raw[TRAIN_MASK] > 0]
wind_scale_s = train_nonzero.std()
station_wind_edge_weight = (station_wind_raw / wind_scale_s).astype(np.float32)
region_train_nonzero = region_wind_raw[TRAIN_MASK][region_wind_raw[TRAIN_MASK] > 0]
wind_scale_r = region_train_nonzero.std()
region_wind_edge_weight = (region_wind_raw / wind_scale_r).astype(np.float32)
del train_nonzero, region_train_nonzero, station_wind_raw, region_wind_raw
gc.collect()

t_mean = np.nanmean(time_arr_raw[TRAIN_MASK], axis=(0, 1), keepdims=True)
t_std = np.nanstd(time_arr_raw[TRAIN_MASK], axis=(0, 1), keepdims=True) + 1e-6
time_arr_std = np.nan_to_num((time_arr_raw - t_mean) / t_std, nan=0.0)
s_mean, s_std = static_arr.mean(axis=0, keepdims=True), static_arr.std(axis=0, keepdims=True) + 1e-6
static_tensor = torch.tensor((static_arr - s_mean) / s_std, dtype=torch.float32).to(DEVICE)

p_buckets = {0: ([], [], [], []), 1: ([], [], [], []), 2: ([], [], [], [])}
for t in range(0, n_time - WINDOW - HORIZON + 1):
    target_t = t + WINDOW + HORIZON - 1
    s_start, s_target = split_id_per_hour[t], split_id_per_hour[target_t]
    if s_start != s_target or s_start == -1:
        continue
    x_win = time_arr_std[t:t + WINDOW]
    traj = region_episode_label[t + WINDOW: t + WINDOW + HORIZON]
    Xl, yregl, ytrajl, sl = p_buckets[s_start]
    Xl.append(x_win)
    yregl.append(time_arr_std[target_t, :, pm25_col_idx])
    ytrajl.append(traj.T)
    sl.append(t)

X0, yreg0, ytraj0, starts0 = (np.stack(v) for v in p_buckets[0])
X1, yreg1, ytraj1, starts1 = (np.stack(v) for v in p_buckets[1])
X2, yreg2, ytraj2, starts2 = (np.stack(v) for v in p_buckets[2])
del p_buckets, time_arr_std
gc.collect()
print(f"windows: train={len(X0)} val={len(X1)} test={len(X2)}")

X0_t = torch.tensor(X0, dtype=torch.float32); del X0
X1_t = torch.tensor(X1, dtype=torch.float32); del X1
X2_t = torch.tensor(X2, dtype=torch.float32); del X2
gc.collect()

yreg0_t, yreg1_t = torch.tensor(yreg0, dtype=torch.float32), torch.tensor(yreg1, dtype=torch.float32)
ytraj0_t = torch.tensor(ytraj0, dtype=torch.float32)
ytraj1_t = torch.tensor(ytraj1, dtype=torch.float32)
starts0_t, starts1_t, starts2_t = (torch.tensor(a, dtype=torch.long) for a in (starts0, starts1, starts2))
del yreg0, yreg1
gc.collect()

POS_WEIGHT = min(float((ytraj0_t.numel() - ytraj0_t.sum()) / ytraj0_t.sum().clamp(min=1)), 50.0)
print(f"pos_weight (per-hour trajectory): {POS_WEIGHT:.2f}")

wind_ewt_s = torch.tensor(station_wind_edge_weight)
wind_ewt_r = torch.tensor(region_wind_edge_weight)

region_criterion = nn.BCEWithLogitsLoss(pos_weight=torch.tensor(POS_WEIGHT))

def run_p1_epoch(model, ewt, X, y, starts, optimizer, train):
    n = X.shape[0]
    idx = torch.randperm(n) if train else torch.arange(n)
    model.train(train)
    total_loss, total_n = 0.0, 0
    eff_batch = MICRO_BATCH * ACCUM_STEPS
    for start in range(0, n, eff_batch):
        if train: optimizer.zero_grad()
        batch_idx = idx[start:start + eff_batch]
        for ms in range(0, len(batch_idx), MICRO_BATCH):
            mb_idx = batch_idx[ms:ms + MICRO_BATCH]
            if len(mb_idx) == 0: continue
            xb = add_static_fn(X[mb_idx].to(DEVICE), static_tensor)
            yb = y[mb_idx].to(DEVICE)
            ew_seq = gather_seq(ewt, starts[mb_idx]).to(DEVICE)
            with torch.set_grad_enabled(train):
                pred = model(xb, edge_index, ew_seq)
                loss = pinball_loss(pred, yb, QUANTILES)
            if train: (loss * len(mb_idx) / len(batch_idx)).backward()
            total_loss += loss.item() * len(mb_idx); total_n += len(mb_idx)
        if train: optimizer.step()
    return total_loss / total_n

def run_p2_epoch(model, ewt_s, ewt_r, X, y_traj, starts, optimizer, train):
    n = X.shape[0]
    idx = torch.randperm(n) if train else torch.arange(n)
    model.train(train)
    total_loss, total_n = 0.0, 0
    eff_batch = MICRO_BATCH * ACCUM_STEPS
    for start in range(0, n, eff_batch):
        if train: optimizer.zero_grad()
        batch_idx = idx[start:start + eff_batch]
        for ms in range(0, len(batch_idx), MICRO_BATCH):
            mb_idx = batch_idx[ms:ms + MICRO_BATCH]
            if len(mb_idx) == 0: continue
            xb = add_static_fn(X[mb_idx].to(DEVICE), static_tensor)
            yb = y_traj[mb_idx].to(DEVICE)
            ew_s = gather_seq(ewt_s, starts[mb_idx]).to(DEVICE)
            ew_r = gather_seq(ewt_r, starts[mb_idx]).to(DEVICE)
            with torch.set_grad_enabled(train):
                logits = model(xb, edge_index, ew_s, region_edge_index, ew_r, n_regions)
                loss = region_criterion(logits, yb)
            if train: loss.backward()
            total_loss += loss.item() * len(mb_idx); total_n += len(mb_idx)
        if train: optimizer.step()
    return total_loss / max(total_n, 1)

def predict_traj_probs(model, ewt_s, ewt_r, X, starts, batch_size=64):
    model.eval()
    n = X.shape[0]
    out = np.zeros((n, n_regions, HORIZON), dtype=np.float32)
    with torch.no_grad():
        for start in range(0, n, batch_size):
            idx = torch.arange(start, min(start + batch_size, n))
            xb = add_static_fn(X[idx].to(DEVICE), static_tensor)
            ew_s = gather_seq(ewt_s, starts[idx]).to(DEVICE)
            ew_r = gather_seq(ewt_r, starts[idx]).to(DEVICE)
            logits = model(xb, edge_index, ew_s, region_edge_index, ew_r, n_regions)
            out[idx.numpy()] = torch.sigmoid(logits).cpu().numpy()
    return out

def run_regularization_config(cfg, seed=0, max_epochs=25, patience=4):
    lr, dropout, weight_decay = cfg["lr"], cfg["dropout"], cfg["weight_decay"]
    torch.manual_seed(seed); np.random.seed(seed)

    p1 = StationQuantileGCN(n_feats, dropout).to(DEVICE)
    opt1 = torch.optim.Adam(p1.parameters(), lr=lr, weight_decay=weight_decay)
    best_p1_val, best_p1_state, bad_epochs, p1_history = float("inf"), None, 0, []
    for epoch in range(1, max_epochs + 1):
        train_loss = run_p1_epoch(p1, wind_ewt_s, X0_t, yreg0_t, starts0_t, opt1, True)
        val_loss = run_p1_epoch(p1, wind_ewt_s, X1_t, yreg1_t, starts1_t, opt1, False)
        improved = val_loss < best_p1_val
        if improved:
            best_p1_val, best_p1_state, bad_epochs = val_loss, copy.deepcopy(p1.state_dict()), 0
        else:
            bad_epochs += 1
        p1_history.append({"epoch": epoch, "val_loss": float(val_loss)})
        if bad_epochs >= patience:
            break
    p1_best_epoch = len(p1_history) - bad_epochs
    p1.load_state_dict(best_p1_state)
    encoder_copy = copy.deepcopy(p1.station_conv)
    del p1; gc.collect()
    if DEVICE.type == "mps": torch.mps.empty_cache()

    p2 = RegionFromTrajectoryGCN(encoder_copy, region_membership_t, dropout).to(DEVICE)
    opt2 = torch.optim.Adam(p2.parameters(), lr=lr, weight_decay=weight_decay)
    y_val_agg = ytraj1_t.numpy().max(axis=-1).reshape(-1)
    best_val_aucpr, best_state, bad_epochs, p2_history = -1.0, None, 0, []
    for epoch in range(1, max_epochs + 1):
        run_p2_epoch(p2, wind_ewt_s, wind_ewt_r, X0_t, ytraj0_t, starts0_t, opt2, True)
        run_p2_epoch(p2, wind_ewt_s, wind_ewt_r, X1_t, ytraj1_t, starts1_t, opt2, False)
        val_probs = predict_traj_probs(p2, wind_ewt_s, wind_ewt_r, X1_t, starts1_t)
        val_aucpr = average_precision_score(y_val_agg, val_probs.max(axis=-1).reshape(-1))
        improved = val_aucpr > best_val_aucpr
        if improved:
            best_val_aucpr, best_state, bad_epochs = val_aucpr, copy.deepcopy(p2.state_dict()), 0
        else:
            bad_epochs += 1
        p2_history.append({"epoch": epoch, "val_aucpr": float(val_aucpr)})
        if bad_epochs >= patience:
            break
    p2_best_epoch = len(p2_history) - bad_epochs
    p2.load_state_dict(best_state)
    p2.eval()

    test_probs = predict_traj_probs(p2, wind_ewt_s, wind_ewt_r, X2_t, starts2_t)
    y_test_agg = ytraj2.max(axis=-1).reshape(-1)
    test_agg_score = test_probs.max(axis=-1).reshape(-1)
    test_aucroc = roc_auc_score(y_test_agg, test_agg_score)
    test_aucpr = average_precision_score(y_test_agg, test_agg_score)

    del p2; gc.collect()
    if DEVICE.type == "mps": torch.mps.empty_cache()

    return {
        "config": cfg, "p1_best_epoch": p1_best_epoch, "p1_epochs_run": len(p1_history),
        "p2_best_epoch": p2_best_epoch, "p2_epochs_run": len(p2_history),
        "test_aucroc": float(test_aucroc), "test_aucpr": float(test_aucpr),
    }

# ================================================================
# RUN: baseline vs higher-dropout vs higher-weight_decay, single seed=0
# ================================================================
CONFIGS = [
    {"label": "baseline (dropout=0.5, wd=0.0005)", "lr": 0.003, "dropout": 0.5, "weight_decay": 0.0005},
    {"label": "higher dropout (0.65, wd=0.0005)", "lr": 0.003, "dropout": 0.65, "weight_decay": 0.0005},
    {"label": "higher weight_decay (dropout=0.5, wd=0.002)", "lr": 0.003, "dropout": 0.5, "weight_decay": 0.002},
]

results = []
for cfg in CONFIGS:
    print(f"\n{'='*20} {cfg['label']} {'='*20}")
    t0 = time.time()
    r = run_regularization_config(cfg, seed=0)
    print(f"  phase1 best epoch: {r['p1_best_epoch']}/{r['p1_epochs_run']} run")
    print(f"  phase2 best epoch: {r['p2_best_epoch']}/{r['p2_epochs_run']} run")
    print(f"  test AUC-ROC={r['test_aucroc']:.4f}  AUC-PR={r['test_aucpr']:.4f}  ({time.time()-t0:.0f}s)")
    results.append(r)

print(f"\n{'='*20} SUMMARY {'='*20}")
print(f"(compare to 2-epoch headline: AUC-ROC=0.9594  AUC-PR=0.7124)")
for r in results:
    print(f"{r['config']['label']:<45} phase2_best_epoch={r['p2_best_epoch']:<3} test_AUC-ROC={r['test_aucroc']:.4f}  test_AUC-PR={r['test_aucpr']:.4f}")

with open(f"{BASE}/regularization_experiment.json", "w") as f:
    json.dump(results, f, indent=2)
print(f"\nsaved to {BASE}/regularization_experiment.json")


device=mps
train hours (2016-2017): 17544  val hours (2018): 8760  test hours (2019): 8760
windows: train=17473 val=8689 test=8689
pos_weight (per-hour trajectory): 50.00

==================== baseline (dropout=0.5, wd=0.0005) ====================
  phase1 best epoch: 4/8 run
  phase2 best epoch: 1/5 run
  test AUC-ROC=0.9597  AUC-PR=0.7144  (1969s)

==================== higher dropout (0.65, wd=0.0005) ====================
  phase1 best epoch: 4/8 run
  phase2 best epoch: 1/5 run
  test AUC-ROC=0.9590  AUC-PR=0.6851  (1964s)

==================== higher weight_decay (dropout=0.5, wd=0.002) ====================
  phase1 best epoch: 4/8 run
  phase2 best epoch: 1/5 run
  test AUC-ROC=0.9584  AUC-PR=0.7031  (1950s)

==================== SUMMARY ====================
(compare to 2-epoch headline: AUC-ROC=0.9594  AUC-PR=0.7124)
baseline (dropout=0.5, wd=0.0005)             phase2_best_epoch=1   test_AUC-ROC=0.9597  test_AUC-PR=0.7144
higher dropout (0.65, wd=0.0005)              phase2_best

In [1]:
# ================================================================
# TRAIN=2016-2019, VAL=2020, TEST=2021 -- matches TransNet's evaluation
# regime (train pre-COVID, val/test across the confirmed COVID-era
# distributional shift), for direct comparability.
# wind_graph, no_graph, minimal_gru -- each independently hyperparameter-
# tuned (search on train=2016-2019/val=2020 only, test=2021 never touched),
# then 10 final seeds each with its own best config.
# ================================================================
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import os, time, copy, gc, json
from sklearn.metrics import average_precision_score, roc_auc_score
from scipy import stats

BASE = "/Users/drewbaldwin/PM2_5 Research"
df = pd.read_pickle(f"{BASE}/air_korea_final_imputed_with_blh.pkl")
station_order = sorted(df["Station_ID"].unique())
n_stations = len(station_order)

stations = df[["Station_ID", "lat", "lon"]].drop_duplicates("Station_ID").set_index("Station_ID").loc[station_order]
lats, lons = stations["lat"].to_numpy(), stations["lon"].to_numpy()

def haversine_km(lat1, lon1, lat2, lon2):
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
    dlat, dlon = lat2 - lat1, lon2 - lon1
    a = np.sin(dlat/2)**2 + np.cos(lat1)*np.cos(lat2)*np.sin(dlon/2)**2
    return 2 * 6371.0 * np.arcsin(np.sqrt(a))

def bearing_matrix(lat, lon):
    lat_r, lon_r = np.radians(lat), np.radians(lon)
    lat1, lat2 = lat_r[:, None], lat_r[None, :]
    dlon = lon_r[None, :] - lon_r[:, None]
    x = np.sin(dlon) * np.cos(lat2)
    y = np.cos(lat1) * np.sin(lat2) - np.sin(lat1) * np.cos(lat2) * np.cos(dlon)
    return (np.degrees(np.arctan2(x, y)) + 360) % 360

dist_km = haversine_km(lats[:, None], lons[:, None], lats[None, :], lons[None, :])
DIST_CUTOFF, RHO_KM = 250.0, 250.0
HYBRID_THRESHOLD_KM = 20.0
dist_edges = (dist_km <= DIST_CUTOFF) & (dist_km > 0)
bearing_from = bearing_matrix(lats, lons)

src_idx, dst_idx = np.nonzero(dist_edges)
edge_index_np = np.stack([src_idx, dst_idx])
decay_edge = np.exp(-dist_km[src_idx, dst_idx] / RHO_KM).astype(np.float32)
bearing_edge = bearing_from[src_idx, dst_idx].astype(np.float32)
close_edge_mask = (dist_km[src_idx, dst_idx] <= HYBRID_THRESHOLD_KM)

REGION_CENTROIDS = {
    "Seoul": (37.566, 126.978), "Busan": (35.180, 129.075), "Daegu": (35.872, 128.602),
    "Incheon": (37.483, 126.633), "Gwangju": (35.155, 126.916), "Daejeon": (36.350, 127.385),
    "Ulsan": (35.550, 129.317), "Sejong": (36.487, 127.282), "Gyeonggi": (37.500, 127.250),
    "Gangwon": (37.867, 127.733), "Chungbuk": (36.633, 127.483), "Chungnam": (36.500, 126.750),
    "Jeonbuk": (35.824, 127.148), "Jeonnam": (34.750, 127.000), "Gyeongbuk": (36.559, 128.729),
    "Gyeongnam": (35.271, 128.663), "Jeju": (33.513, 126.523),
}
region_names = list(REGION_CENTROIDS.keys())
n_regions = len(region_names)
region_lats = np.array([REGION_CENTROIDS[r][0] for r in region_names])
region_lons = np.array([REGION_CENTROIDS[r][1] for r in region_names])
dist_to_region = haversine_km(lats[:, None], lons[:, None], region_lats[None, :], region_lons[None, :])
station_region_idx = dist_to_region.argmin(axis=1)

region_membership = np.zeros((n_stations, n_regions), dtype=np.float32)
region_membership[np.arange(n_stations), station_region_idx] = 1.0
region_membership_t = torch.tensor(region_membership)

region_dist_km = haversine_km(region_lats[:, None], region_lons[:, None], region_lats[None, :], region_lons[None, :])
region_bearing = bearing_matrix(region_lats, region_lons)
r_src_idx, r_dst_idx = np.nonzero(~np.eye(n_regions, dtype=bool))
region_edge_index_np = np.stack([r_src_idx, r_dst_idx])
region_decay_edge = np.exp(-region_dist_km[r_src_idx, r_dst_idx] / RHO_KM).astype(np.float32)
region_bearing_edge = region_bearing[r_src_idx, r_dst_idx].astype(np.float32)

WINDOW, HORIZON = 36, 36
GRAPH_RECENT_HOURS = 18
EVENT_THRESHOLD = 75.0
SUSTAIN_HOURS = 2
QUANTILES = [0.50, 0.75, 0.90, 0.95, 0.99]
N_QUANTILES = len(QUANTILES)
TIME_FEATS = ["SO2", "CO", "NO2", "O3", "PM10", "PM25"]
STATIC_COLS = ["elevation_m", "urban_landuse_area_m2_3km", "green_space_area_3km",
               "building_footprint_area_3km", "railway_length_3km", "dist_to_coast_km",
               "dist_to_major_road_km", "industrial_area_m2_3km", "traffic_points_count_3km",
               "major_roads_count_3km", "total_road_length_3km"]

time_panels = {c: df.pivot(index="Datetime", columns="Station_ID", values=c)[station_order] for c in TIME_FEATS}
dt_index = time_panels["PM25"].index
n_time = len(dt_index)
years = dt_index.year.to_numpy()
doy = dt_index.dayofyear.to_numpy().astype(float)

# ---- SPLIT: train=2016-2019, val=2020, test=2021 (matches TransNet's regime) ----
split_id_per_hour = np.where(years.astype(int) <= 2019, 0, np.where(years == 2020, 1, np.where(years == 2021, 2, -1)))
TRAIN_MASK = split_id_per_hour == 0

wind_dir_arr = df.pivot(index="Datetime", columns="Station_ID", values="winddirection_10m")[station_order].reindex(dt_index).to_numpy().astype(float)
wind_speed_arr = df.pivot(index="Datetime", columns="Station_ID", values="windspeed_10m")[station_order].reindex(dt_index).to_numpy().astype(float)
blh_arr = df.pivot(index="Datetime", columns="Station_ID", values="boundary_layer_height")[station_order].reindex(dt_index).to_numpy().astype(float)
pm25_raw_arr = time_panels["PM25"].to_numpy()

def regional_flat_mean(arr):
    out = np.zeros((n_time, n_regions), dtype=np.float32)
    for r in range(n_regions):
        cols = station_region_idx == r
        out[:, r] = np.nanmean(arr[:, cols], axis=1)
    return out

region_pm25 = regional_flat_mean(pm25_raw_arr)
wdir_sin_station = np.sin(np.radians(wind_dir_arr))
wdir_cos_station = np.cos(np.radians(wind_dir_arr))
region_windspeed = regional_flat_mean(wind_speed_arr)
region_wdir_sin = regional_flat_mean(wdir_sin_station)
region_wdir_cos = regional_flat_mean(wdir_cos_station)
region_wind_dir_deg = (np.degrees(np.arctan2(region_wdir_sin, region_wdir_cos)) + 360) % 360
region_wind_blows_toward = (region_wind_dir_deg + 180) % 360

season_sin_1d = np.sin(2 * np.pi * doy / 365.25)
season_cos_1d = np.cos(2 * np.pi * doy / 365.25)
season_sin = np.tile(season_sin_1d[:, None], (1, n_stations))
season_cos = np.tile(season_cos_1d[:, None], (1, n_stations))

TIME_FEATS_FULL = TIME_FEATS + ["windspeed_10m", "wdir_sin", "wdir_cos", "boundary_layer_height", "season_sin", "season_cos"]
n_time_feats, n_static_feats = len(TIME_FEATS_FULL), len(STATIC_COLS)
n_feats = n_time_feats + n_static_feats
pm25_col_idx = TIME_FEATS_FULL.index("PM25")

time_arr_raw = np.stack([time_panels[c].to_numpy() for c in TIME_FEATS] +
                         [wind_speed_arr, wdir_sin_station, wdir_cos_station, blh_arr, season_sin, season_cos], axis=-1)
del time_panels, season_sin, season_cos, blh_arr
gc.collect()

static_df = df[["Station_ID"] + STATIC_COLS].drop_duplicates("Station_ID").set_index("Station_ID").loc[station_order]
static_arr = static_df[STATIC_COLS].to_numpy()
del df
gc.collect()

rev = region_pm25[::-1]
roll_min_rev = pd.DataFrame(rev).rolling(window=SUSTAIN_HOURS, min_periods=SUSTAIN_HOURS).min().to_numpy()
region_episode_label = (roll_min_rev[::-1] >= EVENT_THRESHOLD).astype(np.float32)
del rev, roll_min_rev, pm25_raw_arr
gc.collect()

def pinball_loss(preds, target, quantiles):
    target_exp = target.unsqueeze(-1)
    diff = target_exp - preds
    q_tensor = torch.tensor(quantiles, device=preds.device, dtype=preds.dtype).view(*([1] * (preds.dim() - 1)), -1)
    return torch.max(q_tensor * diff, (q_tensor - 1) * diff).mean()

def monotonic_quantiles(raw):
    first = raw[..., :1]
    deltas = F.softplus(raw[..., 1:])
    return torch.cat([first, first + torch.cumsum(deltas, dim=-1)], dim=-1)

class WindConvLayer(nn.Module):
    def __init__(self, in_dim, out_dim):
        super().__init__()
        self.lin_self = nn.Linear(in_dim, out_dim)
        self.lin_neigh = nn.Linear(in_dim, out_dim)
        self.lin_connectivity = nn.Linear(1, out_dim)

    def forward(self, x, edge_index, edge_weight, num_nodes):
        src, dst = edge_index[0], edge_index[1]
        messages = x[src] * edge_weight.unsqueeze(-1)
        agg_sum = x.new_zeros(num_nodes, x.size(-1))
        agg_sum.index_add_(0, dst, messages)
        weight_sum = x.new_zeros(num_nodes)
        weight_sum.index_add_(0, dst, edge_weight)
        agg_mean = agg_sum / (weight_sum.unsqueeze(-1) + 1e-8)
        connectivity = torch.log1p(weight_sum.clamp(min=0)).unsqueeze(-1)
        return self.lin_self(x) + self.lin_neigh(agg_mean) + self.lin_connectivity(connectivity)

class AttentionPool(nn.Module):
    def __init__(self, hidden):
        super().__init__()
        self.attn_score = nn.Linear(hidden, 1)

    def forward(self, h_station, region_membership_t_local):
        B, N, H = h_station.shape
        scores = self.attn_score(h_station).squeeze(-1)
        scores = scores - scores.max(dim=1, keepdim=True).values
        exp_scores = torch.exp(scores)
        weighted_exp = exp_scores.unsqueeze(-1) * region_membership_t_local.unsqueeze(0)
        region_denom = weighted_exp.sum(dim=1)
        region_numer = torch.einsum('bnr,bnh->brh', weighted_exp, h_station)
        return region_numer / (region_denom.unsqueeze(-1) + 1e-8)

class StationQuantileGCN(nn.Module):
    def __init__(self, in_dim, dropout, hidden=32, gru_hidden=32):
        super().__init__()
        self.station_conv = WindConvLayer(in_dim, hidden)
        self.drop = nn.Dropout(dropout)
        self.gru = nn.GRU(hidden, gru_hidden, batch_first=True)
        self.head = nn.Linear(gru_hidden, N_QUANTILES)

    def forward(self, x_window, edge_index, edge_weight_seq):
        B, W, N, Fin = x_window.shape
        ei_b = torch.cat([edge_index + i * N for i in range(B)], dim=1)
        num_nodes = B * N
        h_seq = []
        for w in range(W):
            xt = x_window[:, w].reshape(B * N, Fin)
            ew_b = edge_weight_seq[:, w].reshape(-1)
            h = torch.relu(self.station_conv(xt, ei_b, ew_b, num_nodes))
            h = self.drop(h)
            h_seq.append(h.reshape(B, N, -1))
        h_seq = torch.stack(h_seq, dim=1).permute(0, 2, 1, 3).reshape(B * N, W, -1)
        _, h_final = self.gru(h_seq)
        embed = self.drop(h_final.squeeze(0).reshape(B, N, -1))
        return monotonic_quantiles(self.head(embed))

class RegionFromTrajectoryGCN(nn.Module):
    def __init__(self, station_conv, region_membership_t_local, dropout, hidden=32, gru_hidden=32):
        super().__init__()
        self.station_conv = station_conv
        self.attn_pool = AttentionPool(hidden)
        self.region_conv = WindConvLayer(hidden, hidden)
        self.drop = nn.Dropout(dropout)
        self.region_gru = nn.GRU(hidden, gru_hidden, batch_first=True)
        self.region_head = nn.Linear(gru_hidden, HORIZON)
        self.rmem = region_membership_t_local

    def forward(self, x_window, station_edge_index, station_edge_weight_seq, region_edge_index, region_edge_weight_seq, n_reg):
        B, W, N, Fin = x_window.shape
        station_ei_b = torch.cat([station_edge_index + i * N for i in range(B)], dim=1)
        region_ei_b = torch.cat([region_edge_index + i * n_reg for i in range(B)], dim=1)
        num_station_nodes = B * N
        num_region_nodes = B * n_reg
        h_region_seq = []
        for w in range(W):
            xt = x_window[:, w].reshape(B * N, Fin)
            ew_station_b = station_edge_weight_seq[:, w].reshape(-1)
            h_station = torch.relu(self.station_conv(xt, station_ei_b, ew_station_b, num_station_nodes))
            h_station = self.drop(h_station).reshape(B, N, -1)
            h_region_pooled = self.attn_pool(h_station, self.rmem).reshape(B * n_reg, -1)
            ew_region_b = region_edge_weight_seq[:, w].reshape(-1)
            h_region = torch.relu(self.region_conv(h_region_pooled, region_ei_b, ew_region_b, num_region_nodes))
            h_region = self.drop(h_region)
            h_region_seq.append(h_region.reshape(B, n_reg, -1))
        h_region_seq = torch.stack(h_region_seq, dim=1).permute(0, 2, 1, 3).reshape(B * n_reg, W, -1)
        _, h_final = self.region_gru(h_region_seq)
        embed = self.drop(h_final.squeeze(0).reshape(B, n_reg, -1))
        return self.region_head(embed)

class StationGRU(nn.Module):
    def __init__(self, in_dim, dropout, hidden=32, gru_hidden=32):
        super().__init__()
        self.station_lin = nn.Linear(in_dim, hidden)
        self.drop = nn.Dropout(dropout)
        self.gru = nn.GRU(hidden, gru_hidden, batch_first=True)
        self.head = nn.Linear(gru_hidden, N_QUANTILES)

    def forward(self, x_window):
        B, W, N, Fin = x_window.shape
        h_seq = []
        for w in range(W):
            xt = x_window[:, w].reshape(B * N, Fin)
            h = torch.relu(self.station_lin(xt))
            h = self.drop(h)
            h_seq.append(h.reshape(B, N, -1))
        h_seq = torch.stack(h_seq, dim=1).permute(0, 2, 1, 3).reshape(B * N, W, -1)
        _, h_final = self.gru(h_seq)
        embed = self.drop(h_final.squeeze(0).reshape(B, N, -1))
        return monotonic_quantiles(self.head(embed))

class RegionFromTrajectoryGRU(nn.Module):
    def __init__(self, station_lin, region_membership_t_local, dropout, hidden=32, gru_hidden=32):
        super().__init__()
        self.station_lin = station_lin
        self.attn_pool = AttentionPool(hidden)
        self.drop = nn.Dropout(dropout)
        self.region_gru = nn.GRU(hidden, gru_hidden, batch_first=True)
        self.region_head = nn.Linear(gru_hidden, HORIZON)
        self.rmem = region_membership_t_local

    def forward(self, x_window, n_reg):
        B, W, N, Fin = x_window.shape
        h_region_seq = []
        for w in range(W):
            xt = x_window[:, w].reshape(B * N, Fin)
            h_station = torch.relu(self.station_lin(xt)).reshape(B, N, -1)
            h_station = self.drop(h_station)
            h_region_pooled = self.attn_pool(h_station, self.rmem)
            h_region_pooled = self.drop(h_region_pooled)
            h_region_seq.append(h_region_pooled)
        h_region_seq = torch.stack(h_region_seq, dim=1).permute(0, 2, 1, 3).reshape(B * n_reg, W, -1)
        _, h_final = self.region_gru(h_region_seq)
        embed = self.drop(h_final.squeeze(0).reshape(B, n_reg, -1))
        return self.region_head(embed)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else ("mps" if torch.backends.mps.is_available() else "cpu"))
MICRO_BATCH, ACCUM_STEPS = 16, 4
MAX_EPOCHS_P1, MAX_EPOCHS_P2 = 2, 2
print(f"device={DEVICE}")

region_membership_t = region_membership_t.to(DEVICE)
edge_index = torch.tensor(edge_index_np, dtype=torch.long).to(DEVICE)
region_edge_index = torch.tensor(region_edge_index_np, dtype=torch.long).to(DEVICE)

def add_static_fn(x_time_batch, static_tensor_local):
    B, W, N, _ = x_time_batch.shape
    static_b = static_tensor_local.unsqueeze(0).unsqueeze(0).expand(B, W, N, n_static_feats)
    return torch.cat([x_time_batch, static_b], dim=-1)

def gather_seq(ew_by_hour_t, starts_subset):
    idx = starts_subset.unsqueeze(1) + torch.arange(WINDOW).unsqueeze(0)
    ew = ew_by_hour_t[idx]
    ew = ew.clone()
    ew[:, :WINDOW - GRAPH_RECENT_HOURS, :] = 0.0
    return ew

print(f"train hours (2016-2019): {TRAIN_MASK.sum()}  val hours (2020): {(split_id_per_hour==1).sum()}  test hours (2021): {(split_id_per_hour==2).sum()}")

wind_blows_toward = (wind_dir_arr + 180) % 360
wbt_src = wind_blows_toward[:, src_idx]
cos_align = np.maximum(np.cos(np.radians(wbt_src - bearing_edge[None, :])), 0.0)
speed_src = wind_speed_arr[:, src_idx]
wind_component = (cos_align * speed_src).astype(np.float32)
del wbt_src, cos_align, speed_src
gc.collect()
ref_speed = np.float32(np.nanmean(wind_speed_arr[TRAIN_MASK]))
component = np.where(close_edge_mask[None, :], ref_speed, wind_component)
station_wind_raw = np.nan_to_num(decay_edge[None, :] * component, nan=0.0).astype(np.float32)
del component
gc.collect()

r_wbt_src = region_wind_blows_toward[:, r_src_idx]
r_cos_align = np.maximum(np.cos(np.radians(r_wbt_src - region_bearing_edge[None, :])), 0.0)
r_speed_src = region_windspeed[:, r_src_idx]
region_wind_raw = np.nan_to_num(region_decay_edge[None, :] * r_cos_align * r_speed_src, nan=0.0).astype(np.float32)
del r_wbt_src, r_cos_align, r_speed_src
gc.collect()

train_nonzero = station_wind_raw[TRAIN_MASK][station_wind_raw[TRAIN_MASK] > 0]
station_wind_edge_weight = (station_wind_raw / train_nonzero.std()).astype(np.float32)
region_train_nonzero = region_wind_raw[TRAIN_MASK][region_wind_raw[TRAIN_MASK] > 0]
region_wind_edge_weight = (region_wind_raw / region_train_nonzero.std()).astype(np.float32)
del train_nonzero, region_train_nonzero, station_wind_raw, region_wind_raw
gc.collect()

t_mean = np.nanmean(time_arr_raw[TRAIN_MASK], axis=(0, 1), keepdims=True)
t_std = np.nanstd(time_arr_raw[TRAIN_MASK], axis=(0, 1), keepdims=True) + 1e-6
time_arr_std = np.nan_to_num((time_arr_raw - t_mean) / t_std, nan=0.0)
s_mean, s_std = static_arr.mean(axis=0, keepdims=True), static_arr.std(axis=0, keepdims=True) + 1e-6
static_tensor = torch.tensor((static_arr - s_mean) / s_std, dtype=torch.float32).to(DEVICE)

p_buckets = {0: ([], [], [], []), 1: ([], [], [], []), 2: ([], [], [], [])}
for t in range(0, n_time - WINDOW - HORIZON + 1):
    target_t = t + WINDOW + HORIZON - 1
    s_start, s_target = split_id_per_hour[t], split_id_per_hour[target_t]
    if s_start != s_target or s_start == -1:
        continue
    x_win = time_arr_std[t:t + WINDOW]
    traj = region_episode_label[t + WINDOW: t + WINDOW + HORIZON]
    Xl, yregl, ytrajl, sl = p_buckets[s_start]
    Xl.append(x_win)
    yregl.append(time_arr_std[target_t, :, pm25_col_idx])
    ytrajl.append(traj.T)
    sl.append(t)

X0, yreg0, ytraj0, starts0 = (np.stack(v) for v in p_buckets[0])
X1, yreg1, ytraj1, starts1 = (np.stack(v) for v in p_buckets[1])
X2, yreg2, ytraj2, starts2 = (np.stack(v) for v in p_buckets[2])
del p_buckets, time_arr_std
gc.collect()
print(f"windows: train={len(X0)} val={len(X1)} test={len(X2)}")

X0_t = torch.tensor(X0, dtype=torch.float32); del X0
X1_t = torch.tensor(X1, dtype=torch.float32); del X1
X2_t = torch.tensor(X2, dtype=torch.float32); del X2
gc.collect()

yreg0_t, yreg1_t = torch.tensor(yreg0, dtype=torch.float32), torch.tensor(yreg1, dtype=torch.float32)
ytraj0_t = torch.tensor(ytraj0, dtype=torch.float32)
ytraj1_t = torch.tensor(ytraj1, dtype=torch.float32)
starts0_t, starts1_t, starts2_t = (torch.tensor(a, dtype=torch.long) for a in (starts0, starts1, starts2))
del yreg0, yreg1
gc.collect()

POS_WEIGHT = min(float((ytraj0_t.numel() - ytraj0_t.sum()) / ytraj0_t.sum().clamp(min=1)), 50.0)
print(f"pos_weight: {POS_WEIGHT:.2f}")

wind_ewt_s = torch.tensor(station_wind_edge_weight)
wind_ewt_r = torch.tensor(region_wind_edge_weight)
zero_ewt_s = torch.zeros_like(wind_ewt_s)
zero_ewt_r = torch.zeros_like(wind_ewt_r)

region_criterion = nn.BCEWithLogitsLoss(pos_weight=torch.tensor(POS_WEIGHT))

def run_p1_epoch_graph(model, ewt, X, y, starts, optimizer, train):
    n = X.shape[0]
    idx = torch.randperm(n) if train else torch.arange(n)
    model.train(train)
    total_loss, total_n = 0.0, 0
    eff_batch = MICRO_BATCH * ACCUM_STEPS
    for start in range(0, n, eff_batch):
        if train: optimizer.zero_grad()
        batch_idx = idx[start:start + eff_batch]
        for ms in range(0, len(batch_idx), MICRO_BATCH):
            mb_idx = batch_idx[ms:ms + MICRO_BATCH]
            if len(mb_idx) == 0: continue
            xb = add_static_fn(X[mb_idx].to(DEVICE), static_tensor)
            yb = y[mb_idx].to(DEVICE)
            ew_seq = gather_seq(ewt, starts[mb_idx]).to(DEVICE)
            with torch.set_grad_enabled(train):
                pred = model(xb, edge_index, ew_seq)
                loss = pinball_loss(pred, yb, QUANTILES)
            if train: (loss * len(mb_idx) / len(batch_idx)).backward()
            total_loss += loss.item() * len(mb_idx); total_n += len(mb_idx)
        if train: optimizer.step()
    return total_loss / total_n

def run_p2_epoch_graph(model, ewt_s, ewt_r, X, y_traj, starts, optimizer, train):
    n = X.shape[0]
    idx = torch.randperm(n) if train else torch.arange(n)
    model.train(train)
    total_loss, total_n = 0.0, 0
    eff_batch = MICRO_BATCH * ACCUM_STEPS
    for start in range(0, n, eff_batch):
        if train: optimizer.zero_grad()
        batch_idx = idx[start:start + eff_batch]
        for ms in range(0, len(batch_idx), MICRO_BATCH):
            mb_idx = batch_idx[ms:ms + MICRO_BATCH]
            if len(mb_idx) == 0: continue
            xb = add_static_fn(X[mb_idx].to(DEVICE), static_tensor)
            yb = y_traj[mb_idx].to(DEVICE)
            ew_s = gather_seq(ewt_s, starts[mb_idx]).to(DEVICE)
            ew_r = gather_seq(ewt_r, starts[mb_idx]).to(DEVICE)
            with torch.set_grad_enabled(train):
                logits = model(xb, edge_index, ew_s, region_edge_index, ew_r, n_regions)
                loss = region_criterion(logits, yb)
            if train: loss.backward()
            total_loss += loss.item() * len(mb_idx); total_n += len(mb_idx)
        if train: optimizer.step()
    return total_loss / max(total_n, 1)

def predict_probs_graph(model, ewt_s, ewt_r, X, starts, batch_size=64):
    model.eval()
    n = X.shape[0]
    out = np.zeros((n, n_regions, HORIZON), dtype=np.float32)
    with torch.no_grad():
        for start in range(0, n, batch_size):
            idx = torch.arange(start, min(start + batch_size, n))
            xb = add_static_fn(X[idx].to(DEVICE), static_tensor)
            ew_s = gather_seq(ewt_s, starts[idx]).to(DEVICE)
            ew_r = gather_seq(ewt_r, starts[idx]).to(DEVICE)
            logits = model(xb, edge_index, ew_s, region_edge_index, ew_r, n_regions)
            out[idx.numpy()] = torch.sigmoid(logits).cpu().numpy()
    return out

def run_p1_epoch_gru(model, X, y, optimizer, train):
    n = X.shape[0]
    idx = torch.randperm(n) if train else torch.arange(n)
    model.train(train)
    total_loss, total_n = 0.0, 0
    eff_batch = MICRO_BATCH * ACCUM_STEPS
    for start in range(0, n, eff_batch):
        if train: optimizer.zero_grad()
        batch_idx = idx[start:start + eff_batch]
        for ms in range(0, len(batch_idx), MICRO_BATCH):
            mb_idx = batch_idx[ms:ms + MICRO_BATCH]
            if len(mb_idx) == 0: continue
            xb = add_static_fn(X[mb_idx].to(DEVICE), static_tensor)
            yb = y[mb_idx].to(DEVICE)
            with torch.set_grad_enabled(train):
                pred = model(xb)
                loss = pinball_loss(pred, yb, QUANTILES)
            if train: (loss * len(mb_idx) / len(batch_idx)).backward()
            total_loss += loss.item() * len(mb_idx); total_n += len(mb_idx)
        if train: optimizer.step()
    return total_loss / total_n

def run_p2_epoch_gru(model, X, y_traj, optimizer, train):
    n = X.shape[0]
    idx = torch.randperm(n) if train else torch.arange(n)
    model.train(train)
    total_loss, total_n = 0.0, 0
    eff_batch = MICRO_BATCH * ACCUM_STEPS
    for start in range(0, n, eff_batch):
        if train: optimizer.zero_grad()
        batch_idx = idx[start:start + eff_batch]
        for ms in range(0, len(batch_idx), MICRO_BATCH):
            mb_idx = batch_idx[ms:ms + MICRO_BATCH]
            if len(mb_idx) == 0: continue
            xb = add_static_fn(X[mb_idx].to(DEVICE), static_tensor)
            yb = y_traj[mb_idx].to(DEVICE)
            with torch.set_grad_enabled(train):
                logits = model(xb, n_regions)
                loss = region_criterion(logits, yb)
            if train: loss.backward()
            total_loss += loss.item() * len(mb_idx); total_n += len(mb_idx)
        if train: optimizer.step()
    return total_loss / max(total_n, 1)

def predict_probs_gru(model, X, batch_size=64):
    model.eval()
    n = X.shape[0]
    out = np.zeros((n, n_regions, HORIZON), dtype=np.float32)
    with torch.no_grad():
        for start in range(0, n, batch_size):
            idx = torch.arange(start, min(start + batch_size, n))
            xb = add_static_fn(X[idx].to(DEVICE), static_tensor)
            logits = model(xb, n_regions)
            out[idx.numpy()] = torch.sigmoid(logits).cpu().numpy()
    return out

def train_one_config_graph(seed, ewt_s, ewt_r, lr, dropout, weight_decay):
    torch.manual_seed(seed); np.random.seed(seed)
    p1 = StationQuantileGCN(n_feats, dropout).to(DEVICE)
    opt1 = torch.optim.Adam(p1.parameters(), lr=lr, weight_decay=weight_decay)
    best_p1_val, best_p1_state = float("inf"), None
    for epoch in range(1, MAX_EPOCHS_P1 + 1):
        run_p1_epoch_graph(p1, ewt_s, X0_t, yreg0_t, starts0_t, opt1, True)
        vl = run_p1_epoch_graph(p1, ewt_s, X1_t, yreg1_t, starts1_t, opt1, False)
        if vl < best_p1_val:
            best_p1_val, best_p1_state = vl, copy.deepcopy(p1.state_dict())
    p1.load_state_dict(best_p1_state)
    encoder_copy = copy.deepcopy(p1.station_conv)
    p2 = RegionFromTrajectoryGCN(encoder_copy, region_membership_t, dropout).to(DEVICE)
    del p1; gc.collect()
    if DEVICE.type == "mps": torch.mps.empty_cache()
    opt2 = torch.optim.Adam(p2.parameters(), lr=lr, weight_decay=weight_decay)
    y_val_agg = ytraj1_t.numpy().max(axis=-1).reshape(-1)
    best_val_aucpr, best_state = -1.0, None
    for epoch in range(1, MAX_EPOCHS_P2 + 1):
        run_p2_epoch_graph(p2, ewt_s, ewt_r, X0_t, ytraj0_t, starts0_t, opt2, True)
        run_p2_epoch_graph(p2, ewt_s, ewt_r, X1_t, ytraj1_t, starts1_t, opt2, False)
        val_probs = predict_probs_graph(p2, ewt_s, ewt_r, X1_t, starts1_t)
        va = average_precision_score(y_val_agg, val_probs.max(axis=-1).reshape(-1))
        if va > best_val_aucpr:
            best_val_aucpr, best_state = va, copy.deepcopy(p2.state_dict())
    p2.load_state_dict(best_state)
    p2.eval()
    return p2, best_val_aucpr

def train_one_config_gru(seed, lr, dropout, weight_decay):
    torch.manual_seed(seed); np.random.seed(seed)
    p1 = StationGRU(n_feats, dropout).to(DEVICE)
    opt1 = torch.optim.Adam(p1.parameters(), lr=lr, weight_decay=weight_decay)
    best_p1_val, best_p1_state = float("inf"), None
    for epoch in range(1, MAX_EPOCHS_P1 + 1):
        run_p1_epoch_gru(p1, X0_t, yreg0_t, opt1, True)
        vl = run_p1_epoch_gru(p1, X1_t, yreg1_t, opt1, False)
        if vl < best_p1_val:
            best_p1_val, best_p1_state = vl, copy.deepcopy(p1.state_dict())
    p1.load_state_dict(best_p1_state)
    encoder_copy = copy.deepcopy(p1.station_lin)
    p2 = RegionFromTrajectoryGRU(encoder_copy, region_membership_t, dropout).to(DEVICE)
    del p1; gc.collect()
    if DEVICE.type == "mps": torch.mps.empty_cache()
    opt2 = torch.optim.Adam(p2.parameters(), lr=lr, weight_decay=weight_decay)
    y_val_agg = ytraj1_t.numpy().max(axis=-1).reshape(-1)
    best_val_aucpr, best_state = -1.0, None
    for epoch in range(1, MAX_EPOCHS_P2 + 1):
        run_p2_epoch_gru(p2, X0_t, ytraj0_t, opt2, True)
        run_p2_epoch_gru(p2, X1_t, ytraj1_t, opt2, False)
        val_probs = predict_probs_gru(p2, X1_t)
        va = average_precision_score(y_val_agg, val_probs.max(axis=-1).reshape(-1))
        if va > best_val_aucpr:
            best_val_aucpr, best_state = va, copy.deepcopy(p2.state_dict())
    p2.load_state_dict(best_state)
    p2.eval()
    return p2, best_val_aucpr

GRID = [
    {"lr": 0.003, "dropout": 0.5, "weight_decay": 0.0005},
    {"lr": 0.001, "dropout": 0.5, "weight_decay": 0.0005},
    {"lr": 0.003, "dropout": 0.3, "weight_decay": 0.0005},
    {"lr": 0.003, "dropout": 0.5, "weight_decay": 0.001},
]
N_SEEDS = 5
y_test_agg = ytraj2.max(axis=-1).reshape(-1)
all_results = {}

def run_full_pipeline(model_name, ewt_s=None, ewt_r=None, is_gru=False):
    print(f"\n{'#'*15} {model_name}: hyperparameter search on 2016-2019/2020 only {'#'*15}")
    grid_results = []
    for cfg in GRID:
        t0 = time.time()
        if is_gru:
            p2, val_aucpr = train_one_config_gru(seed=0, **cfg)
        else:
            p2, val_aucpr = train_one_config_graph(seed=0, ewt_s=ewt_s, ewt_r=ewt_r, **cfg)
        print(f"  {cfg}  ->  val_AUCPR={val_aucpr:.4f}  ({time.time()-t0:.0f}s)")
        grid_results.append({**cfg, "val_aucpr": float(val_aucpr)})
        del p2; gc.collect()
        if DEVICE.type == "mps": torch.mps.empty_cache()
    best_cfg = max(grid_results, key=lambda r: r["val_aucpr"])
    print(f"  best config for {model_name}: {best_cfg}")

    print(f"\n{'#'*15} {model_name}: {N_SEEDS} final seeds {'#'*15}")
    seed_results = []
    for seed in range(N_SEEDS):
        t0 = time.time()
        if is_gru:
            p2, val_aucpr = train_one_config_gru(seed, best_cfg["lr"], best_cfg["dropout"], best_cfg["weight_decay"])
            test_probs = predict_probs_gru(p2, X2_t)
        else:
            p2, val_aucpr = train_one_config_graph(seed, ewt_s, ewt_r, best_cfg["lr"], best_cfg["dropout"], best_cfg["weight_decay"])
            test_probs = predict_probs_graph(p2, ewt_s, ewt_r, X2_t, starts2_t)
        test_agg_score = test_probs.max(axis=-1).reshape(-1)
        test_aucroc = roc_auc_score(y_test_agg, test_agg_score)
        test_aucpr = average_precision_score(y_test_agg, test_agg_score)
        print(f"  seed={seed}  ({time.time()-t0:.0f}s)  test_AUCROC={test_aucroc:.4f}  test_AUCPR={test_aucpr:.4f}")
        seed_results.append({"seed": seed, "val_aucpr": float(val_aucpr), "test_aucroc": float(test_aucroc), "test_aucpr": float(test_aucpr)})
        del p2; gc.collect()
        if DEVICE.type == "mps": torch.mps.empty_cache()

    return {"grid_search": grid_results, "best_config": best_cfg, "results": seed_results}

all_results["wind_graph"] = run_full_pipeline("wind_graph", wind_ewt_s, wind_ewt_r, is_gru=False)
all_results["no_graph"] = run_full_pipeline("no_graph", zero_ewt_s, zero_ewt_r, is_gru=False)
all_results["minimal_gru"] = run_full_pipeline("minimal_gru", is_gru=True)

# ================================================================
# SUMMARY + PAIRED SIGNIFICANCE TESTS
# ================================================================
print(f"\n{'='*20} SUMMARY: train=2016-2019 / val=2020 / test=2021 (TransNet-matched regime) {'='*20}")
arrs = {}
for name in ["wind_graph", "no_graph", "minimal_gru"]:
    aucpr = np.array([r["test_aucpr"] for r in all_results[name]["results"]])
    aucroc = np.array([r["test_aucroc"] for r in all_results[name]["results"]])
    arrs[name] = {"aucpr": aucpr, "aucroc": aucroc}
    print(f"{name:<15} AUC-PR mean={aucpr.mean():.4f} std={aucpr.std():.4f}  |  AUC-ROC mean={aucroc.mean():.4f} std={aucroc.std():.4f}  "
          f"(best_cfg={all_results[name]['best_config']})")

for other in ["no_graph", "minimal_gru"]:
    t_stat, p_ttest = stats.ttest_rel(arrs["wind_graph"]["aucpr"], arrs[other]["aucpr"])
    w_stat, p_wilcoxon = stats.wilcoxon(arrs["wind_graph"]["aucpr"], arrs[other]["aucpr"])
    n_wins = int((arrs["wind_graph"]["aucpr"] > arrs[other]["aucpr"]).sum())
    print(f"\nwind_graph vs {other} (AUC-PR, {N_SEEDS} seeds): paired t={t_stat:.3f} p={p_ttest:.6f}  |  Wilcoxon p={p_wilcoxon:.4f}  |  wind_graph wins {n_wins}/{N_SEEDS}")

with open(f"{BASE}/2021_transnet_regime_10seed_independent_tuning.json", "w") as f:
    json.dump(all_results, f, indent=2)
print(f"\nsaved to {BASE}/2021_transnet_regime_10seed_independent_tuning.json")


device=mps
train hours (2016-2019): 35064  val hours (2020): 8784  test hours (2021): 8760
windows: train=34993 val=8713 test=8689
pos_weight: 50.00

############### wind_graph: hyperparameter search on 2016-2019/2020 only ###############
  {'lr': 0.003, 'dropout': 0.5, 'weight_decay': 0.0005}  ->  val_AUCPR=0.2657  (1046s)
  {'lr': 0.001, 'dropout': 0.5, 'weight_decay': 0.0005}  ->  val_AUCPR=0.2538  (1031s)
  {'lr': 0.003, 'dropout': 0.3, 'weight_decay': 0.0005}  ->  val_AUCPR=0.2891  (1041s)
  {'lr': 0.003, 'dropout': 0.5, 'weight_decay': 0.001}  ->  val_AUCPR=0.2638  (1022s)
  best config for wind_graph: {'lr': 0.003, 'dropout': 0.3, 'weight_decay': 0.0005, 'val_aucpr': 0.28907405627633986}

############### wind_graph: 5 final seeds ###############
  seed=0  (1098s)  test_AUCROC=0.8666  test_AUCPR=0.4018
  seed=1  (1076s)  test_AUCROC=0.8740  test_AUCPR=0.3869
  seed=2  (1118s)  test_AUCROC=0.8589  test_AUCPR=0.3770
  seed=3  (1120s)  test_AUCROC=0.8748  test_AUCPR=0.3569
  seed=4 

KeyboardInterrupt: 

In [1]:
# ================================================================
# train=2016-2018 / val=2019 / test=2020
# val stays within the stable pre-COVID regime (fixes the train/val
# mismatch from the earlier cross-regime run); only test crosses into
# the COVID-shifted period, isolating genuine forward generalization
# from the wind-direction-consistency confound found on 2021.
# wind_graph, no_graph, minimal_gru -- each independently hyperparameter-
# tuned (search on train=2016-2018/val=2019 only, test=2020 never touched),
# then 5 final seeds each with its own best config.
# ================================================================
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import os, time, copy, gc, json
from sklearn.metrics import average_precision_score, roc_auc_score
from scipy import stats

BASE = "/Users/drewbaldwin/PM2_5 Research"
df = pd.read_pickle(f"{BASE}/air_korea_final_imputed_with_blh.pkl")
station_order = sorted(df["Station_ID"].unique())
n_stations = len(station_order)

stations = df[["Station_ID", "lat", "lon"]].drop_duplicates("Station_ID").set_index("Station_ID").loc[station_order]
lats, lons = stations["lat"].to_numpy(), stations["lon"].to_numpy()

def haversine_km(lat1, lon1, lat2, lon2):
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
    dlat, dlon = lat2 - lat1, lon2 - lon1
    a = np.sin(dlat/2)**2 + np.cos(lat1)*np.cos(lat2)*np.sin(dlon/2)**2
    return 2 * 6371.0 * np.arcsin(np.sqrt(a))

def bearing_matrix(lat, lon):
    lat_r, lon_r = np.radians(lat), np.radians(lon)
    lat1, lat2 = lat_r[:, None], lat_r[None, :]
    dlon = lon_r[None, :] - lon_r[:, None]
    x = np.sin(dlon) * np.cos(lat2)
    y = np.cos(lat1) * np.sin(lat2) - np.sin(lat1) * np.cos(lat2) * np.cos(dlon)
    return (np.degrees(np.arctan2(x, y)) + 360) % 360

dist_km = haversine_km(lats[:, None], lons[:, None], lats[None, :], lons[None, :])
DIST_CUTOFF, RHO_KM = 250.0, 250.0
HYBRID_THRESHOLD_KM = 20.0
dist_edges = (dist_km <= DIST_CUTOFF) & (dist_km > 0)
bearing_from = bearing_matrix(lats, lons)

src_idx, dst_idx = np.nonzero(dist_edges)
edge_index_np = np.stack([src_idx, dst_idx])
decay_edge = np.exp(-dist_km[src_idx, dst_idx] / RHO_KM).astype(np.float32)
bearing_edge = bearing_from[src_idx, dst_idx].astype(np.float32)
close_edge_mask = (dist_km[src_idx, dst_idx] <= HYBRID_THRESHOLD_KM)

REGION_CENTROIDS = {
    "Seoul": (37.566, 126.978), "Busan": (35.180, 129.075), "Daegu": (35.872, 128.602),
    "Incheon": (37.483, 126.633), "Gwangju": (35.155, 126.916), "Daejeon": (36.350, 127.385),
    "Ulsan": (35.550, 129.317), "Sejong": (36.487, 127.282), "Gyeonggi": (37.500, 127.250),
    "Gangwon": (37.867, 127.733), "Chungbuk": (36.633, 127.483), "Chungnam": (36.500, 126.750),
    "Jeonbuk": (35.824, 127.148), "Jeonnam": (34.750, 127.000), "Gyeongbuk": (36.559, 128.729),
    "Gyeongnam": (35.271, 128.663), "Jeju": (33.513, 126.523),
}
region_names = list(REGION_CENTROIDS.keys())
n_regions = len(region_names)
region_lats = np.array([REGION_CENTROIDS[r][0] for r in region_names])
region_lons = np.array([REGION_CENTROIDS[r][1] for r in region_names])
dist_to_region = haversine_km(lats[:, None], lons[:, None], region_lats[None, :], region_lons[None, :])
station_region_idx = dist_to_region.argmin(axis=1)

region_membership = np.zeros((n_stations, n_regions), dtype=np.float32)
region_membership[np.arange(n_stations), station_region_idx] = 1.0
region_membership_t = torch.tensor(region_membership)

region_dist_km = haversine_km(region_lats[:, None], region_lons[:, None], region_lats[None, :], region_lons[None, :])
region_bearing = bearing_matrix(region_lats, region_lons)
r_src_idx, r_dst_idx = np.nonzero(~np.eye(n_regions, dtype=bool))
region_edge_index_np = np.stack([r_src_idx, r_dst_idx])
region_decay_edge = np.exp(-region_dist_km[r_src_idx, r_dst_idx] / RHO_KM).astype(np.float32)
region_bearing_edge = region_bearing[r_src_idx, r_dst_idx].astype(np.float32)

WINDOW, HORIZON = 36, 36
GRAPH_RECENT_HOURS = 18
EVENT_THRESHOLD = 75.0
SUSTAIN_HOURS = 2
QUANTILES = [0.50, 0.75, 0.90, 0.95, 0.99]
N_QUANTILES = len(QUANTILES)
TIME_FEATS = ["SO2", "CO", "NO2", "O3", "PM10", "PM25"]
STATIC_COLS = ["elevation_m", "urban_landuse_area_m2_3km", "green_space_area_3km",
               "building_footprint_area_3km", "railway_length_3km", "dist_to_coast_km",
               "dist_to_major_road_km", "industrial_area_m2_3km", "traffic_points_count_3km",
               "major_roads_count_3km", "total_road_length_3km"]

time_panels = {c: df.pivot(index="Datetime", columns="Station_ID", values=c)[station_order] for c in TIME_FEATS}
dt_index = time_panels["PM25"].index
n_time = len(dt_index)
years = dt_index.year.to_numpy()
doy = dt_index.dayofyear.to_numpy().astype(float)

# ---- SPLIT: train=2016-2018, val=2019 (stable regime, fixes val/train mismatch), test=2020 ----
split_id_per_hour = np.where(years.astype(int) <= 2018, 0, np.where(years == 2019, 1, np.where(years == 2020, 2, -1)))
TRAIN_MASK = split_id_per_hour == 0

wind_dir_arr = df.pivot(index="Datetime", columns="Station_ID", values="winddirection_10m")[station_order].reindex(dt_index).to_numpy().astype(float)
wind_speed_arr = df.pivot(index="Datetime", columns="Station_ID", values="windspeed_10m")[station_order].reindex(dt_index).to_numpy().astype(float)
blh_arr = df.pivot(index="Datetime", columns="Station_ID", values="boundary_layer_height")[station_order].reindex(dt_index).to_numpy().astype(float)
pm25_raw_arr = time_panels["PM25"].to_numpy()

def regional_flat_mean(arr):
    out = np.zeros((n_time, n_regions), dtype=np.float32)
    for r in range(n_regions):
        cols = station_region_idx == r
        out[:, r] = np.nanmean(arr[:, cols], axis=1)
    return out

region_pm25 = regional_flat_mean(pm25_raw_arr)
wdir_sin_station = np.sin(np.radians(wind_dir_arr))
wdir_cos_station = np.cos(np.radians(wind_dir_arr))
region_windspeed = regional_flat_mean(wind_speed_arr)
region_wdir_sin = regional_flat_mean(wdir_sin_station)
region_wdir_cos = regional_flat_mean(wdir_cos_station)
region_wind_dir_deg = (np.degrees(np.arctan2(region_wdir_sin, region_wdir_cos)) + 360) % 360
region_wind_blows_toward = (region_wind_dir_deg + 180) % 360

season_sin_1d = np.sin(2 * np.pi * doy / 365.25)
season_cos_1d = np.cos(2 * np.pi * doy / 365.25)
season_sin = np.tile(season_sin_1d[:, None], (1, n_stations))
season_cos = np.tile(season_cos_1d[:, None], (1, n_stations))

TIME_FEATS_FULL = TIME_FEATS + ["windspeed_10m", "wdir_sin", "wdir_cos", "boundary_layer_height", "season_sin", "season_cos"]
n_time_feats, n_static_feats = len(TIME_FEATS_FULL), len(STATIC_COLS)
n_feats = n_time_feats + n_static_feats
pm25_col_idx = TIME_FEATS_FULL.index("PM25")

time_arr_raw = np.stack([time_panels[c].to_numpy() for c in TIME_FEATS] +
                         [wind_speed_arr, wdir_sin_station, wdir_cos_station, blh_arr, season_sin, season_cos], axis=-1)
del time_panels, season_sin, season_cos, blh_arr
gc.collect()

static_df = df[["Station_ID"] + STATIC_COLS].drop_duplicates("Station_ID").set_index("Station_ID").loc[station_order]
static_arr = static_df[STATIC_COLS].to_numpy()
del df
gc.collect()

rev = region_pm25[::-1]
roll_min_rev = pd.DataFrame(rev).rolling(window=SUSTAIN_HOURS, min_periods=SUSTAIN_HOURS).min().to_numpy()
region_episode_label = (roll_min_rev[::-1] >= EVENT_THRESHOLD).astype(np.float32)
del rev, roll_min_rev, pm25_raw_arr
gc.collect()

def pinball_loss(preds, target, quantiles):
    target_exp = target.unsqueeze(-1)
    diff = target_exp - preds
    q_tensor = torch.tensor(quantiles, device=preds.device, dtype=preds.dtype).view(*([1] * (preds.dim() - 1)), -1)
    return torch.max(q_tensor * diff, (q_tensor - 1) * diff).mean()

def monotonic_quantiles(raw):
    first = raw[..., :1]
    deltas = F.softplus(raw[..., 1:])
    return torch.cat([first, first + torch.cumsum(deltas, dim=-1)], dim=-1)

class WindConvLayer(nn.Module):
    def __init__(self, in_dim, out_dim):
        super().__init__()
        self.lin_self = nn.Linear(in_dim, out_dim)
        self.lin_neigh = nn.Linear(in_dim, out_dim)
        self.lin_connectivity = nn.Linear(1, out_dim)

    def forward(self, x, edge_index, edge_weight, num_nodes):
        src, dst = edge_index[0], edge_index[1]
        messages = x[src] * edge_weight.unsqueeze(-1)
        agg_sum = x.new_zeros(num_nodes, x.size(-1))
        agg_sum.index_add_(0, dst, messages)
        weight_sum = x.new_zeros(num_nodes)
        weight_sum.index_add_(0, dst, edge_weight)
        agg_mean = agg_sum / (weight_sum.unsqueeze(-1) + 1e-8)
        connectivity = torch.log1p(weight_sum.clamp(min=0)).unsqueeze(-1)
        return self.lin_self(x) + self.lin_neigh(agg_mean) + self.lin_connectivity(connectivity)

class AttentionPool(nn.Module):
    def __init__(self, hidden):
        super().__init__()
        self.attn_score = nn.Linear(hidden, 1)

    def forward(self, h_station, region_membership_t_local):
        B, N, H = h_station.shape
        scores = self.attn_score(h_station).squeeze(-1)
        scores = scores - scores.max(dim=1, keepdim=True).values
        exp_scores = torch.exp(scores)
        weighted_exp = exp_scores.unsqueeze(-1) * region_membership_t_local.unsqueeze(0)
        region_denom = weighted_exp.sum(dim=1)
        region_numer = torch.einsum('bnr,bnh->brh', weighted_exp, h_station)
        return region_numer / (region_denom.unsqueeze(-1) + 1e-8)

class StationQuantileGCN(nn.Module):
    def __init__(self, in_dim, dropout, hidden=32, gru_hidden=32):
        super().__init__()
        self.station_conv = WindConvLayer(in_dim, hidden)
        self.drop = nn.Dropout(dropout)
        self.gru = nn.GRU(hidden, gru_hidden, batch_first=True)
        self.head = nn.Linear(gru_hidden, N_QUANTILES)

    def forward(self, x_window, edge_index, edge_weight_seq):
        B, W, N, Fin = x_window.shape
        ei_b = torch.cat([edge_index + i * N for i in range(B)], dim=1)
        num_nodes = B * N
        h_seq = []
        for w in range(W):
            xt = x_window[:, w].reshape(B * N, Fin)
            ew_b = edge_weight_seq[:, w].reshape(-1)
            h = torch.relu(self.station_conv(xt, ei_b, ew_b, num_nodes))
            h = self.drop(h)
            h_seq.append(h.reshape(B, N, -1))
        h_seq = torch.stack(h_seq, dim=1).permute(0, 2, 1, 3).reshape(B * N, W, -1)
        _, h_final = self.gru(h_seq)
        embed = self.drop(h_final.squeeze(0).reshape(B, N, -1))
        return monotonic_quantiles(self.head(embed))

class RegionFromTrajectoryGCN(nn.Module):
    def __init__(self, station_conv, region_membership_t_local, dropout, hidden=32, gru_hidden=32):
        super().__init__()
        self.station_conv = station_conv
        self.attn_pool = AttentionPool(hidden)
        self.region_conv = WindConvLayer(hidden, hidden)
        self.drop = nn.Dropout(dropout)
        self.region_gru = nn.GRU(hidden, gru_hidden, batch_first=True)
        self.region_head = nn.Linear(gru_hidden, HORIZON)
        self.rmem = region_membership_t_local

    def forward(self, x_window, station_edge_index, station_edge_weight_seq, region_edge_index, region_edge_weight_seq, n_reg):
        B, W, N, Fin = x_window.shape
        station_ei_b = torch.cat([station_edge_index + i * N for i in range(B)], dim=1)
        region_ei_b = torch.cat([region_edge_index + i * n_reg for i in range(B)], dim=1)
        num_station_nodes = B * N
        num_region_nodes = B * n_reg
        h_region_seq = []
        for w in range(W):
            xt = x_window[:, w].reshape(B * N, Fin)
            ew_station_b = station_edge_weight_seq[:, w].reshape(-1)
            h_station = torch.relu(self.station_conv(xt, station_ei_b, ew_station_b, num_station_nodes))
            h_station = self.drop(h_station).reshape(B, N, -1)
            h_region_pooled = self.attn_pool(h_station, self.rmem).reshape(B * n_reg, -1)
            ew_region_b = region_edge_weight_seq[:, w].reshape(-1)
            h_region = torch.relu(self.region_conv(h_region_pooled, region_ei_b, ew_region_b, num_region_nodes))
            h_region = self.drop(h_region)
            h_region_seq.append(h_region.reshape(B, n_reg, -1))
        h_region_seq = torch.stack(h_region_seq, dim=1).permute(0, 2, 1, 3).reshape(B * n_reg, W, -1)
        _, h_final = self.region_gru(h_region_seq)
        embed = self.drop(h_final.squeeze(0).reshape(B, n_reg, -1))
        return self.region_head(embed)

class StationGRU(nn.Module):
    def __init__(self, in_dim, dropout, hidden=32, gru_hidden=32):
        super().__init__()
        self.station_lin = nn.Linear(in_dim, hidden)
        self.drop = nn.Dropout(dropout)
        self.gru = nn.GRU(hidden, gru_hidden, batch_first=True)
        self.head = nn.Linear(gru_hidden, N_QUANTILES)

    def forward(self, x_window):
        B, W, N, Fin = x_window.shape
        h_seq = []
        for w in range(W):
            xt = x_window[:, w].reshape(B * N, Fin)
            h = torch.relu(self.station_lin(xt))
            h = self.drop(h)
            h_seq.append(h.reshape(B, N, -1))
        h_seq = torch.stack(h_seq, dim=1).permute(0, 2, 1, 3).reshape(B * N, W, -1)
        _, h_final = self.gru(h_seq)
        embed = self.drop(h_final.squeeze(0).reshape(B, N, -1))
        return monotonic_quantiles(self.head(embed))

class RegionFromTrajectoryGRU(nn.Module):
    def __init__(self, station_lin, region_membership_t_local, dropout, hidden=32, gru_hidden=32):
        super().__init__()
        self.station_lin = station_lin
        self.attn_pool = AttentionPool(hidden)
        self.drop = nn.Dropout(dropout)
        self.region_gru = nn.GRU(hidden, gru_hidden, batch_first=True)
        self.region_head = nn.Linear(gru_hidden, HORIZON)
        self.rmem = region_membership_t_local

    def forward(self, x_window, n_reg):
        B, W, N, Fin = x_window.shape
        h_region_seq = []
        for w in range(W):
            xt = x_window[:, w].reshape(B * N, Fin)
            h_station = torch.relu(self.station_lin(xt)).reshape(B, N, -1)
            h_station = self.drop(h_station)
            h_region_pooled = self.attn_pool(h_station, self.rmem)
            h_region_pooled = self.drop(h_region_pooled)
            h_region_seq.append(h_region_pooled)
        h_region_seq = torch.stack(h_region_seq, dim=1).permute(0, 2, 1, 3).reshape(B * n_reg, W, -1)
        _, h_final = self.region_gru(h_region_seq)
        embed = self.drop(h_final.squeeze(0).reshape(B, n_reg, -1))
        return self.region_head(embed)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else ("mps" if torch.backends.mps.is_available() else "cpu"))
MICRO_BATCH, ACCUM_STEPS = 16, 4
MAX_EPOCHS_P1, MAX_EPOCHS_P2 = 2, 2
print(f"device={DEVICE}")

region_membership_t = region_membership_t.to(DEVICE)
edge_index = torch.tensor(edge_index_np, dtype=torch.long).to(DEVICE)
region_edge_index = torch.tensor(region_edge_index_np, dtype=torch.long).to(DEVICE)

def add_static_fn(x_time_batch, static_tensor_local):
    B, W, N, _ = x_time_batch.shape
    static_b = static_tensor_local.unsqueeze(0).unsqueeze(0).expand(B, W, N, n_static_feats)
    return torch.cat([x_time_batch, static_b], dim=-1)

def gather_seq(ew_by_hour_t, starts_subset):
    idx = starts_subset.unsqueeze(1) + torch.arange(WINDOW).unsqueeze(0)
    ew = ew_by_hour_t[idx]
    ew = ew.clone()
    ew[:, :WINDOW - GRAPH_RECENT_HOURS, :] = 0.0
    return ew

print(f"train hours (2016-2018): {TRAIN_MASK.sum()}  val hours (2019): {(split_id_per_hour==1).sum()}  test hours (2020): {(split_id_per_hour==2).sum()}")

wind_blows_toward = (wind_dir_arr + 180) % 360
wbt_src = wind_blows_toward[:, src_idx]
cos_align = np.maximum(np.cos(np.radians(wbt_src - bearing_edge[None, :])), 0.0)
speed_src = wind_speed_arr[:, src_idx]
wind_component = (cos_align * speed_src).astype(np.float32)
del wbt_src, cos_align, speed_src
gc.collect()
ref_speed = np.float32(np.nanmean(wind_speed_arr[TRAIN_MASK]))
component = np.where(close_edge_mask[None, :], ref_speed, wind_component)
station_wind_raw = np.nan_to_num(decay_edge[None, :] * component, nan=0.0).astype(np.float32)
del component
gc.collect()

r_wbt_src = region_wind_blows_toward[:, r_src_idx]
r_cos_align = np.maximum(np.cos(np.radians(r_wbt_src - region_bearing_edge[None, :])), 0.0)
r_speed_src = region_windspeed[:, r_src_idx]
region_wind_raw = np.nan_to_num(region_decay_edge[None, :] * r_cos_align * r_speed_src, nan=0.0).astype(np.float32)
del r_wbt_src, r_cos_align, r_speed_src
gc.collect()

train_nonzero = station_wind_raw[TRAIN_MASK][station_wind_raw[TRAIN_MASK] > 0]
station_wind_edge_weight = (station_wind_raw / train_nonzero.std()).astype(np.float32)
region_train_nonzero = region_wind_raw[TRAIN_MASK][region_wind_raw[TRAIN_MASK] > 0]
region_wind_edge_weight = (region_wind_raw / region_train_nonzero.std()).astype(np.float32)
del train_nonzero, region_train_nonzero, station_wind_raw, region_wind_raw
gc.collect()

t_mean = np.nanmean(time_arr_raw[TRAIN_MASK], axis=(0, 1), keepdims=True)
t_std = np.nanstd(time_arr_raw[TRAIN_MASK], axis=(0, 1), keepdims=True) + 1e-6
time_arr_std = np.nan_to_num((time_arr_raw - t_mean) / t_std, nan=0.0)
s_mean, s_std = static_arr.mean(axis=0, keepdims=True), static_arr.std(axis=0, keepdims=True) + 1e-6
static_tensor = torch.tensor((static_arr - s_mean) / s_std, dtype=torch.float32).to(DEVICE)

p_buckets = {0: ([], [], [], []), 1: ([], [], [], []), 2: ([], [], [], [])}
for t in range(0, n_time - WINDOW - HORIZON + 1):
    target_t = t + WINDOW + HORIZON - 1
    s_start, s_target = split_id_per_hour[t], split_id_per_hour[target_t]
    if s_start != s_target or s_start == -1:
        continue
    x_win = time_arr_std[t:t + WINDOW]
    traj = region_episode_label[t + WINDOW: t + WINDOW + HORIZON]
    Xl, yregl, ytrajl, sl = p_buckets[s_start]
    Xl.append(x_win)
    yregl.append(time_arr_std[target_t, :, pm25_col_idx])
    ytrajl.append(traj.T)
    sl.append(t)

X0, yreg0, ytraj0, starts0 = (np.stack(v) for v in p_buckets[0])
X1, yreg1, ytraj1, starts1 = (np.stack(v) for v in p_buckets[1])
X2, yreg2, ytraj2, starts2 = (np.stack(v) for v in p_buckets[2])
del p_buckets, time_arr_std
gc.collect()
print(f"windows: train={len(X0)} val={len(X1)} test={len(X2)}")

X0_t = torch.tensor(X0, dtype=torch.float32); del X0
X1_t = torch.tensor(X1, dtype=torch.float32); del X1
X2_t = torch.tensor(X2, dtype=torch.float32); del X2
gc.collect()

yreg0_t, yreg1_t = torch.tensor(yreg0, dtype=torch.float32), torch.tensor(yreg1, dtype=torch.float32)
ytraj0_t = torch.tensor(ytraj0, dtype=torch.float32)
ytraj1_t = torch.tensor(ytraj1, dtype=torch.float32)
starts0_t, starts1_t, starts2_t = (torch.tensor(a, dtype=torch.long) for a in (starts0, starts1, starts2))
del yreg0, yreg1
gc.collect()

POS_WEIGHT = min(float((ytraj0_t.numel() - ytraj0_t.sum()) / ytraj0_t.sum().clamp(min=1)), 50.0)
print(f"pos_weight: {POS_WEIGHT:.2f}")

wind_ewt_s = torch.tensor(station_wind_edge_weight)
wind_ewt_r = torch.tensor(region_wind_edge_weight)
zero_ewt_s = torch.zeros_like(wind_ewt_s)
zero_ewt_r = torch.zeros_like(wind_ewt_r)

region_criterion = nn.BCEWithLogitsLoss(pos_weight=torch.tensor(POS_WEIGHT))

def run_p1_epoch_graph(model, ewt, X, y, starts, optimizer, train):
    n = X.shape[0]
    idx = torch.randperm(n) if train else torch.arange(n)
    model.train(train)
    total_loss, total_n = 0.0, 0
    eff_batch = MICRO_BATCH * ACCUM_STEPS
    for start in range(0, n, eff_batch):
        if train: optimizer.zero_grad()
        batch_idx = idx[start:start + eff_batch]
        for ms in range(0, len(batch_idx), MICRO_BATCH):
            mb_idx = batch_idx[ms:ms + MICRO_BATCH]
            if len(mb_idx) == 0: continue
            xb = add_static_fn(X[mb_idx].to(DEVICE), static_tensor)
            yb = y[mb_idx].to(DEVICE)
            ew_seq = gather_seq(ewt, starts[mb_idx]).to(DEVICE)
            with torch.set_grad_enabled(train):
                pred = model(xb, edge_index, ew_seq)
                loss = pinball_loss(pred, yb, QUANTILES)
            if train: (loss * len(mb_idx) / len(batch_idx)).backward()
            total_loss += loss.item() * len(mb_idx); total_n += len(mb_idx)
        if train: optimizer.step()
    return total_loss / total_n

def run_p2_epoch_graph(model, ewt_s, ewt_r, X, y_traj, starts, optimizer, train):
    n = X.shape[0]
    idx = torch.randperm(n) if train else torch.arange(n)
    model.train(train)
    total_loss, total_n = 0.0, 0
    eff_batch = MICRO_BATCH * ACCUM_STEPS
    for start in range(0, n, eff_batch):
        if train: optimizer.zero_grad()
        batch_idx = idx[start:start + eff_batch]
        for ms in range(0, len(batch_idx), MICRO_BATCH):
            mb_idx = batch_idx[ms:ms + MICRO_BATCH]
            if len(mb_idx) == 0: continue
            xb = add_static_fn(X[mb_idx].to(DEVICE), static_tensor)
            yb = y_traj[mb_idx].to(DEVICE)
            ew_s = gather_seq(ewt_s, starts[mb_idx]).to(DEVICE)
            ew_r = gather_seq(ewt_r, starts[mb_idx]).to(DEVICE)
            with torch.set_grad_enabled(train):
                logits = model(xb, edge_index, ew_s, region_edge_index, ew_r, n_regions)
                loss = region_criterion(logits, yb)
            if train: loss.backward()
            total_loss += loss.item() * len(mb_idx); total_n += len(mb_idx)
        if train: optimizer.step()
    return total_loss / max(total_n, 1)

def predict_probs_graph(model, ewt_s, ewt_r, X, starts, batch_size=64):
    model.eval()
    n = X.shape[0]
    out = np.zeros((n, n_regions, HORIZON), dtype=np.float32)
    with torch.no_grad():
        for start in range(0, n, batch_size):
            idx = torch.arange(start, min(start + batch_size, n))
            xb = add_static_fn(X[idx].to(DEVICE), static_tensor)
            ew_s = gather_seq(ewt_s, starts[idx]).to(DEVICE)
            ew_r = gather_seq(ewt_r, starts[idx]).to(DEVICE)
            logits = model(xb, edge_index, ew_s, region_edge_index, ew_r, n_regions)
            out[idx.numpy()] = torch.sigmoid(logits).cpu().numpy()
    return out

def run_p1_epoch_gru(model, X, y, optimizer, train):
    n = X.shape[0]
    idx = torch.randperm(n) if train else torch.arange(n)
    model.train(train)
    total_loss, total_n = 0.0, 0
    eff_batch = MICRO_BATCH * ACCUM_STEPS
    for start in range(0, n, eff_batch):
        if train: optimizer.zero_grad()
        batch_idx = idx[start:start + eff_batch]
        for ms in range(0, len(batch_idx), MICRO_BATCH):
            mb_idx = batch_idx[ms:ms + MICRO_BATCH]
            if len(mb_idx) == 0: continue
            xb = add_static_fn(X[mb_idx].to(DEVICE), static_tensor)
            yb = y[mb_idx].to(DEVICE)
            with torch.set_grad_enabled(train):
                pred = model(xb)
                loss = pinball_loss(pred, yb, QUANTILES)
            if train: (loss * len(mb_idx) / len(batch_idx)).backward()
            total_loss += loss.item() * len(mb_idx); total_n += len(mb_idx)
        if train: optimizer.step()
    return total_loss / total_n

def run_p2_epoch_gru(model, X, y_traj, optimizer, train):
    n = X.shape[0]
    idx = torch.randperm(n) if train else torch.arange(n)
    model.train(train)
    total_loss, total_n = 0.0, 0
    eff_batch = MICRO_BATCH * ACCUM_STEPS
    for start in range(0, n, eff_batch):
        if train: optimizer.zero_grad()
        batch_idx = idx[start:start + eff_batch]
        for ms in range(0, len(batch_idx), MICRO_BATCH):
            mb_idx = batch_idx[ms:ms + MICRO_BATCH]
            if len(mb_idx) == 0: continue
            xb = add_static_fn(X[mb_idx].to(DEVICE), static_tensor)
            yb = y_traj[mb_idx].to(DEVICE)
            with torch.set_grad_enabled(train):
                logits = model(xb, n_regions)
                loss = region_criterion(logits, yb)
            if train: loss.backward()
            total_loss += loss.item() * len(mb_idx); total_n += len(mb_idx)
        if train: optimizer.step()
    return total_loss / max(total_n, 1)

def predict_probs_gru(model, X, batch_size=64):
    model.eval()
    n = X.shape[0]
    out = np.zeros((n, n_regions, HORIZON), dtype=np.float32)
    with torch.no_grad():
        for start in range(0, n, batch_size):
            idx = torch.arange(start, min(start + batch_size, n))
            xb = add_static_fn(X[idx].to(DEVICE), static_tensor)
            logits = model(xb, n_regions)
            out[idx.numpy()] = torch.sigmoid(logits).cpu().numpy()
    return out

def train_one_config_graph(seed, ewt_s, ewt_r, lr, dropout, weight_decay):
    torch.manual_seed(seed); np.random.seed(seed)
    p1 = StationQuantileGCN(n_feats, dropout).to(DEVICE)
    opt1 = torch.optim.Adam(p1.parameters(), lr=lr, weight_decay=weight_decay)
    best_p1_val, best_p1_state = float("inf"), None
    for epoch in range(1, MAX_EPOCHS_P1 + 1):
        run_p1_epoch_graph(p1, ewt_s, X0_t, yreg0_t, starts0_t, opt1, True)
        vl = run_p1_epoch_graph(p1, ewt_s, X1_t, yreg1_t, starts1_t, opt1, False)
        if vl < best_p1_val:
            best_p1_val, best_p1_state = vl, copy.deepcopy(p1.state_dict())
    p1.load_state_dict(best_p1_state)
    encoder_copy = copy.deepcopy(p1.station_conv)
    p2 = RegionFromTrajectoryGCN(encoder_copy, region_membership_t, dropout).to(DEVICE)
    del p1; gc.collect()
    if DEVICE.type == "mps": torch.mps.empty_cache()
    opt2 = torch.optim.Adam(p2.parameters(), lr=lr, weight_decay=weight_decay)
    y_val_agg = ytraj1_t.numpy().max(axis=-1).reshape(-1)
    best_val_aucpr, best_state = -1.0, None
    for epoch in range(1, MAX_EPOCHS_P2 + 1):
        run_p2_epoch_graph(p2, ewt_s, ewt_r, X0_t, ytraj0_t, starts0_t, opt2, True)
        run_p2_epoch_graph(p2, ewt_s, ewt_r, X1_t, ytraj1_t, starts1_t, opt2, False)
        val_probs = predict_probs_graph(p2, ewt_s, ewt_r, X1_t, starts1_t)
        va = average_precision_score(y_val_agg, val_probs.max(axis=-1).reshape(-1))
        if va > best_val_aucpr:
            best_val_aucpr, best_state = va, copy.deepcopy(p2.state_dict())
    p2.load_state_dict(best_state)
    p2.eval()
    return p2, best_val_aucpr

def train_one_config_gru(seed, lr, dropout, weight_decay):
    torch.manual_seed(seed); np.random.seed(seed)
    p1 = StationGRU(n_feats, dropout).to(DEVICE)
    opt1 = torch.optim.Adam(p1.parameters(), lr=lr, weight_decay=weight_decay)
    best_p1_val, best_p1_state = float("inf"), None
    for epoch in range(1, MAX_EPOCHS_P1 + 1):
        run_p1_epoch_gru(p1, X0_t, yreg0_t, opt1, True)
        vl = run_p1_epoch_gru(p1, X1_t, yreg1_t, opt1, False)
        if vl < best_p1_val:
            best_p1_val, best_p1_state = vl, copy.deepcopy(p1.state_dict())
    p1.load_state_dict(best_p1_state)
    encoder_copy = copy.deepcopy(p1.station_lin)
    p2 = RegionFromTrajectoryGRU(encoder_copy, region_membership_t, dropout).to(DEVICE)
    del p1; gc.collect()
    if DEVICE.type == "mps": torch.mps.empty_cache()
    opt2 = torch.optim.Adam(p2.parameters(), lr=lr, weight_decay=weight_decay)
    y_val_agg = ytraj1_t.numpy().max(axis=-1).reshape(-1)
    best_val_aucpr, best_state = -1.0, None
    for epoch in range(1, MAX_EPOCHS_P2 + 1):
        run_p2_epoch_gru(p2, X0_t, ytraj0_t, opt2, True)
        run_p2_epoch_gru(p2, X1_t, ytraj1_t, opt2, False)
        val_probs = predict_probs_gru(p2, X1_t)
        va = average_precision_score(y_val_agg, val_probs.max(axis=-1).reshape(-1))
        if va > best_val_aucpr:
            best_val_aucpr, best_state = va, copy.deepcopy(p2.state_dict())
    p2.load_state_dict(best_state)
    p2.eval()
    return p2, best_val_aucpr

GRID = [
    {"lr": 0.003, "dropout": 0.5, "weight_decay": 0.0005},
    {"lr": 0.001, "dropout": 0.5, "weight_decay": 0.0005},
    {"lr": 0.003, "dropout": 0.3, "weight_decay": 0.0005},
    {"lr": 0.003, "dropout": 0.5, "weight_decay": 0.001},
]
N_SEEDS = 5
y_test_agg = ytraj2.max(axis=-1).reshape(-1)
all_results = {}

def run_full_pipeline(model_name, ewt_s=None, ewt_r=None, is_gru=False):
    print(f"\n{'#'*15} {model_name}: hyperparameter search on train=2016-2018/val=2019 only {'#'*15}")
    grid_results = []
    for cfg in GRID:
        t0 = time.time()
        if is_gru:
            p2, val_aucpr = train_one_config_gru(seed=0, **cfg)
        else:
            p2, val_aucpr = train_one_config_graph(seed=0, ewt_s=ewt_s, ewt_r=ewt_r, **cfg)
        print(f"  {cfg}  ->  val_AUCPR={val_aucpr:.4f}  ({time.time()-t0:.0f}s)")
        grid_results.append({**cfg, "val_aucpr": float(val_aucpr)})
        del p2; gc.collect()
        if DEVICE.type == "mps": torch.mps.empty_cache()
    best_cfg = max(grid_results, key=lambda r: r["val_aucpr"])
    print(f"  best config for {model_name}: {best_cfg}")

    print(f"\n{'#'*15} {model_name}: {N_SEEDS} final seeds {'#'*15}")
    seed_results = []
    for seed in range(N_SEEDS):
        t0 = time.time()
        if is_gru:
            p2, val_aucpr = train_one_config_gru(seed, best_cfg["lr"], best_cfg["dropout"], best_cfg["weight_decay"])
            test_probs = predict_probs_gru(p2, X2_t)
        else:
            p2, val_aucpr = train_one_config_graph(seed, ewt_s, ewt_r, best_cfg["lr"], best_cfg["dropout"], best_cfg["weight_decay"])
            test_probs = predict_probs_graph(p2, ewt_s, ewt_r, X2_t, starts2_t)
        test_agg_score = test_probs.max(axis=-1).reshape(-1)
        test_aucroc = roc_auc_score(y_test_agg, test_agg_score)
        test_aucpr = average_precision_score(y_test_agg, test_agg_score)
        print(f"  seed={seed}  ({time.time()-t0:.0f}s)  test_AUCROC={test_aucroc:.4f}  test_AUCPR={test_aucpr:.4f}")
        seed_results.append({"seed": seed, "val_aucpr": float(val_aucpr), "test_aucroc": float(test_aucroc), "test_aucpr": float(test_aucpr)})
        del p2; gc.collect()
        if DEVICE.type == "mps": torch.mps.empty_cache()

    return {"grid_search": grid_results, "best_config": best_cfg, "results": seed_results}

all_results["wind_graph"] = run_full_pipeline("wind_graph", wind_ewt_s, wind_ewt_r, is_gru=False)
all_results["no_graph"] = run_full_pipeline("no_graph", zero_ewt_s, zero_ewt_r, is_gru=False)
all_results["minimal_gru"] = run_full_pipeline("minimal_gru", is_gru=True)

print(f"\n{'='*20} SUMMARY: train=2016-2018/val=2019(stable)/test=2020, {N_SEEDS} seeds {'='*20}")
arrs = {}
for name in ["wind_graph", "no_graph", "minimal_gru"]:
    aucpr = np.array([r["test_aucpr"] for r in all_results[name]["results"]])
    aucroc = np.array([r["test_aucroc"] for r in all_results[name]["results"]])
    arrs[name] = {"aucpr": aucpr, "aucroc": aucroc}
    print(f"{name:<15} AUC-PR mean={aucpr.mean():.4f} std={aucpr.std():.4f}  |  AUC-ROC mean={aucroc.mean():.4f} std={aucroc.std():.4f}  "
          f"(best_cfg={all_results[name]['best_config']})")

for other in ["no_graph", "minimal_gru"]:
    t_stat, p_ttest = stats.ttest_rel(arrs["wind_graph"]["aucpr"], arrs[other]["aucpr"])
    w_stat, p_wilcoxon = stats.wilcoxon(arrs["wind_graph"]["aucpr"], arrs[other]["aucpr"])
    n_wins = int((arrs["wind_graph"]["aucpr"] > arrs[other]["aucpr"]).sum())
    print(f"\nwind_graph vs {other} (AUC-PR, {N_SEEDS} seeds): paired t={t_stat:.3f} p={p_ttest:.6f}  |  Wilcoxon p={p_wilcoxon:.4f}  |  wind_graph wins {n_wins}/{N_SEEDS}")

with open(f"{BASE}/2020_stable_val_3way.json", "w") as f:
    json.dump(all_results, f, indent=2)
print(f"\nsaved to {BASE}/2020_stable_val_3way.json")


device=mps
train hours (2016-2018): 26304  val hours (2019): 8760  test hours (2020): 8784
windows: train=26233 val=8689 test=8713
pos_weight: 50.00

############### wind_graph: hyperparameter search on train=2016-2018/val=2019 only ###############
  {'lr': 0.003, 'dropout': 0.5, 'weight_decay': 0.0005}  ->  val_AUCPR=0.7480  (821s)
  {'lr': 0.001, 'dropout': 0.5, 'weight_decay': 0.0005}  ->  val_AUCPR=0.7339  (798s)
  {'lr': 0.003, 'dropout': 0.3, 'weight_decay': 0.0005}  ->  val_AUCPR=0.7459  (789s)
  {'lr': 0.003, 'dropout': 0.5, 'weight_decay': 0.001}  ->  val_AUCPR=0.7478  (794s)
  best config for wind_graph: {'lr': 0.003, 'dropout': 0.5, 'weight_decay': 0.0005, 'val_aucpr': 0.7480201365200111}

############### wind_graph: 5 final seeds ###############
  seed=0  (836s)  test_AUCROC=0.9628  test_AUCPR=0.2228
  seed=1  (853s)  test_AUCROC=0.9600  test_AUCPR=0.2101
  seed=2  (851s)  test_AUCROC=0.9610  test_AUCPR=0.2255
  seed=3  (871s)  test_AUCROC=0.9594  test_AUCPR=0.2440
  seed=4

KeyboardInterrupt: 

In [1]:
# ================================================================
# CLOSE THE FAIRNESS LOOP ON THE 2019 HEADLINE RESULT:
# no_graph and minimal_gru each get their OWN independent hyperparameter
# search (on train=2016-2017/val=2018 only, test=2019 never touched),
# then 10 final seeds each. wind_graph is NOT retrained -- its already-
# established 10-seed result (from wind_vs_nograph_10seed.json) is reused.
# ================================================================
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import os, time, copy, gc, json
from sklearn.metrics import average_precision_score, roc_auc_score
from scipy import stats

BASE = "/Users/drewbaldwin/PM2_5 Research"

with open(f"{BASE}/wind_vs_nograph_10seed.json") as f:
    prior_wind_graph = json.load(f)["wind_graph"]
print(f"loaded {len(prior_wind_graph)} existing wind_graph results (already validated, not retrained)")

df = pd.read_pickle(f"{BASE}/air_korea_final_imputed_with_blh.pkl")
station_order = sorted(df["Station_ID"].unique())
n_stations = len(station_order)

stations = df[["Station_ID", "lat", "lon"]].drop_duplicates("Station_ID").set_index("Station_ID").loc[station_order]
lats, lons = stations["lat"].to_numpy(), stations["lon"].to_numpy()

def haversine_km(lat1, lon1, lat2, lon2):
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
    dlat, dlon = lat2 - lat1, lon2 - lon1
    a = np.sin(dlat/2)**2 + np.cos(lat1)*np.cos(lat2)*np.sin(dlon/2)**2
    return 2 * 6371.0 * np.arcsin(np.sqrt(a))

DIST_CUTOFF = 250.0
dist_km = haversine_km(lats[:, None], lons[:, None], lats[None, :], lons[None, :])
dist_edges = (dist_km <= DIST_CUTOFF) & (dist_km > 0)
src_idx, dst_idx = np.nonzero(dist_edges)
edge_index_np = np.stack([src_idx, dst_idx])
n_station_edges = len(src_idx)

REGION_CENTROIDS = {
    "Seoul": (37.566, 126.978), "Busan": (35.180, 129.075), "Daegu": (35.872, 128.602),
    "Incheon": (37.483, 126.633), "Gwangju": (35.155, 126.916), "Daejeon": (36.350, 127.385),
    "Ulsan": (35.550, 129.317), "Sejong": (36.487, 127.282), "Gyeonggi": (37.500, 127.250),
    "Gangwon": (37.867, 127.733), "Chungbuk": (36.633, 127.483), "Chungnam": (36.500, 126.750),
    "Jeonbuk": (35.824, 127.148), "Jeonnam": (34.750, 127.000), "Gyeongbuk": (36.559, 128.729),
    "Gyeongnam": (35.271, 128.663), "Jeju": (33.513, 126.523),
}
region_names = list(REGION_CENTROIDS.keys())
n_regions = len(region_names)
region_lats = np.array([REGION_CENTROIDS[r][0] for r in region_names])
region_lons = np.array([REGION_CENTROIDS[r][1] for r in region_names])
dist_to_region = haversine_km(lats[:, None], lons[:, None], region_lats[None, :], region_lons[None, :])
station_region_idx = dist_to_region.argmin(axis=1)

region_membership = np.zeros((n_stations, n_regions), dtype=np.float32)
region_membership[np.arange(n_stations), station_region_idx] = 1.0
region_membership_t = torch.tensor(region_membership)

r_src_idx, r_dst_idx = np.nonzero(~np.eye(n_regions, dtype=bool))
region_edge_index_np = np.stack([r_src_idx, r_dst_idx])
n_region_edges = len(r_src_idx)

WINDOW, HORIZON = 36, 36
EVENT_THRESHOLD = 75.0
SUSTAIN_HOURS = 2
QUANTILES = [0.50, 0.75, 0.90, 0.95, 0.99]
N_QUANTILES = len(QUANTILES)
GRAPH_RECENT_HOURS = 18
TIME_FEATS = ["SO2", "CO", "NO2", "O3", "PM10", "PM25"]
STATIC_COLS = ["elevation_m", "urban_landuse_area_m2_3km", "green_space_area_3km",
               "building_footprint_area_3km", "railway_length_3km", "dist_to_coast_km",
               "dist_to_major_road_km", "industrial_area_m2_3km", "traffic_points_count_3km",
               "major_roads_count_3km", "total_road_length_3km"]

time_panels = {c: df.pivot(index="Datetime", columns="Station_ID", values=c)[station_order] for c in TIME_FEATS}
dt_index = time_panels["PM25"].index
n_time = len(dt_index)
years = dt_index.year.to_numpy()
doy = dt_index.dayofyear.to_numpy().astype(float)

split_id_per_hour = np.where((years == 2016) | (years == 2017), 0, np.where(years == 2018, 1, np.where(years == 2019, 2, -1)))
TRAIN_MASK = split_id_per_hour == 0

wind_speed_arr = df.pivot(index="Datetime", columns="Station_ID", values="windspeed_10m")[station_order].reindex(dt_index).to_numpy().astype(float)
wind_dir_arr = df.pivot(index="Datetime", columns="Station_ID", values="winddirection_10m")[station_order].reindex(dt_index).to_numpy().astype(float)
blh_arr = df.pivot(index="Datetime", columns="Station_ID", values="boundary_layer_height")[station_order].reindex(dt_index).to_numpy().astype(float)

wdir_sin_station = np.sin(np.radians(wind_dir_arr))
wdir_cos_station = np.cos(np.radians(wind_dir_arr))
season_sin_1d = np.sin(2 * np.pi * doy / 365.25)
season_cos_1d = np.cos(2 * np.pi * doy / 365.25)
season_sin = np.tile(season_sin_1d[:, None], (1, n_stations))
season_cos = np.tile(season_cos_1d[:, None], (1, n_stations))

TIME_FEATS_FULL = TIME_FEATS + ["windspeed_10m", "wdir_sin", "wdir_cos", "boundary_layer_height", "season_sin", "season_cos"]
n_time_feats, n_static_feats = len(TIME_FEATS_FULL), len(STATIC_COLS)
n_feats = n_time_feats + n_static_feats
pm25_col_idx = TIME_FEATS_FULL.index("PM25")

time_arr_raw = np.stack([time_panels[c].to_numpy() for c in TIME_FEATS] +
                         [wind_speed_arr, wdir_sin_station, wdir_cos_station, blh_arr, season_sin, season_cos], axis=-1)
del time_panels, season_sin, season_cos, blh_arr, wind_speed_arr, wind_dir_arr, wdir_sin_station, wdir_cos_station
gc.collect()

static_df = df[["Station_ID"] + STATIC_COLS].drop_duplicates("Station_ID").set_index("Station_ID").loc[station_order]
static_arr = static_df[STATIC_COLS].to_numpy()

region_pm25 = np.zeros((n_time, n_regions), dtype=np.float32)
pm25_station = time_arr_raw[:, :, pm25_col_idx]
for r in range(n_regions):
    cols = station_region_idx == r
    region_pm25[:, r] = np.nanmean(pm25_station[:, cols], axis=1)
del df
gc.collect()

rev = region_pm25[::-1]
roll_min_rev = pd.DataFrame(rev).rolling(window=SUSTAIN_HOURS, min_periods=SUSTAIN_HOURS).min().to_numpy()
region_episode_label = (roll_min_rev[::-1] >= EVENT_THRESHOLD).astype(np.float32)
del rev, roll_min_rev
gc.collect()

def pinball_loss(preds, target, quantiles):
    target_exp = target.unsqueeze(-1)
    diff = target_exp - preds
    q_tensor = torch.tensor(quantiles, device=preds.device, dtype=preds.dtype).view(*([1] * (preds.dim() - 1)), -1)
    return torch.max(q_tensor * diff, (q_tensor - 1) * diff).mean()

def monotonic_quantiles(raw):
    first = raw[..., :1]
    deltas = F.softplus(raw[..., 1:])
    return torch.cat([first, first + torch.cumsum(deltas, dim=-1)], dim=-1)

class WindConvLayer(nn.Module):
    def __init__(self, in_dim, out_dim):
        super().__init__()
        self.lin_self = nn.Linear(in_dim, out_dim)
        self.lin_neigh = nn.Linear(in_dim, out_dim)
        self.lin_connectivity = nn.Linear(1, out_dim)

    def forward(self, x, edge_index, edge_weight, num_nodes):
        src, dst = edge_index[0], edge_index[1]
        messages = x[src] * edge_weight.unsqueeze(-1)
        agg_sum = x.new_zeros(num_nodes, x.size(-1))
        agg_sum.index_add_(0, dst, messages)
        weight_sum = x.new_zeros(num_nodes)
        weight_sum.index_add_(0, dst, edge_weight)
        agg_mean = agg_sum / (weight_sum.unsqueeze(-1) + 1e-8)
        connectivity = torch.log1p(weight_sum.clamp(min=0)).unsqueeze(-1)
        return self.lin_self(x) + self.lin_neigh(agg_mean) + self.lin_connectivity(connectivity)

class AttentionPool(nn.Module):
    def __init__(self, hidden):
        super().__init__()
        self.attn_score = nn.Linear(hidden, 1)

    def forward(self, h_station, region_membership_t_local):
        B, N, H = h_station.shape
        scores = self.attn_score(h_station).squeeze(-1)
        scores = scores - scores.max(dim=1, keepdim=True).values
        exp_scores = torch.exp(scores)
        weighted_exp = exp_scores.unsqueeze(-1) * region_membership_t_local.unsqueeze(0)
        region_denom = weighted_exp.sum(dim=1)
        region_numer = torch.einsum('bnr,bnh->brh', weighted_exp, h_station)
        return region_numer / (region_denom.unsqueeze(-1) + 1e-8)

class StationQuantileGCN(nn.Module):
    def __init__(self, in_dim, dropout, hidden=32, gru_hidden=32):
        super().__init__()
        self.station_conv = WindConvLayer(in_dim, hidden)
        self.drop = nn.Dropout(dropout)
        self.gru = nn.GRU(hidden, gru_hidden, batch_first=True)
        self.head = nn.Linear(gru_hidden, N_QUANTILES)

    def forward(self, x_window, edge_index, edge_weight_seq):
        B, W, N, Fin = x_window.shape
        ei_b = torch.cat([edge_index + i * N for i in range(B)], dim=1)
        num_nodes = B * N
        h_seq = []
        for w in range(W):
            xt = x_window[:, w].reshape(B * N, Fin)
            ew_b = edge_weight_seq[:, w].reshape(-1)
            h = torch.relu(self.station_conv(xt, ei_b, ew_b, num_nodes))
            h = self.drop(h)
            h_seq.append(h.reshape(B, N, -1))
        h_seq = torch.stack(h_seq, dim=1).permute(0, 2, 1, 3).reshape(B * N, W, -1)
        _, h_final = self.gru(h_seq)
        embed = self.drop(h_final.squeeze(0).reshape(B, N, -1))
        return monotonic_quantiles(self.head(embed))

class RegionFromTrajectoryGCN(nn.Module):
    def __init__(self, station_conv, region_membership_t_local, dropout, hidden=32, gru_hidden=32):
        super().__init__()
        self.station_conv = station_conv
        self.attn_pool = AttentionPool(hidden)
        self.region_conv = WindConvLayer(hidden, hidden)
        self.drop = nn.Dropout(dropout)
        self.region_gru = nn.GRU(hidden, gru_hidden, batch_first=True)
        self.region_head = nn.Linear(gru_hidden, HORIZON)
        self.rmem = region_membership_t_local

    def forward(self, x_window, station_edge_index, station_edge_weight_seq, region_edge_index, region_edge_weight_seq, n_reg):
        B, W, N, Fin = x_window.shape
        station_ei_b = torch.cat([station_edge_index + i * N for i in range(B)], dim=1)
        region_ei_b = torch.cat([region_edge_index + i * n_reg for i in range(B)], dim=1)
        num_station_nodes = B * N
        num_region_nodes = B * n_reg
        h_region_seq = []
        for w in range(W):
            xt = x_window[:, w].reshape(B * N, Fin)
            ew_station_b = station_edge_weight_seq[:, w].reshape(-1)
            h_station = torch.relu(self.station_conv(xt, station_ei_b, ew_station_b, num_station_nodes))
            h_station = self.drop(h_station).reshape(B, N, -1)
            h_region_pooled = self.attn_pool(h_station, self.rmem).reshape(B * n_reg, -1)
            ew_region_b = region_edge_weight_seq[:, w].reshape(-1)
            h_region = torch.relu(self.region_conv(h_region_pooled, region_ei_b, ew_region_b, num_region_nodes))
            h_region = self.drop(h_region)
            h_region_seq.append(h_region.reshape(B, n_reg, -1))
        h_region_seq = torch.stack(h_region_seq, dim=1).permute(0, 2, 1, 3).reshape(B * n_reg, W, -1)
        _, h_final = self.region_gru(h_region_seq)
        embed = self.drop(h_final.squeeze(0).reshape(B, n_reg, -1))
        return self.region_head(embed)

class StationGRU(nn.Module):
    def __init__(self, in_dim, dropout, hidden=32, gru_hidden=32):
        super().__init__()
        self.station_lin = nn.Linear(in_dim, hidden)
        self.drop = nn.Dropout(dropout)
        self.gru = nn.GRU(hidden, gru_hidden, batch_first=True)
        self.head = nn.Linear(gru_hidden, N_QUANTILES)

    def forward(self, x_window):
        B, W, N, Fin = x_window.shape
        h_seq = []
        for w in range(W):
            xt = x_window[:, w].reshape(B * N, Fin)
            h = torch.relu(self.station_lin(xt))
            h = self.drop(h)
            h_seq.append(h.reshape(B, N, -1))
        h_seq = torch.stack(h_seq, dim=1).permute(0, 2, 1, 3).reshape(B * N, W, -1)
        _, h_final = self.gru(h_seq)
        embed = self.drop(h_final.squeeze(0).reshape(B, N, -1))
        return monotonic_quantiles(self.head(embed))

class RegionFromTrajectoryGRU(nn.Module):
    def __init__(self, station_lin, region_membership_t_local, dropout, hidden=32, gru_hidden=32):
        super().__init__()
        self.station_lin = station_lin
        self.attn_pool = AttentionPool(hidden)
        self.drop = nn.Dropout(dropout)
        self.region_gru = nn.GRU(hidden, gru_hidden, batch_first=True)
        self.region_head = nn.Linear(gru_hidden, HORIZON)
        self.rmem = region_membership_t_local

    def forward(self, x_window, n_reg):
        B, W, N, Fin = x_window.shape
        h_region_seq = []
        for w in range(W):
            xt = x_window[:, w].reshape(B * N, Fin)
            h_station = torch.relu(self.station_lin(xt)).reshape(B, N, -1)
            h_station = self.drop(h_station)
            h_region_pooled = self.attn_pool(h_station, self.rmem)
            h_region_pooled = self.drop(h_region_pooled)
            h_region_seq.append(h_region_pooled)
        h_region_seq = torch.stack(h_region_seq, dim=1).permute(0, 2, 1, 3).reshape(B * n_reg, W, -1)
        _, h_final = self.region_gru(h_region_seq)
        embed = self.drop(h_final.squeeze(0).reshape(B, n_reg, -1))
        return self.region_head(embed)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else ("mps" if torch.backends.mps.is_available() else "cpu"))
MICRO_BATCH, ACCUM_STEPS = 16, 4
MAX_EPOCHS_P1, MAX_EPOCHS_P2 = 2, 2
print(f"device={DEVICE}")

region_membership_t = region_membership_t.to(DEVICE)
edge_index = torch.tensor(edge_index_np, dtype=torch.long).to(DEVICE)
region_edge_index = torch.tensor(region_edge_index_np, dtype=torch.long).to(DEVICE)

def add_static_fn(x_time_batch, static_tensor_local):
    B, W, N, _ = x_time_batch.shape
    static_b = static_tensor_local.unsqueeze(0).unsqueeze(0).expand(B, W, N, n_static_feats)
    return torch.cat([x_time_batch, static_b], dim=-1)

def gather_seq(ew_by_hour_t, starts_subset):
    idx = starts_subset.unsqueeze(1) + torch.arange(WINDOW).unsqueeze(0)
    ew = ew_by_hour_t[idx]
    ew = ew.clone()
    ew[:, :WINDOW - GRAPH_RECENT_HOURS, :] = 0.0
    return ew

print(f"train hours (2016-2017): {TRAIN_MASK.sum()}  val hours (2018): {(split_id_per_hour==1).sum()}  test hours (2019): {(split_id_per_hour==2).sum()}")

t_mean = np.nanmean(time_arr_raw[TRAIN_MASK], axis=(0, 1), keepdims=True)
t_std = np.nanstd(time_arr_raw[TRAIN_MASK], axis=(0, 1), keepdims=True) + 1e-6
time_arr_std = np.nan_to_num((time_arr_raw - t_mean) / t_std, nan=0.0)
s_mean, s_std = static_arr.mean(axis=0, keepdims=True), static_arr.std(axis=0, keepdims=True) + 1e-6
static_tensor = torch.tensor((static_arr - s_mean) / s_std, dtype=torch.float32).to(DEVICE)

p_buckets = {0: ([], [], [], []), 1: ([], [], [], []), 2: ([], [], [], [])}
for t in range(0, n_time - WINDOW - HORIZON + 1):
    target_t = t + WINDOW + HORIZON - 1
    s_start, s_target = split_id_per_hour[t], split_id_per_hour[target_t]
    if s_start != s_target or s_start == -1:
        continue
    x_win = time_arr_std[t:t + WINDOW]
    traj = region_episode_label[t + WINDOW: t + WINDOW + HORIZON]
    Xl, yregl, ytrajl, sl = p_buckets[s_start]
    Xl.append(x_win)
    yregl.append(time_arr_std[target_t, :, pm25_col_idx])
    ytrajl.append(traj.T)
    sl.append(t)

X0, yreg0, ytraj0, starts0 = (np.stack(v) for v in p_buckets[0])
X1, yreg1, ytraj1, starts1 = (np.stack(v) for v in p_buckets[1])
X2, yreg2, ytraj2, starts2 = (np.stack(v) for v in p_buckets[2])
del p_buckets, time_arr_std
gc.collect()
print(f"windows: train={len(X0)} val={len(X1)} test={len(X2)}")

X0_t = torch.tensor(X0, dtype=torch.float32); del X0
X1_t = torch.tensor(X1, dtype=torch.float32); del X1
X2_t = torch.tensor(X2, dtype=torch.float32); del X2
gc.collect()

yreg0_t, yreg1_t = torch.tensor(yreg0, dtype=torch.float32), torch.tensor(yreg1, dtype=torch.float32)
ytraj0_t = torch.tensor(ytraj0, dtype=torch.float32)
ytraj1_t = torch.tensor(ytraj1, dtype=torch.float32)
starts0_t, starts1_t, starts2_t = (torch.tensor(a, dtype=torch.long) for a in (starts0, starts1, starts2))
del yreg0, yreg1
gc.collect()

POS_WEIGHT = min(float((ytraj0_t.numel() - ytraj0_t.sum()) / ytraj0_t.sum().clamp(min=1)), 50.0)
print(f"pos_weight: {POS_WEIGHT:.2f}")

zero_ewt_s = torch.zeros((n_time, n_station_edges), dtype=torch.float32)
zero_ewt_r = torch.zeros((n_time, n_region_edges), dtype=torch.float32)

region_criterion = nn.BCEWithLogitsLoss(pos_weight=torch.tensor(POS_WEIGHT))

def run_p1_epoch_graph(model, ewt, X, y, starts, optimizer, train):
    n = X.shape[0]
    idx = torch.randperm(n) if train else torch.arange(n)
    model.train(train)
    total_loss, total_n = 0.0, 0
    eff_batch = MICRO_BATCH * ACCUM_STEPS
    for start in range(0, n, eff_batch):
        if train: optimizer.zero_grad()
        batch_idx = idx[start:start + eff_batch]
        for ms in range(0, len(batch_idx), MICRO_BATCH):
            mb_idx = batch_idx[ms:ms + MICRO_BATCH]
            if len(mb_idx) == 0: continue
            xb = add_static_fn(X[mb_idx].to(DEVICE), static_tensor)
            yb = y[mb_idx].to(DEVICE)
            ew_seq = gather_seq(ewt, starts[mb_idx]).to(DEVICE)
            with torch.set_grad_enabled(train):
                pred = model(xb, edge_index, ew_seq)
                loss = pinball_loss(pred, yb, QUANTILES)
            if train: (loss * len(mb_idx) / len(batch_idx)).backward()
            total_loss += loss.item() * len(mb_idx); total_n += len(mb_idx)
        if train: optimizer.step()
    return total_loss / total_n

def run_p2_epoch_graph(model, ewt_s, ewt_r, X, y_traj, starts, optimizer, train):
    n = X.shape[0]
    idx = torch.randperm(n) if train else torch.arange(n)
    model.train(train)
    total_loss, total_n = 0.0, 0
    eff_batch = MICRO_BATCH * ACCUM_STEPS
    for start in range(0, n, eff_batch):
        if train: optimizer.zero_grad()
        batch_idx = idx[start:start + eff_batch]
        for ms in range(0, len(batch_idx), MICRO_BATCH):
            mb_idx = batch_idx[ms:ms + MICRO_BATCH]
            if len(mb_idx) == 0: continue
            xb = add_static_fn(X[mb_idx].to(DEVICE), static_tensor)
            yb = y_traj[mb_idx].to(DEVICE)
            ew_s = gather_seq(ewt_s, starts[mb_idx]).to(DEVICE)
            ew_r = gather_seq(ewt_r, starts[mb_idx]).to(DEVICE)
            with torch.set_grad_enabled(train):
                logits = model(xb, edge_index, ew_s, region_edge_index, ew_r, n_regions)
                loss = region_criterion(logits, yb)
            if train: loss.backward()
            total_loss += loss.item() * len(mb_idx); total_n += len(mb_idx)
        if train: optimizer.step()
    return total_loss / max(total_n, 1)

def predict_probs_graph(model, ewt_s, ewt_r, X, starts, batch_size=64):
    model.eval()
    n = X.shape[0]
    out = np.zeros((n, n_regions, HORIZON), dtype=np.float32)
    with torch.no_grad():
        for start in range(0, n, batch_size):
            idx = torch.arange(start, min(start + batch_size, n))
            xb = add_static_fn(X[idx].to(DEVICE), static_tensor)
            ew_s = gather_seq(ewt_s, starts[idx]).to(DEVICE)
            ew_r = gather_seq(ewt_r, starts[idx]).to(DEVICE)
            logits = model(xb, edge_index, ew_s, region_edge_index, ew_r, n_regions)
            out[idx.numpy()] = torch.sigmoid(logits).cpu().numpy()
    return out

def run_p1_epoch_gru(model, X, y, optimizer, train):
    n = X.shape[0]
    idx = torch.randperm(n) if train else torch.arange(n)
    model.train(train)
    total_loss, total_n = 0.0, 0
    eff_batch = MICRO_BATCH * ACCUM_STEPS
    for start in range(0, n, eff_batch):
        if train: optimizer.zero_grad()
        batch_idx = idx[start:start + eff_batch]
        for ms in range(0, len(batch_idx), MICRO_BATCH):
            mb_idx = batch_idx[ms:ms + MICRO_BATCH]
            if len(mb_idx) == 0: continue
            xb = add_static_fn(X[mb_idx].to(DEVICE), static_tensor)
            yb = y[mb_idx].to(DEVICE)
            with torch.set_grad_enabled(train):
                pred = model(xb)
                loss = pinball_loss(pred, yb, QUANTILES)
            if train: (loss * len(mb_idx) / len(batch_idx)).backward()
            total_loss += loss.item() * len(mb_idx); total_n += len(mb_idx)
        if train: optimizer.step()
    return total_loss / total_n

def run_p2_epoch_gru(model, X, y_traj, optimizer, train):
    n = X.shape[0]
    idx = torch.randperm(n) if train else torch.arange(n)
    model.train(train)
    total_loss, total_n = 0.0, 0
    eff_batch = MICRO_BATCH * ACCUM_STEPS
    for start in range(0, n, eff_batch):
        if train: optimizer.zero_grad()
        batch_idx = idx[start:start + eff_batch]
        for ms in range(0, len(batch_idx), MICRO_BATCH):
            mb_idx = batch_idx[ms:ms + MICRO_BATCH]
            if len(mb_idx) == 0: continue
            xb = add_static_fn(X[mb_idx].to(DEVICE), static_tensor)
            yb = y_traj[mb_idx].to(DEVICE)
            with torch.set_grad_enabled(train):
                logits = model(xb, n_regions)
                loss = region_criterion(logits, yb)
            if train: loss.backward()
            total_loss += loss.item() * len(mb_idx); total_n += len(mb_idx)
        if train: optimizer.step()
    return total_loss / max(total_n, 1)

def predict_probs_gru(model, X, batch_size=64):
    model.eval()
    n = X.shape[0]
    out = np.zeros((n, n_regions, HORIZON), dtype=np.float32)
    with torch.no_grad():
        for start in range(0, n, batch_size):
            idx = torch.arange(start, min(start + batch_size, n))
            xb = add_static_fn(X[idx].to(DEVICE), static_tensor)
            logits = model(xb, n_regions)
            out[idx.numpy()] = torch.sigmoid(logits).cpu().numpy()
    return out

def train_one_config_graph(seed, ewt_s, ewt_r, lr, dropout, weight_decay):
    torch.manual_seed(seed); np.random.seed(seed)
    p1 = StationQuantileGCN(n_feats, dropout).to(DEVICE)
    opt1 = torch.optim.Adam(p1.parameters(), lr=lr, weight_decay=weight_decay)
    best_p1_val, best_p1_state = float("inf"), None
    for epoch in range(1, MAX_EPOCHS_P1 + 1):
        run_p1_epoch_graph(p1, ewt_s, X0_t, yreg0_t, starts0_t, opt1, True)
        vl = run_p1_epoch_graph(p1, ewt_s, X1_t, yreg1_t, starts1_t, opt1, False)
        if vl < best_p1_val:
            best_p1_val, best_p1_state = vl, copy.deepcopy(p1.state_dict())
    p1.load_state_dict(best_p1_state)
    encoder_copy = copy.deepcopy(p1.station_conv)
    p2 = RegionFromTrajectoryGCN(encoder_copy, region_membership_t, dropout).to(DEVICE)
    del p1; gc.collect()
    if DEVICE.type == "mps": torch.mps.empty_cache()
    opt2 = torch.optim.Adam(p2.parameters(), lr=lr, weight_decay=weight_decay)
    y_val_agg = ytraj1_t.numpy().max(axis=-1).reshape(-1)
    best_val_aucpr, best_state = -1.0, None
    for epoch in range(1, MAX_EPOCHS_P2 + 1):
        run_p2_epoch_graph(p2, ewt_s, ewt_r, X0_t, ytraj0_t, starts0_t, opt2, True)
        run_p2_epoch_graph(p2, ewt_s, ewt_r, X1_t, ytraj1_t, starts1_t, opt2, False)
        val_probs = predict_probs_graph(p2, ewt_s, ewt_r, X1_t, starts1_t)
        va = average_precision_score(y_val_agg, val_probs.max(axis=-1).reshape(-1))
        if va > best_val_aucpr:
            best_val_aucpr, best_state = va, copy.deepcopy(p2.state_dict())
    p2.load_state_dict(best_state)
    p2.eval()
    return p2, best_val_aucpr

def train_one_config_gru(seed, lr, dropout, weight_decay):
    torch.manual_seed(seed); np.random.seed(seed)
    p1 = StationGRU(n_feats, dropout).to(DEVICE)
    opt1 = torch.optim.Adam(p1.parameters(), lr=lr, weight_decay=weight_decay)
    best_p1_val, best_p1_state = float("inf"), None
    for epoch in range(1, MAX_EPOCHS_P1 + 1):
        run_p1_epoch_gru(p1, X0_t, yreg0_t, opt1, True)
        vl = run_p1_epoch_gru(p1, X1_t, yreg1_t, opt1, False)
        if vl < best_p1_val:
            best_p1_val, best_p1_state = vl, copy.deepcopy(p1.state_dict())
    p1.load_state_dict(best_p1_state)
    encoder_copy = copy.deepcopy(p1.station_lin)
    p2 = RegionFromTrajectoryGRU(encoder_copy, region_membership_t, dropout).to(DEVICE)
    del p1; gc.collect()
    if DEVICE.type == "mps": torch.mps.empty_cache()
    opt2 = torch.optim.Adam(p2.parameters(), lr=lr, weight_decay=weight_decay)
    y_val_agg = ytraj1_t.numpy().max(axis=-1).reshape(-1)
    best_val_aucpr, best_state = -1.0, None
    for epoch in range(1, MAX_EPOCHS_P2 + 1):
        run_p2_epoch_gru(p2, X0_t, ytraj0_t, opt2, True)
        run_p2_epoch_gru(p2, X1_t, ytraj1_t, opt2, False)
        val_probs = predict_probs_gru(p2, X1_t)
        va = average_precision_score(y_val_agg, val_probs.max(axis=-1).reshape(-1))
        if va > best_val_aucpr:
            best_val_aucpr, best_state = va, copy.deepcopy(p2.state_dict())
    p2.load_state_dict(best_state)
    p2.eval()
    return p2, best_val_aucpr

GRID = [
    {"lr": 0.003, "dropout": 0.5, "weight_decay": 0.0005},
    {"lr": 0.001, "dropout": 0.5, "weight_decay": 0.0005},
    {"lr": 0.003, "dropout": 0.3, "weight_decay": 0.0005},
    {"lr": 0.003, "dropout": 0.5, "weight_decay": 0.001},
]
N_SEEDS = 10
y_test_agg = ytraj2.max(axis=-1).reshape(-1)
all_results = {}

def run_full_pipeline(model_name, ewt_s=None, ewt_r=None, is_gru=False):
    print(f"\n{'#'*15} {model_name}: hyperparameter search on train=2016-2017/val=2018 only {'#'*15}")
    grid_results = []
    for cfg in GRID:
        t0 = time.time()
        if is_gru:
            p2, val_aucpr = train_one_config_gru(seed=0, **cfg)
        else:
            p2, val_aucpr = train_one_config_graph(seed=0, ewt_s=ewt_s, ewt_r=ewt_r, **cfg)
        print(f"  {cfg}  ->  val_AUCPR={val_aucpr:.4f}  ({time.time()-t0:.0f}s)")
        grid_results.append({**cfg, "val_aucpr": float(val_aucpr)})
        del p2; gc.collect()
        if DEVICE.type == "mps": torch.mps.empty_cache()
    best_cfg = max(grid_results, key=lambda r: r["val_aucpr"])
    print(f"  best config for {model_name}: {best_cfg}")

    print(f"\n{'#'*15} {model_name}: {N_SEEDS} final seeds {'#'*15}")
    seed_results = []
    for seed in range(N_SEEDS):
        t0 = time.time()
        if is_gru:
            p2, val_aucpr = train_one_config_gru(seed, best_cfg["lr"], best_cfg["dropout"], best_cfg["weight_decay"])
            test_probs = predict_probs_gru(p2, X2_t)
        else:
            p2, val_aucpr = train_one_config_graph(seed, ewt_s, ewt_r, best_cfg["lr"], best_cfg["dropout"], best_cfg["weight_decay"])
            test_probs = predict_probs_graph(p2, ewt_s, ewt_r, X2_t, starts2_t)
        test_agg_score = test_probs.max(axis=-1).reshape(-1)
        test_aucroc = roc_auc_score(y_test_agg, test_agg_score)
        test_aucpr = average_precision_score(y_test_agg, test_agg_score)
        print(f"  seed={seed}  ({time.time()-t0:.0f}s)  test_AUCROC={test_aucroc:.4f}  test_AUCPR={test_aucpr:.4f}")
        seed_results.append({"seed": seed, "val_aucpr": float(val_aucpr), "test_aucroc": float(test_aucroc), "test_aucpr": float(test_aucpr)})
        del p2; gc.collect()
        if DEVICE.type == "mps": torch.mps.empty_cache()

    return {"grid_search": grid_results, "best_config": best_cfg, "results": seed_results}

all_results["no_graph"] = run_full_pipeline("no_graph", zero_ewt_s, zero_ewt_r, is_gru=False)
all_results["minimal_gru"] = run_full_pipeline("minimal_gru", is_gru=True)

print(f"\n{'='*20} SUMMARY: 2019 headline, fair independent tuning, {N_SEEDS} seeds {'='*20}")
wind_aucpr = np.array([r["test_aucpr"] for r in prior_wind_graph])
wind_aucroc = np.array([r["test_aucroc"] for r in prior_wind_graph])
print(f"{'wind_graph (existing)':<25} AUC-PR mean={wind_aucpr.mean():.4f} std={wind_aucpr.std():.4f}  |  AUC-ROC mean={wind_aucroc.mean():.4f} std={wind_aucroc.std():.4f}")
arrs = {"wind_graph": {"aucpr": wind_aucpr, "aucroc": wind_aucroc}}
for name in ["no_graph", "minimal_gru"]:
    aucpr = np.array([r["test_aucpr"] for r in all_results[name]["results"]])
    aucroc = np.array([r["test_aucroc"] for r in all_results[name]["results"]])
    arrs[name] = {"aucpr": aucpr, "aucroc": aucroc}
    print(f"{name:<25} AUC-PR mean={aucpr.mean():.4f} std={aucpr.std():.4f}  |  AUC-ROC mean={aucroc.mean():.4f} std={aucroc.std():.4f}  (best_cfg={all_results[name]['best_config']})")

for other in ["no_graph", "minimal_gru"]:
    t_stat, p_ttest = stats.ttest_rel(arrs["wind_graph"]["aucpr"], arrs[other]["aucpr"])
    w_stat, p_wilcoxon = stats.wilcoxon(arrs["wind_graph"]["aucpr"], arrs[other]["aucpr"])
    n_wins = int((arrs["wind_graph"]["aucpr"] > arrs[other]["aucpr"]).sum())
    print(f"\nwind_graph vs {other} (AUC-PR, {N_SEEDS} seeds): paired t={t_stat:.3f} p={p_ttest:.6f}  |  Wilcoxon p={p_wilcoxon:.4f}  |  wind_graph wins {n_wins}/{N_SEEDS}")

with open(f"{BASE}/2019_headline_fair_tuning_check.json", "w") as f:
    json.dump({"wind_graph_existing": prior_wind_graph, "no_graph": all_results["no_graph"], "minimal_gru": all_results["minimal_gru"]}, f, indent=2)
print(f"\nsaved to {BASE}/2019_headline_fair_tuning_check.json")


loaded 10 existing wind_graph results (already validated, not retrained)
device=mps
train hours (2016-2017): 17544  val hours (2018): 8760  test hours (2019): 8760
windows: train=17473 val=8689 test=8689
pos_weight: 50.00

############### no_graph: hyperparameter search on train=2016-2017/val=2018 only ###############
  {'lr': 0.003, 'dropout': 0.5, 'weight_decay': 0.0005}  ->  val_AUCPR=0.4604  (544s)
  {'lr': 0.001, 'dropout': 0.5, 'weight_decay': 0.0005}  ->  val_AUCPR=0.4681  (540s)
  {'lr': 0.003, 'dropout': 0.3, 'weight_decay': 0.0005}  ->  val_AUCPR=0.4672  (539s)
  {'lr': 0.003, 'dropout': 0.5, 'weight_decay': 0.001}  ->  val_AUCPR=0.4637  (538s)
  best config for no_graph: {'lr': 0.001, 'dropout': 0.5, 'weight_decay': 0.0005, 'val_aucpr': 0.4681068234932387}

############### no_graph: 10 final seeds ###############
  seed=0  (573s)  test_AUCROC=0.9474  test_AUCPR=0.6777
  seed=1  (577s)  test_AUCROC=0.9495  test_AUCPR=0.6823
  seed=2  (579s)  test_AUCROC=0.9501  test_AUCPR=0.6

In [1]:
# ================================================================
# 2018 TEST YEAR, FULL 10-SEED FAIR COMPARISON:
# wind_graph: reuse existing tuned config (lr=0.001, dropout=0.5, wd=0.0005,
#   found via grid search on train=2016/val=2017) + reuse existing 5 seeds
#   + train 5 NEW seeds (5-9) with that same config = 10 total.
# no_graph, minimal_gru: each get their OWN independent 4-config grid search
#   (train=2016/val=2017 only, test=2018 never touched), then 10 seeds each.
# ================================================================
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import os, time, copy, gc, json
from sklearn.metrics import average_precision_score, roc_auc_score
from scipy import stats

BASE = "/Users/drewbaldwin/PM2_5 Research"

with open(f"{BASE}/wind_vs_gru_2016_2017_2018_split_5seed.json") as f:
    prior = json.load(f)
wind_graph_existing_5 = prior["wind_graph"]  # seeds 0-4
wind_graph_best_cfg = prior["best_config"]   # {'lr': 0.001, 'dropout': 0.5, 'weight_decay': 0.0005, ...}
print(f"loaded wind_graph's existing 5 seeds and its tuned config: {wind_graph_best_cfg}")

df = pd.read_pickle(f"{BASE}/air_korea_final_imputed_with_blh.pkl")
station_order = sorted(df["Station_ID"].unique())
n_stations = len(station_order)

stations = df[["Station_ID", "lat", "lon"]].drop_duplicates("Station_ID").set_index("Station_ID").loc[station_order]
lats, lons = stations["lat"].to_numpy(), stations["lon"].to_numpy()

def haversine_km(lat1, lon1, lat2, lon2):
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
    dlat, dlon = lat2 - lat1, lon2 - lon1
    a = np.sin(dlat/2)**2 + np.cos(lat1)*np.cos(lat2)*np.sin(dlon/2)**2
    return 2 * 6371.0 * np.arcsin(np.sqrt(a))

def bearing_matrix(lat, lon):
    lat_r, lon_r = np.radians(lat), np.radians(lon)
    lat1, lat2 = lat_r[:, None], lat_r[None, :]
    dlon = lon_r[None, :] - lon_r[:, None]
    x = np.sin(dlon) * np.cos(lat2)
    y = np.cos(lat1) * np.sin(lat2) - np.sin(lat1) * np.cos(lat2) * np.cos(dlon)
    return (np.degrees(np.arctan2(x, y)) + 360) % 360

dist_km = haversine_km(lats[:, None], lons[:, None], lats[None, :], lons[None, :])
DIST_CUTOFF, RHO_KM = 250.0, 250.0
HYBRID_THRESHOLD_KM = 20.0
dist_edges = (dist_km <= DIST_CUTOFF) & (dist_km > 0)
bearing_from = bearing_matrix(lats, lons)

src_idx, dst_idx = np.nonzero(dist_edges)
edge_index_np = np.stack([src_idx, dst_idx])
decay_edge = np.exp(-dist_km[src_idx, dst_idx] / RHO_KM).astype(np.float32)
bearing_edge = bearing_from[src_idx, dst_idx].astype(np.float32)
close_edge_mask = (dist_km[src_idx, dst_idx] <= HYBRID_THRESHOLD_KM)

REGION_CENTROIDS = {
    "Seoul": (37.566, 126.978), "Busan": (35.180, 129.075), "Daegu": (35.872, 128.602),
    "Incheon": (37.483, 126.633), "Gwangju": (35.155, 126.916), "Daejeon": (36.350, 127.385),
    "Ulsan": (35.550, 129.317), "Sejong": (36.487, 127.282), "Gyeonggi": (37.500, 127.250),
    "Gangwon": (37.867, 127.733), "Chungbuk": (36.633, 127.483), "Chungnam": (36.500, 126.750),
    "Jeonbuk": (35.824, 127.148), "Jeonnam": (34.750, 127.000), "Gyeongbuk": (36.559, 128.729),
    "Gyeongnam": (35.271, 128.663), "Jeju": (33.513, 126.523),
}
region_names = list(REGION_CENTROIDS.keys())
n_regions = len(region_names)
region_lats = np.array([REGION_CENTROIDS[r][0] for r in region_names])
region_lons = np.array([REGION_CENTROIDS[r][1] for r in region_names])
dist_to_region = haversine_km(lats[:, None], lons[:, None], region_lats[None, :], region_lons[None, :])
station_region_idx = dist_to_region.argmin(axis=1)

region_membership = np.zeros((n_stations, n_regions), dtype=np.float32)
region_membership[np.arange(n_stations), station_region_idx] = 1.0
region_membership_t = torch.tensor(region_membership)

region_dist_km = haversine_km(region_lats[:, None], region_lons[:, None], region_lats[None, :], region_lons[None, :])
region_bearing = bearing_matrix(region_lats, region_lons)
r_src_idx, r_dst_idx = np.nonzero(~np.eye(n_regions, dtype=bool))
region_edge_index_np = np.stack([r_src_idx, r_dst_idx])
region_decay_edge = np.exp(-region_dist_km[r_src_idx, r_dst_idx] / RHO_KM).astype(np.float32)
region_bearing_edge = region_bearing[r_src_idx, r_dst_idx].astype(np.float32)

WINDOW, HORIZON = 36, 36
GRAPH_RECENT_HOURS = 18
EVENT_THRESHOLD = 75.0
SUSTAIN_HOURS = 2
QUANTILES = [0.50, 0.75, 0.90, 0.95, 0.99]
N_QUANTILES = len(QUANTILES)
TIME_FEATS = ["SO2", "CO", "NO2", "O3", "PM10", "PM25"]
STATIC_COLS = ["elevation_m", "urban_landuse_area_m2_3km", "green_space_area_3km",
               "building_footprint_area_3km", "railway_length_3km", "dist_to_coast_km",
               "dist_to_major_road_km", "industrial_area_m2_3km", "traffic_points_count_3km",
               "major_roads_count_3km", "total_road_length_3km"]

time_panels = {c: df.pivot(index="Datetime", columns="Station_ID", values=c)[station_order] for c in TIME_FEATS}
dt_index = time_panels["PM25"].index
n_time = len(dt_index)
years = dt_index.year.to_numpy()
doy = dt_index.dayofyear.to_numpy().astype(float)

split_id_per_hour = np.where(years == 2016, 0, np.where(years == 2017, 1, np.where(years == 2018, 2, -1)))
TRAIN_MASK = split_id_per_hour == 0

wind_dir_arr = df.pivot(index="Datetime", columns="Station_ID", values="winddirection_10m")[station_order].reindex(dt_index).to_numpy().astype(float)
wind_speed_arr = df.pivot(index="Datetime", columns="Station_ID", values="windspeed_10m")[station_order].reindex(dt_index).to_numpy().astype(float)
blh_arr = df.pivot(index="Datetime", columns="Station_ID", values="boundary_layer_height")[station_order].reindex(dt_index).to_numpy().astype(float)
pm25_raw_arr = time_panels["PM25"].to_numpy()

def regional_flat_mean(arr):
    out = np.zeros((n_time, n_regions), dtype=np.float32)
    for r in range(n_regions):
        cols = station_region_idx == r
        out[:, r] = np.nanmean(arr[:, cols], axis=1)
    return out

region_pm25 = regional_flat_mean(pm25_raw_arr)
wdir_sin_station = np.sin(np.radians(wind_dir_arr))
wdir_cos_station = np.cos(np.radians(wind_dir_arr))
region_windspeed = regional_flat_mean(wind_speed_arr)
region_wdir_sin = regional_flat_mean(wdir_sin_station)
region_wdir_cos = regional_flat_mean(wdir_cos_station)
region_wind_dir_deg = (np.degrees(np.arctan2(region_wdir_sin, region_wdir_cos)) + 360) % 360
region_wind_blows_toward = (region_wind_dir_deg + 180) % 360

season_sin_1d = np.sin(2 * np.pi * doy / 365.25)
season_cos_1d = np.cos(2 * np.pi * doy / 365.25)
season_sin = np.tile(season_sin_1d[:, None], (1, n_stations))
season_cos = np.tile(season_cos_1d[:, None], (1, n_stations))

TIME_FEATS_FULL = TIME_FEATS + ["windspeed_10m", "wdir_sin", "wdir_cos", "boundary_layer_height", "season_sin", "season_cos"]
n_time_feats, n_static_feats = len(TIME_FEATS_FULL), len(STATIC_COLS)
n_feats = n_time_feats + n_static_feats
pm25_col_idx = TIME_FEATS_FULL.index("PM25")

time_arr_raw = np.stack([time_panels[c].to_numpy() for c in TIME_FEATS] +
                         [wind_speed_arr, wdir_sin_station, wdir_cos_station, blh_arr, season_sin, season_cos], axis=-1)
del time_panels, season_sin, season_cos, blh_arr
gc.collect()

static_df = df[["Station_ID"] + STATIC_COLS].drop_duplicates("Station_ID").set_index("Station_ID").loc[station_order]
static_arr = static_df[STATIC_COLS].to_numpy()
del df
gc.collect()

rev = region_pm25[::-1]
roll_min_rev = pd.DataFrame(rev).rolling(window=SUSTAIN_HOURS, min_periods=SUSTAIN_HOURS).min().to_numpy()
region_episode_label = (roll_min_rev[::-1] >= EVENT_THRESHOLD).astype(np.float32)
del rev, roll_min_rev, pm25_raw_arr
gc.collect()

def pinball_loss(preds, target, quantiles):
    target_exp = target.unsqueeze(-1)
    diff = target_exp - preds
    q_tensor = torch.tensor(quantiles, device=preds.device, dtype=preds.dtype).view(*([1] * (preds.dim() - 1)), -1)
    return torch.max(q_tensor * diff, (q_tensor - 1) * diff).mean()

def monotonic_quantiles(raw):
    first = raw[..., :1]
    deltas = F.softplus(raw[..., 1:])
    return torch.cat([first, first + torch.cumsum(deltas, dim=-1)], dim=-1)

class WindConvLayer(nn.Module):
    def __init__(self, in_dim, out_dim):
        super().__init__()
        self.lin_self = nn.Linear(in_dim, out_dim)
        self.lin_neigh = nn.Linear(in_dim, out_dim)
        self.lin_connectivity = nn.Linear(1, out_dim)

    def forward(self, x, edge_index, edge_weight, num_nodes):
        src, dst = edge_index[0], edge_index[1]
        messages = x[src] * edge_weight.unsqueeze(-1)
        agg_sum = x.new_zeros(num_nodes, x.size(-1))
        agg_sum.index_add_(0, dst, messages)
        weight_sum = x.new_zeros(num_nodes)
        weight_sum.index_add_(0, dst, edge_weight)
        agg_mean = agg_sum / (weight_sum.unsqueeze(-1) + 1e-8)
        connectivity = torch.log1p(weight_sum.clamp(min=0)).unsqueeze(-1)
        return self.lin_self(x) + self.lin_neigh(agg_mean) + self.lin_connectivity(connectivity)

class AttentionPool(nn.Module):
    def __init__(self, hidden):
        super().__init__()
        self.attn_score = nn.Linear(hidden, 1)

    def forward(self, h_station, region_membership_t_local):
        B, N, H = h_station.shape
        scores = self.attn_score(h_station).squeeze(-1)
        scores = scores - scores.max(dim=1, keepdim=True).values
        exp_scores = torch.exp(scores)
        weighted_exp = exp_scores.unsqueeze(-1) * region_membership_t_local.unsqueeze(0)
        region_denom = weighted_exp.sum(dim=1)
        region_numer = torch.einsum('bnr,bnh->brh', weighted_exp, h_station)
        return region_numer / (region_denom.unsqueeze(-1) + 1e-8)

class StationQuantileGCN(nn.Module):
    def __init__(self, in_dim, dropout, hidden=32, gru_hidden=32):
        super().__init__()
        self.station_conv = WindConvLayer(in_dim, hidden)
        self.drop = nn.Dropout(dropout)
        self.gru = nn.GRU(hidden, gru_hidden, batch_first=True)
        self.head = nn.Linear(gru_hidden, N_QUANTILES)

    def forward(self, x_window, edge_index, edge_weight_seq):
        B, W, N, Fin = x_window.shape
        ei_b = torch.cat([edge_index + i * N for i in range(B)], dim=1)
        num_nodes = B * N
        h_seq = []
        for w in range(W):
            xt = x_window[:, w].reshape(B * N, Fin)
            ew_b = edge_weight_seq[:, w].reshape(-1)
            h = torch.relu(self.station_conv(xt, ei_b, ew_b, num_nodes))
            h = self.drop(h)
            h_seq.append(h.reshape(B, N, -1))
        h_seq = torch.stack(h_seq, dim=1).permute(0, 2, 1, 3).reshape(B * N, W, -1)
        _, h_final = self.gru(h_seq)
        embed = self.drop(h_final.squeeze(0).reshape(B, N, -1))
        return monotonic_quantiles(self.head(embed))

class RegionFromTrajectoryGCN(nn.Module):
    def __init__(self, station_conv, region_membership_t_local, dropout, hidden=32, gru_hidden=32):
        super().__init__()
        self.station_conv = station_conv
        self.attn_pool = AttentionPool(hidden)
        self.region_conv = WindConvLayer(hidden, hidden)
        self.drop = nn.Dropout(dropout)
        self.region_gru = nn.GRU(hidden, gru_hidden, batch_first=True)
        self.region_head = nn.Linear(gru_hidden, HORIZON)
        self.rmem = region_membership_t_local

    def forward(self, x_window, station_edge_index, station_edge_weight_seq, region_edge_index, region_edge_weight_seq, n_reg):
        B, W, N, Fin = x_window.shape
        station_ei_b = torch.cat([station_edge_index + i * N for i in range(B)], dim=1)
        region_ei_b = torch.cat([region_edge_index + i * n_reg for i in range(B)], dim=1)
        num_station_nodes = B * N
        num_region_nodes = B * n_reg
        h_region_seq = []
        for w in range(W):
            xt = x_window[:, w].reshape(B * N, Fin)
            ew_station_b = station_edge_weight_seq[:, w].reshape(-1)
            h_station = torch.relu(self.station_conv(xt, station_ei_b, ew_station_b, num_station_nodes))
            h_station = self.drop(h_station).reshape(B, N, -1)
            h_region_pooled = self.attn_pool(h_station, self.rmem).reshape(B * n_reg, -1)
            ew_region_b = region_edge_weight_seq[:, w].reshape(-1)
            h_region = torch.relu(self.region_conv(h_region_pooled, region_ei_b, ew_region_b, num_region_nodes))
            h_region = self.drop(h_region)
            h_region_seq.append(h_region.reshape(B, n_reg, -1))
        h_region_seq = torch.stack(h_region_seq, dim=1).permute(0, 2, 1, 3).reshape(B * n_reg, W, -1)
        _, h_final = self.region_gru(h_region_seq)
        embed = self.drop(h_final.squeeze(0).reshape(B, n_reg, -1))
        return self.region_head(embed)

class StationGRU(nn.Module):
    def __init__(self, in_dim, dropout, hidden=32, gru_hidden=32):
        super().__init__()
        self.station_lin = nn.Linear(in_dim, hidden)
        self.drop = nn.Dropout(dropout)
        self.gru = nn.GRU(hidden, gru_hidden, batch_first=True)
        self.head = nn.Linear(gru_hidden, N_QUANTILES)

    def forward(self, x_window):
        B, W, N, Fin = x_window.shape
        h_seq = []
        for w in range(W):
            xt = x_window[:, w].reshape(B * N, Fin)
            h = torch.relu(self.station_lin(xt))
            h = self.drop(h)
            h_seq.append(h.reshape(B, N, -1))
        h_seq = torch.stack(h_seq, dim=1).permute(0, 2, 1, 3).reshape(B * N, W, -1)
        _, h_final = self.gru(h_seq)
        embed = self.drop(h_final.squeeze(0).reshape(B, N, -1))
        return monotonic_quantiles(self.head(embed))

class RegionFromTrajectoryGRU(nn.Module):
    def __init__(self, station_lin, region_membership_t_local, dropout, hidden=32, gru_hidden=32):
        super().__init__()
        self.station_lin = station_lin
        self.attn_pool = AttentionPool(hidden)
        self.drop = nn.Dropout(dropout)
        self.region_gru = nn.GRU(hidden, gru_hidden, batch_first=True)
        self.region_head = nn.Linear(gru_hidden, HORIZON)
        self.rmem = region_membership_t_local

    def forward(self, x_window, n_reg):
        B, W, N, Fin = x_window.shape
        h_region_seq = []
        for w in range(W):
            xt = x_window[:, w].reshape(B * N, Fin)
            h_station = torch.relu(self.station_lin(xt)).reshape(B, N, -1)
            h_station = self.drop(h_station)
            h_region_pooled = self.attn_pool(h_station, self.rmem)
            h_region_pooled = self.drop(h_region_pooled)
            h_region_seq.append(h_region_pooled)
        h_region_seq = torch.stack(h_region_seq, dim=1).permute(0, 2, 1, 3).reshape(B * n_reg, W, -1)
        _, h_final = self.region_gru(h_region_seq)
        embed = self.drop(h_final.squeeze(0).reshape(B, n_reg, -1))
        return self.region_head(embed)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else ("mps" if torch.backends.mps.is_available() else "cpu"))
MICRO_BATCH, ACCUM_STEPS = 16, 4
MAX_EPOCHS_P1, MAX_EPOCHS_P2 = 2, 2
print(f"device={DEVICE}")

region_membership_t = region_membership_t.to(DEVICE)
edge_index = torch.tensor(edge_index_np, dtype=torch.long).to(DEVICE)
region_edge_index = torch.tensor(region_edge_index_np, dtype=torch.long).to(DEVICE)

def add_static_fn(x_time_batch, static_tensor_local):
    B, W, N, _ = x_time_batch.shape
    static_b = static_tensor_local.unsqueeze(0).unsqueeze(0).expand(B, W, N, n_static_feats)
    return torch.cat([x_time_batch, static_b], dim=-1)

def gather_seq(ew_by_hour_t, starts_subset):
    idx = starts_subset.unsqueeze(1) + torch.arange(WINDOW).unsqueeze(0)
    ew = ew_by_hour_t[idx]
    ew = ew.clone()
    ew[:, :WINDOW - GRAPH_RECENT_HOURS, :] = 0.0
    return ew

print(f"train hours (2016): {TRAIN_MASK.sum()}  val hours (2017): {(split_id_per_hour==1).sum()}  test hours (2018): {(split_id_per_hour==2).sum()}")

wind_blows_toward = (wind_dir_arr + 180) % 360
wbt_src = wind_blows_toward[:, src_idx]
cos_align = np.maximum(np.cos(np.radians(wbt_src - bearing_edge[None, :])), 0.0)
speed_src = wind_speed_arr[:, src_idx]
wind_component = (cos_align * speed_src).astype(np.float32)
del wbt_src, cos_align, speed_src
gc.collect()
ref_speed = np.float32(np.nanmean(wind_speed_arr[TRAIN_MASK]))
component = np.where(close_edge_mask[None, :], ref_speed, wind_component)
station_wind_raw = np.nan_to_num(decay_edge[None, :] * component, nan=0.0).astype(np.float32)
del component
gc.collect()

r_wbt_src = region_wind_blows_toward[:, r_src_idx]
r_cos_align = np.maximum(np.cos(np.radians(r_wbt_src - region_bearing_edge[None, :])), 0.0)
r_speed_src = region_windspeed[:, r_src_idx]
region_wind_raw = np.nan_to_num(region_decay_edge[None, :] * r_cos_align * r_speed_src, nan=0.0).astype(np.float32)
del r_wbt_src, r_cos_align, r_speed_src
gc.collect()

train_nonzero = station_wind_raw[TRAIN_MASK][station_wind_raw[TRAIN_MASK] > 0]
station_wind_edge_weight = (station_wind_raw / train_nonzero.std()).astype(np.float32)
region_train_nonzero = region_wind_raw[TRAIN_MASK][region_wind_raw[TRAIN_MASK] > 0]
region_wind_edge_weight = (region_wind_raw / region_train_nonzero.std()).astype(np.float32)
del train_nonzero, region_train_nonzero, station_wind_raw, region_wind_raw
gc.collect()

t_mean = np.nanmean(time_arr_raw[TRAIN_MASK], axis=(0, 1), keepdims=True)
t_std = np.nanstd(time_arr_raw[TRAIN_MASK], axis=(0, 1), keepdims=True) + 1e-6
time_arr_std = np.nan_to_num((time_arr_raw - t_mean) / t_std, nan=0.0)
s_mean, s_std = static_arr.mean(axis=0, keepdims=True), static_arr.std(axis=0, keepdims=True) + 1e-6
static_tensor = torch.tensor((static_arr - s_mean) / s_std, dtype=torch.float32).to(DEVICE)

p_buckets = {0: ([], [], [], []), 1: ([], [], [], []), 2: ([], [], [], [])}
for t in range(0, n_time - WINDOW - HORIZON + 1):
    target_t = t + WINDOW + HORIZON - 1
    s_start, s_target = split_id_per_hour[t], split_id_per_hour[target_t]
    if s_start != s_target or s_start == -1:
        continue
    x_win = time_arr_std[t:t + WINDOW]
    traj = region_episode_label[t + WINDOW: t + WINDOW + HORIZON]
    Xl, yregl, ytrajl, sl = p_buckets[s_start]
    Xl.append(x_win)
    yregl.append(time_arr_std[target_t, :, pm25_col_idx])
    ytrajl.append(traj.T)
    sl.append(t)

X0, yreg0, ytraj0, starts0 = (np.stack(v) for v in p_buckets[0])
X1, yreg1, ytraj1, starts1 = (np.stack(v) for v in p_buckets[1])
X2, yreg2, ytraj2, starts2 = (np.stack(v) for v in p_buckets[2])
del p_buckets, time_arr_std
gc.collect()
print(f"windows: train={len(X0)} val={len(X1)} test={len(X2)}")

X0_t = torch.tensor(X0, dtype=torch.float32); del X0
X1_t = torch.tensor(X1, dtype=torch.float32); del X1
X2_t = torch.tensor(X2, dtype=torch.float32); del X2
gc.collect()

yreg0_t, yreg1_t = torch.tensor(yreg0, dtype=torch.float32), torch.tensor(yreg1, dtype=torch.float32)
ytraj0_t = torch.tensor(ytraj0, dtype=torch.float32)
ytraj1_t = torch.tensor(ytraj1, dtype=torch.float32)
starts0_t, starts1_t, starts2_t = (torch.tensor(a, dtype=torch.long) for a in (starts0, starts1, starts2))
del yreg0, yreg1
gc.collect()

POS_WEIGHT = min(float((ytraj0_t.numel() - ytraj0_t.sum()) / ytraj0_t.sum().clamp(min=1)), 50.0)
print(f"pos_weight: {POS_WEIGHT:.2f}")

wind_ewt_s = torch.tensor(station_wind_edge_weight)
wind_ewt_r = torch.tensor(region_wind_edge_weight)
zero_ewt_s = torch.zeros_like(wind_ewt_s)
zero_ewt_r = torch.zeros_like(wind_ewt_r)

region_criterion = nn.BCEWithLogitsLoss(pos_weight=torch.tensor(POS_WEIGHT))

def run_p1_epoch_graph(model, ewt, X, y, starts, optimizer, train):
    n = X.shape[0]
    idx = torch.randperm(n) if train else torch.arange(n)
    model.train(train)
    total_loss, total_n = 0.0, 0
    eff_batch = MICRO_BATCH * ACCUM_STEPS
    for start in range(0, n, eff_batch):
        if train: optimizer.zero_grad()
        batch_idx = idx[start:start + eff_batch]
        for ms in range(0, len(batch_idx), MICRO_BATCH):
            mb_idx = batch_idx[ms:ms + MICRO_BATCH]
            if len(mb_idx) == 0: continue
            xb = add_static_fn(X[mb_idx].to(DEVICE), static_tensor)
            yb = y[mb_idx].to(DEVICE)
            ew_seq = gather_seq(ewt, starts[mb_idx]).to(DEVICE)
            with torch.set_grad_enabled(train):
                pred = model(xb, edge_index, ew_seq)
                loss = pinball_loss(pred, yb, QUANTILES)
            if train: (loss * len(mb_idx) / len(batch_idx)).backward()
            total_loss += loss.item() * len(mb_idx); total_n += len(mb_idx)
        if train: optimizer.step()
    return total_loss / total_n

def run_p2_epoch_graph(model, ewt_s, ewt_r, X, y_traj, starts, optimizer, train):
    n = X.shape[0]
    idx = torch.randperm(n) if train else torch.arange(n)
    model.train(train)
    total_loss, total_n = 0.0, 0
    eff_batch = MICRO_BATCH * ACCUM_STEPS
    for start in range(0, n, eff_batch):
        if train: optimizer.zero_grad()
        batch_idx = idx[start:start + eff_batch]
        for ms in range(0, len(batch_idx), MICRO_BATCH):
            mb_idx = batch_idx[ms:ms + MICRO_BATCH]
            if len(mb_idx) == 0: continue
            xb = add_static_fn(X[mb_idx].to(DEVICE), static_tensor)
            yb = y_traj[mb_idx].to(DEVICE)
            ew_s = gather_seq(ewt_s, starts[mb_idx]).to(DEVICE)
            ew_r = gather_seq(ewt_r, starts[mb_idx]).to(DEVICE)
            with torch.set_grad_enabled(train):
                logits = model(xb, edge_index, ew_s, region_edge_index, ew_r, n_regions)
                loss = region_criterion(logits, yb)
            if train: loss.backward()
            total_loss += loss.item() * len(mb_idx); total_n += len(mb_idx)
        if train: optimizer.step()
    return total_loss / max(total_n, 1)

def predict_probs_graph(model, ewt_s, ewt_r, X, starts, batch_size=64):
    model.eval()
    n = X.shape[0]
    out = np.zeros((n, n_regions, HORIZON), dtype=np.float32)
    with torch.no_grad():
        for start in range(0, n, batch_size):
            idx = torch.arange(start, min(start + batch_size, n))
            xb = add_static_fn(X[idx].to(DEVICE), static_tensor)
            ew_s = gather_seq(ewt_s, starts[idx]).to(DEVICE)
            ew_r = gather_seq(ewt_r, starts[idx]).to(DEVICE)
            logits = model(xb, edge_index, ew_s, region_edge_index, ew_r, n_regions)
            out[idx.numpy()] = torch.sigmoid(logits).cpu().numpy()
    return out

def run_p1_epoch_gru(model, X, y, optimizer, train):
    n = X.shape[0]
    idx = torch.randperm(n) if train else torch.arange(n)
    model.train(train)
    total_loss, total_n = 0.0, 0
    eff_batch = MICRO_BATCH * ACCUM_STEPS
    for start in range(0, n, eff_batch):
        if train: optimizer.zero_grad()
        batch_idx = idx[start:start + eff_batch]
        for ms in range(0, len(batch_idx), MICRO_BATCH):
            mb_idx = batch_idx[ms:ms + MICRO_BATCH]
            if len(mb_idx) == 0: continue
            xb = add_static_fn(X[mb_idx].to(DEVICE), static_tensor)
            yb = y[mb_idx].to(DEVICE)
            with torch.set_grad_enabled(train):
                pred = model(xb)
                loss = pinball_loss(pred, yb, QUANTILES)
            if train: (loss * len(mb_idx) / len(batch_idx)).backward()
            total_loss += loss.item() * len(mb_idx); total_n += len(mb_idx)
        if train: optimizer.step()
    return total_loss / total_n

def run_p2_epoch_gru(model, X, y_traj, optimizer, train):
    n = X.shape[0]
    idx = torch.randperm(n) if train else torch.arange(n)
    model.train(train)
    total_loss, total_n = 0.0, 0
    eff_batch = MICRO_BATCH * ACCUM_STEPS
    for start in range(0, n, eff_batch):
        if train: optimizer.zero_grad()
        batch_idx = idx[start:start + eff_batch]
        for ms in range(0, len(batch_idx), MICRO_BATCH):
            mb_idx = batch_idx[ms:ms + MICRO_BATCH]
            if len(mb_idx) == 0: continue
            xb = add_static_fn(X[mb_idx].to(DEVICE), static_tensor)
            yb = y_traj[mb_idx].to(DEVICE)
            with torch.set_grad_enabled(train):
                logits = model(xb, n_regions)
                loss = region_criterion(logits, yb)
            if train: loss.backward()
            total_loss += loss.item() * len(mb_idx); total_n += len(mb_idx)
        if train: optimizer.step()
    return total_loss / max(total_n, 1)

def predict_probs_gru(model, X, batch_size=64):
    model.eval()
    n = X.shape[0]
    out = np.zeros((n, n_regions, HORIZON), dtype=np.float32)
    with torch.no_grad():
        for start in range(0, n, batch_size):
            idx = torch.arange(start, min(start + batch_size, n))
            xb = add_static_fn(X[idx].to(DEVICE), static_tensor)
            logits = model(xb, n_regions)
            out[idx.numpy()] = torch.sigmoid(logits).cpu().numpy()
    return out

def train_one_config_graph(seed, ewt_s, ewt_r, lr, dropout, weight_decay):
    torch.manual_seed(seed); np.random.seed(seed)
    p1 = StationQuantileGCN(n_feats, dropout).to(DEVICE)
    opt1 = torch.optim.Adam(p1.parameters(), lr=lr, weight_decay=weight_decay)
    best_p1_val, best_p1_state = float("inf"), None
    for epoch in range(1, MAX_EPOCHS_P1 + 1):
        run_p1_epoch_graph(p1, ewt_s, X0_t, yreg0_t, starts0_t, opt1, True)
        vl = run_p1_epoch_graph(p1, ewt_s, X1_t, yreg1_t, starts1_t, opt1, False)
        if vl < best_p1_val:
            best_p1_val, best_p1_state = vl, copy.deepcopy(p1.state_dict())
    p1.load_state_dict(best_p1_state)
    encoder_copy = copy.deepcopy(p1.station_conv)
    p2 = RegionFromTrajectoryGCN(encoder_copy, region_membership_t, dropout).to(DEVICE)
    del p1; gc.collect()
    if DEVICE.type == "mps": torch.mps.empty_cache()
    opt2 = torch.optim.Adam(p2.parameters(), lr=lr, weight_decay=weight_decay)
    y_val_agg = ytraj1_t.numpy().max(axis=-1).reshape(-1)
    best_val_aucpr, best_state = -1.0, None
    for epoch in range(1, MAX_EPOCHS_P2 + 1):
        run_p2_epoch_graph(p2, ewt_s, ewt_r, X0_t, ytraj0_t, starts0_t, opt2, True)
        run_p2_epoch_graph(p2, ewt_s, ewt_r, X1_t, ytraj1_t, starts1_t, opt2, False)
        val_probs = predict_probs_graph(p2, ewt_s, ewt_r, X1_t, starts1_t)
        va = average_precision_score(y_val_agg, val_probs.max(axis=-1).reshape(-1))
        if va > best_val_aucpr:
            best_val_aucpr, best_state = va, copy.deepcopy(p2.state_dict())
    p2.load_state_dict(best_state)
    p2.eval()
    return p2, best_val_aucpr

def train_one_config_gru(seed, lr, dropout, weight_decay):
    torch.manual_seed(seed); np.random.seed(seed)
    p1 = StationGRU(n_feats, dropout).to(DEVICE)
    opt1 = torch.optim.Adam(p1.parameters(), lr=lr, weight_decay=weight_decay)
    best_p1_val, best_p1_state = float("inf"), None
    for epoch in range(1, MAX_EPOCHS_P1 + 1):
        run_p1_epoch_gru(p1, X0_t, yreg0_t, opt1, True)
        vl = run_p1_epoch_gru(p1, X1_t, yreg1_t, opt1, False)
        if vl < best_p1_val:
            best_p1_val, best_p1_state = vl, copy.deepcopy(p1.state_dict())
    p1.load_state_dict(best_p1_state)
    encoder_copy = copy.deepcopy(p1.station_lin)
    p2 = RegionFromTrajectoryGRU(encoder_copy, region_membership_t, dropout).to(DEVICE)
    del p1; gc.collect()
    if DEVICE.type == "mps": torch.mps.empty_cache()
    opt2 = torch.optim.Adam(p2.parameters(), lr=lr, weight_decay=weight_decay)
    y_val_agg = ytraj1_t.numpy().max(axis=-1).reshape(-1)
    best_val_aucpr, best_state = -1.0, None
    for epoch in range(1, MAX_EPOCHS_P2 + 1):
        run_p2_epoch_gru(p2, X0_t, ytraj0_t, opt2, True)
        run_p2_epoch_gru(p2, X1_t, ytraj1_t, opt2, False)
        val_probs = predict_probs_gru(p2, X1_t)
        va = average_precision_score(y_val_agg, val_probs.max(axis=-1).reshape(-1))
        if va > best_val_aucpr:
            best_val_aucpr, best_state = va, copy.deepcopy(p2.state_dict())
    p2.load_state_dict(best_state)
    p2.eval()
    return p2, best_val_aucpr

GRID = [
    {"lr": 0.003, "dropout": 0.5, "weight_decay": 0.0005},
    {"lr": 0.001, "dropout": 0.5, "weight_decay": 0.0005},
    {"lr": 0.003, "dropout": 0.3, "weight_decay": 0.0005},
    {"lr": 0.003, "dropout": 0.5, "weight_decay": 0.001},
]
N_SEEDS = 10
y_test_agg = ytraj2.max(axis=-1).reshape(-1)
all_results = {}

# ================================================================
# wind_graph: reuse existing tuned config + existing 5 seeds, train 5 NEW seeds (5-9)
# ================================================================
print(f"\n{'#'*15} wind_graph: reusing tuned config {wind_graph_best_cfg}, training seeds 5-9 {'#'*15}")
wg_lr, wg_dropout, wg_wd = wind_graph_best_cfg["lr"], wind_graph_best_cfg["dropout"], wind_graph_best_cfg["weight_decay"]
wind_graph_new_seeds = []
for seed in range(5, 10):
    t0 = time.time()
    p2, val_aucpr = train_one_config_graph(seed, wind_ewt_s, wind_ewt_r, wg_lr, wg_dropout, wg_wd)
    test_probs = predict_probs_graph(p2, wind_ewt_s, wind_ewt_r, X2_t, starts2_t)
    test_agg_score = test_probs.max(axis=-1).reshape(-1)
    test_aucroc = roc_auc_score(y_test_agg, test_agg_score)
    test_aucpr = average_precision_score(y_test_agg, test_agg_score)
    print(f"  seed={seed}  ({time.time()-t0:.0f}s)  test_AUCROC={test_aucroc:.4f}  test_AUCPR={test_aucpr:.4f}")
    wind_graph_new_seeds.append({"seed": seed, "val_aucpr": float(val_aucpr), "test_aucroc": float(test_aucroc), "test_aucpr": float(test_aucpr)})
    del p2; gc.collect()
    if DEVICE.type == "mps": torch.mps.empty_cache()

all_results["wind_graph"] = {"best_config": wind_graph_best_cfg, "results": wind_graph_existing_5 + wind_graph_new_seeds}

# ================================================================
# no_graph, minimal_gru: independent search + 10 seeds each
# ================================================================
def run_full_pipeline(model_name, ewt_s=None, ewt_r=None, is_gru=False):
    print(f"\n{'#'*15} {model_name}: hyperparameter search on train=2016/val=2017 only {'#'*15}")
    grid_results = []
    for cfg in GRID:
        t0 = time.time()
        if is_gru:
            p2, val_aucpr = train_one_config_gru(seed=0, **cfg)
        else:
            p2, val_aucpr = train_one_config_graph(seed=0, ewt_s=ewt_s, ewt_r=ewt_r, **cfg)
        print(f"  {cfg}  ->  val_AUCPR={val_aucpr:.4f}  ({time.time()-t0:.0f}s)")
        grid_results.append({**cfg, "val_aucpr": float(val_aucpr)})
        del p2; gc.collect()
        if DEVICE.type == "mps": torch.mps.empty_cache()
    best_cfg = max(grid_results, key=lambda r: r["val_aucpr"])
    print(f"  best config for {model_name}: {best_cfg}")

    print(f"\n{'#'*15} {model_name}: {N_SEEDS} final seeds {'#'*15}")
    seed_results = []
    for seed in range(N_SEEDS):
        t0 = time.time()
        if is_gru:
            p2, val_aucpr = train_one_config_gru(seed, best_cfg["lr"], best_cfg["dropout"], best_cfg["weight_decay"])
            test_probs = predict_probs_gru(p2, X2_t)
        else:
            p2, val_aucpr = train_one_config_graph(seed, ewt_s, ewt_r, best_cfg["lr"], best_cfg["dropout"], best_cfg["weight_decay"])
            test_probs = predict_probs_graph(p2, ewt_s, ewt_r, X2_t, starts2_t)
        test_agg_score = test_probs.max(axis=-1).reshape(-1)
        test_aucroc = roc_auc_score(y_test_agg, test_agg_score)
        test_aucpr = average_precision_score(y_test_agg, test_agg_score)
        print(f"  seed={seed}  ({time.time()-t0:.0f}s)  test_AUCROC={test_aucroc:.4f}  test_AUCPR={test_aucpr:.4f}")
        seed_results.append({"seed": seed, "val_aucpr": float(val_aucpr), "test_aucroc": float(test_aucroc), "test_aucpr": float(test_aucpr)})
        del p2; gc.collect()
        if DEVICE.type == "mps": torch.mps.empty_cache()

    return {"grid_search": grid_results, "best_config": best_cfg, "results": seed_results}

all_results["no_graph"] = run_full_pipeline("no_graph", zero_ewt_s, zero_ewt_r, is_gru=False)
all_results["minimal_gru"] = run_full_pipeline("minimal_gru", is_gru=True)

# ================================================================
# SUMMARY
# ================================================================
print(f"\n{'='*20} SUMMARY: 2018 test year, full 10-seed fair comparison {'='*20}")
arrs = {}
for name in ["wind_graph", "no_graph", "minimal_gru"]:
    aucpr = np.array([r["test_aucpr"] for r in all_results[name]["results"]])
    aucroc = np.array([r["test_aucroc"] for r in all_results[name]["results"]])
    arrs[name] = {"aucpr": aucpr, "aucroc": aucroc}
    print(f"{name:<15} AUC-PR mean={aucpr.mean():.4f} std={aucpr.std():.4f}  |  AUC-ROC mean={aucroc.mean():.4f} std={aucroc.std():.4f}  (best_cfg={all_results[name]['best_config']})")

for other in ["no_graph", "minimal_gru"]:
    t_stat, p_ttest = stats.ttest_rel(arrs["wind_graph"]["aucpr"], arrs[other]["aucpr"])
    w_stat, p_wilcoxon = stats.wilcoxon(arrs["wind_graph"]["aucpr"], arrs[other]["aucpr"])
    n_wins = int((arrs["wind_graph"]["aucpr"] > arrs[other]["aucpr"]).sum())
    print(f"\nwind_graph vs {other} (AUC-PR, {N_SEEDS} seeds): paired t={t_stat:.3f} p={p_ttest:.6f}  |  Wilcoxon p={p_wilcoxon:.4f}  |  wind_graph wins {n_wins}/{N_SEEDS}")

with open(f"{BASE}/2018_full_10seed_fair_comparison.json", "w") as f:
    json.dump(all_results, f, indent=2)
print(f"\nsaved to {BASE}/2018_full_10seed_fair_comparison.json")


loaded wind_graph's existing 5 seeds and its tuned config: {'lr': 0.001, 'dropout': 0.5, 'weight_decay': 0.0005, 'val_aucpr': 0.42421804918960565}
device=mps
train hours (2016): 8784  val hours (2017): 8760  test hours (2018): 8760
windows: train=8713 val=8689 test=8689
pos_weight: 50.00

############### wind_graph: reusing tuned config {'lr': 0.001, 'dropout': 0.5, 'weight_decay': 0.0005, 'val_aucpr': 0.42421804918960565}, training seeds 5-9 ###############
  seed=5  (460s)  test_AUCROC=0.9349  test_AUCPR=0.4468
  seed=6  (459s)  test_AUCROC=0.9371  test_AUCPR=0.4745
  seed=7  (462s)  test_AUCROC=0.9366  test_AUCPR=0.4562
  seed=8  (465s)  test_AUCROC=0.9284  test_AUCPR=0.4286
  seed=9  (462s)  test_AUCROC=0.9391  test_AUCPR=0.4859

############### no_graph: hyperparameter search on train=2016/val=2017 only ###############
  {'lr': 0.003, 'dropout': 0.5, 'weight_decay': 0.0005}  ->  val_AUCPR=0.4233  (432s)
  {'lr': 0.001, 'dropout': 0.5, 'weight_decay': 0.0005}  ->  val_AUCPR=0.4208 

In [2]:
# ================================================================
# FIGURE: distributional shift in PM2.5, 2016-2019 vs. 2020-2021
# Two panels: (A) overlaid distribution of all hourly PM2.5 values,
# (B) per-year boxplots showing the trend is real and multi-year,
# not just a two-point comparison.
# ================================================================
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from scipy import stats

BASE = "/Users/drewbaldwin/PM2_5 Research"
df = pd.read_pickle(f"{BASE}/air_korea_final_imputed_with_blh.pkl")
df["year"] = df["Datetime"].dt.year

EVENT_THRESHOLD = 75.0

pre = df[df["year"].between(2016, 2019)]["PM25"].dropna()
post = df[df["year"].between(2020, 2021)]["PM25"].dropna()

ks_stat, ks_p = stats.ks_2samp(pre, post)

fig, axes = plt.subplots(1, 2, figsize=(14, 5.5))

# ---- Panel A: overlaid distribution (log-scale y-axis to show the tail clearly) ----
ax = axes[0]
bins = np.linspace(0, 250, 80)
ax.hist(pre, bins=bins, density=True, alpha=0.55, color="#4575b4", label=f"2016-2019 (pre-COVID)\nmean={pre.mean():.1f} $\\mu g/m^3$, n={len(pre):,}")
ax.hist(post, bins=bins, density=True, alpha=0.55, color="#d73027", label=f"2020-2021 (COVID-era)\nmean={post.mean():.1f} $\\mu g/m^3$, n={len(post):,}")
ax.axvline(EVENT_THRESHOLD, color="black", linestyle="--", linewidth=1.5, label=f"Sustained-episode threshold ({EVENT_THRESHOLD:.0f} $\\mu g/m^3$)")
ax.set_yscale("log")
ax.set_xlabel("Hourly PM$_{2.5}$ concentration ($\\mu g/m^3$)")
ax.set_ylabel("Density (log scale)")
ax.set_title(f"Distribution of hourly PM$_{{2.5}}$\nKS statistic={ks_stat:.3f}, p<0.000001")
ax.legend(fontsize=9, loc="upper right")
ax.set_xlim(0, 250)
ax.grid(alpha=0.25)

# ---- Panel B: per-year boxplots showing the trend is real across individual years ----
ax2 = axes[1]
years_list = [2016, 2017, 2018, 2019, 2020, 2021]
data_by_year = [df[df["year"] == y]["PM25"].dropna().values for y in years_list]
colors = ["#4575b4"] * 4 + ["#d73027"] * 2  # pre-COVID blue, COVID-era red

bp = ax2.boxplot(data_by_year, labels=[str(y) for y in years_list], showfliers=False,
                  patch_artist=True, widths=0.6, medianprops=dict(color="black", linewidth=1.5))
for patch, color in zip(bp["boxes"], colors):
    patch.set_facecolor(color)
    patch.set_alpha(0.65)

means = [d.mean() for d in data_by_year]
ax2.plot(range(1, len(years_list) + 1), means, "D", color="black", markersize=6, zorder=5, label="Annual mean")
for i, m in enumerate(means):
    ax2.annotate(f"{m:.1f}", (i + 1, m), textcoords="offset points", xytext=(0, 10), ha="center", fontsize=8)

ax2.axhline(EVENT_THRESHOLD, color="black", linestyle="--", linewidth=1, alpha=0.6)
ax2.set_ylabel("Hourly PM$_{2.5}$ concentration ($\\mu g/m^3$)")
ax2.set_title("Per-year distribution (outliers hidden for readability)")
ax2.legend(fontsize=9, loc="upper right")
ax2.grid(alpha=0.25, axis="y")
ax2.set_ylim(0, 100)

# shade the COVID-era years for visual clarity
ax2.axvspan(4.5, 6.5, color="#d73027", alpha=0.08, zorder=0)

fig.suptitle("PM$_{2.5}$ distributional shift: pre-COVID (2016-2019) vs. COVID-era (2020-2021)", fontsize=13, y=1.02)
fig.tight_layout()
fig.savefig(f"{BASE}/fig_pm25_distributional_shift.png", dpi=150, bbox_inches="tight")
print(f"saved fig_pm25_distributional_shift.png")
print(f"KS test: statistic={ks_stat:.4f}, p={ks_p:.2e}")
print(f"pre-COVID mean={pre.mean():.2f}  post-COVID mean={post.mean():.2f}  relative change={100*(post.mean()/pre.mean()-1):.1f}%")


/var/folders/ft/33z7sjln3fj9bz_flrbhpbsc0000gn/T/ipykernel_892/833024584.py:47: MatplotlibDeprecationWarning: The 'labels' parameter of boxplot() has been renamed 'tick_labels' since Matplotlib 3.9; support for the old name will be dropped in 3.11.
  bp = ax2.boxplot(data_by_year, labels=[str(y) for y in years_list], showfliers=False,


saved fig_pm25_distributional_shift.png
KS test: statistic=0.1753, p=0.00e+00
pre-COVID mean=24.17  post-COVID mean=18.22  relative change=-24.6%


In [3]:
# ================================================================
# FIGURE: PM2.5 time series, 2016-2021, showing the COVID-era shift
# directly over time (daily mean + 30-day rolling trend line),
# with the COVID-era period shaded for visual clarity.
# ================================================================
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

BASE = "/Users/drewbaldwin/PM2_5 Research"
df = pd.read_pickle(f"{BASE}/air_korea_final_imputed_with_blh.pkl")

EVENT_THRESHOLD = 75.0

# national daily mean PM2.5 (averaged across all stations, then across all hours in a day)
daily = df.set_index("Datetime")["PM25"].resample("D").mean()
daily = daily.dropna()

# 30-day rolling mean to smooth out day-to-day noise and reveal the underlying trend
rolling_30d = daily.rolling(window=30, center=True, min_periods=15).mean()

fig, ax = plt.subplots(figsize=(15, 6))

# shade the COVID-era period
covid_start = pd.Timestamp("2020-01-01")
covid_end = pd.Timestamp("2022-01-01")
ax.axvspan(covid_start, covid_end, color="#d73027", alpha=0.08, zorder=0, label="COVID-era (2020-2021)")

# raw daily mean (thin, light)
ax.plot(daily.index, daily.values, color="#4575b4", linewidth=0.4, alpha=0.4, zorder=1, label="Daily mean PM$_{2.5}$")

# 30-day rolling trend (bold)
ax.plot(rolling_30d.index, rolling_30d.values, color="#08306b", linewidth=1.8, zorder=3, label="30-day rolling mean")

# reference threshold line
ax.axhline(EVENT_THRESHOLD, color="black", linestyle="--", linewidth=1, alpha=0.6, zorder=2,
           label=f"Sustained-episode threshold ({EVENT_THRESHOLD:.0f} $\\mu g/m^3$)")

# annotate pre/post period means
pre_mean = daily[daily.index < covid_start].mean()
post_mean = daily[(daily.index >= covid_start) & (daily.index < covid_end)].mean()
ax.axhline(pre_mean, xmin=0, xmax=(covid_start - daily.index[0]) / (daily.index[-1] - daily.index[0]),
           color="#4575b4", linewidth=1.2, linestyle=":", zorder=2)
ax.annotate(f"2016-2019 mean: {pre_mean:.1f} $\\mu g/m^3$", xy=(pd.Timestamp("2017-06-01"), pre_mean + 8),
            fontsize=10, color="#08306b", fontweight="bold")
ax.annotate(f"2020-2021 mean: {post_mean:.1f} $\\mu g/m^3$", xy=(pd.Timestamp("2020-06-01"), post_mean + 8),
            fontsize=10, color="#d73027", fontweight="bold")

ax.set_xlim(daily.index[0], daily.index[-1])
ax.xaxis.set_major_locator(mdates.YearLocator())
ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
ax.set_xlabel("Date")
ax.set_ylabel("PM$_{2.5}$ concentration ($\\mu g/m^3$)")
ax.set_title("National PM$_{2.5}$ time series (2016-2021), showing the COVID-era distributional shift")
ax.legend(loc="upper right", fontsize=9, ncol=2)
ax.grid(alpha=0.25)

fig.tight_layout()
fig.savefig(f"{BASE}/fig_pm25_timeseries.png", dpi=150, bbox_inches="tight")
print("saved fig_pm25_timeseries.png")
print(f"pre-COVID (2016-2019) daily mean: {pre_mean:.2f}")
print(f"COVID-era (2020-2021) daily mean: {post_mean:.2f}")


saved fig_pm25_timeseries.png
pre-COVID (2016-2019) daily mean: 24.17
COVID-era (2020-2021) daily mean: 18.22


In [1]:
# ================================================================
# WITHIN-COVID-REGIME, FULL 10-SEED FAIR COMPARISON:
# train=2020, val=Jan-Jun 2021, test=Jul-Dec 2021 (all within the same
# COVID-shifted regime, isolating "does the graph help" from the
# cross-regime generalization question).
# wind_graph, no_graph, minimal_gru -- ALL THREE independently
# hyperparameter-tuned (search on train=2020/val=H1-2021 only, test
# never touched), all trained in this single script execution, then
# 10 final seeds each with its own best config.
# ================================================================
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import os, time, copy, gc, json
from sklearn.metrics import average_precision_score, roc_auc_score
from scipy import stats

BASE = "/Users/drewbaldwin/PM2_5 Research"
df = pd.read_pickle(f"{BASE}/air_korea_final_imputed_with_blh.pkl")
station_order = sorted(df["Station_ID"].unique())
n_stations = len(station_order)

stations = df[["Station_ID", "lat", "lon"]].drop_duplicates("Station_ID").set_index("Station_ID").loc[station_order]
lats, lons = stations["lat"].to_numpy(), stations["lon"].to_numpy()

def haversine_km(lat1, lon1, lat2, lon2):
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
    dlat, dlon = lat2 - lat1, lon2 - lon1
    a = np.sin(dlat/2)**2 + np.cos(lat1)*np.cos(lat2)*np.sin(dlon/2)**2
    return 2 * 6371.0 * np.arcsin(np.sqrt(a))

def bearing_matrix(lat, lon):
    lat_r, lon_r = np.radians(lat), np.radians(lon)
    lat1, lat2 = lat_r[:, None], lat_r[None, :]
    dlon = lon_r[None, :] - lon_r[:, None]
    x = np.sin(dlon) * np.cos(lat2)
    y = np.cos(lat1) * np.sin(lat2) - np.sin(lat1) * np.cos(lat2) * np.cos(dlon)
    return (np.degrees(np.arctan2(x, y)) + 360) % 360

dist_km = haversine_km(lats[:, None], lons[:, None], lats[None, :], lons[None, :])
DIST_CUTOFF, RHO_KM = 250.0, 250.0
HYBRID_THRESHOLD_KM = 20.0
dist_edges = (dist_km <= DIST_CUTOFF) & (dist_km > 0)
bearing_from = bearing_matrix(lats, lons)

src_idx, dst_idx = np.nonzero(dist_edges)
edge_index_np = np.stack([src_idx, dst_idx])
decay_edge = np.exp(-dist_km[src_idx, dst_idx] / RHO_KM).astype(np.float32)
bearing_edge = bearing_from[src_idx, dst_idx].astype(np.float32)
close_edge_mask = (dist_km[src_idx, dst_idx] <= HYBRID_THRESHOLD_KM)

REGION_CENTROIDS = {
    "Seoul": (37.566, 126.978), "Busan": (35.180, 129.075), "Daegu": (35.872, 128.602),
    "Incheon": (37.483, 126.633), "Gwangju": (35.155, 126.916), "Daejeon": (36.350, 127.385),
    "Ulsan": (35.550, 129.317), "Sejong": (36.487, 127.282), "Gyeonggi": (37.500, 127.250),
    "Gangwon": (37.867, 127.733), "Chungbuk": (36.633, 127.483), "Chungnam": (36.500, 126.750),
    "Jeonbuk": (35.824, 127.148), "Jeonnam": (34.750, 127.000), "Gyeongbuk": (36.559, 128.729),
    "Gyeongnam": (35.271, 128.663), "Jeju": (33.513, 126.523),
}
region_names = list(REGION_CENTROIDS.keys())
n_regions = len(region_names)
region_lats = np.array([REGION_CENTROIDS[r][0] for r in region_names])
region_lons = np.array([REGION_CENTROIDS[r][1] for r in region_names])
dist_to_region = haversine_km(lats[:, None], lons[:, None], region_lats[None, :], region_lons[None, :])
station_region_idx = dist_to_region.argmin(axis=1)

region_membership = np.zeros((n_stations, n_regions), dtype=np.float32)
region_membership[np.arange(n_stations), station_region_idx] = 1.0
region_membership_t = torch.tensor(region_membership)

region_dist_km = haversine_km(region_lats[:, None], region_lons[:, None], region_lats[None, :], region_lons[None, :])
region_bearing = bearing_matrix(region_lats, region_lons)
r_src_idx, r_dst_idx = np.nonzero(~np.eye(n_regions, dtype=bool))
region_edge_index_np = np.stack([r_src_idx, r_dst_idx])
region_decay_edge = np.exp(-region_dist_km[r_src_idx, r_dst_idx] / RHO_KM).astype(np.float32)
region_bearing_edge = region_bearing[r_src_idx, r_dst_idx].astype(np.float32)

WINDOW, HORIZON = 36, 36
GRAPH_RECENT_HOURS = 18
EVENT_THRESHOLD = 75.0
SUSTAIN_HOURS = 2
QUANTILES = [0.50, 0.75, 0.90, 0.95, 0.99]
N_QUANTILES = len(QUANTILES)
TIME_FEATS = ["SO2", "CO", "NO2", "O3", "PM10", "PM25"]
STATIC_COLS = ["elevation_m", "urban_landuse_area_m2_3km", "green_space_area_3km",
               "building_footprint_area_3km", "railway_length_3km", "dist_to_coast_km",
               "dist_to_major_road_km", "industrial_area_m2_3km", "traffic_points_count_3km",
               "major_roads_count_3km", "total_road_length_3km"]

time_panels = {c: df.pivot(index="Datetime", columns="Station_ID", values=c)[station_order] for c in TIME_FEATS}
dt_index = time_panels["PM25"].index
n_time = len(dt_index)
years = dt_index.year.to_numpy()
months = dt_index.month.to_numpy()
doy = dt_index.dayofyear.to_numpy().astype(float)

# ---- SPLIT: train=2020, val=Jan-Jun 2021, test=Jul-Dec 2021 ----
split_id_per_hour = np.where(
    years == 2020, 0,
    np.where((years == 2021) & (months <= 6), 1,
    np.where((years == 2021) & (months >= 7), 2, -1)))
TRAIN_MASK = split_id_per_hour == 0

wind_dir_arr = df.pivot(index="Datetime", columns="Station_ID", values="winddirection_10m")[station_order].reindex(dt_index).to_numpy().astype(float)
wind_speed_arr = df.pivot(index="Datetime", columns="Station_ID", values="windspeed_10m")[station_order].reindex(dt_index).to_numpy().astype(float)
blh_arr = df.pivot(index="Datetime", columns="Station_ID", values="boundary_layer_height")[station_order].reindex(dt_index).to_numpy().astype(float)
pm25_raw_arr = time_panels["PM25"].to_numpy()

def regional_flat_mean(arr):
    out = np.zeros((n_time, n_regions), dtype=np.float32)
    for r in range(n_regions):
        cols = station_region_idx == r
        out[:, r] = np.nanmean(arr[:, cols], axis=1)
    return out

region_pm25 = regional_flat_mean(pm25_raw_arr)
wdir_sin_station = np.sin(np.radians(wind_dir_arr))
wdir_cos_station = np.cos(np.radians(wind_dir_arr))
region_windspeed = regional_flat_mean(wind_speed_arr)
region_wdir_sin = regional_flat_mean(wdir_sin_station)
region_wdir_cos = regional_flat_mean(wdir_cos_station)
region_wind_dir_deg = (np.degrees(np.arctan2(region_wdir_sin, region_wdir_cos)) + 360) % 360
region_wind_blows_toward = (region_wind_dir_deg + 180) % 360

season_sin_1d = np.sin(2 * np.pi * doy / 365.25)
season_cos_1d = np.cos(2 * np.pi * doy / 365.25)
season_sin = np.tile(season_sin_1d[:, None], (1, n_stations))
season_cos = np.tile(season_cos_1d[:, None], (1, n_stations))

TIME_FEATS_FULL = TIME_FEATS + ["windspeed_10m", "wdir_sin", "wdir_cos", "boundary_layer_height", "season_sin", "season_cos"]
n_time_feats, n_static_feats = len(TIME_FEATS_FULL), len(STATIC_COLS)
n_feats = n_time_feats + n_static_feats
pm25_col_idx = TIME_FEATS_FULL.index("PM25")

time_arr_raw = np.stack([time_panels[c].to_numpy() for c in TIME_FEATS] +
                         [wind_speed_arr, wdir_sin_station, wdir_cos_station, blh_arr, season_sin, season_cos], axis=-1)
del time_panels, season_sin, season_cos, blh_arr
gc.collect()

static_df = df[["Station_ID"] + STATIC_COLS].drop_duplicates("Station_ID").set_index("Station_ID").loc[station_order]
static_arr = static_df[STATIC_COLS].to_numpy()
del df
gc.collect()

rev = region_pm25[::-1]
roll_min_rev = pd.DataFrame(rev).rolling(window=SUSTAIN_HOURS, min_periods=SUSTAIN_HOURS).min().to_numpy()
region_episode_label = (roll_min_rev[::-1] >= EVENT_THRESHOLD).astype(np.float32)
del rev, roll_min_rev, pm25_raw_arr
gc.collect()

def pinball_loss(preds, target, quantiles):
    target_exp = target.unsqueeze(-1)
    diff = target_exp - preds
    q_tensor = torch.tensor(quantiles, device=preds.device, dtype=preds.dtype).view(*([1] * (preds.dim() - 1)), -1)
    return torch.max(q_tensor * diff, (q_tensor - 1) * diff).mean()

def monotonic_quantiles(raw):
    first = raw[..., :1]
    deltas = F.softplus(raw[..., 1:])
    return torch.cat([first, first + torch.cumsum(deltas, dim=-1)], dim=-1)

class WindConvLayer(nn.Module):
    def __init__(self, in_dim, out_dim):
        super().__init__()
        self.lin_self = nn.Linear(in_dim, out_dim)
        self.lin_neigh = nn.Linear(in_dim, out_dim)
        self.lin_connectivity = nn.Linear(1, out_dim)

    def forward(self, x, edge_index, edge_weight, num_nodes):
        src, dst = edge_index[0], edge_index[1]
        messages = x[src] * edge_weight.unsqueeze(-1)
        agg_sum = x.new_zeros(num_nodes, x.size(-1))
        agg_sum.index_add_(0, dst, messages)
        weight_sum = x.new_zeros(num_nodes)
        weight_sum.index_add_(0, dst, edge_weight)
        agg_mean = agg_sum / (weight_sum.unsqueeze(-1) + 1e-8)
        connectivity = torch.log1p(weight_sum.clamp(min=0)).unsqueeze(-1)
        return self.lin_self(x) + self.lin_neigh(agg_mean) + self.lin_connectivity(connectivity)

class AttentionPool(nn.Module):
    def __init__(self, hidden):
        super().__init__()
        self.attn_score = nn.Linear(hidden, 1)

    def forward(self, h_station, region_membership_t_local):
        B, N, H = h_station.shape
        scores = self.attn_score(h_station).squeeze(-1)
        scores = scores - scores.max(dim=1, keepdim=True).values
        exp_scores = torch.exp(scores)
        weighted_exp = exp_scores.unsqueeze(-1) * region_membership_t_local.unsqueeze(0)
        region_denom = weighted_exp.sum(dim=1)
        region_numer = torch.einsum('bnr,bnh->brh', weighted_exp, h_station)
        return region_numer / (region_denom.unsqueeze(-1) + 1e-8)

class StationQuantileGCN(nn.Module):
    def __init__(self, in_dim, dropout, hidden=32, gru_hidden=32):
        super().__init__()
        self.station_conv = WindConvLayer(in_dim, hidden)
        self.drop = nn.Dropout(dropout)
        self.gru = nn.GRU(hidden, gru_hidden, batch_first=True)
        self.head = nn.Linear(gru_hidden, N_QUANTILES)

    def forward(self, x_window, edge_index, edge_weight_seq):
        B, W, N, Fin = x_window.shape
        ei_b = torch.cat([edge_index + i * N for i in range(B)], dim=1)
        num_nodes = B * N
        h_seq = []
        for w in range(W):
            xt = x_window[:, w].reshape(B * N, Fin)
            ew_b = edge_weight_seq[:, w].reshape(-1)
            h = torch.relu(self.station_conv(xt, ei_b, ew_b, num_nodes))
            h = self.drop(h)
            h_seq.append(h.reshape(B, N, -1))
        h_seq = torch.stack(h_seq, dim=1).permute(0, 2, 1, 3).reshape(B * N, W, -1)
        _, h_final = self.gru(h_seq)
        embed = self.drop(h_final.squeeze(0).reshape(B, N, -1))
        return monotonic_quantiles(self.head(embed))

class RegionFromTrajectoryGCN(nn.Module):
    def __init__(self, station_conv, region_membership_t_local, dropout, hidden=32, gru_hidden=32):
        super().__init__()
        self.station_conv = station_conv
        self.attn_pool = AttentionPool(hidden)
        self.region_conv = WindConvLayer(hidden, hidden)
        self.drop = nn.Dropout(dropout)
        self.region_gru = nn.GRU(hidden, gru_hidden, batch_first=True)
        self.region_head = nn.Linear(gru_hidden, HORIZON)
        self.rmem = region_membership_t_local

    def forward(self, x_window, station_edge_index, station_edge_weight_seq, region_edge_index, region_edge_weight_seq, n_reg):
        B, W, N, Fin = x_window.shape
        station_ei_b = torch.cat([station_edge_index + i * N for i in range(B)], dim=1)
        region_ei_b = torch.cat([region_edge_index + i * n_reg for i in range(B)], dim=1)
        num_station_nodes = B * N
        num_region_nodes = B * n_reg
        h_region_seq = []
        for w in range(W):
            xt = x_window[:, w].reshape(B * N, Fin)
            ew_station_b = station_edge_weight_seq[:, w].reshape(-1)
            h_station = torch.relu(self.station_conv(xt, station_ei_b, ew_station_b, num_station_nodes))
            h_station = self.drop(h_station).reshape(B, N, -1)
            h_region_pooled = self.attn_pool(h_station, self.rmem).reshape(B * n_reg, -1)
            ew_region_b = region_edge_weight_seq[:, w].reshape(-1)
            h_region = torch.relu(self.region_conv(h_region_pooled, region_ei_b, ew_region_b, num_region_nodes))
            h_region = self.drop(h_region)
            h_region_seq.append(h_region.reshape(B, n_reg, -1))
        h_region_seq = torch.stack(h_region_seq, dim=1).permute(0, 2, 1, 3).reshape(B * n_reg, W, -1)
        _, h_final = self.region_gru(h_region_seq)
        embed = self.drop(h_final.squeeze(0).reshape(B, n_reg, -1))
        return self.region_head(embed)

class StationGRU(nn.Module):
    def __init__(self, in_dim, dropout, hidden=32, gru_hidden=32):
        super().__init__()
        self.station_lin = nn.Linear(in_dim, hidden)
        self.drop = nn.Dropout(dropout)
        self.gru = nn.GRU(hidden, gru_hidden, batch_first=True)
        self.head = nn.Linear(gru_hidden, N_QUANTILES)

    def forward(self, x_window):
        B, W, N, Fin = x_window.shape
        h_seq = []
        for w in range(W):
            xt = x_window[:, w].reshape(B * N, Fin)
            h = torch.relu(self.station_lin(xt))
            h = self.drop(h)
            h_seq.append(h.reshape(B, N, -1))
        h_seq = torch.stack(h_seq, dim=1).permute(0, 2, 1, 3).reshape(B * N, W, -1)
        _, h_final = self.gru(h_seq)
        embed = self.drop(h_final.squeeze(0).reshape(B, N, -1))
        return monotonic_quantiles(self.head(embed))

class RegionFromTrajectoryGRU(nn.Module):
    def __init__(self, station_lin, region_membership_t_local, dropout, hidden=32, gru_hidden=32):
        super().__init__()
        self.station_lin = station_lin
        self.attn_pool = AttentionPool(hidden)
        self.drop = nn.Dropout(dropout)
        self.region_gru = nn.GRU(hidden, gru_hidden, batch_first=True)
        self.region_head = nn.Linear(gru_hidden, HORIZON)
        self.rmem = region_membership_t_local

    def forward(self, x_window, n_reg):
        B, W, N, Fin = x_window.shape
        h_region_seq = []
        for w in range(W):
            xt = x_window[:, w].reshape(B * N, Fin)
            h_station = torch.relu(self.station_lin(xt)).reshape(B, N, -1)
            h_station = self.drop(h_station)
            h_region_pooled = self.attn_pool(h_station, self.rmem)
            h_region_pooled = self.drop(h_region_pooled)
            h_region_seq.append(h_region_pooled)
        h_region_seq = torch.stack(h_region_seq, dim=1).permute(0, 2, 1, 3).reshape(B * n_reg, W, -1)
        _, h_final = self.region_gru(h_region_seq)
        embed = self.drop(h_final.squeeze(0).reshape(B, n_reg, -1))
        return self.region_head(embed)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else ("mps" if torch.backends.mps.is_available() else "cpu"))
MICRO_BATCH, ACCUM_STEPS = 16, 4
MAX_EPOCHS_P1, MAX_EPOCHS_P2 = 2, 2
print(f"device={DEVICE}")

region_membership_t = region_membership_t.to(DEVICE)
edge_index = torch.tensor(edge_index_np, dtype=torch.long).to(DEVICE)
region_edge_index = torch.tensor(region_edge_index_np, dtype=torch.long).to(DEVICE)

def add_static_fn(x_time_batch, static_tensor_local):
    B, W, N, _ = x_time_batch.shape
    static_b = static_tensor_local.unsqueeze(0).unsqueeze(0).expand(B, W, N, n_static_feats)
    return torch.cat([x_time_batch, static_b], dim=-1)

def gather_seq(ew_by_hour_t, starts_subset):
    idx = starts_subset.unsqueeze(1) + torch.arange(WINDOW).unsqueeze(0)
    ew = ew_by_hour_t[idx]
    ew = ew.clone()
    ew[:, :WINDOW - GRAPH_RECENT_HOURS, :] = 0.0
    return ew

print(f"train hours (2020): {TRAIN_MASK.sum()}  val hours (Jan-Jun 2021): {(split_id_per_hour==1).sum()}  test hours (Jul-Dec 2021): {(split_id_per_hour==2).sum()}")

wind_blows_toward = (wind_dir_arr + 180) % 360
wbt_src = wind_blows_toward[:, src_idx]
cos_align = np.maximum(np.cos(np.radians(wbt_src - bearing_edge[None, :])), 0.0)
speed_src = wind_speed_arr[:, src_idx]
wind_component = (cos_align * speed_src).astype(np.float32)
del wbt_src, cos_align, speed_src
gc.collect()
ref_speed = np.float32(np.nanmean(wind_speed_arr[TRAIN_MASK]))
component = np.where(close_edge_mask[None, :], ref_speed, wind_component)
station_wind_raw = np.nan_to_num(decay_edge[None, :] * component, nan=0.0).astype(np.float32)
del component
gc.collect()

r_wbt_src = region_wind_blows_toward[:, r_src_idx]
r_cos_align = np.maximum(np.cos(np.radians(r_wbt_src - region_bearing_edge[None, :])), 0.0)
r_speed_src = region_windspeed[:, r_src_idx]
region_wind_raw = np.nan_to_num(region_decay_edge[None, :] * r_cos_align * r_speed_src, nan=0.0).astype(np.float32)
del r_wbt_src, r_cos_align, r_speed_src
gc.collect()

train_nonzero = station_wind_raw[TRAIN_MASK][station_wind_raw[TRAIN_MASK] > 0]
station_wind_edge_weight = (station_wind_raw / train_nonzero.std()).astype(np.float32)
region_train_nonzero = region_wind_raw[TRAIN_MASK][region_wind_raw[TRAIN_MASK] > 0]
region_wind_edge_weight = (region_wind_raw / region_train_nonzero.std()).astype(np.float32)
del train_nonzero, region_train_nonzero, station_wind_raw, region_wind_raw
gc.collect()

t_mean = np.nanmean(time_arr_raw[TRAIN_MASK], axis=(0, 1), keepdims=True)
t_std = np.nanstd(time_arr_raw[TRAIN_MASK], axis=(0, 1), keepdims=True) + 1e-6
time_arr_std = np.nan_to_num((time_arr_raw - t_mean) / t_std, nan=0.0)
s_mean, s_std = static_arr.mean(axis=0, keepdims=True), static_arr.std(axis=0, keepdims=True) + 1e-6
static_tensor = torch.tensor((static_arr - s_mean) / s_std, dtype=torch.float32).to(DEVICE)

p_buckets = {0: ([], [], [], []), 1: ([], [], [], []), 2: ([], [], [], [])}
for t in range(0, n_time - WINDOW - HORIZON + 1):
    target_t = t + WINDOW + HORIZON - 1
    s_start, s_target = split_id_per_hour[t], split_id_per_hour[target_t]
    if s_start != s_target or s_start == -1:
        continue
    x_win = time_arr_std[t:t + WINDOW]
    traj = region_episode_label[t + WINDOW: t + WINDOW + HORIZON]
    Xl, yregl, ytrajl, sl = p_buckets[s_start]
    Xl.append(x_win)
    yregl.append(time_arr_std[target_t, :, pm25_col_idx])
    ytrajl.append(traj.T)
    sl.append(t)

X0, yreg0, ytraj0, starts0 = (np.stack(v) for v in p_buckets[0])
X1, yreg1, ytraj1, starts1 = (np.stack(v) for v in p_buckets[1])
X2, yreg2, ytraj2, starts2 = (np.stack(v) for v in p_buckets[2])
del p_buckets, time_arr_std
gc.collect()
print(f"windows: train={len(X0)} val={len(X1)} test={len(X2)}")

X0_t = torch.tensor(X0, dtype=torch.float32); del X0
X1_t = torch.tensor(X1, dtype=torch.float32); del X1
X2_t = torch.tensor(X2, dtype=torch.float32); del X2
gc.collect()

yreg0_t, yreg1_t = torch.tensor(yreg0, dtype=torch.float32), torch.tensor(yreg1, dtype=torch.float32)
ytraj0_t = torch.tensor(ytraj0, dtype=torch.float32)
ytraj1_t = torch.tensor(ytraj1, dtype=torch.float32)
starts0_t, starts1_t, starts2_t = (torch.tensor(a, dtype=torch.long) for a in (starts0, starts1, starts2))
del yreg0, yreg1
gc.collect()

POS_WEIGHT = min(float((ytraj0_t.numel() - ytraj0_t.sum()) / ytraj0_t.sum().clamp(min=1)), 50.0)
print(f"pos_weight: {POS_WEIGHT:.2f}")

wind_ewt_s = torch.tensor(station_wind_edge_weight)
wind_ewt_r = torch.tensor(region_wind_edge_weight)
zero_ewt_s = torch.zeros_like(wind_ewt_s)
zero_ewt_r = torch.zeros_like(wind_ewt_r)

region_criterion = nn.BCEWithLogitsLoss(pos_weight=torch.tensor(POS_WEIGHT))

def run_p1_epoch_graph(model, ewt, X, y, starts, optimizer, train):
    n = X.shape[0]
    idx = torch.randperm(n) if train else torch.arange(n)
    model.train(train)
    total_loss, total_n = 0.0, 0
    eff_batch = MICRO_BATCH * ACCUM_STEPS
    for start in range(0, n, eff_batch):
        if train: optimizer.zero_grad()
        batch_idx = idx[start:start + eff_batch]
        for ms in range(0, len(batch_idx), MICRO_BATCH):
            mb_idx = batch_idx[ms:ms + MICRO_BATCH]
            if len(mb_idx) == 0: continue
            xb = add_static_fn(X[mb_idx].to(DEVICE), static_tensor)
            yb = y[mb_idx].to(DEVICE)
            ew_seq = gather_seq(ewt, starts[mb_idx]).to(DEVICE)
            with torch.set_grad_enabled(train):
                pred = model(xb, edge_index, ew_seq)
                loss = pinball_loss(pred, yb, QUANTILES)
            if train: (loss * len(mb_idx) / len(batch_idx)).backward()
            total_loss += loss.item() * len(mb_idx); total_n += len(mb_idx)
        if train: optimizer.step()
    return total_loss / total_n

def run_p2_epoch_graph(model, ewt_s, ewt_r, X, y_traj, starts, optimizer, train):
    n = X.shape[0]
    idx = torch.randperm(n) if train else torch.arange(n)
    model.train(train)
    total_loss, total_n = 0.0, 0
    eff_batch = MICRO_BATCH * ACCUM_STEPS
    for start in range(0, n, eff_batch):
        if train: optimizer.zero_grad()
        batch_idx = idx[start:start + eff_batch]
        for ms in range(0, len(batch_idx), MICRO_BATCH):
            mb_idx = batch_idx[ms:ms + MICRO_BATCH]
            if len(mb_idx) == 0: continue
            xb = add_static_fn(X[mb_idx].to(DEVICE), static_tensor)
            yb = y_traj[mb_idx].to(DEVICE)
            ew_s = gather_seq(ewt_s, starts[mb_idx]).to(DEVICE)
            ew_r = gather_seq(ewt_r, starts[mb_idx]).to(DEVICE)
            with torch.set_grad_enabled(train):
                logits = model(xb, edge_index, ew_s, region_edge_index, ew_r, n_regions)
                loss = region_criterion(logits, yb)
            if train: loss.backward()
            total_loss += loss.item() * len(mb_idx); total_n += len(mb_idx)
        if train: optimizer.step()
    return total_loss / max(total_n, 1)

def predict_probs_graph(model, ewt_s, ewt_r, X, starts, batch_size=64):
    model.eval()
    n = X.shape[0]
    out = np.zeros((n, n_regions, HORIZON), dtype=np.float32)
    with torch.no_grad():
        for start in range(0, n, batch_size):
            idx = torch.arange(start, min(start + batch_size, n))
            xb = add_static_fn(X[idx].to(DEVICE), static_tensor)
            ew_s = gather_seq(ewt_s, starts[idx]).to(DEVICE)
            ew_r = gather_seq(ewt_r, starts[idx]).to(DEVICE)
            logits = model(xb, edge_index, ew_s, region_edge_index, ew_r, n_regions)
            out[idx.numpy()] = torch.sigmoid(logits).cpu().numpy()
    return out

def run_p1_epoch_gru(model, X, y, optimizer, train):
    n = X.shape[0]
    idx = torch.randperm(n) if train else torch.arange(n)
    model.train(train)
    total_loss, total_n = 0.0, 0
    eff_batch = MICRO_BATCH * ACCUM_STEPS
    for start in range(0, n, eff_batch):
        if train: optimizer.zero_grad()
        batch_idx = idx[start:start + eff_batch]
        for ms in range(0, len(batch_idx), MICRO_BATCH):
            mb_idx = batch_idx[ms:ms + MICRO_BATCH]
            if len(mb_idx) == 0: continue
            xb = add_static_fn(X[mb_idx].to(DEVICE), static_tensor)
            yb = y[mb_idx].to(DEVICE)
            with torch.set_grad_enabled(train):
                pred = model(xb)
                loss = pinball_loss(pred, yb, QUANTILES)
            if train: (loss * len(mb_idx) / len(batch_idx)).backward()
            total_loss += loss.item() * len(mb_idx); total_n += len(mb_idx)
        if train: optimizer.step()
    return total_loss / total_n

def run_p2_epoch_gru(model, X, y_traj, optimizer, train):
    n = X.shape[0]
    idx = torch.randperm(n) if train else torch.arange(n)
    model.train(train)
    total_loss, total_n = 0.0, 0
    eff_batch = MICRO_BATCH * ACCUM_STEPS
    for start in range(0, n, eff_batch):
        if train: optimizer.zero_grad()
        batch_idx = idx[start:start + eff_batch]
        for ms in range(0, len(batch_idx), MICRO_BATCH):
            mb_idx = batch_idx[ms:ms + MICRO_BATCH]
            if len(mb_idx) == 0: continue
            xb = add_static_fn(X[mb_idx].to(DEVICE), static_tensor)
            yb = y_traj[mb_idx].to(DEVICE)
            with torch.set_grad_enabled(train):
                logits = model(xb, n_regions)
                loss = region_criterion(logits, yb)
            if train: loss.backward()
            total_loss += loss.item() * len(mb_idx); total_n += len(mb_idx)
        if train: optimizer.step()
    return total_loss / max(total_n, 1)

def predict_probs_gru(model, X, batch_size=64):
    model.eval()
    n = X.shape[0]
    out = np.zeros((n, n_regions, HORIZON), dtype=np.float32)
    with torch.no_grad():
        for start in range(0, n, batch_size):
            idx = torch.arange(start, min(start + batch_size, n))
            xb = add_static_fn(X[idx].to(DEVICE), static_tensor)
            logits = model(xb, n_regions)
            out[idx.numpy()] = torch.sigmoid(logits).cpu().numpy()
    return out

def train_one_config_graph(seed, ewt_s, ewt_r, lr, dropout, weight_decay):
    torch.manual_seed(seed); np.random.seed(seed)
    p1 = StationQuantileGCN(n_feats, dropout).to(DEVICE)
    opt1 = torch.optim.Adam(p1.parameters(), lr=lr, weight_decay=weight_decay)
    best_p1_val, best_p1_state = float("inf"), None
    for epoch in range(1, MAX_EPOCHS_P1 + 1):
        run_p1_epoch_graph(p1, ewt_s, X0_t, yreg0_t, starts0_t, opt1, True)
        vl = run_p1_epoch_graph(p1, ewt_s, X1_t, yreg1_t, starts1_t, opt1, False)
        if vl < best_p1_val:
            best_p1_val, best_p1_state = vl, copy.deepcopy(p1.state_dict())
    p1.load_state_dict(best_p1_state)
    encoder_copy = copy.deepcopy(p1.station_conv)
    p2 = RegionFromTrajectoryGCN(encoder_copy, region_membership_t, dropout).to(DEVICE)
    del p1; gc.collect()
    if DEVICE.type == "mps": torch.mps.empty_cache()
    opt2 = torch.optim.Adam(p2.parameters(), lr=lr, weight_decay=weight_decay)
    y_val_agg = ytraj1_t.numpy().max(axis=-1).reshape(-1)
    best_val_aucpr, best_state = -1.0, None
    for epoch in range(1, MAX_EPOCHS_P2 + 1):
        run_p2_epoch_graph(p2, ewt_s, ewt_r, X0_t, ytraj0_t, starts0_t, opt2, True)
        run_p2_epoch_graph(p2, ewt_s, ewt_r, X1_t, ytraj1_t, starts1_t, opt2, False)
        val_probs = predict_probs_graph(p2, ewt_s, ewt_r, X1_t, starts1_t)
        va = average_precision_score(y_val_agg, val_probs.max(axis=-1).reshape(-1))
        if va > best_val_aucpr:
            best_val_aucpr, best_state = va, copy.deepcopy(p2.state_dict())
    p2.load_state_dict(best_state)
    p2.eval()
    return p2, best_val_aucpr

def train_one_config_gru(seed, lr, dropout, weight_decay):
    torch.manual_seed(seed); np.random.seed(seed)
    p1 = StationGRU(n_feats, dropout).to(DEVICE)
    opt1 = torch.optim.Adam(p1.parameters(), lr=lr, weight_decay=weight_decay)
    best_p1_val, best_p1_state = float("inf"), None
    for epoch in range(1, MAX_EPOCHS_P1 + 1):
        run_p1_epoch_gru(p1, X0_t, yreg0_t, opt1, True)
        vl = run_p1_epoch_gru(p1, X1_t, yreg1_t, opt1, False)
        if vl < best_p1_val:
            best_p1_val, best_p1_state = vl, copy.deepcopy(p1.state_dict())
    p1.load_state_dict(best_p1_state)
    encoder_copy = copy.deepcopy(p1.station_lin)
    p2 = RegionFromTrajectoryGRU(encoder_copy, region_membership_t, dropout).to(DEVICE)
    del p1; gc.collect()
    if DEVICE.type == "mps": torch.mps.empty_cache()
    opt2 = torch.optim.Adam(p2.parameters(), lr=lr, weight_decay=weight_decay)
    y_val_agg = ytraj1_t.numpy().max(axis=-1).reshape(-1)
    best_val_aucpr, best_state = -1.0, None
    for epoch in range(1, MAX_EPOCHS_P2 + 1):
        run_p2_epoch_gru(p2, X0_t, ytraj0_t, opt2, True)
        run_p2_epoch_gru(p2, X1_t, ytraj1_t, opt2, False)
        val_probs = predict_probs_gru(p2, X1_t)
        va = average_precision_score(y_val_agg, val_probs.max(axis=-1).reshape(-1))
        if va > best_val_aucpr:
            best_val_aucpr, best_state = va, copy.deepcopy(p2.state_dict())
    p2.load_state_dict(best_state)
    p2.eval()
    return p2, best_val_aucpr

GRID = [
    {"lr": 0.003, "dropout": 0.5, "weight_decay": 0.0005},
    {"lr": 0.001, "dropout": 0.5, "weight_decay": 0.0005},
    {"lr": 0.003, "dropout": 0.3, "weight_decay": 0.0005},
    {"lr": 0.003, "dropout": 0.5, "weight_decay": 0.001},
]
N_SEEDS = 10
y_test_agg = ytraj2.max(axis=-1).reshape(-1)
all_results = {}

def run_full_pipeline(model_name, ewt_s=None, ewt_r=None, is_gru=False):
    print(f"\n{'#'*15} {model_name}: hyperparameter search on train=2020/val=H1-2021 only {'#'*15}")
    grid_results = []
    for cfg in GRID:
        t0 = time.time()
        if is_gru:
            p2, val_aucpr = train_one_config_gru(seed=0, **cfg)
        else:
            p2, val_aucpr = train_one_config_graph(seed=0, ewt_s=ewt_s, ewt_r=ewt_r, **cfg)
        print(f"  {cfg}  ->  val_AUCPR={val_aucpr:.4f}  ({time.time()-t0:.0f}s)")
        grid_results.append({**cfg, "val_aucpr": float(val_aucpr)})
        del p2; gc.collect()
        if DEVICE.type == "mps": torch.mps.empty_cache()
    best_cfg = max(grid_results, key=lambda r: r["val_aucpr"])
    print(f"  best config for {model_name}: {best_cfg}")

    print(f"\n{'#'*15} {model_name}: {N_SEEDS} final seeds {'#'*15}")
    seed_results = []
    for seed in range(N_SEEDS):
        t0 = time.time()
        if is_gru:
            p2, val_aucpr = train_one_config_gru(seed, best_cfg["lr"], best_cfg["dropout"], best_cfg["weight_decay"])
            test_probs = predict_probs_gru(p2, X2_t)
        else:
            p2, val_aucpr = train_one_config_graph(seed, ewt_s, ewt_r, best_cfg["lr"], best_cfg["dropout"], best_cfg["weight_decay"])
            test_probs = predict_probs_graph(p2, ewt_s, ewt_r, X2_t, starts2_t)
        test_agg_score = test_probs.max(axis=-1).reshape(-1)
        test_aucroc = roc_auc_score(y_test_agg, test_agg_score)
        test_aucpr = average_precision_score(y_test_agg, test_agg_score)
        print(f"  seed={seed}  ({time.time()-t0:.0f}s)  test_AUCROC={test_aucroc:.4f}  test_AUCPR={test_aucpr:.4f}")
        seed_results.append({"seed": seed, "val_aucpr": float(val_aucpr), "test_aucroc": float(test_aucroc), "test_aucpr": float(test_aucpr)})
        del p2; gc.collect()
        if DEVICE.type == "mps": torch.mps.empty_cache()

    return {"grid_search": grid_results, "best_config": best_cfg, "results": seed_results}

all_results["wind_graph"] = run_full_pipeline("wind_graph", wind_ewt_s, wind_ewt_r, is_gru=False)
all_results["no_graph"] = run_full_pipeline("no_graph", zero_ewt_s, zero_ewt_r, is_gru=False)
all_results["minimal_gru"] = run_full_pipeline("minimal_gru", is_gru=True)

print(f"\n{'='*20} SUMMARY: within-COVID-regime (train=2020/val=H1-2021/test=H2-2021), {N_SEEDS} seeds {'='*20}")
arrs = {}
for name in ["wind_graph", "no_graph", "minimal_gru"]:
    aucpr = np.array([r["test_aucpr"] for r in all_results[name]["results"]])
    aucroc = np.array([r["test_aucroc"] for r in all_results[name]["results"]])
    arrs[name] = {"aucpr": aucpr, "aucroc": aucroc}
    print(f"{name:<15} AUC-PR mean={aucpr.mean():.4f} std={aucpr.std():.4f}  |  AUC-ROC mean={aucroc.mean():.4f} std={aucroc.std():.4f}  (best_cfg={all_results[name]['best_config']})")

for other in ["no_graph", "minimal_gru"]:
    t_stat, p_ttest = stats.ttest_rel(arrs["wind_graph"]["aucpr"], arrs[other]["aucpr"])
    w_stat, p_wilcoxon = stats.wilcoxon(arrs["wind_graph"]["aucpr"], arrs[other]["aucpr"])
    n_wins = int((arrs["wind_graph"]["aucpr"] > arrs[other]["aucpr"]).sum())
    print(f"\nwind_graph vs {other} (AUC-PR, {N_SEEDS} seeds): paired t={t_stat:.3f} p={p_ttest:.6f}  |  Wilcoxon p={p_wilcoxon:.4f}  |  wind_graph wins {n_wins}/{N_SEEDS}")

with open(f"{BASE}/within_covid_regime_full_10seed.json", "w") as f:
    json.dump(all_results, f, indent=2)
print(f"\nsaved to {BASE}/within_covid_regime_full_10seed.json")


device=mps
train hours (2020): 8784  val hours (Jan-Jun 2021): 4344  test hours (Jul-Dec 2021): 4416
windows: train=8713 val=4273 test=4345
pos_weight: 50.00

############### wind_graph: hyperparameter search on train=2020/val=H1-2021 only ###############
  {'lr': 0.003, 'dropout': 0.5, 'weight_decay': 0.0005}  ->  val_AUCPR=0.2529  (303s)
  {'lr': 0.001, 'dropout': 0.5, 'weight_decay': 0.0005}  ->  val_AUCPR=0.3031  (298s)
  {'lr': 0.003, 'dropout': 0.3, 'weight_decay': 0.0005}  ->  val_AUCPR=0.2693  (296s)
  {'lr': 0.003, 'dropout': 0.5, 'weight_decay': 0.001}  ->  val_AUCPR=0.2692  (297s)
  best config for wind_graph: {'lr': 0.001, 'dropout': 0.5, 'weight_decay': 0.0005, 'val_aucpr': 0.3030616831182246}

############### wind_graph: 10 final seeds ###############
  seed=0  (316s)  test_AUCROC=0.9508  test_AUCPR=0.4918
  seed=1  (318s)  test_AUCROC=0.9500  test_AUCPR=0.4569
  seed=2  (316s)  test_AUCROC=0.9533  test_AUCPR=0.4690
  seed=3  (316s)  test_AUCROC=0.9595  test_AUCPR=0.5218


In [1]:
# ================================================================
# GRADIENT-BOOSTED-TREE BASELINE (with hyperparameter tuning) on the
# within-COVID-regime split: train=2020, val=Jan-Jun 2021, test=Jul-Dec 2021.
# Uses sklearn's HistGradientBoostingClassifier -- XGBoost still has the
# unresolved libomp/Homebrew issue on this machine, so we're reusing the
# working substitute rather than re-hitting that error.
# Same single-region feature set as the earlier tree-baseline comparison
# (mean + last value per channel over the 36h window, plus static features).
# ================================================================
import numpy as np
import pandas as pd
import os, gc, json, time
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import average_precision_score, roc_auc_score, precision_score, recall_score, matthews_corrcoef, roc_curve

BASE = "/Users/drewbaldwin/PM2_5 Research"
df = pd.read_pickle(f"{BASE}/air_korea_final_imputed_with_blh.pkl")
station_order = sorted(df["Station_ID"].unique())
n_stations = len(station_order)

stations = df[["Station_ID", "lat", "lon"]].drop_duplicates("Station_ID").set_index("Station_ID").loc[station_order]
lats, lons = stations["lat"].to_numpy(), stations["lon"].to_numpy()

def haversine_km(lat1, lon1, lat2, lon2):
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
    dlat, dlon = lat2 - lat1, lon2 - lon1
    a = np.sin(dlat/2)**2 + np.cos(lat1)*np.cos(lat2)*np.sin(dlon/2)**2
    return 2 * 6371.0 * np.arcsin(np.sqrt(a))

REGION_CENTROIDS = {
    "Seoul": (37.566, 126.978), "Busan": (35.180, 129.075), "Daegu": (35.872, 128.602),
    "Incheon": (37.483, 126.633), "Gwangju": (35.155, 126.916), "Daejeon": (36.350, 127.385),
    "Ulsan": (35.550, 129.317), "Sejong": (36.487, 127.282), "Gyeonggi": (37.500, 127.250),
    "Gangwon": (37.867, 127.733), "Chungbuk": (36.633, 127.483), "Chungnam": (36.500, 126.750),
    "Jeonbuk": (35.824, 127.148), "Jeonnam": (34.750, 127.000), "Gyeongbuk": (36.559, 128.729),
    "Gyeongnam": (35.271, 128.663), "Jeju": (33.513, 126.523),
}
region_names = list(REGION_CENTROIDS.keys())
n_regions = len(region_names)
region_lats = np.array([REGION_CENTROIDS[r][0] for r in region_names])
region_lons = np.array([REGION_CENTROIDS[r][1] for r in region_names])
dist_to_region = haversine_km(lats[:, None], lons[:, None], region_lats[None, :], region_lons[None, :])
station_region_idx = dist_to_region.argmin(axis=1)

WINDOW, HORIZON = 36, 36
EVENT_THRESHOLD = 75.0
SUSTAIN_HOURS = 2
TIME_FEATS = ["SO2", "CO", "NO2", "O3", "PM10", "PM25"]
STATIC_COLS = ["elevation_m", "urban_landuse_area_m2_3km", "green_space_area_3km",
               "building_footprint_area_3km", "railway_length_3km", "dist_to_coast_km",
               "dist_to_major_road_km", "industrial_area_m2_3km", "traffic_points_count_3km",
               "major_roads_count_3km", "total_road_length_3km"]

time_panels = {c: df.pivot(index="Datetime", columns="Station_ID", values=c)[station_order] for c in TIME_FEATS}
dt_index = time_panels["PM25"].index
n_time = len(dt_index)
years = dt_index.year.to_numpy()
months = dt_index.month.to_numpy()
doy = dt_index.dayofyear.to_numpy().astype(float)

# ---- SPLIT: train=2020, val=Jan-Jun 2021, test=Jul-Dec 2021 ----
split_id_per_hour = np.where(
    years == 2020, 0,
    np.where((years == 2021) & (months <= 6), 1,
    np.where((years == 2021) & (months >= 7), 2, -1)))
TRAIN_MASK = split_id_per_hour == 0

wind_speed_arr = df.pivot(index="Datetime", columns="Station_ID", values="windspeed_10m")[station_order].reindex(dt_index).to_numpy().astype(float)
wind_dir_arr = df.pivot(index="Datetime", columns="Station_ID", values="winddirection_10m")[station_order].reindex(dt_index).to_numpy().astype(float)
blh_arr = df.pivot(index="Datetime", columns="Station_ID", values="boundary_layer_height")[station_order].reindex(dt_index).to_numpy().astype(float)
pm25_raw_arr = time_panels["PM25"].to_numpy()

def regional_flat_mean(arr):
    out = np.zeros((n_time, n_regions), dtype=np.float32)
    for r in range(n_regions):
        cols = station_region_idx == r
        out[:, r] = np.nanmean(arr[:, cols], axis=1)
    return out

region_pm25 = regional_flat_mean(pm25_raw_arr)
wdir_sin_station = np.sin(np.radians(wind_dir_arr))
wdir_cos_station = np.cos(np.radians(wind_dir_arr))
season_sin_1d = np.sin(2 * np.pi * doy / 365.25)
season_cos_1d = np.cos(2 * np.pi * doy / 365.25)
season_sin = np.tile(season_sin_1d[:, None], (1, n_stations))
season_cos = np.tile(season_cos_1d[:, None], (1, n_stations))

TIME_FEATS_FULL = TIME_FEATS + ["windspeed_10m", "wdir_sin", "wdir_cos", "boundary_layer_height", "season_sin", "season_cos"]
n_channels = len(TIME_FEATS_FULL)

time_arr_raw = np.stack([time_panels[c].to_numpy() for c in TIME_FEATS] +
                         [wind_speed_arr, wdir_sin_station, wdir_cos_station, blh_arr, season_sin, season_cos], axis=-1)
del time_panels, season_sin, season_cos, blh_arr
gc.collect()

region_channels = np.zeros((n_time, n_regions, n_channels), dtype=np.float32)
for r in range(n_regions):
    cols = station_region_idx == r
    for c in range(n_channels):
        region_channels[:, r, c] = np.nanmean(time_arr_raw[:, cols, c], axis=1)
del time_arr_raw
gc.collect()

static_df = df[["Station_ID"] + STATIC_COLS].drop_duplicates("Station_ID").set_index("Station_ID").loc[station_order]
static_arr = static_df[STATIC_COLS].to_numpy()
region_static = np.zeros((n_regions, len(STATIC_COLS)), dtype=np.float32)
for r in range(n_regions):
    cols = station_region_idx == r
    region_static[r] = static_arr[cols].mean(axis=0)
del df, static_arr
gc.collect()

rev = region_pm25[::-1]
roll_min_rev = pd.DataFrame(rev).rolling(window=SUSTAIN_HOURS, min_periods=SUSTAIN_HOURS).min().to_numpy()
region_episode_label = (roll_min_rev[::-1] >= EVENT_THRESHOLD).astype(np.float32)
del rev, roll_min_rev, pm25_raw_arr
gc.collect()

print(f"train hours (2020): {TRAIN_MASK.sum()}  val hours (H1-2021): {(split_id_per_hour==1).sum()}  test hours (H2-2021): {(split_id_per_hour==2).sum()}")

# ================================================================
# Build flattened tabular features (single-region only, matching the
# earlier tree-baseline "Model A" comparison point)
# ================================================================
buckets = {0: {"X": [], "y": []}, 1: {"X": [], "y": []}, 2: {"X": [], "y": []}}

for t in range(0, n_time - WINDOW - HORIZON + 1):
    target_t = t + WINDOW + HORIZON - 1
    s_start, s_target = split_id_per_hour[t], split_id_per_hour[target_t]
    if s_start != s_target or s_start == -1:
        continue
    window = region_channels[t:t + WINDOW]
    window_mean = window.mean(axis=0)
    window_last = window[-1]
    labels = region_episode_label[t + WINDOW: t + WINDOW + HORIZON].max(axis=0)

    for r in range(n_regions):
        feat = np.concatenate([window_mean[r], window_last[r], region_static[r]])
        buckets[s_start]["X"].append(feat)
        buckets[s_start]["y"].append(labels[r])

X0 = np.stack(buckets[0]["X"]); y0 = np.array(buckets[0]["y"])
X1 = np.stack(buckets[1]["X"]); y1 = np.array(buckets[1]["y"])
X2 = np.stack(buckets[2]["X"]); y2 = np.array(buckets[2]["y"])
del buckets, region_channels
gc.collect()
print(f"rows: train={len(y0)} val={len(y1)} test={len(y2)}  (positive rate train={y0.mean():.4f})")

POS_WEIGHT = min(float((len(y0) - y0.sum()) / max(y0.sum(), 1)), 50.0)
print(f"pos_weight (capped at 50): {POS_WEIGHT:.2f}")
sample_weight_0 = np.where(y0 == 1, POS_WEIGHT, 1.0)

# ================================================================
# Hyperparameter grid search (train=2020, val=H1-2021, test never touched)
# ================================================================
GRID = [
    {"learning_rate": 0.05, "max_depth": 4, "l2_regularization": 0.0},
    {"learning_rate": 0.05, "max_depth": 6, "l2_regularization": 0.0},
    {"learning_rate": 0.05, "max_depth": 8, "l2_regularization": 0.0},
    {"learning_rate": 0.1,  "max_depth": 4, "l2_regularization": 0.0},
    {"learning_rate": 0.1,  "max_depth": 6, "l2_regularization": 0.0},
    {"learning_rate": 0.1,  "max_depth": 6, "l2_regularization": 1.0},
]

print(f"\n{'#'*15} HistGB: hyperparameter search on train=2020/val=H1-2021 only {'#'*15}")
grid_results = []
for cfg in GRID:
    t0 = time.time()
    model = HistGradientBoostingClassifier(
        max_iter=300, **cfg,
        early_stopping=True, validation_fraction=0.1, n_iter_no_change=20,
        random_state=0,
    )
    model.fit(X0, y0, sample_weight=sample_weight_0)
    val_score = model.predict_proba(X1)[:, 1]
    val_aucpr = average_precision_score(y1, val_score)
    print(f"  {cfg}  ->  val_AUCPR={val_aucpr:.4f}  n_iter={model.n_iter_}  ({time.time()-t0:.0f}s)")
    grid_results.append({**cfg, "val_aucpr": float(val_aucpr), "n_iter": int(model.n_iter_)})

best_cfg = max(grid_results, key=lambda r: r["val_aucpr"])
print(f"\nbest config: {best_cfg}")

# ================================================================
# Final model: retrain with best config, evaluate on test=H2-2021
# ================================================================
final_model = HistGradientBoostingClassifier(
    max_iter=300, learning_rate=best_cfg["learning_rate"], max_depth=best_cfg["max_depth"],
    l2_regularization=best_cfg["l2_regularization"],
    early_stopping=True, validation_fraction=0.1, n_iter_no_change=20,
    random_state=0,
)
final_model.fit(X0, y0, sample_weight=sample_weight_0)

val_score = final_model.predict_proba(X1)[:, 1]
test_score = final_model.predict_proba(X2)[:, 1]

test_aucroc = roc_auc_score(y2, test_score)
test_aucpr = average_precision_score(y2, test_score)

fpr, tpr, thresholds = roc_curve(y1, val_score)
f1_scores = []
for th in thresholds:
    pred = (val_score >= th).astype(int)
    p = precision_score(y1, pred, zero_division=0)
    r = recall_score(y1, pred, zero_division=0)
    f1_scores.append(2 * p * r / (p + r) if (p + r) > 0 else 0.0)
best_thresh = thresholds[np.argmax(f1_scores)]

pred_test = (test_score >= best_thresh).astype(int)
precision = precision_score(y2, pred_test, zero_division=0)
recall = recall_score(y2, pred_test, zero_division=0)
mcc = matthews_corrcoef(y2, pred_test)
far = 1 - precision

print(f"\n{'='*20} FINAL: HistGB on within-COVID-regime split {'='*20}")
print(f"test AUC-ROC={test_aucroc:.4f}  AUC-PR={test_aucpr:.4f}")
print(f"F1-optimal thresh={best_thresh:.4f}  precision={precision:.4f}  recall={recall:.4f}  FAR={far:.4f}  MCC={mcc:.4f}")

print(f"\n{'='*20} COMPARE TO NEURAL MODELS ON SAME SPLIT {'='*20}")
print(f"wind_graph (10-seed mean):    AUC-PR=0.4904  AUC-ROC=0.9553")
print(f"no_graph (10-seed mean):      AUC-PR=0.4373  AUC-ROC=0.9338")
print(f"minimal_gru (10-seed mean):   AUC-PR=0.4003  AUC-ROC=0.9412")
print(f"HistGB (tree baseline):        AUC-PR={test_aucpr:.4f}  AUC-ROC={test_aucroc:.4f}")

with open(f"{BASE}/histgb_within_covid_regime.json", "w") as f:
    json.dump({"grid_search": grid_results, "best_config": best_cfg,
               "test_aucroc": float(test_aucroc), "test_aucpr": float(test_aucpr),
               "threshold": float(best_thresh), "precision": float(precision),
               "recall": float(recall), "far": float(far), "mcc": float(mcc)}, f, indent=2)
print(f"\nsaved to {BASE}/histgb_within_covid_regime.json")


train hours (2020): 8784  val hours (H1-2021): 4344  test hours (H2-2021): 4416
rows: train=148121 val=72641 test=73865  (positive rate train=0.0134)
pos_weight (capped at 50): 50.00

############### HistGB: hyperparameter search on train=2020/val=H1-2021 only ###############
  {'learning_rate': 0.05, 'max_depth': 4, 'l2_regularization': 0.0}  ->  val_AUCPR=0.2441  n_iter=300  (1s)
  {'learning_rate': 0.05, 'max_depth': 6, 'l2_regularization': 0.0}  ->  val_AUCPR=0.2440  n_iter=300  (1s)
  {'learning_rate': 0.05, 'max_depth': 8, 'l2_regularization': 0.0}  ->  val_AUCPR=0.2361  n_iter=300  (2s)
  {'learning_rate': 0.1, 'max_depth': 4, 'l2_regularization': 0.0}  ->  val_AUCPR=0.2234  n_iter=300  (1s)
  {'learning_rate': 0.1, 'max_depth': 6, 'l2_regularization': 0.0}  ->  val_AUCPR=0.2192  n_iter=252  (1s)
  {'learning_rate': 0.1, 'max_depth': 6, 'l2_regularization': 1.0}  ->  val_AUCPR=0.2250  n_iter=300  (1s)

best config: {'learning_rate': 0.05, 'max_depth': 4, 'l2_regularization': 0.0

In [2]:
# ================================================================
# GRADIENT-BOOSTED-TREE BASELINE (with hyperparameter tuning) on the
# 2018 split: train=2016, val=2017, test=2018.
# Single-region features only. Uses HistGradientBoostingClassifier
# (XGBoost still blocked by the missing libomp/Homebrew dependency).
# ================================================================
import numpy as np
import pandas as pd
import os, gc, json, time
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import average_precision_score, roc_auc_score, precision_score, recall_score, matthews_corrcoef, roc_curve

BASE = "/Users/drewbaldwin/PM2_5 Research"
df = pd.read_pickle(f"{BASE}/air_korea_final_imputed_with_blh.pkl")
station_order = sorted(df["Station_ID"].unique())
n_stations = len(station_order)

stations = df[["Station_ID", "lat", "lon"]].drop_duplicates("Station_ID").set_index("Station_ID").loc[station_order]
lats, lons = stations["lat"].to_numpy(), stations["lon"].to_numpy()

def haversine_km(lat1, lon1, lat2, lon2):
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
    dlat, dlon = lat2 - lat1, lon2 - lon1
    a = np.sin(dlat/2)**2 + np.cos(lat1)*np.cos(lat2)*np.sin(dlon/2)**2
    return 2 * 6371.0 * np.arcsin(np.sqrt(a))

REGION_CENTROIDS = {
    "Seoul": (37.566, 126.978), "Busan": (35.180, 129.075), "Daegu": (35.872, 128.602),
    "Incheon": (37.483, 126.633), "Gwangju": (35.155, 126.916), "Daejeon": (36.350, 127.385),
    "Ulsan": (35.550, 129.317), "Sejong": (36.487, 127.282), "Gyeonggi": (37.500, 127.250),
    "Gangwon": (37.867, 127.733), "Chungbuk": (36.633, 127.483), "Chungnam": (36.500, 126.750),
    "Jeonbuk": (35.824, 127.148), "Jeonnam": (34.750, 127.000), "Gyeongbuk": (36.559, 128.729),
    "Gyeongnam": (35.271, 128.663), "Jeju": (33.513, 126.523),
}
region_names = list(REGION_CENTROIDS.keys())
n_regions = len(region_names)
region_lats = np.array([REGION_CENTROIDS[r][0] for r in region_names])
region_lons = np.array([REGION_CENTROIDS[r][1] for r in region_names])
dist_to_region = haversine_km(lats[:, None], lons[:, None], region_lats[None, :], region_lons[None, :])
station_region_idx = dist_to_region.argmin(axis=1)

WINDOW, HORIZON = 36, 36
EVENT_THRESHOLD = 75.0
SUSTAIN_HOURS = 2
TIME_FEATS = ["SO2", "CO", "NO2", "O3", "PM10", "PM25"]
STATIC_COLS = ["elevation_m", "urban_landuse_area_m2_3km", "green_space_area_3km",
               "building_footprint_area_3km", "railway_length_3km", "dist_to_coast_km",
               "dist_to_major_road_km", "industrial_area_m2_3km", "traffic_points_count_3km",
               "major_roads_count_3km", "total_road_length_3km"]

time_panels = {c: df.pivot(index="Datetime", columns="Station_ID", values=c)[station_order] for c in TIME_FEATS}
dt_index = time_panels["PM25"].index
n_time = len(dt_index)
years = dt_index.year.to_numpy()
doy = dt_index.dayofyear.to_numpy().astype(float)

# ---- SPLIT: train=2016, val=2017, test=2018 ----
split_id_per_hour = np.where(years == 2016, 0, np.where(years == 2017, 1, np.where(years == 2018, 2, -1)))
TRAIN_MASK = split_id_per_hour == 0

wind_speed_arr = df.pivot(index="Datetime", columns="Station_ID", values="windspeed_10m")[station_order].reindex(dt_index).to_numpy().astype(float)
wind_dir_arr = df.pivot(index="Datetime", columns="Station_ID", values="winddirection_10m")[station_order].reindex(dt_index).to_numpy().astype(float)
blh_arr = df.pivot(index="Datetime", columns="Station_ID", values="boundary_layer_height")[station_order].reindex(dt_index).to_numpy().astype(float)
pm25_raw_arr = time_panels["PM25"].to_numpy()

def regional_flat_mean(arr):
    out = np.zeros((n_time, n_regions), dtype=np.float32)
    for r in range(n_regions):
        cols = station_region_idx == r
        out[:, r] = np.nanmean(arr[:, cols], axis=1)
    return out

region_pm25 = regional_flat_mean(pm25_raw_arr)
wdir_sin_station = np.sin(np.radians(wind_dir_arr))
wdir_cos_station = np.cos(np.radians(wind_dir_arr))
season_sin_1d = np.sin(2 * np.pi * doy / 365.25)
season_cos_1d = np.cos(2 * np.pi * doy / 365.25)
season_sin = np.tile(season_sin_1d[:, None], (1, n_stations))
season_cos = np.tile(season_cos_1d[:, None], (1, n_stations))

TIME_FEATS_FULL = TIME_FEATS + ["windspeed_10m", "wdir_sin", "wdir_cos", "boundary_layer_height", "season_sin", "season_cos"]
n_channels = len(TIME_FEATS_FULL)

time_arr_raw = np.stack([time_panels[c].to_numpy() for c in TIME_FEATS] +
                         [wind_speed_arr, wdir_sin_station, wdir_cos_station, blh_arr, season_sin, season_cos], axis=-1)
del time_panels, season_sin, season_cos, blh_arr
gc.collect()

region_channels = np.zeros((n_time, n_regions, n_channels), dtype=np.float32)
for r in range(n_regions):
    cols = station_region_idx == r
    for c in range(n_channels):
        region_channels[:, r, c] = np.nanmean(time_arr_raw[:, cols, c], axis=1)
del time_arr_raw
gc.collect()

static_df = df[["Station_ID"] + STATIC_COLS].drop_duplicates("Station_ID").set_index("Station_ID").loc[station_order]
static_arr = static_df[STATIC_COLS].to_numpy()
region_static = np.zeros((n_regions, len(STATIC_COLS)), dtype=np.float32)
for r in range(n_regions):
    cols = station_region_idx == r
    region_static[r] = static_arr[cols].mean(axis=0)
del df, static_arr
gc.collect()

rev = region_pm25[::-1]
roll_min_rev = pd.DataFrame(rev).rolling(window=SUSTAIN_HOURS, min_periods=SUSTAIN_HOURS).min().to_numpy()
region_episode_label = (roll_min_rev[::-1] >= EVENT_THRESHOLD).astype(np.float32)
del rev, roll_min_rev, pm25_raw_arr
gc.collect()

print(f"train hours (2016): {TRAIN_MASK.sum()}  val hours (2017): {(split_id_per_hour==1).sum()}  test hours (2018): {(split_id_per_hour==2).sum()}")

# ================================================================
# Build flattened tabular features (single-region only)
# ================================================================
buckets = {0: {"X": [], "y": []}, 1: {"X": [], "y": []}, 2: {"X": [], "y": []}}

for t in range(0, n_time - WINDOW - HORIZON + 1):
    target_t = t + WINDOW + HORIZON - 1
    s_start, s_target = split_id_per_hour[t], split_id_per_hour[target_t]
    if s_start != s_target or s_start == -1:
        continue
    window = region_channels[t:t + WINDOW]
    window_mean = window.mean(axis=0)
    window_last = window[-1]
    labels = region_episode_label[t + WINDOW: t + WINDOW + HORIZON].max(axis=0)

    for r in range(n_regions):
        feat = np.concatenate([window_mean[r], window_last[r], region_static[r]])
        buckets[s_start]["X"].append(feat)
        buckets[s_start]["y"].append(labels[r])

X0 = np.stack(buckets[0]["X"]); y0 = np.array(buckets[0]["y"])
X1 = np.stack(buckets[1]["X"]); y1 = np.array(buckets[1]["y"])
X2 = np.stack(buckets[2]["X"]); y2 = np.array(buckets[2]["y"])
del buckets, region_channels
gc.collect()
print(f"rows: train={len(y0)} val={len(y1)} test={len(y2)}  (positive rate train={y0.mean():.4f})")

POS_WEIGHT = min(float((len(y0) - y0.sum()) / max(y0.sum(), 1)), 50.0)
print(f"pos_weight (capped at 50): {POS_WEIGHT:.2f}")
sample_weight_0 = np.where(y0 == 1, POS_WEIGHT, 1.0)

# ================================================================
# Hyperparameter grid search (train=2016, val=2017, test never touched)
# ================================================================
GRID = [
    {"learning_rate": 0.05, "max_depth": 4, "l2_regularization": 0.0},
    {"learning_rate": 0.05, "max_depth": 6, "l2_regularization": 0.0},
    {"learning_rate": 0.05, "max_depth": 8, "l2_regularization": 0.0},
    {"learning_rate": 0.1,  "max_depth": 4, "l2_regularization": 0.0},
    {"learning_rate": 0.1,  "max_depth": 6, "l2_regularization": 0.0},
    {"learning_rate": 0.1,  "max_depth": 6, "l2_regularization": 1.0},
]

print(f"\n{'#'*15} HistGB: hyperparameter search on train=2016/val=2017 only {'#'*15}")
grid_results = []
for cfg in GRID:
    t0 = time.time()
    model = HistGradientBoostingClassifier(
        max_iter=300, **cfg,
        early_stopping=True, validation_fraction=0.1, n_iter_no_change=20,
        random_state=0,
    )
    model.fit(X0, y0, sample_weight=sample_weight_0)
    val_score = model.predict_proba(X1)[:, 1]
    val_aucpr = average_precision_score(y1, val_score)
    print(f"  {cfg}  ->  val_AUCPR={val_aucpr:.4f}  n_iter={model.n_iter_}  ({time.time()-t0:.0f}s)")
    grid_results.append({**cfg, "val_aucpr": float(val_aucpr), "n_iter": int(model.n_iter_)})

best_cfg = max(grid_results, key=lambda r: r["val_aucpr"])
print(f"\nbest config: {best_cfg}")

# ================================================================
# Final model: retrain with best config, evaluate on test=2018
# ================================================================
final_model = HistGradientBoostingClassifier(
    max_iter=300, learning_rate=best_cfg["learning_rate"], max_depth=best_cfg["max_depth"],
    l2_regularization=best_cfg["l2_regularization"],
    early_stopping=True, validation_fraction=0.1, n_iter_no_change=20,
    random_state=0,
)
final_model.fit(X0, y0, sample_weight=sample_weight_0)

val_score = final_model.predict_proba(X1)[:, 1]
test_score = final_model.predict_proba(X2)[:, 1]

test_aucroc = roc_auc_score(y2, test_score)
test_aucpr = average_precision_score(y2, test_score)

fpr, tpr, thresholds = roc_curve(y1, val_score)
f1_scores = []
for th in thresholds:
    pred = (val_score >= th).astype(int)
    p = precision_score(y1, pred, zero_division=0)
    r = recall_score(y1, pred, zero_division=0)
    f1_scores.append(2 * p * r / (p + r) if (p + r) > 0 else 0.0)
best_thresh = thresholds[np.argmax(f1_scores)]

pred_test = (test_score >= best_thresh).astype(int)
precision = precision_score(y2, pred_test, zero_division=0)
recall = recall_score(y2, pred_test, zero_division=0)
mcc = matthews_corrcoef(y2, pred_test)
far = 1 - precision

print(f"\n{'='*20} FINAL: HistGB on 2018 split (train=2016/val=2017/test=2018) {'='*20}")
print(f"test AUC-ROC={test_aucroc:.4f}  AUC-PR={test_aucpr:.4f}")
print(f"F1-optimal thresh={best_thresh:.4f}  precision={precision:.4f}  recall={recall:.4f}  FAR={far:.4f}  MCC={mcc:.4f}")

print(f"\n{'='*20} COMPARE TO NEURAL MODELS ON SAME SPLIT {'='*20}")
print(f"wind_graph (10-seed mean):    AUC-PR=0.4628")
print(f"no_graph (10-seed mean):      AUC-PR=0.4356")
print(f"minimal_gru (10-seed mean):   AUC-PR=0.4253")
print(f"HistGB (tree baseline):        AUC-PR={test_aucpr:.4f}  AUC-ROC={test_aucroc:.4f}")

with open(f"{BASE}/histgb_2018_split.json", "w") as f:
    json.dump({"grid_search": grid_results, "best_config": best_cfg,
               "test_aucroc": float(test_aucroc), "test_aucpr": float(test_aucpr),
               "threshold": float(best_thresh), "precision": float(precision),
               "recall": float(recall), "far": float(far), "mcc": float(mcc)}, f, indent=2)
print(f"\nsaved to {BASE}/histgb_2018_split.json")


train hours (2016): 8784  val hours (2017): 8760  test hours (2018): 8760
rows: train=148121 val=147713 test=147713  (positive rate train=0.0368)
pos_weight (capped at 50): 26.15

############### HistGB: hyperparameter search on train=2016/val=2017 only ###############
  {'learning_rate': 0.05, 'max_depth': 4, 'l2_regularization': 0.0}  ->  val_AUCPR=0.2798  n_iter=300  (1s)
  {'learning_rate': 0.05, 'max_depth': 6, 'l2_regularization': 0.0}  ->  val_AUCPR=0.2500  n_iter=300  (1s)
  {'learning_rate': 0.05, 'max_depth': 8, 'l2_regularization': 0.0}  ->  val_AUCPR=0.2481  n_iter=300  (2s)
  {'learning_rate': 0.1, 'max_depth': 4, 'l2_regularization': 0.0}  ->  val_AUCPR=0.2079  n_iter=300  (1s)
  {'learning_rate': 0.1, 'max_depth': 6, 'l2_regularization': 0.0}  ->  val_AUCPR=0.2143  n_iter=262  (1s)
  {'learning_rate': 0.1, 'max_depth': 6, 'l2_regularization': 1.0}  ->  val_AUCPR=0.2089  n_iter=300  (1s)

best config: {'learning_rate': 0.05, 'max_depth': 4, 'l2_regularization': 0.0, 'val_